# Atalaya · piloto acústico YAMNet · iteración 6 (hito 2: contra el atajo de dominio)

**Exploración técnica; no es un detector operativo.** Prueba técnica aislada en Colab (ADR 0016
y ADR 0017): nada de este cuaderno entra en el producto. Las cifras no son métricas de campo.

**Por qué esta iteración.** La iteración 5 separaba bien dentro de las fuentes vistas, pero ante
una fuente nunca vista fallaba (leave-one-source-out: ROC-AUC 0,29–0,39 en ITU ARIS, recall ≈ 0
con el umbral, hasta 40–52 % de falsas alarmas en FSD50K): aprendía el canal de cada dataset.
El micrófono del Mac frente a un altavoz es otra fuente nueva.

**Qué cambia (todo el código probado en `pilots/acoustic/tests` e incrustado tal cual):**
1. **Normalización de canal** compartida con el Mac (`channel_norm`): 16 kHz, paso banda
   100–7 500 Hz, sonoridad por ventana y PCEN; YAMNet ve ventanas de 0,96 s normalizadas.
2. **Aumento de canal** solo en entrenamiento (`channel_augment`): EQ, micrófono, altavoz
   pequeño, sala sintética, códec mu-law y ruido; `synthetic=true`.
3. **Mezclas cruzadas** (`augment.plan_cross_mixes`): dron de una fuente sobre fondo de OTRA.
4. **Rasgo armónico de hélices** (`harmonic`): 8 rasgos que se concatenan al embedding.
5. **Contra el atajo de dominio** (`domain_robust`): INLP (se proyecta fuera lo que distingue una
   fuente de otra dentro de cada clase), group-DRO por fuente × clase y **umbral por peor fuente**
   (FPR de validación ≤ 5 % en cada fuente); calibración de sitio como variante aparte (D-7).
6. **Evaluación:** **LOSO es la métrica principal**; criterio **C-LOSO** (ADR 0018, propuesto) y
   D-6 como referencia. Modelo principal **preinscrito** (no se elige mirando LOSO):
   `iter6 · completo (canal + cruzadas + armónico + INLP + DRO)`.
7. **Exportación:** YAMNet a ONNX con `tf2onnx` y paridad ≤ 1e-4 frente a TensorFlow;
   `modelo_hito2.json` (pesos, Platt, alerta, hashes y vectores dorados), `predicciones.csv`,
   `registro_iter6.json` y `SHA256SUMS`. Recálculo independiente:
   `uv run python recompute_metrics.py predicciones.csv --sha256 <hash> --record registro_iter6.json --artifact modelo_hito2.json --criterio c-loso`.
8. **Prueba tu audio:** al final, sube un fichero propio y el cuaderno lo analiza por la misma
   ruta que el modelo exportado (ventanas, puerta de voz, alerta 3 de 5 con histéresis).

**Fuentes:** las mismas que la iteración 5 (Svanström CC0, ESC-10 CC BY 3.0, FSD50K solo CC0/CC BY
por clip, ITU ARIS CC BY 4.0). Las fuentes de dron extra (Salford DroneNoise CC BY 4.0,
DroneAudioSet solo-dron MIT, Yi-KAIST CC BY 4.0) están **desactivadas** salvo autorización expresa
registrada en `docs/requirements/DECISIONES-USUARIO.md`: sin ellas, el criterio C-LOSO queda
**NO EVALUABLE** en los pliegues con una sola fuente de dron entrenando. **Prohibidas** (nunca se
descargan): DADS, Al-Emadi, UrbanSound8K, ESC-50 fuera de ESC-10, Sonniss y AeroSonicDB.

**Cómo ejecutarlo:** Colab gratuito (CPU), «Entorno de ejecución → Ejecutar todas». No hace falta
reiniciar tras instalar dependencias (solo se instala lo que falta, sin tocar numpy ni protobuf).
Puntos de control en `/content/atalaya_piloto` (sin Drive): si el kernel se reinicia, «Ejecutar
todas» salta lo ya descargado y calculado; si Colab recicla la máquina, se empieza de cero.
Tiempo orientativo **NO VERIFICADO**: 1,5–3 h.


In [ ]:
# Dependencias: SOLO lo que falta, fijando numpy/protobuf/tensorflow a lo que ya trae Colab para no
# obligar a reiniciar el kernel. tf2onnx va sin dependencias (las suyas exigen protobuf 3.20).
import importlib, importlib.metadata as meta, importlib.util, subprocess, sys
PINS = {'tensorflow-hub': '0.16.1', 'onnx': '1.17.0', 'onnxruntime': '1.20.1', 'tf2onnx': '1.16.1'}
def _missing(module):
    return importlib.util.find_spec(module) is None
keep = [f"{name}=={meta.version(name)}" for name in ("numpy", "protobuf", "tensorflow")]
tf_minor = ".".join(meta.version("tensorflow").split(".")[:2])
wanted = [(f"tensorflow-hub=={PINS['tensorflow-hub']}", "tensorflow_hub"), (f"tf-keras=={tf_minor}.*", "tf_keras"),
          (f"onnx=={PINS['onnx']}", "onnx"), (f"onnxruntime=={PINS['onnxruntime']}", "onnxruntime")]
todo = [spec for spec, module in wanted if _missing(module)]
if todo:
    print("Instalando:", ", ".join(todo))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *todo, *keep])
if _missing("tf2onnx"):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", f"tf2onnx=={PINS['tf2onnx']}"])
importlib.invalidate_caches()
for name in ("tensorflow", "tensorflow-hub", "onnx", "onnxruntime", "tf2onnx", "numpy", "protobuf", "librosa", "soundfile"):
    try:
        print(f"  {name} {meta.version(name)}")
    except meta.PackageNotFoundError:
        print(f"  {name}: NO INSTALADO")
print("Dependencias listas; no hace falta reiniciar el entorno.")


In [ ]:
import csv, hashlib, io, json, math, os, random, tarfile, time, wave, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import tensorflow as tf
import tensorflow_hub as hub
from IPython.display import Audio, display
# Las celdas siguientes definen el código probado en pilots/acoustic/tests (incrustado tal cual).

PILOT = json.loads('''{"source_commit":"e7a6eaf9760fa3f9ecc37d09cd11a8cd20be90dd","license_sha256":"a2010f343487d3f7618affe54f789f5487602331c0a8d03f49e9a7c547cf0499","group_status":"provisional","clips":[{"sha256":"e8790309a7888c58c1d1f0f3bfa7850f193af5f23fceb8fdc830aad610ffeb28","bytes":1764052,"group_id":"BACKGROUND_001","training_excluded":false,"name":"BACKGROUND_001.wav"},{"sha256":"3bdef3845531382240df338a41e555d42023d98e8db63c54380fab82c23bde39","bytes":1764048,"group_id":"BACKGROUND_002","training_excluded":false,"name":"BACKGROUND_002.wav"},{"sha256":"7fcad6fffda1d56ba8c236480b04289e8c079ff518c0770770c0cb8c09c43e95","bytes":1764052,"group_id":"BACKGROUND_003","training_excluded":false,"name":"BACKGROUND_003.wav"},{"sha256":"8db1c6e9977126387f5ab00fe5d09cbda1a91e12d315a59356059d606eab0552","bytes":1764048,"group_id":"BACKGROUND_004","training_excluded":false,"name":"BACKGROUND_004.wav"},{"sha256":"1e86479957e5ef21e5ab4c4b25d3df1f09647a34957e621790f199cac8dfac28","bytes":1764048,"group_id":"BACKGROUND_005","training_excluded":false,"name":"BACKGROUND_005.wav"},{"sha256":"f73498b347bd77f4824cd42887ef72c08eb928e79ad2d2dba54f105d5b23fff5","bytes":1764052,"group_id":"BACKGROUND_006","training_excluded":false,"name":"BACKGROUND_006.wav"},{"sha256":"737b05e544fce41d5d2f2e608bae5fd199a6695a80f4c31d2e6945490f15c91d","bytes":1764048,"group_id":"BACKGROUND_007","training_excluded":false,"name":"BACKGROUND_007.wav"},{"sha256":"7f03a35e12c520b95eeab235ce7bec4d174b5adbf506e398da1f9d8a2f3f51a4","bytes":1764048,"group_id":"BACKGROUND_008","training_excluded":false,"name":"BACKGROUND_008.wav"},{"sha256":"e3cb52f2d2abef93cfe80724de76a96f4de8fa05597139e0ce5ad7dd32db5d7d","bytes":1764048,"group_id":"BACKGROUND_009","training_excluded":false,"name":"BACKGROUND_009.wav"},{"sha256":"37200cb91d6f12b4e5e6c79cfe42576252f033c67e862f737d7147ff52981318","bytes":1764048,"group_id":"BACKGROUND_010","training_excluded":false,"name":"BACKGROUND_010.wav"},{"sha256":"c59c17282d104ca1383efbe9addff87dd3c0669fd0a3edea6923f4fe86a4f9cc","bytes":1764052,"group_id":"BACKGROUND_011","training_excluded":false,"name":"BACKGROUND_011.wav"},{"sha256":"9dc31639da818626ff0f0995c793f1d36c12450ef5fd7e275dfc4fa9e52fe1f9","bytes":1764048,"group_id":"BACKGROUND_012","training_excluded":false,"name":"BACKGROUND_012.wav"},{"sha256":"ed96b240e24ad16fca1c430cccbed95754354567c9c1b4e707772504213864a7","bytes":1764048,"group_id":"BACKGROUND_013","training_excluded":false,"name":"BACKGROUND_013.wav"},{"sha256":"9bc49486d332ea8dc745fb6ec0c6cf46d5c20ebcb2bd43324d06996bd22fb680","bytes":1764048,"group_id":"BACKGROUND_014","training_excluded":false,"name":"BACKGROUND_014.wav"},{"sha256":"37c9ca48639871650a6610f43a08f78574c2cabb02ff69d84fcd4e6b9863c01c","bytes":1764052,"group_id":"BACKGROUND_015","training_excluded":false,"name":"BACKGROUND_015.wav"},{"sha256":"ed26ab3e21d613b3b2400e1fd93cb81435cca26e4ae02283460ada3ebcce754f","bytes":1764048,"group_id":"BACKGROUND_016","training_excluded":false,"name":"BACKGROUND_016.wav"},{"sha256":"c8e44c10e29e7f10452e1bfa228ceffee1ade9aaefd0e7f4e18a1f801af2b647","bytes":1764048,"group_id":"BACKGROUND_017","training_excluded":false,"name":"BACKGROUND_017.wav"},{"sha256":"0d618c8b488c2463de551890ed92499e56a2e203306e7d92ffef7ce20a409d69","bytes":1764052,"group_id":"BACKGROUND_018","training_excluded":false,"name":"BACKGROUND_018.wav"},{"sha256":"37200cb91d6f12b4e5e6c79cfe42576252f033c67e862f737d7147ff52981318","bytes":1764048,"group_id":"BACKGROUND_010","training_excluded":true,"name":"BACKGROUND_019.wav"},{"sha256":"d1bdd8d0fffa1e99b2a672160456e1eb2bd8c2b4ed41c99320dd982d576bd637","bytes":1764052,"group_id":"BACKGROUND_020","training_excluded":false,"name":"BACKGROUND_020.wav"},{"sha256":"88c517a8e97b1735716e2257ae5cd55f355ed5834f427c3746918e1b9411e778","bytes":1764052,"group_id":"BACKGROUND_021","training_excluded":false,"name":"BACKGROUND_021.wav"},{"sha256":"33537f39d3143b27d5a9de3c66e0f1e00c599b73b263989814ff9f17e59ec4f9","bytes":1764052,"group_id":"BACKGROUND_022","training_excluded":false,"name":"BACKGROUND_022.wav"},{"sha256":"2fce1d9350ab2b453676e156000ff292a520ba710bbc571e69e7b19aa51ebe4f","bytes":1764048,"group_id":"BACKGROUND_023","training_excluded":false,"name":"BACKGROUND_023.wav"},{"sha256":"991877ffaa560d376332a878afdeb1c95c080aee7e2a5d1ff0f7da44ecc93ffa","bytes":1764048,"group_id":"BACKGROUND_024","training_excluded":false,"name":"BACKGROUND_024.wav"},{"sha256":"c441558280f77504792a2a0c6a8829422a7763b366a45aab18ac381c7c43f066","bytes":1764048,"group_id":"BACKGROUND_025","training_excluded":false,"name":"BACKGROUND_025.wav"},{"sha256":"55654455627d97f252f30f63ea7f37e5ce7d9a6c076eb3580c630bc46e98a6e9","bytes":1764052,"group_id":"BACKGROUND_026","training_excluded":false,"name":"BACKGROUND_026.wav"},{"sha256":"9f4edfaca8194afbb264120cb002f7e9d4bceb03f0fc2ec39a88d6af2b251cd5","bytes":1764048,"group_id":"BACKGROUND_027","training_excluded":false,"name":"BACKGROUND_027.wav"},{"sha256":"af35ea6d7947b5d520054067233f932637f3852f1dc0f6e3e9479e525d9eac3a","bytes":1764048,"group_id":"BACKGROUND_028","training_excluded":false,"name":"BACKGROUND_028.wav"},{"sha256":"785c4bd25ef23a1a9f66af55accfb3bce6a06149c79bf9d236b8ce042562f397","bytes":1764052,"group_id":"BACKGROUND_029","training_excluded":false,"name":"BACKGROUND_029.wav"},{"sha256":"34f988beed58747337bf53bed8be7fcc3f0a6aadb06d49dc486fc48ef97d6b71","bytes":1764052,"group_id":"BACKGROUND_030","training_excluded":false,"name":"BACKGROUND_030.wav"},{"sha256":"3c3fe29253e442cee58bb02b064e590e060d3e3e1c4039d8eb05d2da3b2c368c","bytes":1764048,"group_id":"DRONE_001","training_excluded":false,"name":"DRONE_001.wav"},{"sha256":"a18e7825b0da4df748bea37d9392d92512192a7fe6447bea679d8980c8cfe3c4","bytes":1764052,"group_id":"DRONE_002","training_excluded":false,"name":"DRONE_002.wav"},{"sha256":"12f04588de940bcd15ae955eb1b9a8a250986cf129f5c481f64598cd32cd302b","bytes":1764048,"group_id":"DRONE_003","training_excluded":false,"name":"DRONE_003.wav"},{"sha256":"37c5062e4f07c9e05a67f7eb92a02785bddbf2fa10da8fd8596526755b54ec82","bytes":1764048,"group_id":"DRONE_004","training_excluded":false,"name":"DRONE_004.wav"},{"sha256":"d56924abc9fe6ae53fae99e3eb5b72d2805c605581fa3e129eac8e578623705a","bytes":1764048,"group_id":"DRONE_005","training_excluded":false,"name":"DRONE_005.wav"},{"sha256":"2079a59ea51d1e3468c29302d3c0c443b365fa27fe7dc53ac57a5fbc6d6b1edf","bytes":1764048,"group_id":"DRONE_006","training_excluded":false,"name":"DRONE_006.wav"},{"sha256":"46bb93a90bec955129f6e2b2043d8d406fe93dedcd50a2a1519f82db1c59e420","bytes":1764048,"group_id":"DRONE_007","training_excluded":false,"name":"DRONE_007.wav"},{"sha256":"3001968764aafe76b604c455a7464f19dde8bd5b91fc24f1f486be6df5daa749","bytes":1764048,"group_id":"DRONE_008","training_excluded":false,"name":"DRONE_008.wav"},{"sha256":"b17bff5669ee90a509af546c0edf0ffb4703c26920b7137c71010edc9186f897","bytes":1764048,"group_id":"DRONE_009","training_excluded":false,"name":"DRONE_009.wav"},{"sha256":"4798cb55d03757b17b4f43f40f9173eec0a0205de9797cce5424420c6833859c","bytes":1764048,"group_id":"DRONE_010","training_excluded":false,"name":"DRONE_010.wav"},{"sha256":"d556d8297499b37f58ac90d0d3f865d553495688230905c52bd9c3019829cec9","bytes":1764048,"group_id":"DRONE_011","training_excluded":false,"name":"DRONE_011.wav"},{"sha256":"7631a3915d5ac73cc704c12fef7de1c6020c0181c04f58be90318ec0f870b2d1","bytes":1764048,"group_id":"DRONE_012","training_excluded":false,"name":"DRONE_012.wav"},{"sha256":"5548aac8a087f83faacd47da455af8a6e1f55489001200f222582ccdc81f7f08","bytes":1764052,"group_id":"DRONE_013","training_excluded":false,"name":"DRONE_013.wav"},{"sha256":"94ace9424b53e69c34ca8184ab3bac397996807047d4abf3a31abf96abc22d9b","bytes":1764052,"group_id":"DRONE_014","training_excluded":false,"name":"DRONE_014.wav"},{"sha256":"154de713d1e6bab713d91dedf66c623b313eb8dd74b9560fdc6199abc4f1d03e","bytes":1764052,"group_id":"DRONE_015","training_excluded":false,"name":"DRONE_015.wav"},{"sha256":"c46e30be454f61aa0dad797daa4dddd4e747e893150eb0b99fc27ede5ee090da","bytes":1764052,"group_id":"DRONE_016","training_excluded":false,"name":"DRONE_016.wav"},{"sha256":"789304d5fecdf52bee3eaea5b2b56ce20e67875b7d0df0b57bd15a7bf488719d","bytes":1764052,"group_id":"DRONE_017","training_excluded":false,"name":"DRONE_017.wav"},{"sha256":"06c806bdbe8499a31976b9c63be89afddc35fc20ebceb5b7fef8d48105d22490","bytes":1764052,"group_id":"DRONE_018","training_excluded":false,"name":"DRONE_018.wav"},{"sha256":"732ed57b672f7d1c6edc36f08fedad116119b4dd0d267c7e33ad20af6693acd0","bytes":1764052,"group_id":"DRONE_019","training_excluded":false,"name":"DRONE_019.wav"},{"sha256":"68b73a293ca70e8791481492122cdc81db56c4967673c9f27d99844127bcb23d","bytes":1764052,"group_id":"DRONE_020","training_excluded":false,"name":"DRONE_020.wav"},{"sha256":"7aba84d6e9edfa6766d18cc481022162e0c9a09734a502a6ac0789e8e9ce88a6","bytes":1764052,"group_id":"DRONE_021","training_excluded":false,"name":"DRONE_021.wav"},{"sha256":"ddc68de1fdb7b5ec7477d897af55e1a3881991bbe1587c9d083844050e685df9","bytes":1764052,"group_id":"DRONE_022","training_excluded":false,"name":"DRONE_022.wav"},{"sha256":"596be4f0dd105f98bc2b9a8fc0bbad69ba88645130f6cf54f683799555be53bf","bytes":1764052,"group_id":"DRONE_023","training_excluded":false,"name":"DRONE_023.wav"},{"sha256":"719c315afc8b9aca1bc8428aeeed1306be1d8f381806e1c4ef89b2338f849b7d","bytes":1764052,"group_id":"DRONE_024","training_excluded":false,"name":"DRONE_024.wav"},{"sha256":"4450b89707d09138e4aaf49b84df9d3188b3f7c778ba9af1fb149dec7534900a","bytes":1764052,"group_id":"DRONE_025","training_excluded":false,"name":"DRONE_025.wav"},{"sha256":"a3fca5caa403979b87122f425db4c04fabd13c7d5505d2112c77d680870905a3","bytes":1764052,"group_id":"DRONE_026","training_excluded":false,"name":"DRONE_026.wav"},{"sha256":"de55752be0ac29995d82cce0f0f9cedaa38a9c4405b0e180c27fed8030903fe5","bytes":1764052,"group_id":"DRONE_027","training_excluded":false,"name":"DRONE_027.wav"},{"sha256":"cf401858d8fc05a969a2ad3389cfeffb9b824b96da344cc9216e37bce597f3bb","bytes":1764052,"group_id":"DRONE_028","training_excluded":false,"name":"DRONE_028.wav"},{"sha256":"ca05e5b073ecc4e6567677d1a760f32ed96198c9f77b53ea90f07ba59915383b","bytes":1764048,"group_id":"DRONE_029","training_excluded":false,"name":"DRONE_029.wav"},{"sha256":"9bcb1ee9e4e80086e238042a90929ea8c43a5549792f617e2669ef438dd50552","bytes":1764052,"group_id":"DRONE_030","training_excluded":false,"name":"DRONE_030.wav"},{"sha256":"ff04901b686c1b5aee3db7a7fcef94df677fd2e8cd49e6a840dc09e025260e90","bytes":1764048,"group_id":"HELICOPTER_001","training_excluded":false,"name":"HELICOPTER_001.wav"},{"sha256":"e9dbf544c189c0a1d80f3d599501432ecbd84e03ce5e1536114c6417d7de30d9","bytes":1764048,"group_id":"HELICOPTER_002","training_excluded":false,"name":"HELICOPTER_002.wav"},{"sha256":"976b0afd62fe6045673d37e0ba585979212603061c27bf9d1de4b23049b8f6af","bytes":1764048,"group_id":"HELICOPTER_003","training_excluded":false,"name":"HELICOPTER_003.wav"},{"sha256":"bdb7524eef645970587e1e96eaf23374255fc1789522f4bb69a42effc017d18b","bytes":1764048,"group_id":"HELICOPTER_004","training_excluded":false,"name":"HELICOPTER_004.wav"},{"sha256":"43961c8304d0de1d4dfb0ead9d97de8949a2e0893fc5bb3098ef9ab9a5e23120","bytes":1764048,"group_id":"HELICOPTER_005","training_excluded":false,"name":"HELICOPTER_005.wav"},{"sha256":"6ee524cabbff355ba4490ca975e284725531b59d2d660ef27ea0279215b95b08","bytes":1764048,"group_id":"HELICOPTER_006","training_excluded":false,"name":"HELICOPTER_006.wav"},{"sha256":"84102b6c5918a672ba8fb12750b7da9fd2d3f9857034627312f9d3a845610efc","bytes":1764048,"group_id":"HELICOPTER_007","training_excluded":false,"name":"HELICOPTER_007.wav"},{"sha256":"900639c6f5b02bbb3b42698550672aee7cd9b1fa77e8e9a8d00a0b0c1ed2e835","bytes":1764048,"group_id":"HELICOPTER_008","training_excluded":false,"name":"HELICOPTER_008.wav"},{"sha256":"7a69c14724199307229fb9fad26738ea4161a2734a8139cc865759a98718cc52","bytes":1764048,"group_id":"HELICOPTER_009","training_excluded":false,"name":"HELICOPTER_009.wav"},{"sha256":"a65b32fa367acfa63ce071b7e0b34b14e9aa82e571c2a5f96ae3b0067f8caa12","bytes":1764048,"group_id":"HELICOPTER_010","training_excluded":false,"name":"HELICOPTER_010.wav"},{"sha256":"cd9effd2f2552f29e6ae40fe6bf5f61ec9ead9cab6d428fbe242e939e9e50324","bytes":1764048,"group_id":"HELICOPTER_011","training_excluded":false,"name":"HELICOPTER_011.wav"},{"sha256":"94b8278d0c2ce8157250c229219505a8e18a1796d6d608de7a928eb896c19edb","bytes":1764048,"group_id":"HELICOPTER_012","training_excluded":false,"name":"HELICOPTER_012.wav"},{"sha256":"f4353777e7cafed31deef733eac90c65b0cc346e27503133acca5923ace0110e","bytes":1764052,"group_id":"HELICOPTER_013","training_excluded":false,"name":"HELICOPTER_013.wav"},{"sha256":"f0dbb7525b3dcb9d5269da1ef79ab1aaa94de912309ac4974caaacc75c8e103c","bytes":1764052,"group_id":"HELICOPTER_014","training_excluded":false,"name":"HELICOPTER_014.wav"},{"sha256":"a87b44b5cd1768bc657be7633c7ada7d93277d18c7214cfdbcbeb12c659f1cf4","bytes":1764052,"group_id":"HELICOPTER_015","training_excluded":false,"name":"HELICOPTER_015.wav"},{"sha256":"c68034426c7f0a97e4e95ebb7c7076893423835710aec2d7467c8b33b3dcef63","bytes":1764052,"group_id":"HELICOPTER_016","training_excluded":false,"name":"HELICOPTER_016.wav"},{"sha256":"7a8dcbbece59a44d276dc6b967ddf5b27a87a596d783eab3481f79510829f882","bytes":1764052,"group_id":"HELICOPTER_017","training_excluded":false,"name":"HELICOPTER_017.wav"},{"sha256":"769efc0d59b2107abb5e798a63781c43a1e382305f063f66d4f7f14a378a83b3","bytes":1764052,"group_id":"HELICOPTER_018","training_excluded":false,"name":"HELICOPTER_018.wav"},{"sha256":"f06f4a129f1ffe7d6ed9547f595d86ecf427e2d43a06c41f243d6a3ad8496ad4","bytes":1764052,"group_id":"HELICOPTER_019","training_excluded":false,"name":"HELICOPTER_019.wav"},{"sha256":"dec81d6f4386afda08ddb0129a2afc8a3c34f8239f30ffe58ddad9cd5cc585e0","bytes":1764052,"group_id":"HELICOPTER_020","training_excluded":false,"name":"HELICOPTER_020.wav"},{"sha256":"8677954e4aa82fe7db2281af1c7f6dbb064e82bc3f3fc66a04ccf23e2e571c05","bytes":1764052,"group_id":"HELICOPTER_021","training_excluded":false,"name":"HELICOPTER_021.wav"},{"sha256":"ff1f8395286e5cac5a24115a3b804480e1e0ebbc2fb903bae8c377ca0bd90fe5","bytes":1764052,"group_id":"HELICOPTER_022","training_excluded":false,"name":"HELICOPTER_022.wav"},{"sha256":"811f1e939c894a6f406d724f51bd6b06ba9dd271f03df3306ac456fc46af13a0","bytes":1764052,"group_id":"HELICOPTER_023","training_excluded":false,"name":"HELICOPTER_023.wav"},{"sha256":"d4b54c14bc09b136c9d7ce7bbd59ac27c6d34675c10bd675c7013d8294fdb3ba","bytes":1764052,"group_id":"HELICOPTER_024","training_excluded":false,"name":"HELICOPTER_024.wav"},{"sha256":"a82b468ae60f483a673b19794c8c43aefcea2bf66f7fa37fc533c918eb8fcb3d","bytes":1764052,"group_id":"HELICOPTER_025","training_excluded":false,"name":"HELICOPTER_025.wav"},{"sha256":"95184efe7292c1e9535a3f11580df0ccb0fff4a8536011690a5e738a1c393856","bytes":1764052,"group_id":"HELICOPTER_026","training_excluded":false,"name":"HELICOPTER_026.wav"},{"sha256":"a9555b719d5401d39c6713fe6dbb121298c1fab2015b29258a0143c9779a6fc0","bytes":1764052,"group_id":"HELICOPTER_027","training_excluded":false,"name":"HELICOPTER_027.wav"},{"sha256":"b4637a283e3175d935b9444cf29e5d47b1dec0736b45d365f517d3a3dc60af80","bytes":1764052,"group_id":"HELICOPTER_028","training_excluded":false,"name":"HELICOPTER_028.wav"},{"sha256":"00d4cb5e960aa0d613f412892993befb1b96f1c391709f8577d1a76cd47eb963","bytes":1764052,"group_id":"HELICOPTER_029","training_excluded":false,"name":"HELICOPTER_029.wav"},{"sha256":"55ecba3904aa9cf9427b6811a1ac69ae36e8c77c29c068742c5f971baf57f9cf","bytes":1764052,"group_id":"HELICOPTER_030","training_excluded":false,"name":"HELICOPTER_030.wav"}],"esc10":{"commit":"33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6","license_git_blob_sha1":"7b7f93640c5d9c62c67bdd68a509991b4db61fbd","metadata_csv_sha256":"ca660da60191a97de289983a05821c9382d852a38a2ba8428980816b68cf6246","files":[{"path":"audio/1-100032-A-0.wav","bytes":441044,"blob":"827f934e88bf940925546573d1386028efe019fe","category":"dog","group_id":"esc10:freesound-100032"},{"path":"audio/1-110389-A-0.wav","bytes":441044,"blob":"6e92e5300a24ea2c6b2ac592ecf99e13a9e2a57e","category":"dog","group_id":"esc10:freesound-110389"},{"path":"audio/1-116765-A-41.wav","bytes":441044,"blob":"f10d5ff9eaca005fe1438d8f1b4c61e56f152fa1","category":"chainsaw","group_id":"esc10:freesound-116765"},{"path":"audio/1-17150-A-12.wav","bytes":441044,"blob":"bd82e0882322c073e0c2d4046bcd214536722424","category":"crackling_fire","group_id":"esc10:freesound-17150"},{"path":"audio/1-172649-A-40.wav","bytes":441044,"blob":"462808a1c8ea1224eed03ed42d8db1c2dcbf79bc","category":"helicopter","group_id":"esc10:freesound-172649"},{"path":"audio/1-172649-B-40.wav","bytes":441044,"blob":"7681f7af50a9618c0f631ce9cd7b85fea0288bb4","category":"helicopter","group_id":"esc10:freesound-172649"},{"path":"audio/1-172649-C-40.wav","bytes":441044,"blob":"859dde63b78ab33983fbbf40954e3b37ada588f0","category":"helicopter","group_id":"esc10:freesound-172649"},{"path":"audio/1-172649-D-40.wav","bytes":441044,"blob":"07f480ed583cff787b36152b864b1ca364e23e2e","category":"helicopter","group_id":"esc10:freesound-172649"},{"path":"audio/1-172649-E-40.wav","bytes":441044,"blob":"c8cce173f210e72c860dcbc52b0fcb6ffe9eb9e1","category":"helicopter","group_id":"esc10:freesound-172649"},{"path":"audio/1-172649-F-40.wav","bytes":441044,"blob":"bd2d2fea17dc4fe9b6e36b6b0d0b675859ebe7a4","category":"helicopter","group_id":"esc10:freesound-172649"},{"path":"audio/1-17367-A-10.wav","bytes":441044,"blob":"600d0516d95f997df769d9f259faa59b69546938","category":"rain","group_id":"esc10:freesound-17367"},{"path":"audio/1-17565-A-12.wav","bytes":441044,"blob":"0d911d64168a4ed9b87a7115114c3e50707afbb0","category":"crackling_fire","group_id":"esc10:freesound-17565"},{"path":"audio/1-17742-A-12.wav","bytes":441044,"blob":"3dd1fe545717150e32ac793ec0a5aef1ccedc2ab","category":"crackling_fire","group_id":"esc10:freesound-17742"},{"path":"audio/1-17808-A-12.wav","bytes":441044,"blob":"8f0a8c62333c503a81f37fa49de2cf377a406d6d","category":"crackling_fire","group_id":"esc10:freesound-17808"},{"path":"audio/1-17808-B-12.wav","bytes":441044,"blob":"ebca2d4ce9188c73af0084af3f979bb3373cf118","category":"crackling_fire","group_id":"esc10:freesound-17808"},{"path":"audio/1-181071-A-40.wav","bytes":441044,"blob":"d10d29c1f6a1fc877abda4cd8b07628c87374e57","category":"helicopter","group_id":"esc10:freesound-181071"},{"path":"audio/1-181071-B-40.wav","bytes":441044,"blob":"acef42945a40e721ac5f19aabf41a686d5792248","category":"helicopter","group_id":"esc10:freesound-181071"},{"path":"audio/1-19898-A-41.wav","bytes":441044,"blob":"e40b96e6e20225d692d581b36ca81a0fd2eaad15","category":"chainsaw","group_id":"esc10:freesound-19898"},{"path":"audio/1-19898-B-41.wav","bytes":441044,"blob":"1338f3e11cbdf498d8d35145fa17e246722bf9da","category":"chainsaw","group_id":"esc10:freesound-19898"},{"path":"audio/1-19898-C-41.wav","bytes":441044,"blob":"d1df3caddf7caf4abbfc3d2f69528aa57603dd7d","category":"chainsaw","group_id":"esc10:freesound-19898"},{"path":"audio/1-21189-A-10.wav","bytes":441044,"blob":"2b8bd44c4e58123af544a10ca4e95104f4995c8e","category":"rain","group_id":"esc10:freesound-21189"},{"path":"audio/1-21934-A-38.wav","bytes":441044,"blob":"85eeede5793009005c5259afa68ff1ec1900ab10","category":"clock_tick","group_id":"esc10:freesound-21934"},{"path":"audio/1-21935-A-38.wav","bytes":441044,"blob":"93426a05dd560969524043823df5309d79a209bb","category":"clock_tick","group_id":"esc10:freesound-21935"},{"path":"audio/1-26222-A-10.wav","bytes":441044,"blob":"9720340897157fda4c4616a2bc90449160c688e4","category":"rain","group_id":"esc10:freesound-26222"},{"path":"audio/1-26806-A-1.wav","bytes":441044,"blob":"f90668192a6fcde4008ce79ac8c5c9e4f4f3169d","category":"rooster","group_id":"esc10:freesound-26806"},{"path":"audio/1-27724-A-1.wav","bytes":441044,"blob":"e0b73778a9eb2f0b136ab2ac24a0c362a572c0cf","category":"rooster","group_id":"esc10:freesound-27724"},{"path":"audio/1-28135-A-11.wav","bytes":441044,"blob":"c101c4f8dd0c0510e4accf2a0e9e2eb859bdd205","category":"sea_waves","group_id":"esc10:freesound-28135"},{"path":"audio/1-28135-B-11.wav","bytes":441044,"blob":"b0bfe1d3a80b0198118614b2e128fd6a340b3864","category":"sea_waves","group_id":"esc10:freesound-28135"},{"path":"audio/1-29561-A-10.wav","bytes":441044,"blob":"e6deb5219c3c5e3fb734dbc58e08a74d27f49647","category":"rain","group_id":"esc10:freesound-29561"},{"path":"audio/1-30226-A-0.wav","bytes":441044,"blob":"0e2a902d8fcc349e88162730514b77aa333bb4ac","category":"dog","group_id":"esc10:freesound-30226"},{"path":"audio/1-30344-A-0.wav","bytes":441044,"blob":"b48c8bb1bce5bd118d279df2715c55934d1f2181","category":"dog","group_id":"esc10:freesound-30344"},{"path":"audio/1-32318-A-0.wav","bytes":441044,"blob":"e1a5bb9563bd24f54ad124ee33a432ea1b03274a","category":"dog","group_id":"esc10:freesound-32318"},{"path":"audio/1-34119-A-1.wav","bytes":441044,"blob":"43de1c092e659313a0e5354b1553a7138feb97f5","category":"rooster","group_id":"esc10:freesound-34119"},{"path":"audio/1-34119-B-1.wav","bytes":441044,"blob":"bbd1eb0adc768d393964bb903c95ebcf3135b365","category":"rooster","group_id":"esc10:freesound-34119"},{"path":"audio/1-35687-A-38.wav","bytes":441044,"blob":"20984e049f2bff4a21a5abffc12fa1bf6b6b6202","category":"clock_tick","group_id":"esc10:freesound-35687"},{"path":"audio/1-39901-A-11.wav","bytes":441044,"blob":"df83d3c83da622a37ad971efb3215bb389354399","category":"sea_waves","group_id":"esc10:freesound-39901"},{"path":"audio/1-39901-B-11.wav","bytes":441044,"blob":"0e5818a8e43dcd83dcfef0f40b8ef57f5d4a5952","category":"sea_waves","group_id":"esc10:freesound-39901"},{"path":"audio/1-39923-A-1.wav","bytes":441044,"blob":"fb7297796b97e912d5968d7882169d9bc6caf340","category":"rooster","group_id":"esc10:freesound-39923"},{"path":"audio/1-4211-A-12.wav","bytes":441044,"blob":"4ba4fc97381bc3b15018da44556de7e8c4dfd4b2","category":"crackling_fire","group_id":"esc10:freesound-4211"},{"path":"audio/1-42139-A-38.wav","bytes":441044,"blob":"1117fb1aa8b2dad8b59d6d83408ecd6340d993c1","category":"clock_tick","group_id":"esc10:freesound-42139"},{"path":"audio/1-43382-A-1.wav","bytes":441044,"blob":"b945857d8b4ae07f9b8666befb83911ed515a934","category":"rooster","group_id":"esc10:freesound-43382"},{"path":"audio/1-43760-A-11.wav","bytes":441044,"blob":"89500e79148fc9e8d2f5f87ca6c6fc79ce32479c","category":"sea_waves","group_id":"esc10:freesound-43760"},{"path":"audio/1-44831-A-1.wav","bytes":441044,"blob":"e6c6be09d638b0cc1831a04dc44883ce291ecb0a","category":"rooster","group_id":"esc10:freesound-44831"},{"path":"audio/1-46272-A-12.wav","bytes":441044,"blob":"a1e5f965ac7978dc1a4dc3c78b16a86ebf181683","category":"crackling_fire","group_id":"esc10:freesound-46272"},{"path":"audio/1-47250-A-41.wav","bytes":441044,"blob":"0c9e41b3a997e59aa92b9698d6e195e385d8f313","category":"chainsaw","group_id":"esc10:freesound-47250"},{"path":"audio/1-47250-B-41.wav","bytes":441044,"blob":"1ab54b48e7c9a9431c35bdb1b21642edffa0b010","category":"chainsaw","group_id":"esc10:freesound-47250"},{"path":"audio/1-48413-A-38.wav","bytes":441044,"blob":"11ecf7f4250bf37f3428eba89291e96ff7032601","category":"clock_tick","group_id":"esc10:freesound-48413"},{"path":"audio/1-50060-A-10.wav","bytes":441044,"blob":"fa97131072e1897acd7f94259cabc4fcf404ffed","category":"rain","group_id":"esc10:freesound-50060"},{"path":"audio/1-54958-A-10.wav","bytes":441044,"blob":"bfe2f997dbf1df865dce4f47cc3975caed229d83","category":"rain","group_id":"esc10:freesound-54958"},{"path":"audio/1-56311-A-10.wav","bytes":441044,"blob":"e0441a8aa72a84d04b3eebed1e9a36506abb9c23","category":"rain","group_id":"esc10:freesound-56311"},{"path":"audio/1-57163-A-38.wav","bytes":441044,"blob":"4368525b3ebf1ad4609309db82cea819475655fb","category":"clock_tick","group_id":"esc10:freesound-57163"},{"path":"audio/1-59513-A-0.wav","bytes":441044,"blob":"4ab3d8324d034bc4b409e00f0cbb90a7ede6bf0b","category":"dog","group_id":"esc10:freesound-59513"},{"path":"audio/1-61252-A-11.wav","bytes":441044,"blob":"80eb54e7eb84a220a9f5764d6d314d8d1d13e594","category":"sea_waves","group_id":"esc10:freesound-61252"},{"path":"audio/1-62849-A-38.wav","bytes":441044,"blob":"4507f43d9776a62c0451cab878bb0a220dbacd8d","category":"clock_tick","group_id":"esc10:freesound-62849"},{"path":"audio/1-62850-A-38.wav","bytes":441044,"blob":"92116c8bcbf71166d696dcdbd3b3190a43ffe2c6","category":"clock_tick","group_id":"esc10:freesound-62850"},{"path":"audio/1-63871-A-10.wav","bytes":441044,"blob":"7f38d00cab65f3a6b4121cccad8f312cef677750","category":"rain","group_id":"esc10:freesound-63871"},{"path":"audio/1-64398-A-41.wav","bytes":441044,"blob":"1353a722e72be83deb664292ceb0650567cb5986","category":"chainsaw","group_id":"esc10:freesound-64398"},{"path":"audio/1-64398-B-41.wav","bytes":441044,"blob":"961c4e263bb2ec2664c387c38abe123fb86c0a14","category":"chainsaw","group_id":"esc10:freesound-64398"},{"path":"audio/1-7057-A-12.wav","bytes":441044,"blob":"0b011429a98ecf0c4f3f033e0359bd0544664c2f","category":"crackling_fire","group_id":"esc10:freesound-7057"},{"path":"audio/1-85362-A-0.wav","bytes":441044,"blob":"e33e993fe343481c082d7c94cb63bf4649525503","category":"dog","group_id":"esc10:freesound-85362"},{"path":"audio/1-91359-A-11.wav","bytes":441044,"blob":"78a4c4a1ecb93a281776cba50adb900989accc2e","category":"sea_waves","group_id":"esc10:freesound-91359"},{"path":"audio/1-91359-B-11.wav","bytes":441044,"blob":"4ba9dc1f64a86f3e8979a0eb3e8f335890a23e77","category":"sea_waves","group_id":"esc10:freesound-91359"},{"path":"audio/1-97392-A-0.wav","bytes":441044,"blob":"2e1663be132678941a71ace8a1362733fd82de47","category":"dog","group_id":"esc10:freesound-97392"},{"path":"audio/2-100786-A-1.wav","bytes":441044,"blob":"e4ad4279f99319d31d31f2a157312a0eac679692","category":"rooster","group_id":"esc10:freesound-100786"},{"path":"audio/2-101676-A-10.wav","bytes":441044,"blob":"1d2d1633f00742f6230c6a8d5d51a482809e0de1","category":"rain","group_id":"esc10:freesound-101676"},{"path":"audio/2-102852-A-11.wav","bytes":441044,"blob":"55dc84ee953945b0d04ddaa3ce25009a55ce6f31","category":"sea_waves","group_id":"esc10:freesound-102852"},{"path":"audio/2-114280-A-0.wav","bytes":441044,"blob":"640300ef66b9eaea9d048d9eb9d621d4d7cbe9b8","category":"dog","group_id":"esc10:freesound-114280"},{"path":"audio/2-114587-A-0.wav","bytes":441044,"blob":"bff55488bf8b6ef59c460c4a0bfdc051647ebc0f","category":"dog","group_id":"esc10:freesound-114587"},{"path":"audio/2-116400-A-0.wav","bytes":441044,"blob":"4780c149b2d3190d0cf624f6e69cdfebd83b61cd","category":"dog","group_id":"esc10:freesound-116400"},{"path":"audio/2-117271-A-0.wav","bytes":441044,"blob":"b2fe19cd797eb3b24e0e1363c0f04c0bf4ad8e59","category":"dog","group_id":"esc10:freesound-117271"},{"path":"audio/2-117625-A-10.wav","bytes":441044,"blob":"710e44189d059c03a4ca70b5d495b621ffab2f04","category":"rain","group_id":"esc10:freesound-117625"},{"path":"audio/2-118072-A-0.wav","bytes":441044,"blob":"0affa9f25eb119883e5c07a2549977cfd3cd2af3","category":"dog","group_id":"esc10:freesound-118072"},{"path":"audio/2-118964-A-0.wav","bytes":441044,"blob":"3aa5fa68cc5dd2e12a85fbb182eda4dcf8046ee1","category":"dog","group_id":"esc10:freesound-118964"},{"path":"audio/2-119748-A-38.wav","bytes":441044,"blob":"df5cab40c9cee24bb6a10568cbfde67a0038b744","category":"clock_tick","group_id":"esc10:freesound-119748"},{"path":"audio/2-122104-A-0.wav","bytes":441044,"blob":"77d7370b682b9b2c458a292762d78fc14e4ef6d9","category":"dog","group_id":"esc10:freesound-122104"},{"path":"audio/2-122104-B-0.wav","bytes":441044,"blob":"f3b49a4640e72d889822c72a49e8200b6c097cd8","category":"dog","group_id":"esc10:freesound-122104"},{"path":"audio/2-124662-A-11.wav","bytes":441044,"blob":"39749ddcec3d6ed32ad3f6712525e13e5c216fe6","category":"sea_waves","group_id":"esc10:freesound-124662"},{"path":"audio/2-125966-A-11.wav","bytes":441044,"blob":"00cb2d37091c8fdd33848853322f522eb10c4741","category":"sea_waves","group_id":"esc10:freesound-125966"},{"path":"audio/2-127108-A-38.wav","bytes":441044,"blob":"ab68bf24c4bb36483f9516ffd1b6c9b42d2ffa22","category":"clock_tick","group_id":"esc10:freesound-127108"},{"path":"audio/2-131943-A-38.wav","bytes":441044,"blob":"a359b94ea1bd194aa9d1457cb77f9a67c93af18c","category":"clock_tick","group_id":"esc10:freesound-131943"},{"path":"audio/2-132157-A-11.wav","bytes":441044,"blob":"0d20d3ac6495cb60d8a38535f1597efd8efec8eb","category":"sea_waves","group_id":"esc10:freesound-132157"},{"path":"audio/2-132157-B-11.wav","bytes":441044,"blob":"e5cfb37fc09ab3c29b97af2ef30280d4b1b74ff6","category":"sea_waves","group_id":"esc10:freesound-132157"},{"path":"audio/2-133863-A-11.wav","bytes":441044,"blob":"d0df45b58b581646db5a789ed898f5bd9ec65995","category":"sea_waves","group_id":"esc10:freesound-133863"},{"path":"audio/2-134700-A-38.wav","bytes":441044,"blob":"c12d36258203cbb61c91983ae1b7a9404ff495f2","category":"clock_tick","group_id":"esc10:freesound-134700"},{"path":"audio/2-135728-A-38.wav","bytes":441044,"blob":"da91425ae99cd0f880dbf236a96bb68e030f055f","category":"clock_tick","group_id":"esc10:freesound-135728"},{"path":"audio/2-137162-A-11.wav","bytes":441044,"blob":"f87e86cf83acd03945a0f672b5d625de927660e0","category":"sea_waves","group_id":"esc10:freesound-137162"},{"path":"audio/2-140147-A-38.wav","bytes":441044,"blob":"e02fd45067b4c0c73ca3c6e606e53faa5ecbdbc8","category":"clock_tick","group_id":"esc10:freesound-140147"},{"path":"audio/2-141584-A-38.wav","bytes":441044,"blob":"698f106155dca96b9146c13f04c761e57a45791b","category":"clock_tick","group_id":"esc10:freesound-141584"},{"path":"audio/2-155801-A-11.wav","bytes":441044,"blob":"f762dda885a2e3a7316098f4ce372f1e19586ac0","category":"sea_waves","group_id":"esc10:freesound-155801"},{"path":"audio/2-18766-A-12.wav","bytes":441044,"blob":"f733048823a7ad9032da75269d7f49845b20800f","category":"crackling_fire","group_id":"esc10:freesound-18766"},{"path":"audio/2-18766-B-12.wav","bytes":441044,"blob":"31030da7fbec3a66cdf54da394da7bc738ab91c3","category":"crackling_fire","group_id":"esc10:freesound-18766"},{"path":"audio/2-188822-A-40.wav","bytes":441044,"blob":"837c04120a1bdf922004ee33bdc0660686d32868","category":"helicopter","group_id":"esc10:freesound-188822"},{"path":"audio/2-188822-B-40.wav","bytes":441044,"blob":"3a68cce2b4b5932c2dc0ea2e815fedee51702367","category":"helicopter","group_id":"esc10:freesound-188822"},{"path":"audio/2-188822-C-40.wav","bytes":441044,"blob":"e39e9a615aaf1fdcf645bccc456624a64b7e3d41","category":"helicopter","group_id":"esc10:freesound-188822"},{"path":"audio/2-188822-D-40.wav","bytes":441044,"blob":"332c7b553edc5c51c5dd887f4266f3f74cbaa16f","category":"helicopter","group_id":"esc10:freesound-188822"},{"path":"audio/2-28314-A-12.wav","bytes":441044,"blob":"a18ca1b39aae92638bb5f100a562243f383a80dd","category":"crackling_fire","group_id":"esc10:freesound-28314"},{"path":"audio/2-28314-B-12.wav","bytes":441044,"blob":"bba2da664e00247728bec8599aee269b4c38d0be","category":"crackling_fire","group_id":"esc10:freesound-28314"},{"path":"audio/2-30322-A-12.wav","bytes":441044,"blob":"853513ebb1c6819303a45e056d95413c29b4d767","category":"crackling_fire","group_id":"esc10:freesound-30322"},{"path":"audio/2-30322-B-12.wav","bytes":441044,"blob":"f90b4a6eed3cf0a0fdd14470ced9cf7a0bd75cf4","category":"crackling_fire","group_id":"esc10:freesound-30322"},{"path":"audio/2-37806-A-40.wav","bytes":441044,"blob":"3bd119bdea7045560ddf0ddad3415b1fba3739ab","category":"helicopter","group_id":"esc10:freesound-37806"},{"path":"audio/2-37806-B-40.wav","bytes":441044,"blob":"52e6835d725a0907a67a30423b3bfb3aca60a8c5","category":"helicopter","group_id":"esc10:freesound-37806"},{"path":"audio/2-37806-C-40.wav","bytes":441044,"blob":"04ec43780724e31db44bacd6367173387d8d5894","category":"helicopter","group_id":"esc10:freesound-37806"},{"path":"audio/2-37806-D-40.wav","bytes":441044,"blob":"a105e0b7c527fd9d03cdf4104c60c7e00756a783","category":"helicopter","group_id":"esc10:freesound-37806"},{"path":"audio/2-50667-A-41.wav","bytes":441044,"blob":"a15bf5fbc5a4be99f25192bb8ef42f0d0948859a","category":"chainsaw","group_id":"esc10:freesound-50667"},{"path":"audio/2-50667-B-41.wav","bytes":441044,"blob":"1e7cc2e549afd0c959c5a097239ca497e8680b55","category":"chainsaw","group_id":"esc10:freesound-50667"},{"path":"audio/2-50668-A-41.wav","bytes":441044,"blob":"d328337672590ab23c7bdc9ae5fca11ee0497957","category":"chainsaw","group_id":"esc10:freesound-50668"},{"path":"audio/2-50668-B-41.wav","bytes":441044,"blob":"24f583083caa2043cc80573161b6c9530cc1f4f7","category":"chainsaw","group_id":"esc10:freesound-50668"},{"path":"audio/2-61311-A-12.wav","bytes":441044,"blob":"62f73675c32dd36b88e27e61ee37eaac717cd888","category":"crackling_fire","group_id":"esc10:freesound-61311"},{"path":"audio/2-65747-A-12.wav","bytes":441044,"blob":"3f7c96193a7469a5b1fce2941c26e83d7194831b","category":"crackling_fire","group_id":"esc10:freesound-65747"},{"path":"audio/2-65750-A-1.wav","bytes":441044,"blob":"12b648a23099d76c40f1c1428a0a6b6883732e55","category":"rooster","group_id":"esc10:freesound-65750"},{"path":"audio/2-68391-A-41.wav","bytes":441044,"blob":"924e0108babe046914fa1e09495235bb32ddc739","category":"chainsaw","group_id":"esc10:freesound-68391"},{"path":"audio/2-68391-B-41.wav","bytes":441044,"blob":"f4523f481fd5552858210b35d1b0a5331d687a6e","category":"chainsaw","group_id":"esc10:freesound-68391"},{"path":"audio/2-71162-A-1.wav","bytes":441044,"blob":"07321949231d710b5e6284cc073085153ad94ddd","category":"rooster","group_id":"esc10:freesound-71162"},{"path":"audio/2-72970-A-10.wav","bytes":441044,"blob":"803c53cb66e414f6514b0213c93e4bfcf480b54b","category":"rain","group_id":"esc10:freesound-72970"},{"path":"audio/2-73027-A-10.wav","bytes":441044,"blob":"68c64ff8be63e7d38a84790c9df96bdf33009645","category":"rain","group_id":"esc10:freesound-73027"},{"path":"audio/2-73260-A-10.wav","bytes":441044,"blob":"935966d408cda6c568d349ba3d4caa125ddab58a","category":"rain","group_id":"esc10:freesound-73260"},{"path":"audio/2-77945-A-41.wav","bytes":441044,"blob":"6bc16d32c45dd1ee297038b89c24290b1f428dc3","category":"chainsaw","group_id":"esc10:freesound-77945"},{"path":"audio/2-77945-B-41.wav","bytes":441044,"blob":"2e391158764f3ea540fe445e1d45739bbda36a34","category":"chainsaw","group_id":"esc10:freesound-77945"},{"path":"audio/2-81270-A-1.wav","bytes":441044,"blob":"5b73c8c7820a5717e7d51e62c42beec4ce157402","category":"rooster","group_id":"esc10:freesound-81270"},{"path":"audio/2-81731-A-10.wav","bytes":441044,"blob":"3a6754bac0184fcbe18a07bbff0d42029a7115f7","category":"rain","group_id":"esc10:freesound-81731"},{"path":"audio/2-82367-A-10.wav","bytes":441044,"blob":"bb8293494c04464ec9115657987b9f86bf3817b9","category":"rain","group_id":"esc10:freesound-82367"},{"path":"audio/2-87781-A-10.wav","bytes":441044,"blob":"8ec1575332e569cb50a0df6f67cbcaae76352989","category":"rain","group_id":"esc10:freesound-87781"},{"path":"audio/2-88724-A-38.wav","bytes":441044,"blob":"6eeee7b7f193b6f03724f033e0d0a4b9600a5dad","category":"clock_tick","group_id":"esc10:freesound-88724"},{"path":"audio/2-95035-A-1.wav","bytes":441044,"blob":"ff85ab4b8e0b0444f249e7fb1a12bb9ebaed4a17","category":"rooster","group_id":"esc10:freesound-95035"},{"path":"audio/2-95258-A-1.wav","bytes":441044,"blob":"063a68b24d87e8d8eca73980861a1b6ec18f7090","category":"rooster","group_id":"esc10:freesound-95258"},{"path":"audio/2-95258-B-1.wav","bytes":441044,"blob":"db8bbc346d4f98e746e27c5afb013ad587d60d33","category":"rooster","group_id":"esc10:freesound-95258"},{"path":"audio/3-104632-A-12.wav","bytes":441044,"blob":"0ef958e6158c3465959b92073704f924a58dbd82","category":"crackling_fire","group_id":"esc10:freesound-104632"},{"path":"audio/3-104958-A-12.wav","bytes":441044,"blob":"9504407f0f89ab7a74ebe4c3c273227dd225f195","category":"crackling_fire","group_id":"esc10:freesound-104958"},{"path":"audio/3-107219-A-1.wav","bytes":441044,"blob":"e7a07f0279ba7ac008c197e23f4b240c0923cf3b","category":"rooster","group_id":"esc10:freesound-107219"},{"path":"audio/3-116135-A-1.wav","bytes":441044,"blob":"cdb3f89bec04d2ed85cc0e9e126f38ff9ff3005a","category":"rooster","group_id":"esc10:freesound-116135"},{"path":"audio/3-118656-A-41.wav","bytes":441044,"blob":"91e14b1e88d287aba37e45c1389fe5933a889747","category":"chainsaw","group_id":"esc10:freesound-118656"},{"path":"audio/3-118657-A-41.wav","bytes":441044,"blob":"4477575c6779b34a0285d53c8acc250c6b1f9708","category":"chainsaw","group_id":"esc10:freesound-118657"},{"path":"audio/3-118657-B-41.wav","bytes":441044,"blob":"77ce79ef29b1daed85407fa901fb70e5c6ee5d23","category":"chainsaw","group_id":"esc10:freesound-118657"},{"path":"audio/3-118658-A-41.wav","bytes":441044,"blob":"ae80452694112ba3edcdb06627dede13c70ad252","category":"chainsaw","group_id":"esc10:freesound-118658"},{"path":"audio/3-118658-B-41.wav","bytes":441044,"blob":"1844ed2fca8dd855ee7fcc350a2cf1ff920cfa06","category":"chainsaw","group_id":"esc10:freesound-118658"},{"path":"audio/3-118972-A-41.wav","bytes":441044,"blob":"9e1f158f1669bbbe2e6943d7083af88840dd052b","category":"chainsaw","group_id":"esc10:freesound-118972"},{"path":"audio/3-118972-B-41.wav","bytes":441044,"blob":"a7bd325578aa5b9c3ded8f3f328ae3e75e2b3629","category":"chainsaw","group_id":"esc10:freesound-118972"},{"path":"audio/3-120644-A-12.wav","bytes":441044,"blob":"1d13ab24a374ef3f92874549b4eefd1469f2b5e8","category":"crackling_fire","group_id":"esc10:freesound-120644"},{"path":"audio/3-132852-A-10.wav","bytes":441044,"blob":"350b379453b933b8d0692cfcf0974ecbc7845d0f","category":"rain","group_id":"esc10:freesound-132852"},{"path":"audio/3-134049-A-1.wav","bytes":441044,"blob":"cbf1e3f01f79205ce837332b8d47945ebdfabd0c","category":"rooster","group_id":"esc10:freesound-134049"},{"path":"audio/3-136288-A-0.wav","bytes":441044,"blob":"d30939bb52969e221d736bd7d228d01779a1e062","category":"dog","group_id":"esc10:freesound-136288"},{"path":"audio/3-137152-A-1.wav","bytes":441044,"blob":"4989bc3d781a91553575eb3744e99dd199b702ae","category":"rooster","group_id":"esc10:freesound-137152"},{"path":"audio/3-140774-A-10.wav","bytes":441044,"blob":"7a66d4087495bbf9f098e03645c3d2f09e33c113","category":"rain","group_id":"esc10:freesound-140774"},{"path":"audio/3-142005-A-10.wav","bytes":441044,"blob":"b5e44f2d9dda76769b11ba2695586ba0175ce9c4","category":"rain","group_id":"esc10:freesound-142005"},{"path":"audio/3-142006-A-10.wav","bytes":441044,"blob":"2e2ebd8f2152566301aca7c9b1485f751b8a6d23","category":"rain","group_id":"esc10:freesound-142006"},{"path":"audio/3-142593-A-38.wav","bytes":441044,"blob":"766b6a8ea252be7f8454fc0bfce21dc1a879840e","category":"clock_tick","group_id":"esc10:freesound-142593"},{"path":"audio/3-143929-A-10.wav","bytes":441044,"blob":"a24552555823b2d73cc455a51ce99b6364aa45cd","category":"rain","group_id":"esc10:freesound-143929"},{"path":"audio/3-143933-A-38.wav","bytes":441044,"blob":"4de62f76162a688d2d9cb92a4686b28d1c5c1ec3","category":"clock_tick","group_id":"esc10:freesound-143933"},{"path":"audio/3-144028-A-0.wav","bytes":441044,"blob":"53d6a3ca61303dba95f4cf63fe3c497b836ff4e2","category":"dog","group_id":"esc10:freesound-144028"},{"path":"audio/3-144827-A-11.wav","bytes":441044,"blob":"4d0bbd203597f6ac1eff6f757212e63718c8ed13","category":"sea_waves","group_id":"esc10:freesound-144827"},{"path":"audio/3-144827-B-11.wav","bytes":441044,"blob":"c3a46056c4b7d1bc67c37a32afd30f05d958123d","category":"sea_waves","group_id":"esc10:freesound-144827"},{"path":"audio/3-145382-A-1.wav","bytes":441044,"blob":"fbd22d98cb02774688bfc77a292a9ec9c6a876f5","category":"rooster","group_id":"esc10:freesound-145382"},{"path":"audio/3-145774-A-12.wav","bytes":441044,"blob":"7884f4f476e32ae6cd925fd1f2317735eaf30882","category":"crackling_fire","group_id":"esc10:freesound-145774"},{"path":"audio/3-147965-A-12.wav","bytes":441044,"blob":"19117eda30f90e50150f89c52cb81fed426388bc","category":"crackling_fire","group_id":"esc10:freesound-147965"},{"path":"audio/3-149189-A-1.wav","bytes":441044,"blob":"64c51a9d08d7401133a2239e184f7b4e8375da65","category":"rooster","group_id":"esc10:freesound-149189"},{"path":"audio/3-150363-A-38.wav","bytes":441044,"blob":"0dbea245e7bfe352153b3b315acb66edfef34f94","category":"clock_tick","group_id":"esc10:freesound-150363"},{"path":"audio/3-150979-A-40.wav","bytes":441044,"blob":"4e584c38d037953721e5f02af9d16a51501a7a3f","category":"helicopter","group_id":"esc10:freesound-150979"},{"path":"audio/3-150979-B-40.wav","bytes":441044,"blob":"e6a43e464a5e6ab5ae61f20422c4c66c6f49e08c","category":"helicopter","group_id":"esc10:freesound-150979"},{"path":"audio/3-150979-C-40.wav","bytes":441044,"blob":"459eaa76863661dc0def8cf673859e6de71139d1","category":"helicopter","group_id":"esc10:freesound-150979"},{"path":"audio/3-154926-A-40.wav","bytes":441044,"blob":"00c6501ab74c1d5d71be2a3f07b5dedc958c75a7","category":"helicopter","group_id":"esc10:freesound-154926"},{"path":"audio/3-154926-B-40.wav","bytes":441044,"blob":"a5281b4a29a7cb28b42580cabbd6d3f57108d922","category":"helicopter","group_id":"esc10:freesound-154926"},{"path":"audio/3-154957-A-1.wav","bytes":441044,"blob":"f653558a18c6941d68ad8955266cfa7cc982b5e4","category":"rooster","group_id":"esc10:freesound-154957"},{"path":"audio/3-155312-A-0.wav","bytes":441044,"blob":"111a621fcbaf8bf2975d4c7f74ba6a3528dde76e","category":"dog","group_id":"esc10:freesound-155312"},{"path":"audio/3-155642-A-11.wav","bytes":441044,"blob":"9d52df706121f4608687369109d9b227f4b2f85e","category":"sea_waves","group_id":"esc10:freesound-155642"},{"path":"audio/3-155642-B-11.wav","bytes":441044,"blob":"0388cc66bc65aa9b42dbca664b610949d04be68a","category":"sea_waves","group_id":"esc10:freesound-155642"},{"path":"audio/3-157149-A-10.wav","bytes":441044,"blob":"9a56658afb3703b43abb5299c99dcffaa6598602","category":"rain","group_id":"esc10:freesound-157149"},{"path":"audio/3-157187-A-12.wav","bytes":441044,"blob":"c2472cb9d05a0ac4e9eed6bda816dd3a0fb580e9","category":"crackling_fire","group_id":"esc10:freesound-157187"},{"path":"audio/3-157487-A-10.wav","bytes":441044,"blob":"474caaf4a6aa41c6ce3f613d33220028d3bf1005","category":"rain","group_id":"esc10:freesound-157487"},{"path":"audio/3-157615-A-10.wav","bytes":441044,"blob":"145a06cfd54a25c222fa62ba8789aa0e6e37e25d","category":"rain","group_id":"esc10:freesound-157615"},{"path":"audio/3-157695-A-0.wav","bytes":441044,"blob":"723a0d24593aa57135a39f91ae30868ccd98f907","category":"dog","group_id":"esc10:freesound-157695"},{"path":"audio/3-158476-A-12.wav","bytes":441044,"blob":"933e631ab4ee729ecfeece84c5ab42344c2af567","category":"crackling_fire","group_id":"esc10:freesound-158476"},{"path":"audio/3-163288-A-1.wav","bytes":441044,"blob":"0fe498209abd0584dbbf5f77204454c0e887c8bc","category":"rooster","group_id":"esc10:freesound-163288"},{"path":"audio/3-163459-A-0.wav","bytes":441044,"blob":"d5a1cb707d47c2e3c7560d2c99a179f29fb9a04f","category":"dog","group_id":"esc10:freesound-163459"},{"path":"audio/3-164120-A-11.wav","bytes":441044,"blob":"99a0e58e53f65cdb4afe119f2b4260cce62209a1","category":"sea_waves","group_id":"esc10:freesound-164120"},{"path":"audio/3-164630-A-11.wav","bytes":441044,"blob":"da3e939db33f15e6ebf21a34ea847c9b24ccbbb0","category":"sea_waves","group_id":"esc10:freesound-164630"},{"path":"audio/3-164688-A-38.wav","bytes":441044,"blob":"f1385273f8d00ee7bccc500caff120ae3a325f38","category":"clock_tick","group_id":"esc10:freesound-164688"},{"path":"audio/3-165856-A-41.wav","bytes":441044,"blob":"1d14858ae1eac581083b06dfce034e31e672edba","category":"chainsaw","group_id":"esc10:freesound-165856"},{"path":"audio/3-166422-A-11.wav","bytes":441044,"blob":"1068c33440cbf67508b238f455442401a911cf36","category":"sea_waves","group_id":"esc10:freesound-166422"},{"path":"audio/3-170015-A-0.wav","bytes":441044,"blob":"3f77e6f92e3fa7c0f1b6b342d9994441f020c97a","category":"dog","group_id":"esc10:freesound-170015"},{"path":"audio/3-170377-A-38.wav","bytes":441044,"blob":"3557422127224742be9c66da558b9f350c227a9c","category":"clock_tick","group_id":"esc10:freesound-170377"},{"path":"audio/3-170383-A-38.wav","bytes":441044,"blob":"6df922f00e1c8f214c6a195e401324778111b0a2","category":"clock_tick","group_id":"esc10:freesound-170383"},{"path":"audio/3-171012-A-38.wav","bytes":441044,"blob":"e0657f40e4ef35c177c3d18db5d73c6ef41f4acb","category":"clock_tick","group_id":"esc10:freesound-171012"},{"path":"audio/3-171041-A-38.wav","bytes":441044,"blob":"b7ab013c820897851894d64263bf4899ba7a67e9","category":"clock_tick","group_id":"esc10:freesound-171041"},{"path":"audio/3-180256-A-0.wav","bytes":441044,"blob":"52be532086787624ca389edddbe2f1df002c6959","category":"dog","group_id":"esc10:freesound-180256"},{"path":"audio/3-180977-A-0.wav","bytes":441044,"blob":"64a3423e151a3d491c381cb7177fa0b306f58a06","category":"dog","group_id":"esc10:freesound-180977"},{"path":"audio/3-187710-A-11.wav","bytes":441044,"blob":"6d75a1ae8d3bcdc97e783e68981b51566166e10f","category":"sea_waves","group_id":"esc10:freesound-187710"},{"path":"audio/3-65748-A-12.wav","bytes":441044,"blob":"b185ff462bb8912f18ffdf7dfa697779ff02ee5f","category":"crackling_fire","group_id":"esc10:freesound-65748"},{"path":"audio/3-68630-A-40.wav","bytes":441044,"blob":"6b74acf79f823fcd2c43fd1f7a754394b807158e","category":"helicopter","group_id":"esc10:freesound-68630"},{"path":"audio/3-68630-B-40.wav","bytes":441044,"blob":"d16f00135bbdf275fd03a6d9edad0f08c4877a5a","category":"helicopter","group_id":"esc10:freesound-68630"},{"path":"audio/3-68630-C-40.wav","bytes":441044,"blob":"1fa31f3f92b57925b0bc293df6a03c5e9e8c8414","category":"helicopter","group_id":"esc10:freesound-68630"},{"path":"audio/4-125929-A-40.wav","bytes":441044,"blob":"b52d76efca994da600c4cce57a18c04ff44af981","category":"helicopter","group_id":"esc10:freesound-125929"},{"path":"audio/4-149294-A-41.wav","bytes":441044,"blob":"3a34164d44a9b58554c9278c6a821cdcc7b3e312","category":"chainsaw","group_id":"esc10:freesound-149294"},{"path":"audio/4-149294-B-41.wav","bytes":441044,"blob":"1be93fd969d547a297f98dc98b49602c9ba18a2b","category":"chainsaw","group_id":"esc10:freesound-149294"},{"path":"audio/4-157611-A-41.wav","bytes":441044,"blob":"17f808c7fdee59a8419aed0e45c16f23152ed8b4","category":"chainsaw","group_id":"esc10:freesound-157611"},{"path":"audio/4-157611-B-41.wav","bytes":441044,"blob":"b9c747cb162b0ae4e60fb482adb3ba4b5b239d2c","category":"chainsaw","group_id":"esc10:freesound-157611"},{"path":"audio/4-160999-A-10.wav","bytes":441044,"blob":"0f7b52a218566cf52cebf0b3b8f80703d95201ca","category":"rain","group_id":"esc10:freesound-160999"},{"path":"audio/4-161127-A-10.wav","bytes":441044,"blob":"1b3bc376feb3daee8bcc51968cb200db831a1381","category":"rain","group_id":"esc10:freesound-161127"},{"path":"audio/4-161579-A-40.wav","bytes":441044,"blob":"9f658f4c033dfefc20e4693c4569e03e6c6b85f9","category":"helicopter","group_id":"esc10:freesound-161579"},{"path":"audio/4-161579-B-40.wav","bytes":441044,"blob":"10c41ded4e61294f4882767eb71ed6073a1939c2","category":"helicopter","group_id":"esc10:freesound-161579"},{"path":"audio/4-163264-A-10.wav","bytes":441044,"blob":"9e28bf11cf883d1f466bdd9ebb3249658afadac3","category":"rain","group_id":"esc10:freesound-163264"},{"path":"audio/4-164021-A-1.wav","bytes":441044,"blob":"b6e3721373f1b88e587f49c09394b02ad2760f60","category":"rooster","group_id":"esc10:freesound-164021"},{"path":"audio/4-164064-A-1.wav","bytes":441044,"blob":"f1b0fde7d857ab9a91e8a814df3e72b9a3bf959f","category":"rooster","group_id":"esc10:freesound-164064"},{"path":"audio/4-164064-B-1.wav","bytes":441044,"blob":"c17a3d6b2ba1bc2c9bc4238c836da65f29232812","category":"rooster","group_id":"esc10:freesound-164064"},{"path":"audio/4-164064-C-1.wav","bytes":441044,"blob":"2b94adfde9b5eff1cc68f530d51c414570aa683c","category":"rooster","group_id":"esc10:freesound-164064"},{"path":"audio/4-164206-A-10.wav","bytes":441044,"blob":"a62c9e3f5acba2246e4adc6646d36b3f733950fc","category":"rain","group_id":"esc10:freesound-164206"},{"path":"audio/4-164661-A-12.wav","bytes":441044,"blob":"e34562c51fd01da6433faa64d3f4ce7aeaec5eaf","category":"crackling_fire","group_id":"esc10:freesound-164661"},{"path":"audio/4-164661-B-12.wav","bytes":441044,"blob":"90ceb871b0420a2844faa004b36262dc9848d413","category":"crackling_fire","group_id":"esc10:freesound-164661"},{"path":"audio/4-164859-A-1.wav","bytes":441044,"blob":"0eb9fda631e1fbf73bbc5a178ceb85504e1f1a7d","category":"rooster","group_id":"esc10:freesound-164859"},{"path":"audio/4-165823-A-41.wav","bytes":441044,"blob":"36d60400a227a4b81d8d86cd0d2e29e36352633b","category":"chainsaw","group_id":"esc10:freesound-165823"},{"path":"audio/4-165823-B-41.wav","bytes":441044,"blob":"056181960eb570fe294962de482909372c698b51","category":"chainsaw","group_id":"esc10:freesound-165823"},{"path":"audio/4-166661-A-10.wav","bytes":441044,"blob":"b3a24e8044c5ca20ff46ee546b71c315eed12a85","category":"rain","group_id":"esc10:freesound-166661"},{"path":"audio/4-167063-A-11.wav","bytes":441044,"blob":"a1f39376d7db5108da179ee59e28b0ce27abd9ea","category":"sea_waves","group_id":"esc10:freesound-167063"},{"path":"audio/4-167063-B-11.wav","bytes":441044,"blob":"b7ed2f5babf7a2fcade6f01f0a5a8e3744f92a0b","category":"sea_waves","group_id":"esc10:freesound-167063"},{"path":"audio/4-167063-C-11.wav","bytes":441044,"blob":"55a3abbfddcd87dda79c10910033b8fd1d152682","category":"sea_waves","group_id":"esc10:freesound-167063"},{"path":"audio/4-169127-A-41.wav","bytes":441044,"blob":"210bed0319549e626033e8c1aed3cf7d661660dd","category":"chainsaw","group_id":"esc10:freesound-169127"},{"path":"audio/4-169127-B-41.wav","bytes":441044,"blob":"21bcfae09e1862d0d9d519886d5cd05442d1b850","category":"chainsaw","group_id":"esc10:freesound-169127"},{"path":"audio/4-170078-A-1.wav","bytes":441044,"blob":"cb63c3d125d3736de53eb70bfe4872689e188ac1","category":"rooster","group_id":"esc10:freesound-170078"},{"path":"audio/4-170247-A-12.wav","bytes":441044,"blob":"d652fcd1c249dd557530e263a82538a012130b98","category":"crackling_fire","group_id":"esc10:freesound-170247"},{"path":"audio/4-170247-B-12.wav","bytes":441044,"blob":"e040aeb22b48935cb788d551dd13adf440224883","category":"crackling_fire","group_id":"esc10:freesound-170247"},{"path":"audio/4-171207-A-12.wav","bytes":441044,"blob":"54a9dd018a1f12f78c60ae3de68077a32a100d4f","category":"crackling_fire","group_id":"esc10:freesound-171207"},{"path":"audio/4-175000-A-40.wav","bytes":441044,"blob":"e814883f5a4e22790e1c7456601954ab507526e6","category":"helicopter","group_id":"esc10:freesound-175000"},{"path":"audio/4-175000-B-40.wav","bytes":441044,"blob":"79e7a37a52b227fc6b0571748d5a9c384e0a2c1c","category":"helicopter","group_id":"esc10:freesound-175000"},{"path":"audio/4-175000-C-40.wav","bytes":441044,"blob":"d1591af60c7e7079bb12c4f8781a8536e08a3720","category":"helicopter","group_id":"esc10:freesound-175000"},{"path":"audio/4-175945-A-38.wav","bytes":441044,"blob":"c48c517b475243d24c3c367a10180a0f2bb3eaa6","category":"clock_tick","group_id":"esc10:freesound-175945"},{"path":"audio/4-177250-A-10.wav","bytes":441044,"blob":"3be0edd48c964e52d7fdb1938174226f68ae796d","category":"rain","group_id":"esc10:freesound-177250"},{"path":"audio/4-179984-A-38.wav","bytes":441044,"blob":"dbab289a5e2be5e348150ed52eba2b97b36a4bd9","category":"clock_tick","group_id":"esc10:freesound-179984"},{"path":"audio/4-180380-A-10.wav","bytes":441044,"blob":"d74299504b7aa733b4b1d60ad76a5dd0d984498d","category":"rain","group_id":"esc10:freesound-180380"},{"path":"audio/4-181035-A-38.wav","bytes":441044,"blob":"a4128d96eb4ca4e1e8055f5232eb48beda9d4412","category":"clock_tick","group_id":"esc10:freesound-181035"},{"path":"audio/4-181286-A-10.wav","bytes":441044,"blob":"ed7bbf10423f099f77ebe420c0c1d526dc733b95","category":"rain","group_id":"esc10:freesound-181286"},{"path":"audio/4-181563-A-12.wav","bytes":441044,"blob":"1f0fafd7c93f2a2c028f304373e3c399d9aa1a91","category":"crackling_fire","group_id":"esc10:freesound-181563"},{"path":"audio/4-181865-A-38.wav","bytes":441044,"blob":"f3686a810519930dea2e2f1233e70815f4fbcd22","category":"clock_tick","group_id":"esc10:freesound-181865"},{"path":"audio/4-182368-A-12.wav","bytes":441044,"blob":"5c64895382166d8d8468768031984cbc7750100a","category":"crackling_fire","group_id":"esc10:freesound-182368"},{"path":"audio/4-182369-A-12.wav","bytes":441044,"blob":"e99203d548d13fbb8567244f7b474cad182e8828","category":"crackling_fire","group_id":"esc10:freesound-182369"},{"path":"audio/4-182395-A-0.wav","bytes":441044,"blob":"ab0dd81ce1598053f846dc728d8c72c514cb2a36","category":"dog","group_id":"esc10:freesound-182395"},{"path":"audio/4-182613-A-11.wav","bytes":441044,"blob":"a9f08fb7ec02445cea5d020a0582e4534c869e04","category":"sea_waves","group_id":"esc10:freesound-182613"},{"path":"audio/4-182613-B-11.wav","bytes":441044,"blob":"4af806e40f1505e9b7072a9cd30577ccefb81830","category":"sea_waves","group_id":"esc10:freesound-182613"},{"path":"audio/4-183487-A-1.wav","bytes":441044,"blob":"a3cce1b23420e1e66985065063f0b37f55c66c8c","category":"rooster","group_id":"esc10:freesound-183487"},{"path":"audio/4-183992-A-0.wav","bytes":441044,"blob":"4b3fd1257a4c0b5f16e292c325e9858fc7191107","category":"dog","group_id":"esc10:freesound-183992"},{"path":"audio/4-184575-A-0.wav","bytes":441044,"blob":"2fed2c957b743de546df489ebe90242aab95c4ec","category":"dog","group_id":"esc10:freesound-184575"},{"path":"audio/4-188033-A-38.wav","bytes":441044,"blob":"1cb17870081499e01bb490ac3b67d291e8d00381","category":"clock_tick","group_id":"esc10:freesound-188033"},{"path":"audio/4-191327-A-38.wav","bytes":441044,"blob":"57904d247f44be8994667e176b21827529744b0e","category":"clock_tick","group_id":"esc10:freesound-191327"},{"path":"audio/4-191687-A-0.wav","bytes":441044,"blob":"b524e89890cd23f4dcb432afb75b2af1e63e78f1","category":"dog","group_id":"esc10:freesound-191687"},{"path":"audio/4-192236-A-0.wav","bytes":441044,"blob":"88618e9d65c74bd42aed6bc0e452f0062787e1d1","category":"dog","group_id":"esc10:freesound-192236"},{"path":"audio/4-193480-A-40.wav","bytes":441044,"blob":"7cb17ab85b7a717fcf2b8d1b014d0d6ca64e4346","category":"helicopter","group_id":"esc10:freesound-193480"},{"path":"audio/4-193480-B-40.wav","bytes":441044,"blob":"57fe7f7a148b17a319d426b89cd559bfbdfe38e1","category":"helicopter","group_id":"esc10:freesound-193480"},{"path":"audio/4-194711-A-38.wav","bytes":441044,"blob":"c9269696ac306408c20808e56aa184bfb9a73cad","category":"clock_tick","group_id":"esc10:freesound-194711"},{"path":"audio/4-194754-A-0.wav","bytes":441044,"blob":"d0d17c6a87df97042ea61947ab39ee56a2e278ea","category":"dog","group_id":"esc10:freesound-194754"},{"path":"audio/4-195497-A-11.wav","bytes":441044,"blob":"2ea5e1066fa4f63fd9f6ccab09dee772bde244b1","category":"sea_waves","group_id":"esc10:freesound-195497"},{"path":"audio/4-195497-B-11.wav","bytes":441044,"blob":"bcea11c38430094c3e7cdeb81b6160d47df253a3","category":"sea_waves","group_id":"esc10:freesound-195497"},{"path":"audio/4-198965-A-38.wav","bytes":441044,"blob":"a4022b23857f9a627125b03cbb63a403e299204f","category":"clock_tick","group_id":"esc10:freesound-198965"},{"path":"audio/4-199261-A-0.wav","bytes":441044,"blob":"102410afc607e42c6af917f402c975834405283f","category":"dog","group_id":"esc10:freesound-199261"},{"path":"audio/4-204618-A-11.wav","bytes":441044,"blob":"849acd1eb4dff5eb940673c7d4ebd97e4c9e9dae","category":"sea_waves","group_id":"esc10:freesound-204618"},{"path":"audio/4-207124-A-0.wav","bytes":441044,"blob":"a0f603ea031b1b82237d713c30d9112501c13acd","category":"dog","group_id":"esc10:freesound-207124"},{"path":"audio/4-208021-A-1.wav","bytes":441044,"blob":"db5df4fe832f7c01667ac4ed966bdd7784670593","category":"rooster","group_id":"esc10:freesound-208021"},{"path":"audio/5-170338-A-41.wav","bytes":441044,"blob":"6329e3119b000c8492c34a984255554804d86499","category":"chainsaw","group_id":"esc10:freesound-170338"},{"path":"audio/5-170338-B-41.wav","bytes":441044,"blob":"05fa269a1368621ae93d8f492685fe9caf235d10","category":"chainsaw","group_id":"esc10:freesound-170338"},{"path":"audio/5-171653-A-41.wav","bytes":441044,"blob":"3bae68ad7d4f4101ae1bcd6c251f1fbdd88459c6","category":"chainsaw","group_id":"esc10:freesound-171653"},{"path":"audio/5-177957-A-40.wav","bytes":441044,"blob":"ef304bf180d8fc11d9337b5615c248f74f319fc3","category":"helicopter","group_id":"esc10:freesound-177957"},{"path":"audio/5-177957-B-40.wav","bytes":441044,"blob":"c57f7a580d5b42b43e79704792ba1b049d0b4f17","category":"helicopter","group_id":"esc10:freesound-177957"},{"path":"audio/5-177957-C-40.wav","bytes":441044,"blob":"9ab7d79b21877479baf9ef9a1f18a423bd69fa7e","category":"helicopter","group_id":"esc10:freesound-177957"},{"path":"audio/5-177957-D-40.wav","bytes":441044,"blob":"5fe0764a33d62dff06aa380de6349c270f147d86","category":"helicopter","group_id":"esc10:freesound-177957"},{"path":"audio/5-177957-E-40.wav","bytes":441044,"blob":"de7402c1430d330926a16b8cc6816b3f674892c8","category":"helicopter","group_id":"esc10:freesound-177957"},{"path":"audio/5-181766-A-10.wav","bytes":441044,"blob":"2a09938d139395389c5962592d766fa729f2b356","category":"rain","group_id":"esc10:freesound-181766"},{"path":"audio/5-185579-A-41.wav","bytes":441044,"blob":"a69e8484f6a56fa932ac668e8c2392a052ffb6cf","category":"chainsaw","group_id":"esc10:freesound-185579"},{"path":"audio/5-185579-B-41.wav","bytes":441044,"blob":"f911fb653e6da49b94e0fae38198518b62d6a1b0","category":"chainsaw","group_id":"esc10:freesound-185579"},{"path":"audio/5-186924-A-12.wav","bytes":441044,"blob":"3cc189aba1ec9c8aa1b1f7b852553203e70c31db","category":"crackling_fire","group_id":"esc10:freesound-186924"},{"path":"audio/5-188655-A-10.wav","bytes":441044,"blob":"db56db957314620866a97bb9b5cf6ddc03e10935","category":"rain","group_id":"esc10:freesound-188655"},{"path":"audio/5-189212-A-12.wav","bytes":441044,"blob":"b84edd3493b67af4d022318a4dcc01e3239a67ce","category":"crackling_fire","group_id":"esc10:freesound-189212"},{"path":"audio/5-189237-A-12.wav","bytes":441044,"blob":"5b219c7d4bb618c20d606f1a6fe30c2bd9d54a5b","category":"crackling_fire","group_id":"esc10:freesound-189237"},{"path":"audio/5-191131-A-40.wav","bytes":441044,"blob":"6b517465c33e1dc09ee0e9890d8790990535f99c","category":"helicopter","group_id":"esc10:freesound-191131"},{"path":"audio/5-193339-A-10.wav","bytes":441044,"blob":"f03fea3ea38a0079966688e1e70454bd7bb8bcaf","category":"rain","group_id":"esc10:freesound-193339"},{"path":"audio/5-193473-A-12.wav","bytes":441044,"blob":"32e9f57259e34dca248f8f78dac5e9a200b70ef0","category":"crackling_fire","group_id":"esc10:freesound-193473"},{"path":"audio/5-193473-B-12.wav","bytes":441044,"blob":"55ba8763cffcdfbb85904af9aab84615b88014a1","category":"crackling_fire","group_id":"esc10:freesound-193473"},{"path":"audio/5-194892-A-10.wav","bytes":441044,"blob":"35b28d26cd8ed89d720924adbbf35da4e14eae0b","category":"rain","group_id":"esc10:freesound-194892"},{"path":"audio/5-194930-A-1.wav","bytes":441044,"blob":"b46c6f6312f9ec41c3f7c9fc0da4d4c9cfb13e99","category":"rooster","group_id":"esc10:freesound-194930"},{"path":"audio/5-194930-B-1.wav","bytes":441044,"blob":"9eefb7c53a095615093d93065874714d768dc811","category":"rooster","group_id":"esc10:freesound-194930"},{"path":"audio/5-195710-A-10.wav","bytes":441044,"blob":"8a75b307855cfbece3c0c40e4fb7565cd3151617","category":"rain","group_id":"esc10:freesound-195710"},{"path":"audio/5-198321-A-10.wav","bytes":441044,"blob":"8fa6cd4d8eb3c6b863274c65fe51726f716ecbb9","category":"rain","group_id":"esc10:freesound-198321"},{"path":"audio/5-200334-A-1.wav","bytes":441044,"blob":"fdb7318f7ac0ee73ee9a736878131975623b0273","category":"rooster","group_id":"esc10:freesound-200334"},{"path":"audio/5-200334-B-1.wav","bytes":441044,"blob":"06fd3ccadf544f8783051e3831f18d79f0255bfb","category":"rooster","group_id":"esc10:freesound-200334"},{"path":"audio/5-200339-A-1.wav","bytes":441044,"blob":"046e3454836c54de0f73a8eac93bbfb90b8d6071","category":"rooster","group_id":"esc10:freesound-200339"},{"path":"audio/5-200461-A-11.wav","bytes":441044,"blob":"586683f1a3b5672a0b88d335c8f7676e776d38ef","category":"sea_waves","group_id":"esc10:freesound-200461"},{"path":"audio/5-200461-B-11.wav","bytes":441044,"blob":"ffb08327cb4d8e8a528dad0d8023a5f32784c220","category":"sea_waves","group_id":"esc10:freesound-200461"},{"path":"audio/5-201194-A-38.wav","bytes":441044,"blob":"4fa6eb01f120e03150af2d48dfe93aea981f82f9","category":"clock_tick","group_id":"esc10:freesound-201194"},{"path":"audio/5-202898-A-10.wav","bytes":441044,"blob":"415e3c2010b32fa0fc35862785abe827018d8736","category":"rain","group_id":"esc10:freesound-202898"},{"path":"audio/5-203128-A-0.wav","bytes":441044,"blob":"55e673d389cc3b2765dd5c6c8328db33d394e8f5","category":"dog","group_id":"esc10:freesound-203128"},{"path":"audio/5-203128-B-0.wav","bytes":441044,"blob":"779596d47f7ac68d0f3b6c833ddb339c4a36b90e","category":"dog","group_id":"esc10:freesound-203128"},{"path":"audio/5-203739-A-10.wav","bytes":441044,"blob":"c7a801acd318b26db41ced3e0497c8c722b2e5a0","category":"rain","group_id":"esc10:freesound-203739"},{"path":"audio/5-205898-A-40.wav","bytes":441044,"blob":"4af5b5e32f8d7fbfc444659764fdfc1435dae33a","category":"helicopter","group_id":"esc10:freesound-205898"},{"path":"audio/5-208030-A-0.wav","bytes":441044,"blob":"a2cd0eee56099c83e4d580d3ce3903e6496af5c9","category":"dog","group_id":"esc10:freesound-208030"},{"path":"audio/5-208624-A-38.wav","bytes":441044,"blob":"b0fdd6df5a89f43442701b5f61ddfc14da389331","category":"clock_tick","group_id":"esc10:freesound-208624"},{"path":"audio/5-208810-A-11.wav","bytes":441044,"blob":"40062170aae2b8379cdf3bc58a204f8e2446a217","category":"sea_waves","group_id":"esc10:freesound-208810"},{"path":"audio/5-208810-B-11.wav","bytes":441044,"blob":"79aa2b8933b29180fd6f69d015a1d69d933bc8ee","category":"sea_waves","group_id":"esc10:freesound-208810"},{"path":"audio/5-209698-A-38.wav","bytes":441044,"blob":"7afd61ec7e77f0a8b003aa12c77837d4a9d76aab","category":"clock_tick","group_id":"esc10:freesound-209698"},{"path":"audio/5-209833-A-38.wav","bytes":441044,"blob":"c2ce047fe58da4097da8e636819710729a62abcb","category":"clock_tick","group_id":"esc10:freesound-209833"},{"path":"audio/5-210571-A-38.wav","bytes":441044,"blob":"139be1b4f1e7fe869c8b8f6b2ae63a02f6c9059f","category":"clock_tick","group_id":"esc10:freesound-210571"},{"path":"audio/5-212181-A-38.wav","bytes":441044,"blob":"504c205aba63d178c19d5c1bd8b5482554f8c34e","category":"clock_tick","group_id":"esc10:freesound-212181"},{"path":"audio/5-212454-A-0.wav","bytes":441044,"blob":"51ce420bc47b24251d50996bc93d1eca6dd7b09f","category":"dog","group_id":"esc10:freesound-212454"},{"path":"audio/5-213077-A-11.wav","bytes":441044,"blob":"f6932fc42e6ee7aa413d070b9b39b7e42eedaba1","category":"sea_waves","group_id":"esc10:freesound-213077"},{"path":"audio/5-213802-A-12.wav","bytes":441044,"blob":"2ee91b3de015cdb2282dd35cb60afda0e009c6dc","category":"crackling_fire","group_id":"esc10:freesound-213802"},{"path":"audio/5-213855-A-0.wav","bytes":441044,"blob":"a96507297dfb559a5a25c069c4a655b56184fa38","category":"dog","group_id":"esc10:freesound-213855"},{"path":"audio/5-215658-A-12.wav","bytes":441044,"blob":"148516f213d0a1f90e8417e7c2177170738bf401","category":"crackling_fire","group_id":"esc10:freesound-215658"},{"path":"audio/5-215658-B-12.wav","bytes":441044,"blob":"69a07f9aeb81a17b70599f5a10198687c85a9320","category":"crackling_fire","group_id":"esc10:freesound-215658"},{"path":"audio/5-216370-A-41.wav","bytes":441044,"blob":"e79f6b6b0ea67f520d101f575cf712dbc597a6d0","category":"chainsaw","group_id":"esc10:freesound-216370"},{"path":"audio/5-216370-B-41.wav","bytes":441044,"blob":"41770900f2e01770ae3063af43b3f8ca56fc4616","category":"chainsaw","group_id":"esc10:freesound-216370"},{"path":"audio/5-217158-A-0.wav","bytes":441044,"blob":"4995db05ea32d297ad2c4d40574c2ff1a1b0e8a2","category":"dog","group_id":"esc10:freesound-217158"},{"path":"audio/5-219342-A-38.wav","bytes":441044,"blob":"2b355b29628ea1a071d863397ca42eba29a5363c","category":"clock_tick","group_id":"esc10:freesound-219342"},{"path":"audio/5-219379-A-11.wav","bytes":441044,"blob":"7157565fcbd582c8e95019a454a37edc21f8671d","category":"sea_waves","group_id":"esc10:freesound-219379"},{"path":"audio/5-219379-B-11.wav","bytes":441044,"blob":"9c35ad6770650a1dd8103443290782af327bde89","category":"sea_waves","group_id":"esc10:freesound-219379"},{"path":"audio/5-219379-C-11.wav","bytes":441044,"blob":"1144457ffd0c941083348d2d9e218a19185798a1","category":"sea_waves","group_id":"esc10:freesound-219379"},{"path":"audio/5-220955-A-40.wav","bytes":441044,"blob":"88447190fb5b6798393e373e31500f56ae1d9455","category":"helicopter","group_id":"esc10:freesound-220955"},{"path":"audio/5-222524-A-41.wav","bytes":441044,"blob":"2b56cafdd04d339e4fc8b37324667d1b6ba6c868","category":"chainsaw","group_id":"esc10:freesound-222524"},{"path":"audio/5-231762-A-0.wav","bytes":441044,"blob":"95e1b0e93e1612381a4e1ff9e272ab2c555f0ed4","category":"dog","group_id":"esc10:freesound-231762"},{"path":"audio/5-233160-A-1.wav","bytes":441044,"blob":"abca45f604c092158ee453b09b6be8452c9d1480","category":"rooster","group_id":"esc10:freesound-233160"},{"path":"audio/5-234879-A-1.wav","bytes":441044,"blob":"e02fe76f39eed6a262592a358bf3c8efc3ca20db","category":"rooster","group_id":"esc10:freesound-234879"},{"path":"audio/5-234879-B-1.wav","bytes":441044,"blob":"30f7b4f60b26766aed25bc4dd688a078c23e4269","category":"rooster","group_id":"esc10:freesound-234879"},{"path":"audio/5-235671-A-38.wav","bytes":441044,"blob":"bb4a68dcfe6ad0ae9ff70ce2a0a9218a04c0a4f7","category":"clock_tick","group_id":"esc10:freesound-235671"},{"path":"audio/5-9032-A-0.wav","bytes":441044,"blob":"12bade49a211c2d73dc0913750ea66761bddc797","category":"dog","group_id":"esc10:freesound-9032"}]},"fsd50k":{"record":"4060432","license_sha256":"ebee1f320405cc09abfcc3c77e14fbc3adb9734d53f2ef5274d35eddcbb476f1","metadata_archives":[{"key":"FSD50K.doc.zip","bytes":6984,"md5":"3516162b82dc2945d3e7feba0904e800","sha256":"a5c314ad2bf93d01b5921aa4270c70c90e28ac9a283ba9f08167490e728deb85"},{"key":"FSD50K.ground_truth.zip","bytes":334701,"md5":"ca27382c195e37d2269c4c866dd73485","sha256":"db2396260a7b1fb06feb6ccac71685794fc89917d71c3a8d64a1de85c2cc0ebf"},{"key":"FSD50K.metadata.zip","bytes":6700838,"md5":"b9ea0c829a411c1d42adb9da539ed237","sha256":"9a738e032546f9a2c6e3d04566928d04a65fb79b422cc9d78bb781723537bd19"}],"audio_archives":{"dev":{"volumes":[{"key":"FSD50K.dev_audio.z01","bytes":3221225472,"md5":"faa7cf4cc076fc34a44a479a5ed862a3"},{"key":"FSD50K.dev_audio.z02","bytes":3221225472,"md5":"8f9b66153e68571164fb1315d00bc7bc"},{"key":"FSD50K.dev_audio.z03","bytes":3221225472,"md5":"1196ef47d267a993d30fa98af54b7159"},{"key":"FSD50K.dev_audio.z04","bytes":3221225472,"md5":"d088ac4e11ba53daf9f7574c11cccac9"},{"key":"FSD50K.dev_audio.z05","bytes":3221225472,"md5":"81356521aa159accd3c35de22da28c7f"},{"key":"FSD50K.dev_audio.zip","bytes":2306663327,"md5":"c480d119b8f7a7e32fdb58f3ea4d6c5a"}],"central_directory":{"disk":5,"offset":2302700512,"bytes":3962717,"n_entries":40967,"sha256":"a52c8e77acf8c01e5108a3872461a65f2a583b2ddc3e0bb6554e51dc7ae12c5f"}},"eval":{"volumes":[{"key":"FSD50K.eval_audio.z01","bytes":3221225472,"md5":"3090670eaeecc013ca1ff84fe4442aeb"},{"key":"FSD50K.eval_audio.zip","bytes":3037675767,"md5":"6fa47636c3a3ad5c7dfeba99f2637982"}],"central_directory":{"disk":1,"offset":3036675385,"bytes":1000284,"n_entries":10232,"sha256":"6cb5e5d62d9a0930e0624a61977b441d7e88b80caa608b9a3cbf2ccab4e9db75"}}},"clips":[{"fname":"21560","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/21560.wav","disk":3,"offset":2974062365,"compressed_bytes":1876631,"size":2072050,"method":8,"crc32":2928156733,"sha256":"24bcce59028fdd0cc6982c2977f005e8cda2fdd79332deaa02875be2eb298e5c"},{"fname":"26128","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"WIM","group_id":"fsd50k:u-07d9d387aebf","member":"FSD50K.dev_audio/26128.wav","disk":3,"offset":85587840,"compressed_bytes":1550070,"size":1712334,"method":8,"crc32":975761476,"sha256":"36ee3ad76e9f906936e47069497ba58e1a73a87779e4adee681fdc0bb01836f2"},{"fname":"39983","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/39983.wav","disk":4,"offset":210298315,"compressed_bytes":2031996,"size":2646078,"method":8,"crc32":2530665553,"sha256":"4b59221f0a7e1edbfc5157fbbfdb4411ab3990d65aeea417eb536274882ed7ed"},{"fname":"73289","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"NoiseCollector","group_id":"fsd50k:u-0697aba1f475","member":"FSD50K.dev_audio/73289.wav","disk":4,"offset":1294062659,"compressed_bytes":1962053,"size":2511438,"method":8,"crc32":1310670317,"sha256":"e05860644a709f8272c6de36384c684ebb9d120e32ee50ca952742f47f765846"},{"fname":"73292","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"NoiseCollector","group_id":"fsd50k:u-0697aba1f475","member":"FSD50K.dev_audio/73292.wav","disk":2,"offset":2705500879,"compressed_bytes":1788555,"size":2115150,"method":8,"crc32":2946402595,"sha256":"0adece3df9c7c346919f0a6e412ca2b4884fba0595aae3ab3ce4da5dc604e04a"},{"fname":"73297","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"NoiseCollector","group_id":"fsd50k:u-0697aba1f475","member":"FSD50K.dev_audio/73297.wav","disk":0,"offset":2738083464,"compressed_bytes":1361540,"size":2147406,"method":8,"crc32":1241033753,"sha256":"43d7d27a9f126462d1a9855ed048b8cf242a124fb49599a247ad0e6f6914b526"},{"fname":"103325","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"miastodzwiekow","group_id":"fsd50k:u-999af4936c85","member":"FSD50K.dev_audio/103325.wav","disk":2,"offset":2136896570,"compressed_bytes":2224582,"size":2474574,"method":8,"crc32":4012189523,"sha256":"dfed2ec55a2797584a82abea0965fb05a0663503c72ad9608466e48272ccda45"},{"fname":"140582","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"LG","group_id":"fsd50k:u-b870b7809747","member":"FSD50K.dev_audio/140582.wav","disk":5,"offset":1583711353,"compressed_bytes":752788,"size":800796,"method":8,"crc32":1719757534,"sha256":"55c4bc86bfa97aef7c9a288a4be63f3bb59b88c77be15ee4a38c07ee572cefd2"},{"fname":"140592","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"LG","group_id":"fsd50k:u-b870b7809747","member":"FSD50K.dev_audio/140592.wav","disk":4,"offset":533081607,"compressed_bytes":515782,"size":568026,"method":8,"crc32":3197108035,"sha256":"4fdf7c74d89ce75278a76198c17cc0b3af9a571453f91875b6168d9045ce92e5"},{"fname":"143556","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Fight2FlyPhoto","group_id":"fsd50k:u-5f673f5a2b54","member":"FSD50K.dev_audio/143556.wav","disk":4,"offset":191669005,"compressed_bytes":1290818,"size":1509806,"method":8,"crc32":1546249240,"sha256":"e004bcf44b039bc51e2476308cb5ef78729bbd4333b4591d3c321834662301ff"},{"fname":"143557","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Fight2FlyPhoto","group_id":"fsd50k:u-5f673f5a2b54","member":"FSD50K.dev_audio/143557.wav","disk":3,"offset":2194399388,"compressed_bytes":1774109,"size":2101338,"method":8,"crc32":2344624093,"sha256":"c271bbd6a20a47eaaaf967f2a2bd94c4d7876f979368b33b706c9ca06c8d3103"},{"fname":"143563","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Fight2FlyPhoto","group_id":"fsd50k:u-5f673f5a2b54","member":"FSD50K.dev_audio/143563.wav","disk":4,"offset":1243522505,"compressed_bytes":2227468,"size":2642840,"method":8,"crc32":4238320251,"sha256":"60e4a2d919a1ce110c4bb6e12ffffb191ac355314fe4615bc80dd1e329171553"},{"fname":"162367","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"qubodup","group_id":"fsd50k:u-2afd5d535ce9","member":"FSD50K.dev_audio/162367.wav","disk":0,"offset":2205531670,"compressed_bytes":1559930,"size":1712336,"method":8,"crc32":1450598409,"sha256":"51d308f8c6acb0e17ea632b25b731277047221cce9bf24bf6ee1b0c55c67b7e1"},{"fname":"162439","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"qubodup","group_id":"fsd50k:u-2afd5d535ce9","member":"FSD50K.dev_audio/162439.wav","disk":3,"offset":2219120981,"compressed_bytes":2282122,"size":2326286,"method":8,"crc32":508775554,"sha256":"f62f9946439f5d6921d07ac243647d5b8c522aae8e0c744700e0d0905000008e"},{"fname":"184723","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"qubodup","group_id":"fsd50k:u-2afd5d535ce9","member":"FSD50K.dev_audio/184723.wav","disk":2,"offset":808804485,"compressed_bytes":852342,"size":882078,"method":8,"crc32":2349564507,"sha256":"f0a6161a44c424a1c9c4b9f1a49ed03d2fb3473f18732f650e1d529e44bc9f0e"},{"fname":"195412","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"beerbelly38","group_id":"fsd50k:u-742e9a13fbda","member":"FSD50K.dev_audio/195412.wav","disk":5,"offset":1478947021,"compressed_bytes":2136775,"size":2370894,"method":8,"crc32":968150699,"sha256":"f4040727523292caa7d307fc113300a80cbee53a499bd13a5e84c3d51076d079"},{"fname":"200436","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"ID_23","group_id":"fsd50k:u-d94533516063","member":"FSD50K.dev_audio/200436.wav","disk":3,"offset":137704921,"compressed_bytes":864814,"size":1448270,"method":8,"crc32":3871336509,"sha256":"76680de5419f079782501a712664a98961de6aa8a441d194bea19e768b4cce3b"},{"fname":"207463","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"AurelioSons","group_id":"fsd50k:u-5d871bb2cd0e","member":"FSD50K.dev_audio/207463.wav","disk":0,"offset":2233065961,"compressed_bytes":858019,"size":902088,"method":8,"crc32":950391549,"sha256":"bbbbb58c0546922cb5c60eec635ac55334e4cf22f4ce5b49b00ba0d93d529b68"},{"fname":"207470","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"AurelioSons","group_id":"fsd50k:u-5d871bb2cd0e","member":"FSD50K.dev_audio/207470.wav","disk":1,"offset":1397482117,"compressed_bytes":642914,"size":697500,"method":8,"crc32":593732824,"sha256":"0143335aa7ab64445dc0c0636cd3f1fe4c719aa0d69cd6bc0b42f8ab3e3e3c3c"},{"fname":"208841","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Insestanyricarda","group_id":"fsd50k:u-568563aa5b58","member":"FSD50K.dev_audio/208841.wav","disk":4,"offset":414014250,"compressed_bytes":378207,"size":406736,"method":8,"crc32":906968462,"sha256":"4fb300859a9491c22da1041eff864ed65cc911250308b64e325fabb0d2fd6fcb"},{"fname":"208854","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Insestanyricarda","group_id":"fsd50k:u-568563aa5b58","member":"FSD50K.dev_audio/208854.wav","disk":3,"offset":556222029,"compressed_bytes":429707,"size":454362,"method":8,"crc32":1957253012,"sha256":"a40529c37c075406f6558aba6a74596ae3a664d64d4dac4e5b445f0852d0cebe"},{"fname":"209042","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Insestanyricarda","group_id":"fsd50k:u-568563aa5b58","member":"FSD50K.dev_audio/209042.wav","disk":2,"offset":321560476,"compressed_bytes":404853,"size":477006,"method":8,"crc32":380191396,"sha256":"d643846309a52475a3ba7bbcd5ad6b19e01b71335a76c511f4ac8c9c1e70864c"},{"fname":"219466","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"bigpickle51","group_id":"fsd50k:u-2091b0470b89","member":"FSD50K.dev_audio/219466.wav","disk":4,"offset":2891861993,"compressed_bytes":1511016,"size":2423226,"method":8,"crc32":3149854240,"sha256":"99a016e7fa1820f173e38c07f9442e70684d1aad27e2736bbbe8301476e24877"},{"fname":"219468","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"bigpickle51","group_id":"fsd50k:u-2091b0470b89","member":"FSD50K.dev_audio/219468.wav","disk":2,"offset":1705735631,"compressed_bytes":1059726,"size":1764078,"method":8,"crc32":1920028785,"sha256":"b2729253a706d304637ed1ee83167f122c32ede4bcb0ade86c2c5219254c3e5a"},{"fname":"219469","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"bigpickle51","group_id":"fsd50k:u-2091b0470b89","member":"FSD50K.dev_audio/219469.wav","disk":1,"offset":1462141108,"compressed_bytes":1061483,"size":1551640,"method":8,"crc32":2974647727,"sha256":"fa538a3640e2c5734cfc0ddd136430bd397b87fcd37a281b28c8175de2ff0a49"},{"fname":"237974","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Zovex","group_id":"fsd50k:u-ba29f3ce14f1","member":"FSD50K.dev_audio/237974.wav","disk":3,"offset":2147745897,"compressed_bytes":847257,"size":974396,"method":8,"crc32":4097133133,"sha256":"8267007a5798fd1f877217c5a4c84083bd2643dd18ce0b1dab361a64af0d9ab4"},{"fname":"251963","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"cliftonmcarlson","group_id":"fsd50k:u-661d286debf5","member":"FSD50K.dev_audio/251963.wav","disk":5,"offset":337523358,"compressed_bytes":1416980,"size":1850452,"method":8,"crc32":1157987830,"sha256":"659125b6e0820635561efbdbeb407efabae80a2cf85b4055b624c9146dd1ef92"},{"fname":"251968","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"cliftonmcarlson","group_id":"fsd50k:u-661d286debf5","member":"FSD50K.dev_audio/251968.wav","disk":4,"offset":468559919,"compressed_bytes":1046694,"size":1455166,"method":8,"crc32":507955222,"sha256":"1ea6e8bbe7616afe5093b749615f0732d6304ff79cc3e0996f2f45b45684f022"},{"fname":"251975","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"cliftonmcarlson","group_id":"fsd50k:u-661d286debf5","member":"FSD50K.dev_audio/251975.wav","disk":3,"offset":653015152,"compressed_bytes":1464250,"size":1890918,"method":8,"crc32":3205882411,"sha256":"e7d451156faf341e688842cf4bdf9ab94a505a8c2f94a14836ec18f93f67979b"},{"fname":"263522","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Alcappuccino","group_id":"fsd50k:u-f656cb959936","member":"FSD50K.dev_audio/263522.wav","disk":2,"offset":1649521963,"compressed_bytes":570020,"size":592782,"method":8,"crc32":2176670138,"sha256":"89e74d50192f8d1b25f9f493171e57713f68a7c5386c7ba1efa6f97e37be32d8"},{"fname":"276291","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Sandermotions","group_id":"fsd50k:u-ce952f98aa65","member":"FSD50K.dev_audio/276291.wav","disk":5,"offset":1584847770,"compressed_bytes":1459702,"size":1815412,"method":8,"crc32":2328577474,"sha256":"2db94f9d826b6d75250600408e76e89eeaef5d574ceee8bc1e68b6773a8da772"},{"fname":"320786","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"freesoundjon01","group_id":"fsd50k:u-56fe33724eb2","member":"FSD50K.dev_audio/320786.wav","disk":3,"offset":670024313,"compressed_bytes":1583315,"size":2368896,"method":8,"crc32":1659998704,"sha256":"2eb541db428edf8f1204e2c81243db918746f430b5ecfd39c7f8f7219a1c0093"},{"fname":"324311","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"kgunessee","group_id":"fsd50k:u-d16d2cb15da8","member":"FSD50K.dev_audio/324311.wav","disk":4,"offset":660080131,"compressed_bytes":2111829,"size":2449204,"method":8,"crc32":3810797892,"sha256":"da1b45ac13fdf790dc0efe8e71b9e4ee46ff8568cbc056534fee4217dee5887e"},{"fname":"324318","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"kgunessee","group_id":"fsd50k:u-d16d2cb15da8","member":"FSD50K.dev_audio/324318.wav","disk":2,"offset":30059006,"compressed_bytes":1290481,"size":1696110,"method":8,"crc32":2066640937,"sha256":"d6184f95e390306580bca02060d58270e39a69734220c7504b2c363f52d3aa50"},{"fname":"324319","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"kgunessee","group_id":"fsd50k:u-d16d2cb15da8","member":"FSD50K.dev_audio/324319.wav","disk":2,"offset":1904088409,"compressed_bytes":1164319,"size":1630878,"method":8,"crc32":2707330963,"sha256":"b633fa2778fdbc98ed110e20f01d22030bb598b94d94388a6d17de6f4366b318"},{"fname":"330816","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Sandermotions","group_id":"fsd50k:u-ce952f98aa65","member":"FSD50K.dev_audio/330816.wav","disk":1,"offset":2531885238,"compressed_bytes":1444914,"size":1582680,"method":8,"crc32":1526014186,"sha256":"db7093619276946a9bb364f515d36bc6065bf8ca636217efbfb729f48df99c07"},{"fname":"343685","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"ElementRS2","group_id":"fsd50k:u-922eb7ef168d","member":"FSD50K.dev_audio/343685.wav","disk":4,"offset":105415030,"compressed_bytes":973741,"size":1185486,"method":8,"crc32":752587539,"sha256":"0e3fb165372fe90ae71e79f1778cd6832e7a3e104dad56bb45ca321aef136136"},{"fname":"351183","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"KenzieVaness","group_id":"fsd50k:u-d4b3522bb1bb","member":"FSD50K.dev_audio/351183.wav","disk":4,"offset":1420046202,"compressed_bytes":995796,"size":1255072,"method":8,"crc32":1194546019,"sha256":"77e181e5b0f0011e537c811ee8a3718e8f90a31e230cee32e4734ac7108037e5"},{"fname":"353265","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"TheSpiderWriter","group_id":"fsd50k:u-aa5afd52529d","member":"FSD50K.dev_audio/353265.wav","disk":3,"offset":1762184415,"compressed_bytes":2138106,"size":2402382,"method":8,"crc32":2753011386,"sha256":"17dec161eff6a88c0b6a2ab4ddb87a20f8f4ad25ceb68002709a628a3eb66888"},{"fname":"369482","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Alcappuccino","group_id":"fsd50k:u-f656cb959936","member":"FSD50K.dev_audio/369482.wav","disk":0,"offset":1116829103,"compressed_bytes":1974818,"size":2402646,"method":8,"crc32":3870975678,"sha256":"a7c2d7c199d0571171a028bbca95fe1a262899855d95d5c14cf9c5cd103ae73b"},{"fname":"386140","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"ldezem","group_id":"fsd50k:u-d246c32be78f","member":"FSD50K.dev_audio/386140.wav","disk":4,"offset":238424500,"compressed_bytes":211544,"size":264678,"method":8,"crc32":919870641,"sha256":"ec025cd1af1edc5d1e006d949c5434dc40908e31e4722f4ae809200f816be69e"},{"fname":"403316","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"straget","group_id":"fsd50k:u-86fe64f5e1ad","member":"FSD50K.dev_audio/403316.wav","disk":2,"offset":1931415868,"compressed_bytes":1430654,"size":2018664,"method":8,"crc32":4287962026,"sha256":"4b6116e5eb25b6f7ed1eadfbd283bc6218a6d2452a4f96fcd30533a3c01cbb90"},{"fname":"420442","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"CussinSailor","group_id":"fsd50k:u-521eebe45290","member":"FSD50K.dev_audio/420442.wav","disk":2,"offset":3138182988,"compressed_bytes":496531,"size":591618,"method":8,"crc32":600344566,"sha256":"ad359b00eaaab46688824006976ceb5ced7dfdde3d88b0cae9892fadcf8c5831"},{"fname":"425268","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"77Pacer","group_id":"fsd50k:u-13a52a7621c3","member":"FSD50K.dev_audio/425268.wav","disk":0,"offset":353881826,"compressed_bytes":674567,"size":746630,"method":8,"crc32":3853355060,"sha256":"c38cd350c69ede4935c1cbe224b922fb1550aff5f60467bb40707650cd06a849"},{"fname":"431473","split":"dev","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"johnnythesalesman","group_id":"fsd50k:u-6cbeba708203","member":"FSD50K.dev_audio/431473.wav","disk":4,"offset":623495189,"compressed_bytes":748458,"size":778924,"method":8,"crc32":1553655328,"sha256":"bc9d4ab2aaceebb5e10ca73d161d55f2eb5a0fbd723740db21878b8c0a372be8"},{"fname":"39865","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"flarn2006","group_id":"fsd50k:u-936f25721919","member":"FSD50K.eval_audio/39865.wav","disk":1,"offset":76304223,"compressed_bytes":231883,"size":260756,"method":8,"crc32":1856635073,"sha256":"ec03f0a9192688bb1000cda2d91356c15fa462a83681aa56ff38ebd18cbc1883"},{"fname":"127724","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"nicStage","group_id":"fsd50k:u-5321096e69f0","member":"FSD50K.eval_audio/127724.wav","disk":1,"offset":1996883361,"compressed_bytes":680719,"size":915676,"method":8,"crc32":792099947,"sha256":"7e7671523fd80c40ab172c599d7c6f46504e8f4a6515d388449efd97f9cb35ca"},{"fname":"127737","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"nicStage","group_id":"fsd50k:u-5321096e69f0","member":"FSD50K.eval_audio/127737.wav","disk":0,"offset":1103798779,"compressed_bytes":742496,"size":967658,"method":8,"crc32":1810684036,"sha256":"4b7e72089982824950c2a62e8967d96755d11d30a614b2118f385d3c8fa2e28f"},{"fname":"138961","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Huggy13ear","group_id":"fsd50k:u-8af2068e509f","member":"FSD50K.eval_audio/138961.wav","disk":0,"offset":2101383723,"compressed_bytes":600758,"size":1041486,"method":8,"crc32":419461335,"sha256":"07fdb59225ecc856132960a1e0d26d44701cfe16fe1fb73a2c2009bb27b02e15"},{"fname":"172667","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"UnderlinedDesigns","group_id":"fsd50k:u-8511b65aa95c","member":"FSD50K.eval_audio/172667.wav","disk":1,"offset":987115525,"compressed_bytes":77455,"size":80078,"method":8,"crc32":3703337685,"sha256":"9fb4903cdc8d027fa1f583d3642d76515105105d2065c38f5c0ab00b3573a641"},{"fname":"219539","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Yap_Audio_Production","group_id":"fsd50k:u-5c99aed1f96d","member":"FSD50K.eval_audio/219539.wav","disk":1,"offset":1586896924,"compressed_bytes":706095,"size":930894,"method":8,"crc32":2716967260,"sha256":"5e480acea7c1d6279378571f8ccf324819fb4f293cce148942308e3502707835"},{"fname":"231963","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"INS_Ribera_Baixa","group_id":"fsd50k:u-ab6afc9c87c6","member":"FSD50K.eval_audio/231963.wav","disk":0,"offset":507002719,"compressed_bytes":547694,"size":689936,"method":8,"crc32":3510495836,"sha256":"543cd135d09fce09b1d3cc7791ebd72e207deb061d1b870722e0bd9b464ee137"},{"fname":"232818","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"lezaarth","group_id":"fsd50k:u-158b0c7488e5","member":"FSD50K.eval_audio/232818.wav","disk":0,"offset":3199833145,"compressed_bytes":617363,"size":736356,"method":8,"crc32":1140433688,"sha256":"fb3a3c1d85cd8844f51799dc7263fcecdf191056e429ff5201385a791bd88e17"},{"fname":"235517","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"ceberation","group_id":"fsd50k:u-82c90b7d3c76","member":"FSD50K.eval_audio/235517.wav","disk":0,"offset":2587818596,"compressed_bytes":1215344,"size":1940478,"method":8,"crc32":1062560526,"sha256":"ee87d503de62a22a39d1fa2257758b020ad70d3912402ca54a5a21a143a27d2c"},{"fname":"249568","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"UncleSigmund","group_id":"fsd50k:u-5151862954fc","member":"FSD50K.eval_audio/249568.wav","disk":1,"offset":1038858451,"compressed_bytes":1420928,"size":1717784,"method":8,"crc32":59279653,"sha256":"1a6c20faecae1127029012f7fa6c09f3a265bc47713644ade91890d909339013"},{"fname":"263458","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Cell31_Sound_Productions","group_id":"fsd50k:u-8d8be27f5bb5","member":"FSD50K.eval_audio/263458.wav","disk":1,"offset":250376591,"compressed_bytes":1761930,"size":2196348,"method":8,"crc32":3503496736,"sha256":"4da6f46530a88d3acb51349d067f4ea8fea38094d16792a7580ca158a56957b2"},{"fname":"322179","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"ikbenraar","group_id":"fsd50k:u-75ab5eb3e592","member":"FSD50K.eval_audio/322179.wav","disk":0,"offset":1925580862,"compressed_bytes":732710,"size":902378,"method":8,"crc32":3489321458,"sha256":"5b9a90b26a48b7d227f05eccd32e6e617633c77caa02480c0260189c1340ea3f"},{"fname":"333821","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"wmquincy101","group_id":"fsd50k:u-f96a7ae34089","member":"FSD50K.eval_audio/333821.wav","disk":1,"offset":640949071,"compressed_bytes":324661,"size":397954,"method":8,"crc32":698158778,"sha256":"bf79f4f65aa5b15603a5f460f7d1dd64e6243028e20495ad20ea202400161e97"},{"fname":"340862","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"passAirmangrace","group_id":"fsd50k:u-2c8f8f11d6ad","member":"FSD50K.eval_audio/340862.wav","disk":1,"offset":2937285450,"compressed_bytes":765356,"size":1048654,"method":8,"crc32":2066211684,"sha256":"29324ee8d7e43752b11b9f582484d074492ab09eb3c153a9dbdd188faa49dd49"},{"fname":"347709","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"desdur","group_id":"fsd50k:u-e9deec462299","member":"FSD50K.eval_audio/347709.wav","disk":0,"offset":276702160,"compressed_bytes":739438,"size":1078350,"method":8,"crc32":487742219,"sha256":"751f249c140140c2f4c960490937f03c872af68060273ce7dff4962765582a8c"},{"fname":"352813","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"sim2710","group_id":"fsd50k:u-2542461a1543","member":"FSD50K.eval_audio/352813.wav","disk":1,"offset":2341796070,"compressed_bytes":2117303,"size":2314184,"method":8,"crc32":2590032150,"sha256":"ce66cb329c257af02dc4459dc41cba9266296deb5dd66b921b3f193797d3972f"},{"fname":"391515","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"relwin","group_id":"fsd50k:u-c826467b98f3","member":"FSD50K.eval_audio/391515.wav","disk":0,"offset":439010952,"compressed_bytes":1859970,"size":2042812,"method":8,"crc32":3955742258,"sha256":"65f6be897ec37a70687663c48502f0422a913ee90328062ce72d11d7b0d282f6"},{"fname":"397164","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"FullMetalJedi","group_id":"fsd50k:u-c0a11a887101","member":"FSD50K.eval_audio/397164.wav","disk":0,"offset":3006883893,"compressed_bytes":2153719,"size":2469944,"method":8,"crc32":1225192254,"sha256":"62bb01452348c51407743d7984a89d8e8b00fcea3197d37e9e53159957c55169"},{"fname":"397165","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"FullMetalJedi","group_id":"fsd50k:u-c0a11a887101","member":"FSD50K.eval_audio/397165.wav","disk":1,"offset":2516910474,"compressed_bytes":2048935,"size":2488624,"method":8,"crc32":3496347766,"sha256":"d2ce1807792c358e59164b66d76ba868c1e1271ced0a82175691c9d5bb02f8de"},{"fname":"401255","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"DRFX","group_id":"fsd50k:u-6ca039334672","member":"FSD50K.eval_audio/401255.wav","disk":1,"offset":1950464841,"compressed_bytes":381459,"size":513456,"method":8,"crc32":2997875067,"sha256":"710485b1b93267145d1a37e78cb42c77a36997f1a47b0e7876a139885d256b34"},{"fname":"420560","split":"eval","target_label":"Aircraft","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Nice_Audio","group_id":"fsd50k:u-c28551eb8018","member":"FSD50K.eval_audio/420560.wav","disk":1,"offset":2623277749,"compressed_bytes":2513460,"size":2624798,"method":8,"crc32":439769641,"sha256":"dd27bb54a0c474e5279a47237fefb559c5a52dc405bc1606d5a01f958bbf8104"},{"fname":"7899","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"schluppipuppie","group_id":"fsd50k:u-925865885dba","member":"FSD50K.dev_audio/7899.wav","disk":1,"offset":3207488228,"compressed_bytes":204187,"size":237114,"method":8,"crc32":1814136557,"sha256":"34f589da8d7ac50f35414423c2e81d47f40ea816b340c740848730d2b9e1c847"},{"fname":"7910","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"schluppipuppie","group_id":"fsd50k:u-925865885dba","member":"FSD50K.dev_audio/7910.wav","disk":0,"offset":2762256065,"compressed_bytes":196762,"size":233394,"method":8,"crc32":541682942,"sha256":"9dac1b1dee9164f792048c6e0e6ddeb0bea74c0fe448ba46ef697ee8ca028195"},{"fname":"14462","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"reinsamba","group_id":"fsd50k:u-1d952795538e","member":"FSD50K.dev_audio/14462.wav","disk":1,"offset":1085501700,"compressed_bytes":29055,"size":35094,"method":8,"crc32":1507245079,"sha256":"af8ab5269661f4cead8ff19b3007d75463afeef93903fc5d40f3a0d616b5e9af"},{"fname":"20021","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"reinsamba","group_id":"fsd50k:u-1d952795538e","member":"FSD50K.dev_audio/20021.wav","disk":0,"offset":1589113635,"compressed_bytes":1819850,"size":2145980,"method":8,"crc32":1674678140,"sha256":"b8fd9cfdbd446211b77d17bd4c5c38859aaedc9f9c9a8524b5b75891800e7adc"},{"fname":"32408","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"digifishmusic","group_id":"fsd50k:u-23878ad72831","member":"FSD50K.dev_audio/32408.wav","disk":2,"offset":2529935219,"compressed_bytes":1449703,"size":2461774,"method":8,"crc32":1608728098,"sha256":"f8c2fd8cc8aa487b548e54f119104595eb1002de29f8ef6a1ff553f856f4642a"},{"fname":"32549","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/32549.wav","disk":1,"offset":2700961137,"compressed_bytes":565217,"size":692782,"method":8,"crc32":3642990878,"sha256":"cd1feaefccf8b89ae7e32e28cd5d718b1d7880e6b6ee8ffab319e9dd667fe78c"},{"fname":"33372","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Coppersmith Barbet","group_id":"fsd50k:u-7f0ee4312a02","member":"FSD50K.dev_audio/33372.wav","disk":0,"offset":614755236,"compressed_bytes":876051,"size":1263594,"method":8,"crc32":3237837477,"sha256":"41656995e1dc9035c96ad9de884e4e6a88929b60d14e1a30267d6e8363d82cf7"},{"fname":"33550","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"reinsamba","group_id":"fsd50k:u-1d952795538e","member":"FSD50K.dev_audio/33550.wav","disk":2,"offset":297294585,"compressed_bytes":62267,"size":67962,"method":8,"crc32":2639263880,"sha256":"848fad0b4aba38f4dc4502f853fcf0601408e235744e7080ee89c130aa1ccc6d"},{"fname":"34696","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"roubignolle","group_id":"fsd50k:u-bba66f09f839","member":"FSD50K.dev_audio/34696.wav","disk":2,"offset":994569898,"compressed_bytes":126934,"size":171304,"method":8,"crc32":237766470,"sha256":"aaa2cf9da95a705a5f8e7108b1a8678c7ad15d4db52964b9cad7448429e2a99d"},{"fname":"47610","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"ReadeOnly","group_id":"fsd50k:u-3665da313492","member":"FSD50K.dev_audio/47610.wav","disk":5,"offset":1092177853,"compressed_bytes":41548,"size":69144,"method":8,"crc32":1870634405,"sha256":"4c686fc95a50ba82d790dfd084913763c2714b77e5e6c2e514adf479ba80938d"},{"fname":"49983","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Sparrer","group_id":"fsd50k:u-95b36c38c6aa","member":"FSD50K.dev_audio/49983.wav","disk":1,"offset":2791384132,"compressed_bytes":1885625,"size":2646078,"method":8,"crc32":3873647532,"sha256":"5277214f64d0939477e3bd823a70789631c278cb654cfa185df6eee930d0e6be"},{"fname":"51094","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Sparrer","group_id":"fsd50k:u-95b36c38c6aa","member":"FSD50K.dev_audio/51094.wav","disk":1,"offset":1635398135,"compressed_bytes":1206204,"size":1435598,"method":8,"crc32":839475849,"sha256":"97157b8fa3889b9b15e4e1854f6ad2f58236a8bd229ce7902c8bc34971323c3a"},{"fname":"53587","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"digifishmusic","group_id":"fsd50k:u-23878ad72831","member":"FSD50K.dev_audio/53587.wav","disk":4,"offset":1614493552,"compressed_bytes":229258,"size":470634,"method":8,"crc32":3971371659,"sha256":"661c391a6982d4b21f88e9035d69e261053e7fd4af8148e94079b74aded32002"},{"fname":"57612","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"digifishmusic","group_id":"fsd50k:u-23878ad72831","member":"FSD50K.dev_audio/57612.wav","disk":1,"offset":2087850002,"compressed_bytes":395136,"size":811820,"method":8,"crc32":248010235,"sha256":"aad5c22d63d5ad27b816e4b52c9ed3dc409c42a6ccbf2f89c6939416a8062060"},{"fname":"57656","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"jih","group_id":"fsd50k:u-23389bf89c7f","member":"FSD50K.dev_audio/57656.wav","disk":2,"offset":2615221139,"compressed_bytes":215948,"size":254198,"method":8,"crc32":2863991141,"sha256":"9245394f6af308a6fdba3399bd39a7f049912f8bf32619f4b04462b7fb9d7d40"},{"fname":"59346","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"suonho","group_id":"fsd50k:u-abc319c5b53c","member":"FSD50K.dev_audio/59346.wav","disk":1,"offset":2603524120,"compressed_bytes":345526,"size":488404,"method":8,"crc32":2023242208,"sha256":"f04036db33079cd3b9c7dc3ec56e3fc0332c511635c623999de98d8f36f9f5a6"},{"fname":"62304","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"thesecretmachine","group_id":"fsd50k:u-b81fad357a7d","member":"FSD50K.dev_audio/62304.wav","disk":3,"offset":16874431,"compressed_bytes":348031,"size":429748,"method":8,"crc32":201587958,"sha256":"406ea26caebfe09ec957f029d66ecfec38e776b69651333efcc486f22ff01b4d"},{"fname":"69082","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"NoiseCollector","group_id":"fsd50k:u-0697aba1f475","member":"FSD50K.dev_audio/69082.wav","disk":0,"offset":2510221785,"compressed_bytes":370126,"size":450592,"method":8,"crc32":3678428652,"sha256":"e736f812a3d236b70478b2c98c5a983d5801733de4631dfad41cb64cdea117c3"},{"fname":"69449","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"IFartInUrGeneralDirection","group_id":"fsd50k:u-83de0247da34","member":"FSD50K.dev_audio/69449.wav","disk":4,"offset":3071172364,"compressed_bytes":1868076,"size":2465358,"method":8,"crc32":2130267032,"sha256":"2d0d63f50b6a2afa1f3c590eb8c2309c3f96436f02304f34c5f7e9c07dbc2241"},{"fname":"72536","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Sparrer","group_id":"fsd50k:u-95b36c38c6aa","member":"FSD50K.dev_audio/72536.wav","disk":0,"offset":1276590170,"compressed_bytes":1399138,"size":2646078,"method":8,"crc32":1715209731,"sha256":"96f720ff366cb27dcb6ad86c72c9a8b1e11cebdf830a9d020b6e5a785f90bda6"},{"fname":"72892","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Benboncan","group_id":"fsd50k:u-05af64f71b1e","member":"FSD50K.dev_audio/72892.wav","disk":0,"offset":1865281002,"compressed_bytes":510395,"size":697068,"method":8,"crc32":4251426750,"sha256":"be9f07388f32460ef8afb933365be37f400a86e343867028a6407ce4cfc2e507"},{"fname":"73617","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"jus","group_id":"fsd50k:u-06db4ae193d8","member":"FSD50K.dev_audio/73617.wav","disk":1,"offset":918388981,"compressed_bytes":1296294,"size":1658780,"method":8,"crc32":2917494541,"sha256":"cfc28b3dcacc1aea81198488ee202153637507d3cafe4d417bfe9255e5dabee1"},{"fname":"75663","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"NoiseCollector","group_id":"fsd50k:u-0697aba1f475","member":"FSD50K.dev_audio/75663.wav","disk":2,"offset":2665254426,"compressed_bytes":349956,"size":557646,"method":8,"crc32":3010245979,"sha256":"42840a0257d83c4a6c2adf507efe1892620c441000097335ba01ef6d3fe85a8a"},{"fname":"81155","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"edbles","group_id":"fsd50k:u-af66ce56fb05","member":"FSD50K.dev_audio/81155.wav","disk":5,"offset":1790333389,"compressed_bytes":2034272,"size":2340942,"method":8,"crc32":2738997495,"sha256":"3021418cc7ed7add186d98692cf5bd92893673408d4eb5d97ab453a9480cb70b"},{"fname":"81766","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Bodhisatva","group_id":"fsd50k:u-8e69e44603c8","member":"FSD50K.dev_audio/81766.wav","disk":2,"offset":245740996,"compressed_bytes":602326,"size":864604,"method":8,"crc32":1318056920,"sha256":"a5e1b9cbbad1b182c7816087ead1f4ec68444e0c79d4eee2e9116d2f7e4c4cf7"},{"fname":"92640","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"CGEffex","group_id":"fsd50k:u-4f7ab92b7591","member":"FSD50K.dev_audio/92640.wav","disk":4,"offset":197607824,"compressed_bytes":405430,"size":725838,"method":8,"crc32":1783614044,"sha256":"368f24dfbb1ad708e1c0c1c334c95f10504fc77206236b89369427f019686bbd"},{"fname":"97389","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"soundbytez","group_id":"fsd50k:u-02e41892b44a","member":"FSD50K.dev_audio/97389.wav","disk":0,"offset":1906510345,"compressed_bytes":959322,"size":1159908,"method":8,"crc32":1034368701,"sha256":"0b42f41f58551c166c3b5115a09d435db3ecc5d125864fcb170f4fa58b7d4b70"},{"fname":"100893","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"paubg_pou","group_id":"fsd50k:u-8ac0f3f255b9","member":"FSD50K.dev_audio/100893.wav","disk":3,"offset":714061505,"compressed_bytes":449514,"size":477268,"method":8,"crc32":296065099,"sha256":"c19bb66623e6d4b79362840c230d9955e603cc266c2739c9893f3343d9fd523d"},{"fname":"103110","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"jeo","group_id":"fsd50k:u-146e6ea2abd8","member":"FSD50K.dev_audio/103110.wav","disk":5,"offset":2244463393,"compressed_bytes":570373,"size":652012,"method":8,"crc32":867117076,"sha256":"3a46eb8ad7dc8bfb0353f78ffb475a720adf37961602c2ffe85795066e21b4e4"},{"fname":"111075","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"soundbytez","group_id":"fsd50k:u-02e41892b44a","member":"FSD50K.dev_audio/111075.wav","disk":4,"offset":2979314029,"compressed_bytes":77060,"size":110328,"method":8,"crc32":4052282026,"sha256":"3db0af5cf2b98bd8623177d6ebad311765a4b3a3c7e9d5f40eab5f5f349cb863"},{"fname":"111094","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"soundbytez","group_id":"fsd50k:u-02e41892b44a","member":"FSD50K.dev_audio/111094.wav","disk":0,"offset":1420579236,"compressed_bytes":948739,"size":1252518,"method":8,"crc32":40756540,"sha256":"01ea61d6746db5e5cd09b82beed2e89cf1c9f1d97b455f5956158282eb1b56a5"},{"fname":"146328","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Kamyarbb","group_id":"fsd50k:u-4d229dedd694","member":"FSD50K.dev_audio/146328.wav","disk":3,"offset":3125201474,"compressed_bytes":192995,"size":347982,"method":8,"crc32":2256456711,"sha256":"a1924163ab1ed7ad91c4bd309cc1772dccf2be1dd5ecc7f4e80e602c328ef566"},{"fname":"152325","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"lomboy","group_id":"fsd50k:u-27efe1407e76","member":"FSD50K.dev_audio/152325.wav","disk":4,"offset":2020144614,"compressed_bytes":342975,"size":513870,"method":8,"crc32":755353702,"sha256":"e9bb8b9ec8e22986156198a24f7c61b11ad4ff3060f698964a4a421bc347349a"},{"fname":"174895","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"toiletrolltube","group_id":"fsd50k:u-898a3f52b9ab","member":"FSD50K.dev_audio/174895.wav","disk":5,"offset":2096790029,"compressed_bytes":659620,"size":801478,"method":8,"crc32":2333603468,"sha256":"fd4d7106cd04a7ed1a60ef8b24c87c8f1b66a84083f1d11379841514a14f7b74"},{"fname":"176965","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"kvgarlic","group_id":"fsd50k:u-b70ab1660d5f","member":"FSD50K.dev_audio/176965.wav","disk":0,"offset":2632099604,"compressed_bytes":1182259,"size":1731294,"method":8,"crc32":2709402852,"sha256":"c52ee6a4f3fd0c0544cb0af310af4caa913484a1fc80236eaeea29ca8277d0d4"},{"fname":"188734","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"kangaroovindaloo","group_id":"fsd50k:u-04d5c1a4baf3","member":"FSD50K.dev_audio/188734.wav","disk":4,"offset":1678966244,"compressed_bytes":1862719,"size":2425578,"method":8,"crc32":3736904470,"sha256":"374c79dff6fb90293cc56559aa478c7d935673383550dbd8c2f1d7d235d9efe9"},{"fname":"189529","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"tinga","group_id":"fsd50k:u-ffd667450ce8","member":"FSD50K.dev_audio/189529.wav","disk":0,"offset":295487529,"compressed_bytes":209400,"size":256374,"method":8,"crc32":2111514899,"sha256":"04e1b1eedd08b8ddfe737b45ff66ddaed3d95f47dcd2fc2872da18e517dfb8de"},{"fname":"189543","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"tinga","group_id":"fsd50k:u-ffd667450ce8","member":"FSD50K.dev_audio/189543.wav","disk":4,"offset":650538953,"compressed_bytes":244234,"size":290402,"method":8,"crc32":2380267798,"sha256":"4d719dcd9b57397f405cdd9739cb4f985b75643c343635c6e14e15558d428630"},{"fname":"189545","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"tinga","group_id":"fsd50k:u-ffd667450ce8","member":"FSD50K.dev_audio/189545.wav","disk":3,"offset":2765317098,"compressed_bytes":157829,"size":188318,"method":8,"crc32":3255899206,"sha256":"ca9b6f9061b77cb9a6c8533026d86dbe17c3058e777814256e91fd7e73f7f109"},{"fname":"195942","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"FestaCienciaTecnologia","group_id":"fsd50k:u-7bb39dee004b","member":"FSD50K.dev_audio/195942.wav","disk":4,"offset":1747846176,"compressed_bytes":483005,"size":552416,"method":8,"crc32":3343073062,"sha256":"dfc642e9bff7bdfb66a0374e13d04da2c40b669dfe16321a6dfc5161fa4fce4e"},{"fname":"209045","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Insestanyricarda","group_id":"fsd50k:u-568563aa5b58","member":"FSD50K.dev_audio/209045.wav","disk":4,"offset":2495993088,"compressed_bytes":1414500,"size":1612878,"method":8,"crc32":222716313,"sha256":"20ff34d63bbcfb84200f97125c7ef9c3bbb2a345e75c9e5abeadd3f85b5a14a4"},{"fname":"213138","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Taira Komori","group_id":"fsd50k:u-d05198fcfdb9","member":"FSD50K.dev_audio/213138.wav","disk":1,"offset":2024586363,"compressed_bytes":151282,"size":176974,"method":8,"crc32":2105760589,"sha256":"0c55a9422cdaa7ab9b4cb9bad2368ce81f6ffe160fb97fbd4e480a3e70a420ab"},{"fname":"216203","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"RSilveira_88","group_id":"fsd50k:u-3c8e9518c05a","member":"FSD50K.dev_audio/216203.wav","disk":1,"offset":3219624834,"compressed_bytes":488842,"size":2440360,"method":8,"crc32":1168440860,"sha256":"3b4d496606391e757d55ed3374024ec70682c3022472c54179f01ed2bd59ccc6"},{"fname":"223125","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"keweldog","group_id":"fsd50k:u-ab5daffa886d","member":"FSD50K.dev_audio/223125.wav","disk":4,"offset":941817828,"compressed_bytes":1798517,"size":2421962,"method":8,"crc32":4289727619,"sha256":"6661425034fa36d0b749312dde95070080660f3e678d0d8ddd837d24fe81e638"},{"fname":"223127","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"keweldog","group_id":"fsd50k:u-ab5daffa886d","member":"FSD50K.dev_audio/223127.wav","disk":2,"offset":600198588,"compressed_bytes":1814583,"size":2421962,"method":8,"crc32":2740467129,"sha256":"77b61e2d0ce7b843d8f8751f32ff8da586cbc45835fb83ce2c1ee540745e6ad4"},{"fname":"231270","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"melc91","group_id":"fsd50k:u-d4d1da60a175","member":"FSD50K.dev_audio/231270.wav","disk":2,"offset":2411842381,"compressed_bytes":624026,"size":834126,"method":8,"crc32":1210084438,"sha256":"e81efa1125061f0c17be2ccd2a760ffa0310b3ec9226f1ea728677b1bb601228"},{"fname":"232054","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"insdali","group_id":"fsd50k:u-fc9af2da2a46","member":"FSD50K.dev_audio/232054.wav","disk":4,"offset":731643732,"compressed_bytes":160725,"size":203882,"method":8,"crc32":425527326,"sha256":"f8fe9891f99c18d24a0681e93c41b6c78ff05474d809a0c6aa6c2b12e98a899c"},{"fname":"237121","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"ejames","group_id":"fsd50k:u-fcf5547b6f79","member":"FSD50K.dev_audio/237121.wav","disk":2,"offset":684001389,"compressed_bytes":446632,"size":658254,"method":8,"crc32":2526461328,"sha256":"da107cc033cecab0d253d6a8c035b4d8914a5d674817cf29c98f56a27473920c"},{"fname":"240476","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Godowan","group_id":"fsd50k:u-c9f9d9106eb9","member":"FSD50K.dev_audio/240476.wav","disk":4,"offset":1268370912,"compressed_bytes":46895,"size":84688,"method":8,"crc32":3820359271,"sha256":"e2f1c08b5da54cd1122dbbe1375987598031b40be53722331ebc73e2c00ee843"},{"fname":"242092","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"abhijitchirde","group_id":"fsd50k:u-3624f7f29a31","member":"FSD50K.dev_audio/242092.wav","disk":0,"offset":2568114030,"compressed_bytes":1659713,"size":2126382,"method":8,"crc32":866061373,"sha256":"e5add9d0a291b21247b3ea93c788b6d1bdfcfd90523ea336463aa914a728d3d4"},{"fname":"245702","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"TicAshfield","group_id":"fsd50k:u-f61aaf37c2fe","member":"FSD50K.dev_audio/245702.wav","disk":4,"offset":2543785448,"compressed_bytes":1248729,"size":1965630,"method":8,"crc32":3316418221,"sha256":"5a3ee78b24120da374c1708e848de2a90b3de32d9e516935ac44ec17f0df175f"},{"fname":"246223","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"AgentDD","group_id":"fsd50k:u-7c6137755a93","member":"FSD50K.dev_audio/246223.wav","disk":1,"offset":3158666664,"compressed_bytes":67243,"size":224594,"method":8,"crc32":2911473884,"sha256":"fb4fb5ef0a0b46fe33b6c652a615969968b8118126ade9198ca1bd689df9d01d"},{"fname":"249546","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"FngerSounds","group_id":"fsd50k:u-3011f2379084","member":"FSD50K.dev_audio/249546.wav","disk":5,"offset":329659106,"compressed_bytes":165651,"size":215992,"method":8,"crc32":3965598860,"sha256":"8f2681bd9193fed04c33de2335cfcf45be7624fbb7203a88fb8aee43b1951f69"},{"fname":"250810","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"trouxel","group_id":"fsd50k:u-abf5155dfbf0","member":"FSD50K.dev_audio/250810.wav","disk":1,"offset":1456711153,"compressed_bytes":830584,"size":1366568,"method":8,"crc32":3242799482,"sha256":"65b5b9ccbe69aec74fd4e344c0cf39dd9d98657f164efdef2f09382d3f1aa932"},{"fname":"250991","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"GammaScorpii","group_id":"fsd50k:u-d5bdd18ff0bc","member":"FSD50K.dev_audio/250991.wav","disk":5,"offset":1300141645,"compressed_bytes":1978682,"size":2624916,"method":8,"crc32":1336373023,"sha256":"3cb89219c176e9aa123106c36ac9dd7e2752808715a946ddad1327aa08b7d90c"},{"fname":"268019","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"luckylittleraven","group_id":"fsd50k:u-1c6f50054379","member":"FSD50K.dev_audio/268019.wav","disk":0,"offset":2962403716,"compressed_bytes":1094558,"size":1615412,"method":8,"crc32":1699210873,"sha256":"6daf819c1f64e71867b1df69d288c1b9f00ed272579372365bdea89445ad2e86"},{"fname":"271150","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Sevin7","group_id":"fsd50k:u-3ada7d296da4","member":"FSD50K.dev_audio/271150.wav","disk":0,"offset":1295465989,"compressed_bytes":99560,"size":116456,"method":8,"crc32":836133864,"sha256":"0fd07653e385aa8f5810a0e9907ff5bd389abd35ea92a816ac065f6f71a59215"},{"fname":"273726","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"MrAuralization","group_id":"fsd50k:u-810d1734e091","member":"FSD50K.dev_audio/273726.wav","disk":2,"offset":1995004783,"compressed_bytes":216140,"size":408824,"method":8,"crc32":1168480858,"sha256":"72c7ef55f9146439ebda0a8f3db82b8dfd97d547e3f4bf5cb34a33f4e2e6ffd7"},{"fname":"274078","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"RemoteLeg","group_id":"fsd50k:u-a7aaab00d537","member":"FSD50K.dev_audio/274078.wav","disk":4,"offset":1348184171,"compressed_bytes":259270,"size":321756,"method":8,"crc32":2852978460,"sha256":"982538cead8e52a9a01becb4cacd80b2f3333b49387dedfd997d6c239537a3c0"},{"fname":"274226","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"ducksingel","group_id":"fsd50k:u-108367c1aaa1","member":"FSD50K.dev_audio/274226.wav","disk":0,"offset":2216307701,"compressed_bytes":1407264,"size":2324554,"method":8,"crc32":2606751948,"sha256":"4c891fe3ab69d1fbbee7b1e724cb9124d96f38a6108c5ed75f4873632a9a92d0"},{"fname":"326704","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"bikamek1","group_id":"fsd50k:u-693bc359fc19","member":"FSD50K.dev_audio/326704.wav","disk":3,"offset":1743970993,"compressed_bytes":375412,"size":494112,"method":8,"crc32":1175949161,"sha256":"dbb4f28f892ac0098f4311c6d2a165ca778d046c32345768c8db9aaf06a127b4"},{"fname":"330967","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Adam_N","group_id":"fsd50k:u-124149624cb4","member":"FSD50K.dev_audio/330967.wav","disk":4,"offset":3004400636,"compressed_bytes":106000,"size":170370,"method":8,"crc32":3636781190,"sha256":"5117647583e39a83d5f31546cd78258ea7a14bd576d509ad79b0c8fc1e5091fb"},{"fname":"347918","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"NickOxytocin","group_id":"fsd50k:u-706093a44f55","member":"FSD50K.dev_audio/347918.wav","disk":5,"offset":214426880,"compressed_bytes":791795,"size":1434360,"method":8,"crc32":70683999,"sha256":"47757a1114423b5b663931e428d22e3c2e09ceb6666514364975f0a84de688ba"},{"fname":"353213","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"tec studios","group_id":"fsd50k:u-be53219926a7","member":"FSD50K.dev_audio/353213.wav","disk":4,"offset":2410966131,"compressed_bytes":80186,"size":88368,"method":8,"crc32":3813268102,"sha256":"90c48408cb52817baeb42ab66fe051dd233aec91d0a6b1558f9b0643aec60f7f"},{"fname":"353214","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"tec studios","group_id":"fsd50k:u-be53219926a7","member":"FSD50K.dev_audio/353214.wav","disk":4,"offset":1874368904,"compressed_bytes":91782,"size":103008,"method":8,"crc32":4084203969,"sha256":"0a12b953ceb9188fa9fdac7b95b7a06bd62ee6caf55d627b51ccccbb974e6438"},{"fname":"361385","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/361385.wav","disk":5,"offset":865240865,"compressed_bytes":231049,"size":299948,"method":8,"crc32":3997647582,"sha256":"f1b04a9a7c211503898b67b1abb3436a6fc1a41f17bd82b093f99e1f2f1559fe"},{"fname":"362427","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"tec studios","group_id":"fsd50k:u-be53219926a7","member":"FSD50K.dev_audio/362427.wav","disk":5,"offset":711703720,"compressed_bytes":176954,"size":189942,"method":8,"crc32":683908780,"sha256":"6f08bd34c1427db9e4314605e346e08ecd33f3ab654f1189cbe8ddef0df683db"},{"fname":"369546","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"cafity","group_id":"fsd50k:u-a98614cb7e6f","member":"FSD50K.dev_audio/369546.wav","disk":4,"offset":318894953,"compressed_bytes":123693,"size":250386,"method":8,"crc32":4190056612,"sha256":"7e63b18fde7862ac55ab0f3ee3432c3ae4ba9f84755e6f8f1eec50d9d4c9c33c"},{"fname":"386511","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"iimck","group_id":"fsd50k:u-edd2cece2381","member":"FSD50K.dev_audio/386511.wav","disk":2,"offset":836006130,"compressed_bytes":103060,"size":110328,"method":8,"crc32":1200836951,"sha256":"1dfe7d2cc9670bd75db194dbcea5f3d916a7e07bf1e8d67865d794615b0dff87"},{"fname":"386697","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"tommy_mooney","group_id":"fsd50k:u-ae6670cc4ff6","member":"FSD50K.dev_audio/386697.wav","disk":4,"offset":1553668391,"compressed_bytes":290196,"size":926178,"method":8,"crc32":3712460392,"sha256":"d8a5c771924945b3706a81a4bf0dc260b5448de676085a37942f86ac33f19bf1"},{"fname":"386985","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"EpicWizard","group_id":"fsd50k:u-402c82c457c7","member":"FSD50K.dev_audio/386985.wav","disk":4,"offset":2690483347,"compressed_bytes":1994495,"size":2619620,"method":8,"crc32":2987344999,"sha256":"088c96fd6e0cfa2b3250777253702a49d023c71fc29f9b06438b046a06a4a25b"},{"fname":"387991","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"DCPoke","group_id":"fsd50k:u-d23d5e0485b0","member":"FSD50K.dev_audio/387991.wav","disk":0,"offset":2409886177,"compressed_bytes":1541956,"size":1993518,"method":8,"crc32":919610794,"sha256":"b3da7262aacfb0a7caf2e109859546dc7edfcc24c3710d69ff0606939b10f7b7"},{"fname":"388072","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"jumpbug99","group_id":"fsd50k:u-c1d6d7c491a5","member":"FSD50K.dev_audio/388072.wav","disk":0,"offset":480680458,"compressed_bytes":573040,"size":845498,"method":8,"crc32":3434426820,"sha256":"cbabc734613830feef55c7b21aed217e98598baa8641fff08650ae65c643a12a"},{"fname":"388725","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Inkahootz81","group_id":"fsd50k:u-3f0efde45e46","member":"FSD50K.dev_audio/388725.wav","disk":2,"offset":1940653521,"compressed_bytes":546069,"size":917070,"method":8,"crc32":3908628166,"sha256":"46a9951e5cfa600fe665fbe06b12e9e8c9276e0bdd6f5e40e2e958f85601defb"},{"fname":"391336","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/391336.wav","disk":5,"offset":346632517,"compressed_bytes":668714,"size":836380,"method":8,"crc32":2665278919,"sha256":"a7b1cbdbfcca5eea101c3bb3e598f5be7d7f9353ba54ddb9ead1f5387d3911a0"},{"fname":"394850","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"islandan","group_id":"fsd50k:u-afb4f617263f","member":"FSD50K.dev_audio/394850.wav","disk":5,"offset":1247268298,"compressed_bytes":1057945,"size":1709886,"method":8,"crc32":3112735102,"sha256":"bfebcbf5d8879d222bd6d7f32fe0d2bfb7f276c0a4c35b3a62c054c7c2e30db3"},{"fname":"394851","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"islandan","group_id":"fsd50k:u-afb4f617263f","member":"FSD50K.dev_audio/394851.wav","disk":5,"offset":1740225619,"compressed_bytes":1182974,"size":2043698,"method":8,"crc32":2147103682,"sha256":"97407f6f838b1b8178336d988ee3eb7c4c4d22b58bb305c330752cae4630539c"},{"fname":"405475","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"penetrermind","group_id":"fsd50k:u-99d9dc39a740","member":"FSD50K.dev_audio/405475.wav","disk":3,"offset":1716410730,"compressed_bytes":32831,"size":38352,"method":8,"crc32":2065819318,"sha256":"fbe76181c82bf6e5cccfcad46d0c86b43042a03dbb57f654ca19658ebb92497c"},{"fname":"405482","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"penetrermind","group_id":"fsd50k:u-99d9dc39a740","member":"FSD50K.dev_audio/405482.wav","disk":4,"offset":315191015,"compressed_bytes":108301,"size":128080,"method":8,"crc32":641370466,"sha256":"175b259ac3d0f9498c24914642107ebe0017fc44ec29783a7fe1814e59b5d152"},{"fname":"405483","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"penetrermind","group_id":"fsd50k:u-99d9dc39a740","member":"FSD50K.dev_audio/405483.wav","disk":2,"offset":2556969956,"compressed_bytes":29554,"size":37646,"method":8,"crc32":2950366157,"sha256":"768c31e35a9972a9b1e1b095be8cae1f996b9dc4fd4984d91c22e23a5c15b1fb"},{"fname":"410519","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"liveware","group_id":"fsd50k:u-f89ae5efe69e","member":"FSD50K.dev_audio/410519.wav","disk":4,"offset":1343789591,"compressed_bytes":392142,"size":567398,"method":8,"crc32":2636837124,"sha256":"c94328e76f12d5504598b29c09b7cd011205cce63a7685eb5bdcf63fb60ca4ff"},{"fname":"410959","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"OhmOhm","group_id":"fsd50k:u-ffaa2da84903","member":"FSD50K.dev_audio/410959.wav","disk":0,"offset":514046692,"compressed_bytes":1856043,"size":2282038,"method":8,"crc32":455467528,"sha256":"7d1f9a270f4d7f49177557ed396a30ddaab1bca8b6ad437ab703f43aaa2d9e39"},{"fname":"415019","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"NickPeeters","group_id":"fsd50k:u-d26b9c272350","member":"FSD50K.dev_audio/415019.wav","disk":0,"offset":2256294073,"compressed_bytes":1624641,"size":2318668,"method":8,"crc32":3623048383,"sha256":"0094076a072a8a58f8bb007a2f7ba076aa09b88883fb4f0c70fbbd2c10ec79c2"},{"fname":"417713","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"ernijsyoyo","group_id":"fsd50k:u-8eb30bfdf9a1","member":"FSD50K.dev_audio/417713.wav","disk":2,"offset":606567596,"compressed_bytes":31276,"size":39252,"method":8,"crc32":2071662637,"sha256":"98d87aca11b500c9d2821ad2c8487fa6021f015f1db4f8df885818926c578481"},{"fname":"417720","split":"dev","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"ernijsyoyo","group_id":"fsd50k:u-8eb30bfdf9a1","member":"FSD50K.dev_audio/417720.wav","disk":2,"offset":440594034,"compressed_bytes":43810,"size":71412,"method":8,"crc32":3936980301,"sha256":"926bb3209e1ef140c8d2c7bca6318f78038f74f4172ea3ffd1be8b6d0a3ec5c5"},{"fname":"35368","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"fresco","group_id":"fsd50k:u-a12de35e2537","member":"FSD50K.eval_audio/35368.wav","disk":0,"offset":2151475444,"compressed_bytes":356575,"size":409480,"method":8,"crc32":3482878975,"sha256":"10cbb803c13a61d0ec357a1154988fc042021ff00058b54fc792961c5d9e45bc"},{"fname":"52790","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"alexbird","group_id":"fsd50k:u-e595f38fe2d6","member":"FSD50K.eval_audio/52790.wav","disk":1,"offset":759397493,"compressed_bytes":1865467,"size":2179150,"method":8,"crc32":1982118691,"sha256":"ee3318ab84434f45c97c7265e1c8b0929121b549c08f0096c67219ca12e15c55"},{"fname":"56720","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"subcidal","group_id":"fsd50k:u-348dfafe687e","member":"FSD50K.eval_audio/56720.wav","disk":1,"offset":594113482,"compressed_bytes":99088,"size":125828,"method":8,"crc32":1644790967,"sha256":"c3603f33b853ffeb54c5b50b123efe6d0ff3c40e2c8de20e7ca2240fd86eff75"},{"fname":"94924","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"higginsdj","group_id":"fsd50k:u-e0cc34d0b164","member":"FSD50K.eval_audio/94924.wav","disk":0,"offset":379710605,"compressed_bytes":937411,"size":1842202,"method":8,"crc32":1224576998,"sha256":"f0b411d181ff301465213131d9b68bac615c5fb302d13e158d1293cc8b533372"},{"fname":"96080","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"higginsdj","group_id":"fsd50k:u-e0cc34d0b164","member":"FSD50K.eval_audio/96080.wav","disk":1,"offset":242523995,"compressed_bytes":1306458,"size":1633170,"method":8,"crc32":2548407278,"sha256":"437986dcb420bf3437461f87a3bcb6d717ee2757752085b510eae7ee4d480c94"},{"fname":"122174","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"daveincamas","group_id":"fsd50k:u-5b6619acc4fd","member":"FSD50K.eval_audio/122174.wav","disk":0,"offset":183032755,"compressed_bytes":137046,"size":180600,"method":8,"crc32":411444556,"sha256":"af1c0f34fbdf29dc5510146aee277bd54b285fc5f915a4e24c48e2c50b0f6c60"},{"fname":"127989","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"1pjladd2","group_id":"fsd50k:u-88438933b8bc","member":"FSD50K.eval_audio/127989.wav","disk":1,"offset":2086252420,"compressed_bytes":1552484,"size":1820526,"method":8,"crc32":594417132,"sha256":"21467b04a18998c0caf5c523e3e78710f4e303390432fd8c2c5fcd700717dc96"},{"fname":"148854","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"martian","group_id":"fsd50k:u-16b33ba57853","member":"FSD50K.eval_audio/148854.wav","disk":0,"offset":1415227487,"compressed_bytes":1105033,"size":2196780,"method":8,"crc32":426165895,"sha256":"25c8d7d37db2f66a857db10c01d2c73ffcf577ffebf4383e46fc6d07fd9a5b8c"},{"fname":"154402","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"semccab","group_id":"fsd50k:u-2ba11bbef5a7","member":"FSD50K.eval_audio/154402.wav","disk":1,"offset":428173169,"compressed_bytes":372106,"size":914302,"method":8,"crc32":4069531283,"sha256":"f29aaa8fbcd324687ccdd920f5cd69a46aeafe63ee80230461f78ae6ebb85ae5"},{"fname":"157311","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"burkay","group_id":"fsd50k:u-a613e41293bf","member":"FSD50K.eval_audio/157311.wav","disk":1,"offset":2534726589,"compressed_bytes":1257437,"size":1743906,"method":8,"crc32":4227386922,"sha256":"4830ef6db5a39c7e85808b560d4793d637a7c61b629d2a8135551581cb437ecc"},{"fname":"166257","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"hpnguyen52","group_id":"fsd50k:u-4c8c88072cc3","member":"FSD50K.eval_audio/166257.wav","disk":1,"offset":975283465,"compressed_bytes":1928699,"size":2363370,"method":8,"crc32":2232929597,"sha256":"760c79fdcca4cb61239d5f4fbd9c917720a61a5bef990cc75a1eafaa851aa7d2"},{"fname":"179343","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"lolamadeus","group_id":"fsd50k:u-52fae1a3dc99","member":"FSD50K.eval_audio/179343.wav","disk":0,"offset":1476938776,"compressed_bytes":352518,"size":436726,"method":8,"crc32":1868423059,"sha256":"b890e2df159cd5cc00477cedba94f871bfa9f7cdf8e5cf4fc3059c42ce8f2039"},{"fname":"187443","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"VlatkoBlazek","group_id":"fsd50k:u-ab880a77d5c6","member":"FSD50K.eval_audio/187443.wav","disk":0,"offset":854317390,"compressed_bytes":1549235,"size":1875336,"method":8,"crc32":1877211704,"sha256":"d04d6a375ab63fec359bc1a3c40c4105437791ebc9d903d9083ee4e428791336"},{"fname":"194948","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"soundmary","group_id":"fsd50k:u-51f62a00c5d6","member":"FSD50K.eval_audio/194948.wav","disk":1,"offset":1084030858,"compressed_bytes":200664,"size":245628,"method":8,"crc32":3605900353,"sha256":"66926ee4eeda59b378bb80c748ee012e395d43e3df79f9b7fd78636c6d18dd8e"},{"fname":"215175","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"john9","group_id":"fsd50k:u-0c134b8f1971","member":"FSD50K.eval_audio/215175.wav","disk":0,"offset":657350760,"compressed_bytes":74095,"size":95124,"method":8,"crc32":3483953337,"sha256":"d1a55b1ce08a1ecb41124942b07cfce30ba5b13033c1df5e41c899f7496a50f0"},{"fname":"223978","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"roman_cgr","group_id":"fsd50k:u-9d4b0c96f5ad","member":"FSD50K.eval_audio/223978.wav","disk":0,"offset":1642336861,"compressed_bytes":1559855,"size":2258940,"method":8,"crc32":164619300,"sha256":"b2df6bb13b9eacc5eb459ac5df762befe71b5392e616b621918a7b2dfb7e81ce"},{"fname":"319512","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"squashy555","group_id":"fsd50k:u-b77ba4e6741a","member":"FSD50K.eval_audio/319512.wav","disk":0,"offset":3112224227,"compressed_bytes":831948,"size":1125124,"method":8,"crc32":4269366485,"sha256":"10d97c33e56b6ecd7b47f0c2fd20bf99746e826aad233b36afe135f7cac1cba9"},{"fname":"326452","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"miekyj","group_id":"fsd50k:u-0be2fda97a45","member":"FSD50K.eval_audio/326452.wav","disk":0,"offset":12556741,"compressed_bytes":373269,"size":991388,"method":8,"crc32":4084776863,"sha256":"b64c044df9c28246539c0dfc14ef0099cfcf99590b0b121896ddfc88d9c25857"},{"fname":"390298","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"pflanigan","group_id":"fsd50k:u-7a8d2aeb4cd8","member":"FSD50K.eval_audio/390298.wav","disk":1,"offset":2330203305,"compressed_bytes":1079297,"size":1928526,"method":8,"crc32":1718687059,"sha256":"4f96fa6c132eb34ecf84ea9e5a8295639e30819fb536ca7220d7593685117da3"},{"fname":"391277","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"ANARKYA","group_id":"fsd50k:u-d256f42e3df4","member":"FSD50K.eval_audio/391277.wav","disk":0,"offset":537568421,"compressed_bytes":578790,"size":868262,"method":8,"crc32":2263484135,"sha256":"260b61edf6c9eed077ec8cc03232dcab4ff683266992ad6bfd9b078b254ab41c"},{"fname":"393782","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"_CAZAPROD_","group_id":"fsd50k:u-906520efa131","member":"FSD50K.eval_audio/393782.wav","disk":0,"offset":2855452822,"compressed_bytes":1986282,"size":2226254,"method":8,"crc32":2615904608,"sha256":"450f1966de2281e7d59b6fdc0a94d66374599f99738c757b5b69edc5e2df7ab3"},{"fname":"401636","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Mystikuum","group_id":"fsd50k:u-210468f34e38","member":"FSD50K.eval_audio/401636.wav","disk":0,"offset":2663507596,"compressed_bytes":317865,"size":485178,"method":8,"crc32":3198319060,"sha256":"e1a0171c5880778f682604ca9be18ef5c71d0fd82e3afabf904c6b8ab81680ab"},{"fname":"414398","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"samwd","group_id":"fsd50k:u-80288fd4f50f","member":"FSD50K.eval_audio/414398.wav","disk":0,"offset":149143129,"compressed_bytes":1130535,"size":1968702,"method":8,"crc32":2051420120,"sha256":"4731ff8a29059be2e962cbeeaf5bea1d7243ff236c645e638571ac802f6d56f0"},{"fname":"427435","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Cribbler","group_id":"fsd50k:u-46e1581221a1","member":"FSD50K.eval_audio/427435.wav","disk":0,"offset":2032571359,"compressed_bytes":1442068,"size":2403528,"method":8,"crc32":116341395,"sha256":"cbfb6606d8dec04f801f845d0e563282b3183f1065ecd253e9cf398687e08265"},{"fname":"428075","split":"eval","target_label":"Bird","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"bolkmar","group_id":"fsd50k:u-d6cc4e0f3dff","member":"FSD50K.eval_audio/428075.wav","disk":1,"offset":1157258460,"compressed_bytes":405020,"size":603928,"method":8,"crc32":2116736682,"sha256":"208dd6b1013ef4a5fc6e5ea12d61ce8580e2742bc0c9111a1d9d32ff34144271"},{"fname":"18834","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"WIM","group_id":"fsd50k:u-07d9d387aebf","member":"FSD50K.dev_audio/18834.wav","disk":2,"offset":545527496,"compressed_bytes":1454594,"size":1914824,"method":8,"crc32":4132653048,"sha256":"3763e6d842a83b1549468ec572a066c7919a94894836d36f56034ee2bfdc4c61"},{"fname":"23452","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"milo","group_id":"fsd50k:u-13adffea089b","member":"FSD50K.dev_audio/23452.wav","disk":3,"offset":2022196315,"compressed_bytes":1175691,"size":1499762,"method":8,"crc32":3784101901,"sha256":"a384b04a0429927ad222aacfaa9318002724de138b772ffdb66a47849a05135b"},{"fname":"23722","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"milo","group_id":"fsd50k:u-13adffea089b","member":"FSD50K.dev_audio/23722.wav","disk":0,"offset":945057030,"compressed_bytes":1119332,"size":1455182,"method":8,"crc32":3905415027,"sha256":"2cbf2e794ac5f827bb0db17d4499085687c5177db474b5a7d334c0e0e8019381"},{"fname":"26972","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"LG","group_id":"fsd50k:u-b870b7809747","member":"FSD50K.dev_audio/26972.wav","disk":2,"offset":2090830284,"compressed_bytes":195444,"size":217104,"method":8,"crc32":778301205,"sha256":"d8bd3e824a6116ef908704552f61bf8f884f107e80492df6ab438c8245b2a62c"},{"fname":"31006","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"hello_flowers","group_id":"fsd50k:u-50ed42df4aac","member":"FSD50K.dev_audio/31006.wav","disk":2,"offset":1134384197,"compressed_bytes":819712,"size":1323078,"method":8,"crc32":2930368203,"sha256":"3e1c9a53c154b73ca22bd5135b3b1d930b1d2505b7eae75e069a077bf7c382f2"},{"fname":"39934","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/39934.wav","disk":3,"offset":385269679,"compressed_bytes":1306568,"size":1678704,"method":8,"crc32":3659324461,"sha256":"6162c752c3354669eaec9af42f8342b24c4880da4201eab88fc7e4cf537386f5"},{"fname":"51688","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"inchadney","group_id":"fsd50k:u-ea67aa4b603e","member":"FSD50K.dev_audio/51688.wav","disk":2,"offset":1953983091,"compressed_bytes":1651895,"size":1871430,"method":8,"crc32":395028651,"sha256":"64baa8e627f50a1e65daba21ddde2146479cf119f018180cac6530cf838ed3e4"},{"fname":"55041","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"suonidigenova","group_id":"fsd50k:u-87e068f4668c","member":"FSD50K.dev_audio/55041.wav","disk":5,"offset":2064134635,"compressed_bytes":1448471,"size":1757262,"method":8,"crc32":1403914219,"sha256":"0897823b90a7ada241ad72866b9de09b361a712bed117567a9191d09fc583611"},{"fname":"55050","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"suonidigenova","group_id":"fsd50k:u-87e068f4668c","member":"FSD50K.dev_audio/55050.wav","disk":4,"offset":949801635,"compressed_bytes":783388,"size":791440,"method":8,"crc32":2218834882,"sha256":"4d6ca5325a65c0c63f15df1fcbde1e45489f79b7f7643f6608c2cb6cf089c409"},{"fname":"55052","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"suonidigenova","group_id":"fsd50k:u-87e068f4668c","member":"FSD50K.dev_audio/55052.wav","disk":1,"offset":1928399743,"compressed_bytes":715724,"size":782970,"method":8,"crc32":2834268279,"sha256":"ccab12cf751373f8cec90fef89bb1fb6c0342332ed356b2bc11a6fe9d5750f92"},{"fname":"56627","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"cmartins10","group_id":"fsd50k:u-8fec10558c50","member":"FSD50K.dev_audio/56627.wav","disk":5,"offset":241934514,"compressed_bytes":1971371,"size":2151724,"method":8,"crc32":3086180046,"sha256":"9f886f7195f654cf9fed50683d95b4c93bba5ced56872afa8cc9d6d654303814"},{"fname":"59201","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/59201.wav","disk":4,"offset":2769260698,"compressed_bytes":2130083,"size":2296318,"method":8,"crc32":2284006254,"sha256":"0520e1185275b4c5165a4c523a55472af903c170638ee53ce0be1414d8e9c87c"},{"fname":"69663","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"schaarsen","group_id":"fsd50k:u-47dd0935100f","member":"FSD50K.dev_audio/69663.wav","disk":3,"offset":788277173,"compressed_bytes":984950,"size":1202766,"method":8,"crc32":2219188337,"sha256":"2ca48b31961398c3b4e12d53441db4dd94b9a2fc3467e0b6bec02ab8d59c157b"},{"fname":"74471","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"LG","group_id":"fsd50k:u-b870b7809747","member":"FSD50K.dev_audio/74471.wav","disk":4,"offset":406713060,"compressed_bytes":775139,"size":973914,"method":8,"crc32":1428153667,"sha256":"379a5aa9c01db3522b5f24858c19920eab839950b78aeefb55f3e7a9c357188b"},{"fname":"75980","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"NoiseCollector","group_id":"fsd50k:u-0697aba1f475","member":"FSD50K.dev_audio/75980.wav","disk":0,"offset":1427226208,"compressed_bytes":594674,"size":700494,"method":8,"crc32":182083495,"sha256":"dbf35f9d9ec96bed0e73fb740dcd410adeca5bf26caf12f31009e92ab0d46260"},{"fname":"75982","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"NoiseCollector","group_id":"fsd50k:u-0697aba1f475","member":"FSD50K.dev_audio/75982.wav","disk":2,"offset":2770460179,"compressed_bytes":74628,"size":82252,"method":8,"crc32":400902740,"sha256":"1168c30584662d7b99516ba065ead2b3a53ad80ec3525bbb7077eaf1e5f569ac"},{"fname":"75994","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"NoiseCollector","group_id":"fsd50k:u-0697aba1f475","member":"FSD50K.dev_audio/75994.wav","disk":5,"offset":1216138053,"compressed_bytes":70951,"size":96846,"method":8,"crc32":3739484564,"sha256":"68aedb03553dd3e1b8a4604d570a36bb6d9c45adecdf1c496f51f1496459d5a6"},{"fname":"77697","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Benboncan","group_id":"fsd50k:u-05af64f71b1e","member":"FSD50K.dev_audio/77697.wav","disk":1,"offset":1756295587,"compressed_bytes":647938,"size":913170,"method":8,"crc32":2198141754,"sha256":"6f07d77aedf779fe7fd94921ad49b0222377f3dd29d637f57f489fbdf45592a0"},{"fname":"90721","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"pkorac","group_id":"fsd50k:u-0fcbd40884c8","member":"FSD50K.dev_audio/90721.wav","disk":3,"offset":448772383,"compressed_bytes":2218920,"size":2579280,"method":8,"crc32":19493628,"sha256":"fbde79e1ce1bca57e98beed66e274b7b70629ea0d457f48576a3fbc9a855521d"},{"fname":"91533","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"LG","group_id":"fsd50k:u-b870b7809747","member":"FSD50K.dev_audio/91533.wav","disk":1,"offset":1477188574,"compressed_bytes":913329,"size":1016308,"method":8,"crc32":3242572799,"sha256":"be75062a5fe91d673cad6ed8cfd11ff29402f4ba94b05e6083dba26cb0a01bc7"},{"fname":"93946","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"CGEffex","group_id":"fsd50k:u-4f7ab92b7591","member":"FSD50K.dev_audio/93946.wav","disk":4,"offset":800810252,"compressed_bytes":1873991,"size":2497086,"method":8,"crc32":695356352,"sha256":"c88968b74aee60905393062d8d81480c8023697aad4b27d9f158cfb62f6f3683"},{"fname":"99630","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"tec studios","group_id":"fsd50k:u-be53219926a7","member":"FSD50K.dev_audio/99630.wav","disk":1,"offset":556054513,"compressed_bytes":622613,"size":705678,"method":8,"crc32":3623491573,"sha256":"23d7dcbfb77ded868e876197879fdca6c8b1cd40eb7b09dbce990b1736d280b2"},{"fname":"100400","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"vnss","group_id":"fsd50k:u-3b892f91e719","member":"FSD50K.dev_audio/100400.wav","disk":0,"offset":1576020915,"compressed_bytes":2125107,"size":2607070,"method":8,"crc32":3185338176,"sha256":"1b0cf72a4061236523c330597fd1902cae32880b14ecb1381b3968c36b85c7f7"},{"fname":"109578","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"vinjatovix","group_id":"fsd50k:u-a89e24cca828","member":"FSD50K.dev_audio/109578.wav","disk":4,"offset":2407987596,"compressed_bytes":1894622,"size":2267268,"method":8,"crc32":2297721499,"sha256":"a4ca9468f1cca72131661ae90c9837d5addc2a7ff583558ed86a4f4a8315298a"},{"fname":"110996","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/110996.wav","disk":5,"offset":1190262186,"compressed_bytes":1029073,"size":1162408,"method":8,"crc32":830069790,"sha256":"13f335ba0e03cbe9799555db54f966c4e8a6fb46b5a0aff813b0b655054390a7"},{"fname":"162252","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"qubodup","group_id":"fsd50k:u-2afd5d535ce9","member":"FSD50K.dev_audio/162252.wav","disk":2,"offset":352304568,"compressed_bytes":1155657,"size":1461042,"method":8,"crc32":426419311,"sha256":"50f4160210b57c2a6e0c0af0bf7c8f8733e3709e0a7515419a502fe89631aa36"},{"fname":"163117","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"qubodup","group_id":"fsd50k:u-2afd5d535ce9","member":"FSD50K.dev_audio/163117.wav","disk":4,"offset":3214345172,"compressed_bytes":150653,"size":224106,"method":8,"crc32":3336235738,"sha256":"d706e92067bf31249cb0987b20b18bb9ff9cb27744d3cf81f13000c3401c40ec"},{"fname":"178457","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"vinjatovix","group_id":"fsd50k:u-a89e24cca828","member":"FSD50K.dev_audio/178457.wav","disk":3,"offset":1379209843,"compressed_bytes":1636930,"size":2152750,"method":8,"crc32":4248854887,"sha256":"df34d6653c53b6efd65464b0496b05dac3818fa34448976843818c786126f04a"},{"fname":"179900","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"timsc","group_id":"fsd50k:u-defe87587941","member":"FSD50K.dev_audio/179900.wav","disk":0,"offset":105684263,"compressed_bytes":876423,"size":1023264,"method":8,"crc32":2717050820,"sha256":"3d4c2a829c684012b70d2f699ad5771b2e83a13199fe76da01c5f3d9377c1ab5"},{"fname":"179904","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"timsc","group_id":"fsd50k:u-defe87587941","member":"FSD50K.dev_audio/179904.wav","disk":0,"offset":2767567568,"compressed_bytes":809474,"size":915500,"method":8,"crc32":3538968614,"sha256":"ed6181c36b328cf029a715039e56a7d08b2ce4ab56e335b14b4ab850abf330e9"},{"fname":"179905","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"timsc","group_id":"fsd50k:u-defe87587941","member":"FSD50K.dev_audio/179905.wav","disk":2,"offset":1525553787,"compressed_bytes":625876,"size":759730,"method":8,"crc32":4100769176,"sha256":"891474118e66098d1451456e2ed2037f641c884dbcac5a2b0a63c3e69944c89b"},{"fname":"195091","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"bunting","group_id":"fsd50k:u-c7ef2a86bd55","member":"FSD50K.dev_audio/195091.wav","disk":3,"offset":74992818,"compressed_bytes":916825,"size":970570,"method":8,"crc32":1650397280,"sha256":"ef37d0bcbabd238311028b5ed2164366a3b7c1c55c284fea2163936e2ed64411"},{"fname":"197734","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"flcellogrl","group_id":"fsd50k:u-0498d86ff642","member":"FSD50K.dev_audio/197734.wav","disk":0,"offset":2900251554,"compressed_bytes":316041,"size":348336,"method":8,"crc32":3723740611,"sha256":"d42d02887c0e6099c3d464eff062d3d96c7e32b86bc4c8ba0504d1ee48c76953"},{"fname":"200289","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"vumseplutten1709","group_id":"fsd50k:u-1aaa137d5630","member":"FSD50K.dev_audio/200289.wav","disk":2,"offset":2069647882,"compressed_bytes":1963761,"size":2286670,"method":8,"crc32":1712837592,"sha256":"4fcce1ceb1d028ec015947601fa7f1fc78b45bf1d4360c889b981de6f0e8356d"},{"fname":"208891","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"T David Pattison","group_id":"fsd50k:u-2f136d800ce1","member":"FSD50K.dev_audio/208891.wav","disk":0,"offset":823090568,"compressed_bytes":1375694,"size":1418320,"method":8,"crc32":2634295056,"sha256":"85038f266296c57562a25e3492e497829e0f094f05d141ca40ad89ccc0c5969f"},{"fname":"224011","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"soundatADM","group_id":"fsd50k:u-25fc7dfc5f9a","member":"FSD50K.dev_audio/224011.wav","disk":1,"offset":227201659,"compressed_bytes":235457,"size":253902,"method":8,"crc32":2132087488,"sha256":"c3738ec729fd52ebf6158d7375665938411b12ebabc852840178e66bce2eb720"},{"fname":"234238","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"vumseplutten1709","group_id":"fsd50k:u-1aaa137d5630","member":"FSD50K.dev_audio/234238.wav","disk":5,"offset":1680613419,"compressed_bytes":1080536,"size":1260546,"method":8,"crc32":167846513,"sha256":"b744e7c11ce271599fffbf94af443255bdcfe072cd91a4e766a617418c2fdf0e"},{"fname":"252865","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"skitchscharff","group_id":"fsd50k:u-d9c46421da85","member":"FSD50K.dev_audio/252865.wav","disk":2,"offset":1725599809,"compressed_bytes":2087698,"size":2268582,"method":8,"crc32":2652654029,"sha256":"8782517b22fec253209af47fa7e3627c5884a507ba03a2feaa87f8b596c44326"},{"fname":"252866","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"skitchscharff","group_id":"fsd50k:u-d9c46421da85","member":"FSD50K.dev_audio/252866.wav","disk":3,"offset":598954122,"compressed_bytes":1109377,"size":1252518,"method":8,"crc32":1259057611,"sha256":"3780f5ecc653804a6c0e2d9f41764602b75ebec6b10fa678665869d580b194a2"},{"fname":"252867","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"skitchscharff","group_id":"fsd50k:u-d9c46421da85","member":"FSD50K.dev_audio/252867.wav","disk":0,"offset":689209739,"compressed_bytes":2161470,"size":2536710,"method":8,"crc32":3134193915,"sha256":"b125d2ee897d7fe3a470d65d896d6f31135cceb86f4576e4c933afb1b2fee4d7"},{"fname":"253580","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"P3PP3R","group_id":"fsd50k:u-ca630ab1bca6","member":"FSD50K.dev_audio/253580.wav","disk":1,"offset":2268900368,"compressed_bytes":452996,"size":617478,"method":8,"crc32":671089113,"sha256":"853b0d080736242297b45dfa5d216f22f2668b317d374e4652497f10b5cbdf71"},{"fname":"261846","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"nathanvladimirovich","group_id":"fsd50k:u-cb8e68ae3849","member":"FSD50K.dev_audio/261846.wav","disk":1,"offset":1111116908,"compressed_bytes":375458,"size":467106,"method":8,"crc32":4225317804,"sha256":"b05c2c66307a90544128fcf401702a37a97c9fc71052c004ea75dd87455d668d"},{"fname":"261847","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"nathanvladimirovich","group_id":"fsd50k:u-cb8e68ae3849","member":"FSD50K.dev_audio/261847.wav","disk":0,"offset":649273356,"compressed_bytes":284836,"size":303274,"method":8,"crc32":1279816162,"sha256":"61c3fe64ca56d4398853f696a84959f489717dff528d22e6b2009af9895c74f2"},{"fname":"263159","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"LoopUdu","group_id":"fsd50k:u-9ff8351bd220","member":"FSD50K.dev_audio/263159.wav","disk":3,"offset":2737421544,"compressed_bytes":1083107,"size":1214064,"method":8,"crc32":763218440,"sha256":"238cf38e80275b1e3b487eac14059a4176a6dc93f38d26331112fae49b0642a3"},{"fname":"263571","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Alcappuccino","group_id":"fsd50k:u-f656cb959936","member":"FSD50K.dev_audio/263571.wav","disk":1,"offset":2388080636,"compressed_bytes":2224515,"size":2399118,"method":8,"crc32":1913879969,"sha256":"952388d51267b9d8d02d7379eb800880d796cbc7383abc8c10ad31dd60c180fe"},{"fname":"317386","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"TiredHippo","group_id":"fsd50k:u-765e1926decf","member":"FSD50K.dev_audio/317386.wav","disk":0,"offset":564447175,"compressed_bytes":1340026,"size":1924850,"method":8,"crc32":2849715660,"sha256":"bc5e311800c7f1946b2044f3f0d3e1440d86e000e7873fd7e6e6ef9e05fb7dfe"},{"fname":"321255","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"freesoundjon01","group_id":"fsd50k:u-56fe33724eb2","member":"FSD50K.dev_audio/321255.wav","disk":3,"offset":2580857397,"compressed_bytes":1850894,"size":2510484,"method":8,"crc32":3671316459,"sha256":"3ad1e33a21df69b87234060590e0349eaeb8344cad1486a69713139a8c7d267e"},{"fname":"321256","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"freesoundjon01","group_id":"fsd50k:u-56fe33724eb2","member":"FSD50K.dev_audio/321256.wav","disk":3,"offset":1990808844,"compressed_bytes":638703,"size":951462,"method":8,"crc32":3596520937,"sha256":"a202be79e3de99ee18ec9a324443a3256bcf18fdea3193a5ca6269585fde1e50"},{"fname":"325542","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"YleArkisto","group_id":"fsd50k:u-d9969ea40981","member":"FSD50K.dev_audio/325542.wav","disk":0,"offset":2946473692,"compressed_bytes":1727716,"size":2218758,"method":8,"crc32":897331360,"sha256":"f8aa9f3a3a7fa168dec8540e27e602b89e344f3a04abe5cbc3261959237f72af"},{"fname":"328616","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Sandermotions","group_id":"fsd50k:u-ce952f98aa65","member":"FSD50K.dev_audio/328616.wav","disk":1,"offset":101467912,"compressed_bytes":2177639,"size":2522952,"method":8,"crc32":3176207019,"sha256":"7049eb3fe780c72382aced8ddd499c280bfc6d9bea655aad2b2f1ca2e058bbd7"},{"fname":"353433","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"ckater","group_id":"fsd50k:u-f2f077b8e2c3","member":"FSD50K.dev_audio/353433.wav","disk":4,"offset":530579703,"compressed_bytes":2125537,"size":2402432,"method":8,"crc32":1225049134,"sha256":"040c20a96bee1503d63b5d820b53587ad86270707e7ea641aa01b26b10b97c04"},{"fname":"394287","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"morganpurkis","group_id":"fsd50k:u-ae05348d0c80","member":"FSD50K.dev_audio/394287.wav","disk":4,"offset":1074209780,"compressed_bytes":485705,"size":494966,"method":8,"crc32":842946502,"sha256":"9b50aa907a6412e55b301636b0efee8be56333d9d5ee595922e42ecc8b4f1448"},{"fname":"394932","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"giddster","group_id":"fsd50k:u-78f4faae483b","member":"FSD50K.dev_audio/394932.wav","disk":4,"offset":1086373223,"compressed_bytes":703418,"size":853454,"method":8,"crc32":1587340604,"sha256":"e80ae43e0287cb94a23f646c8c3a2f918c30efcd60f3327dd84e0aaa4cb2f12e"},{"fname":"397915","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"klankbeeld","group_id":"fsd50k:u-9421960ef6b8","member":"FSD50K.dev_audio/397915.wav","disk":4,"offset":1905283492,"compressed_bytes":1930577,"size":2270688,"method":8,"crc32":1264271113,"sha256":"3a79d801eccb3c765f3077c0015f8f3a398bf73c4961a00dc3cc34d973ad254b"},{"fname":"420716","split":"dev","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"tierrafilm@gmail.com","group_id":"fsd50k:u-04ce0c57d3c6","member":"FSD50K.dev_audio/420716.wav","disk":4,"offset":3097638287,"compressed_bytes":497037,"size":536656,"method":8,"crc32":443714967,"sha256":"57ef5ae11f9b15aff2cced62945d099b4e901788971c75cee48ae9e0b189f402"},{"fname":"31776","split":"eval","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Slanesh","group_id":"fsd50k:u-d93da97a6fb9","member":"FSD50K.eval_audio/31776.wav","disk":1,"offset":1787898169,"compressed_bytes":1863122,"size":2434694,"method":8,"crc32":1959814120,"sha256":"24a5a0b8cd3a7db8e7a1e096072511a82ac72b966bbc28054b757c5286c8d25d"},{"fname":"73257","split":"eval","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"jakobhandersen","group_id":"fsd50k:u-f8187c529e90","member":"FSD50K.eval_audio/73257.wav","disk":1,"offset":2777326422,"compressed_bytes":1431134,"size":1769550,"method":8,"crc32":4051626814,"sha256":"3d42da9a0bcaf5fddea87f9ea3196e59f01ea43dd1e680e3f88fb7ecc488a42f"},{"fname":"118999","split":"eval","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"JoneagleX","group_id":"fsd50k:u-8e757edd43c3","member":"FSD50K.eval_audio/118999.wav","disk":0,"offset":1735176024,"compressed_bytes":1887339,"size":2110542,"method":8,"crc32":2379685714,"sha256":"ddc4d1bba5ebd2c0c0e24f09709a1b949328daf12b7fa2c98561ef8fe84c1ff9"},{"fname":"154833","split":"eval","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"pawsound","group_id":"fsd50k:u-9453f98917c8","member":"FSD50K.eval_audio/154833.wav","disk":1,"offset":1614626758,"compressed_bytes":725510,"size":1356864,"method":8,"crc32":1206269058,"sha256":"f8c192ac5e98d811750fdf7a8f9485781e9760136d7a90b3e8e31f2b7d8b972a"},{"fname":"171674","split":"eval","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Jaturo","group_id":"fsd50k:u-a331beedfb16","member":"FSD50K.eval_audio/171674.wav","disk":1,"offset":221639622,"compressed_bytes":623001,"size":631176,"method":8,"crc32":1048893506,"sha256":"3f3f7248407182c31fa4724078840ca7445d80c46c6cc1b615486ac2fdf04069"},{"fname":"198893","split":"eval","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"bone666138","group_id":"fsd50k:u-9c573414ebf1","member":"FSD50K.eval_audio/198893.wav","disk":0,"offset":472873931,"compressed_bytes":949071,"size":1189128,"method":8,"crc32":4218993788,"sha256":"d48b6aab5a2cf28cab3ad2781007cc4be94ccf2f7489cf83cdb18ac6db151a56"},{"fname":"274906","split":"eval","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"hantorio","group_id":"fsd50k:u-a06a88904be1","member":"FSD50K.eval_audio/274906.wav","disk":1,"offset":222262709,"compressed_bytes":524147,"size":573362,"method":8,"crc32":157170269,"sha256":"40d16883280c1102647124de286b7038ddb220edf8b432099cba59a8876905c5"},{"fname":"274909","split":"eval","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"hantorio","group_id":"fsd50k:u-a06a88904be1","member":"FSD50K.eval_audio/274909.wav","disk":0,"offset":41011955,"compressed_bytes":1161022,"size":1275276,"method":8,"crc32":1063597628,"sha256":"209d8a231b4977bf21fd24d8bbe64911375e55e3c540230c53b78c4107f98117"},{"fname":"274910","split":"eval","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"hantorio","group_id":"fsd50k:u-a06a88904be1","member":"FSD50K.eval_audio/274910.wav","disk":0,"offset":2581420710,"compressed_bytes":2213438,"size":2468860,"method":8,"crc32":3303692687,"sha256":"c60c3f8e262e4d1412b5df56e6a19533347c9b96b75c75b7a600ba700a006746"},{"fname":"321458","split":"eval","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"cartoonrob","group_id":"fsd50k:u-af59f0f90eb2","member":"FSD50K.eval_audio/321458.wav","disk":0,"offset":984939610,"compressed_bytes":422941,"size":2199630,"method":8,"crc32":3006864508,"sha256":"50a833f68ada82275c0634aba282572805ef9e8bdef69653534faaa16909bdff"},{"fname":"328541","split":"eval","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Vendarro","group_id":"fsd50k:u-83754f1be537","member":"FSD50K.eval_audio/328541.wav","disk":0,"offset":3025358397,"compressed_bytes":649472,"size":956494,"method":8,"crc32":3971284720,"sha256":"54f1850f22f56d19cdfaa37bfb615e27125bed6e60e15d67fea631d03b4355e0"},{"fname":"346108","split":"eval","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"limetoe","group_id":"fsd50k:u-6c3833101241","member":"FSD50K.eval_audio/346108.wav","disk":1,"offset":702314879,"compressed_bytes":302246,"size":366448,"method":8,"crc32":618526361,"sha256":"682817099da758cb64fa0173be2f8871d0ea29caa4f4f5702baf851aa818b628"},{"fname":"352374","split":"eval","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Kinoton","group_id":"fsd50k:u-3a502bca35df","member":"FSD50K.eval_audio/352374.wav","disk":0,"offset":413222490,"compressed_bytes":1967632,"size":2469678,"method":8,"crc32":3672934282,"sha256":"70e67ba8b3e6fad21e3b0bb87612e3b66ff0cb735a80528e94cf9bc4e35b59b5"},{"fname":"379125","split":"eval","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"13GPanska_Hronikova_Katerina","group_id":"fsd50k:u-9b67d3e4c726","member":"FSD50K.eval_audio/379125.wav","disk":1,"offset":420981287,"compressed_bytes":270020,"size":306146,"method":8,"crc32":3152112822,"sha256":"fea391708a9a076d5a387c366ef66451044ff4dddba386ef3128331a8908673a"},{"fname":"394221","split":"eval","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"mikewest","group_id":"fsd50k:u-9a4365133130","member":"FSD50K.eval_audio/394221.wav","disk":1,"offset":1822426803,"compressed_bytes":651375,"size":1622582,"method":8,"crc32":1108376594,"sha256":"136de068431cdffca18c33081823e85564216541460b87d1fca1f522e0e908ea"},{"fname":"398415","split":"eval","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"bone666138","group_id":"fsd50k:u-9c573414ebf1","member":"FSD50K.eval_audio/398415.wav","disk":0,"offset":2541939034,"compressed_bytes":368763,"size":569498,"method":8,"crc32":2909494561,"sha256":"323169e06235607f92975532244c2d744e12f92d00cf8b5141206c7570fc0785"},{"fname":"424785","split":"eval","target_label":"Boat_and_Water_vehicle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"barbara.raposo","group_id":"fsd50k:u-b0124d67acd8","member":"FSD50K.eval_audio/424785.wav","disk":1,"offset":2733452933,"compressed_bytes":566901,"size":798798,"method":8,"crc32":1372186035,"sha256":"a18444614c5e1984ef171d1dc0c9c6867f29f7bcd050c18ae7b22ab4e52071db"},{"fname":"17910","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"WIM","group_id":"fsd50k:u-07d9d387aebf","member":"FSD50K.dev_audio/17910.wav","disk":3,"offset":2660644229,"compressed_bytes":325351,"size":358528,"method":8,"crc32":730276633,"sha256":"770299bdeb7278ac7dbe607d260e1910fddb90d62d5101a216b1ec8399ec6c65"},{"fname":"22706","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"terminal","group_id":"fsd50k:u-4e686af7bdcc","member":"FSD50K.dev_audio/22706.wav","disk":4,"offset":1210262795,"compressed_bytes":278187,"size":291088,"method":8,"crc32":3215707459,"sha256":"33df490530f1af02018b676af8a352982be83ca143c4d49aad9d8f00c58fef37"},{"fname":"22882","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Corsica_S","group_id":"fsd50k:u-b8d46530f98a","member":"FSD50K.dev_audio/22882.wav","disk":0,"offset":2153548336,"compressed_bytes":1818415,"size":2075854,"method":8,"crc32":1598249014,"sha256":"2b459a04141b3e0941f2635f2d07c14309f45d1411f4859b439c1bf0a48e7c81"},{"fname":"39120","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Uganda","group_id":"fsd50k:u-893d538563bc","member":"FSD50K.dev_audio/39120.wav","disk":0,"offset":553542071,"compressed_bytes":872215,"size":1791360,"method":8,"crc32":2274741744,"sha256":"9543b00caf23e2dc08fd6c2e2d1438a13e717644c84b9f9ff51d75b62e61fbd0"},{"fname":"45198","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"stomachache","group_id":"fsd50k:u-f1962693b66c","member":"FSD50K.dev_audio/45198.wav","disk":2,"offset":328605033,"compressed_bytes":1182384,"size":1284316,"method":8,"crc32":1872132281,"sha256":"ba7f77852b6fdc0f71a458a6ee4d6ce90521558dcb15fe73253a850103074229"},{"fname":"50896","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"RutgerMuller","group_id":"fsd50k:u-080a8534ba93","member":"FSD50K.dev_audio/50896.wav","disk":3,"offset":791585017,"compressed_bytes":152547,"size":185298,"method":8,"crc32":2264986341,"sha256":"12e2174c60d5738a587c279772291ef9dcd5c278bca767114b53b6e12c543519"},{"fname":"59448","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"rfhache","group_id":"fsd50k:u-63b4f6312277","member":"FSD50K.dev_audio/59448.wav","disk":2,"offset":1176848654,"compressed_bytes":1449556,"size":1687218,"method":8,"crc32":2360806201,"sha256":"110111eb178495a5415aded7fdb895b52e77b32ad3d3c6929e960acd765f0edd"},{"fname":"68662","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"mikaelfernstrom","group_id":"fsd50k:u-d18d78ad1f85","member":"FSD50K.dev_audio/68662.wav","disk":0,"offset":443141642,"compressed_bytes":178201,"size":241742,"method":8,"crc32":603074541,"sha256":"538024adf9ff54fbd7f513d28e03f77f6c683011d5adf7743df77eac0feaf0e5"},{"fname":"70114","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"JustinBW","group_id":"fsd50k:u-b80adf265165","member":"FSD50K.dev_audio/70114.wav","disk":0,"offset":651195026,"compressed_bytes":141861,"size":142106,"method":8,"crc32":366128176,"sha256":"dbda8045e5d529ecff4d6a1628e77fb5ec83158b60e079f454f5d7fa74ba208c"},{"fname":"70121","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"JustinBW","group_id":"fsd50k:u-b80adf265165","member":"FSD50K.dev_audio/70121.wav","disk":4,"offset":3065171897,"compressed_bytes":209024,"size":231290,"method":8,"crc32":3277080830,"sha256":"ce046e117d9ac781aaa6626ba9fe7e0e8353d5aae7c98fd457e47808ead50cfe"},{"fname":"70160","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"JustinBW","group_id":"fsd50k:u-b80adf265165","member":"FSD50K.dev_audio/70160.wav","disk":1,"offset":3130683796,"compressed_bytes":463559,"size":516224,"method":8,"crc32":762238721,"sha256":"a0042c907792b495824d9985a4ce03e499c8902e46f223ab855a1590e6d02aa2"},{"fname":"71740","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"audible-edge","group_id":"fsd50k:u-c89bb2fafd78","member":"FSD50K.dev_audio/71740.wav","disk":4,"offset":167316728,"compressed_bytes":639570,"size":783338,"method":8,"crc32":3454127087,"sha256":"95d58b537eb97f2c7a2aabd1951281c75389a4bf748657de0960281795cb4026"},{"fname":"87246","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"tiredoldwhiteman","group_id":"fsd50k:u-367f7e920c18","member":"FSD50K.dev_audio/87246.wav","disk":2,"offset":717671822,"compressed_bytes":1985856,"size":2603586,"method":8,"crc32":1195533976,"sha256":"64f322f2cb6d9a498b07540afca60771fb5be669b765680206d8d1577f7cf4b5"},{"fname":"103434","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"confusion_music","group_id":"fsd50k:u-3c7d0fe58b74","member":"FSD50K.dev_audio/103434.wav","disk":4,"offset":252359464,"compressed_bytes":611370,"size":785208,"method":8,"crc32":3478580660,"sha256":"1030826fa55d2f2b06ff060650bbd20b00500a982fdf85575389769425440433"},{"fname":"107273","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"tec studios","group_id":"fsd50k:u-be53219926a7","member":"FSD50K.dev_audio/107273.wav","disk":3,"offset":24359845,"compressed_bytes":256476,"size":350082,"method":8,"crc32":4186402210,"sha256":"ed26a52466e45894d5f8679b43caedef0290a0074a1c167debb29aa7663576d4"},{"fname":"107583","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Benboncan","group_id":"fsd50k:u-05af64f71b1e","member":"FSD50K.dev_audio/107583.wav","disk":1,"offset":2785661985,"compressed_bytes":1371873,"size":1587534,"method":8,"crc32":3655951996,"sha256":"2e408450b309f55002558d118c53c8f5599c231490a685492c05b8824bd3e66a"},{"fname":"118971","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"esperri","group_id":"fsd50k:u-feac567e2756","member":"FSD50K.dev_audio/118971.wav","disk":5,"offset":132363926,"compressed_bytes":193339,"size":248432,"method":8,"crc32":3232760185,"sha256":"7b6ac80183f1e7e37e4aa324817aa2e1b51f71cab02bc7e92f36c1c4a563772b"},{"fname":"125560","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"NoiseCollector","group_id":"fsd50k:u-0697aba1f475","member":"FSD50K.dev_audio/125560.wav","disk":1,"offset":928115107,"compressed_bytes":649715,"size":757386,"method":8,"crc32":3383530612,"sha256":"cfdcb73a0f19a6f5cfe7833068824a5e4880093304a6dc684512da00c5b141df"},{"fname":"141450","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"qlevihirs","group_id":"fsd50k:u-9ef33e089b6e","member":"FSD50K.dev_audio/141450.wav","disk":2,"offset":812481389,"compressed_bytes":370874,"size":462088,"method":8,"crc32":238217702,"sha256":"898736e98020f006cf4b055d94de87f762eecfe5ba4f0ee7ffd8f9c4ad6ec2c0"},{"fname":"143658","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"qubodup","group_id":"fsd50k:u-2afd5d535ce9","member":"FSD50K.dev_audio/143658.wav","disk":4,"offset":148595584,"compressed_bytes":150856,"size":153142,"method":8,"crc32":2487706435,"sha256":"0b7d89d8acb0577faa62306d95c09f43ca2d930e50c33f472c907afb9f86230a"},{"fname":"152657","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"alexftw123","group_id":"fsd50k:u-fc70effaa49b","member":"FSD50K.dev_audio/152657.wav","disk":1,"offset":2312850398,"compressed_bytes":327539,"size":573378,"method":8,"crc32":1057369940,"sha256":"699915d3981937e7720965b6e5f56a74f9a2b4d43131f1c1feb3524d7f7bd5f6"},{"fname":"162188","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"tsidilin","group_id":"fsd50k:u-f43b4d7e339f","member":"FSD50K.dev_audio/162188.wav","disk":4,"offset":3206013940,"compressed_bytes":521733,"size":583556,"method":8,"crc32":2671927058,"sha256":"92612430fa2b927410693165492de571796873b68fe2bb5d3c27f0709bf3ee73"},{"fname":"162500","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Chippy569","group_id":"fsd50k:u-49b03e0ea46a","member":"FSD50K.dev_audio/162500.wav","disk":2,"offset":1252182487,"compressed_bytes":194135,"size":209554,"method":8,"crc32":1521316188,"sha256":"0c2324b0787a5debdb6e88e513e77383fd17cbc99315e7d75bf12c3737242258"},{"fname":"164306","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"thegreenguerilla","group_id":"fsd50k:u-167bb4585525","member":"FSD50K.dev_audio/164306.wav","disk":0,"offset":1390843954,"compressed_bytes":536627,"size":582990,"method":8,"crc32":3369849471,"sha256":"58221c938a4c190ab47f8cf6308709bd0ff5b3c7ee0812613290e984e9319c39"},{"fname":"165644","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"pishpirika","group_id":"fsd50k:u-76b36aff4078","member":"FSD50K.dev_audio/165644.wav","disk":0,"offset":2276659134,"compressed_bytes":528987,"size":588552,"method":8,"crc32":1886543731,"sha256":"f97f4dcef1fd18021da26ae4707abcdaea11c7e7ef31267c6be10d58c2f959a1"},{"fname":"166717","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Bradovic","group_id":"fsd50k:u-a9b65c1d7f00","member":"FSD50K.dev_audio/166717.wav","disk":2,"offset":2887016802,"compressed_bytes":1028889,"size":1284412,"method":8,"crc32":2175674329,"sha256":"193dedfa408a300b0f4916f25697ac257e5df5bc3d1100f33c8297d8368aaaf6"},{"fname":"167485","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Zbyhek","group_id":"fsd50k:u-23a4e24a78b2","member":"FSD50K.dev_audio/167485.wav","disk":5,"offset":1739645895,"compressed_bytes":528647,"size":697520,"method":8,"crc32":3174764650,"sha256":"d3bcd756372702af0aac5f11b3f5eaaae34ec43183290f3820d329b7867494ec"},{"fname":"167874","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"gmarchisio","group_id":"fsd50k:u-10e42b4c5bd7","member":"FSD50K.dev_audio/167874.wav","disk":0,"offset":193157557,"compressed_bytes":1166600,"size":1339584,"method":8,"crc32":3572733788,"sha256":"f0978c46efdff6a5729526c70a13f6587b50483b319d985731aca48b1ed4a51d"},{"fname":"176641","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"jorickhoofd","group_id":"fsd50k:u-7fd6ff4561b2","member":"FSD50K.dev_audio/176641.wav","disk":4,"offset":2052117306,"compressed_bytes":1327689,"size":1565568,"method":8,"crc32":264974249,"sha256":"b13d349d4561ab2b86b7f0c7dc46b87e87bd1fa9b25b3b0bd803dc07cb67dfbe"},{"fname":"191174","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"cahen","group_id":"fsd50k:u-b10d88c3d9ba","member":"FSD50K.dev_audio/191174.wav","disk":0,"offset":1140209945,"compressed_bytes":112249,"size":136034,"method":8,"crc32":2647957534,"sha256":"bbbbc3e2af8ee2df5d05b8810d382b529fead39fc646036f3de51661a476d412"},{"fname":"211187","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"qubodup","group_id":"fsd50k:u-2afd5d535ce9","member":"FSD50K.dev_audio/211187.wav","disk":2,"offset":571514551,"compressed_bytes":389790,"size":441078,"method":8,"crc32":3722350744,"sha256":"2346cd7f78c85c24e5305c342aa42edc252f075cbd2239b579d5159ae6bd09c4"},{"fname":"250062","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"kwahmah_02","group_id":"fsd50k:u-249cb9886263","member":"FSD50K.dev_audio/250062.wav","disk":5,"offset":2183269386,"compressed_bytes":497222,"size":591104,"method":8,"crc32":17792254,"sha256":"e49cbf91a7ae78dee3475c557f9660ee239707add93f114c303efb90cbc4af65"},{"fname":"254714","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"GreekIrish","group_id":"fsd50k:u-e9b356149232","member":"FSD50K.dev_audio/254714.wav","disk":0,"offset":2844016370,"compressed_bytes":1895956,"size":2044792,"method":8,"crc32":2550080264,"sha256":"63d151ab29a0207b29ff8037dec601d65c53d1138309a712960acddc8cd1f35c"},{"fname":"256529","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"tymorafarr","group_id":"fsd50k:u-647950e3f971","member":"FSD50K.dev_audio/256529.wav","disk":0,"offset":239407435,"compressed_bytes":889152,"size":1014116,"method":8,"crc32":2081477291,"sha256":"3e24dbde247ffa7e347a8a3eb6dc32602cbb2eba68eb69e03d7f753ea74c2007"},{"fname":"257028","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Jagadamba","group_id":"fsd50k:u-977f555ef13b","member":"FSD50K.dev_audio/257028.wav","disk":1,"offset":1417679907,"compressed_bytes":597022,"size":808336,"method":8,"crc32":4221707566,"sha256":"7b720280948d48d75c70bb1916400dbfd2e115508829e3e48912968ace455344"},{"fname":"266919","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"kwahmah_02","group_id":"fsd50k:u-249cb9886263","member":"FSD50K.dev_audio/266919.wav","disk":5,"offset":1188570411,"compressed_bytes":82579,"size":89910,"method":8,"crc32":713511467,"sha256":"516aa0e295f81e7719f5fe1ee5083f43a7ff09a296c879af9d80f02594974f79"},{"fname":"280204","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"memuse","group_id":"fsd50k:u-01c58d34fc5f","member":"FSD50K.dev_audio/280204.wav","disk":2,"offset":1037737448,"compressed_bytes":1134477,"size":1648848,"method":8,"crc32":1731524056,"sha256":"98166c128bc0671c747627dc985c21541ccd90b9f1a3cfd88bb2fa362309e820"},{"fname":"319152","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"cmorris035","group_id":"fsd50k:u-4755c084e6c2","member":"FSD50K.dev_audio/319152.wav","disk":5,"offset":1487611726,"compressed_bytes":1998160,"size":2128168,"method":8,"crc32":2191234729,"sha256":"4184c5553add3577a5a7bdfca750b0c1696349eb97cde32fd4d13dd644c7d341"},{"fname":"322946","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"salnier","group_id":"fsd50k:u-ccb9686f5bcc","member":"FSD50K.dev_audio/322946.wav","disk":3,"offset":848215470,"compressed_bytes":997477,"size":1379674,"method":8,"crc32":3390495079,"sha256":"0ff9e1e2f5a09c9bbbc24f730c892b7add39341437e96259968801b2d38ba17c"},{"fname":"325808","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"soundjoao","group_id":"fsd50k:u-60172c186b47","member":"FSD50K.dev_audio/325808.wav","disk":2,"offset":2911387581,"compressed_bytes":175628,"size":220578,"method":8,"crc32":4179297585,"sha256":"97195806b0d7f71b1741ccc27afde60bd3e7239ebca1befcb8124b945f0f5d7e"},{"fname":"328224","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"danshousestudio","group_id":"fsd50k:u-458f92a673d0","member":"FSD50K.dev_audio/328224.wav","disk":0,"offset":1292562623,"compressed_bytes":460292,"size":609204,"method":8,"crc32":1370829717,"sha256":"bb75234382b7301f2d9036ad5556eab9a8116e8b934d3988c2f378145f3143c3"},{"fname":"336740","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"steshystesh","group_id":"fsd50k:u-74c3b9d4c539","member":"FSD50K.dev_audio/336740.wav","disk":1,"offset":1054219752,"compressed_bytes":461917,"size":494840,"method":8,"crc32":3550150103,"sha256":"b0cd6a9ae3b71a1e8f02312e4cb3537bb5e3346cae277d9118fe91fdb4cfcf19"},{"fname":"340623","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"mickyman5000","group_id":"fsd50k:u-d762973b7952","member":"FSD50K.dev_audio/340623.wav","disk":2,"offset":2680952579,"compressed_bytes":179416,"size":194658,"method":8,"crc32":2972691174,"sha256":"9cf237d6695c0198d5192b0df07266bb1a507f2f61802f3b00f483e7686f6f42"},{"fname":"340700","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"mickyman5000","group_id":"fsd50k:u-d762973b7952","member":"FSD50K.dev_audio/340700.wav","disk":2,"offset":819822481,"compressed_bytes":389391,"size":441148,"method":8,"crc32":3679084007,"sha256":"bc32ab1524bc54efaed45e0641e49fe4b68eaa8d2efac5d2880564152e461de5"},{"fname":"343369","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"ccdff2","group_id":"fsd50k:u-3de0aae797d5","member":"FSD50K.dev_audio/343369.wav","disk":4,"offset":1617592600,"compressed_bytes":1589828,"size":1834820,"method":8,"crc32":2553915513,"sha256":"007f2fd60e19ce66d9f8a0e87018a7754515fd53cf96c30cccdb4f0e5e30a247"},{"fname":"345337","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"willybilly1984","group_id":"fsd50k:u-e2e4c68e7bdf","member":"FSD50K.dev_audio/345337.wav","disk":2,"offset":1310482388,"compressed_bytes":576020,"size":819420,"method":8,"crc32":711997046,"sha256":"06da2d46b8a97f041c3768eebf94fec899bd18112def6a1d93393aadee466570"},{"fname":"345357","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"iut_Paris8","group_id":"fsd50k:u-e3399b16301a","member":"FSD50K.dev_audio/345357.wav","disk":4,"offset":448024744,"compressed_bytes":376141,"size":417870,"method":8,"crc32":2588706467,"sha256":"7d5fbf2dae9bef15af97ad2ae668846c4166df4f64d4d236141e1eeafc4ff64c"},{"fname":"345761","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"djtiii","group_id":"fsd50k:u-53df539c86c9","member":"FSD50K.dev_audio/345761.wav","disk":4,"offset":1292667480,"compressed_bytes":1395094,"size":1806854,"method":8,"crc32":141510525,"sha256":"06292606a61ad6ba4645c194de3f51150090fb3906a6955db67eb9d4972d53b5"},{"fname":"361403","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"richwise","group_id":"fsd50k:u-e4ab9481470e","member":"FSD50K.dev_audio/361403.wav","disk":4,"offset":2441676337,"compressed_bytes":913170,"size":1551300,"method":8,"crc32":3622261445,"sha256":"f3e2e02fe7d355576250e93a093a3ddf7a85f49d520e7f82fcf8042ade23ed9a"},{"fname":"385929","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Pól","group_id":"fsd50k:u-d7c1ea02d3f5","member":"FSD50K.dev_audio/385929.wav","disk":4,"offset":2906568843,"compressed_bytes":665618,"size":705678,"method":8,"crc32":1411617161,"sha256":"e73af118e3d53511d1881b491519b10e62f6a75506dff2e8b5a19ec1573101a0"},{"fname":"387674","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Jagadamba","group_id":"fsd50k:u-977f555ef13b","member":"FSD50K.dev_audio/387674.wav","disk":0,"offset":3077455239,"compressed_bytes":1761224,"size":1903278,"method":8,"crc32":2357999907,"sha256":"d7d9c5b53143dd66114dd2382877b4c91bae1792950e4f55bc99ffa558322bde"},{"fname":"387814","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Jagadamba","group_id":"fsd50k:u-977f555ef13b","member":"FSD50K.dev_audio/387814.wav","disk":0,"offset":1967949295,"compressed_bytes":2089921,"size":2305158,"method":8,"crc32":1311147263,"sha256":"a78796d258f5dd5788e144d5fb525eae739712fbe37072abf9ff0010623dd440"},{"fname":"398675","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Dominik_W","group_id":"fsd50k:u-f425d94343b7","member":"FSD50K.dev_audio/398675.wav","disk":0,"offset":746777479,"compressed_bytes":659056,"size":844688,"method":8,"crc32":1063298542,"sha256":"3ad87cf7fb8d7e1bd4c7f9d796303c664ac0bf29447d63e5bd2b493da9f791b9"},{"fname":"413904","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"kyles","group_id":"fsd50k:u-2bef0e7cc443","member":"FSD50K.dev_audio/413904.wav","disk":1,"offset":2283235033,"compressed_bytes":797622,"size":1114718,"method":8,"crc32":2444583476,"sha256":"78918108dd85f863e252a963ef0f58612828c43e0ce9348ebb8534bae1d74d58"},{"fname":"423607","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Mativve","group_id":"fsd50k:u-a1e939cf2f7a","member":"FSD50K.dev_audio/423607.wav","disk":1,"offset":1918055009,"compressed_bytes":97896,"size":131590,"method":8,"crc32":1735026953,"sha256":"ddb0594a97fa227bf166fb5e1ec387b49cbfc92d9e476479098a28384d6dfae9"},{"fname":"425861","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Soundholder","group_id":"fsd50k:u-137d123a1098","member":"FSD50K.dev_audio/425861.wav","disk":0,"offset":2291395811,"compressed_bytes":863164,"size":1065064,"method":8,"crc32":1845252533,"sha256":"0af887d39b22bc097f20bf2b7c0e6b67ca826479cf72bbf852db2502fbba4de7"},{"fname":"432304","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"richwise","group_id":"fsd50k:u-e4ab9481470e","member":"FSD50K.dev_audio/432304.wav","disk":5,"offset":1254876918,"compressed_bytes":778585,"size":947182,"method":8,"crc32":3924070604,"sha256":"6df83303ac4e66ed1fb21fbb7fe6794fb7fdaae191903b4ca1029c1fc1d1bccf"},{"fname":"433662","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"namikiri","group_id":"fsd50k:u-b59b46c9f7bb","member":"FSD50K.dev_audio/433662.wav","disk":0,"offset":942210832,"compressed_bytes":484959,"size":523562,"method":8,"crc32":3908699726,"sha256":"2a8ee01b59eae284582c3de8b9bee5e8721f5e91b198aa28ae956423ea7644a3"},{"fname":"433866","split":"dev","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"jodybruchon","group_id":"fsd50k:u-e6bb2503d834","member":"FSD50K.dev_audio/433866.wav","disk":2,"offset":3112581066,"compressed_bytes":774427,"size":1067340,"method":8,"crc32":3997735303,"sha256":"15e984e2abc348f240d06c13a1d2960f64242567d7854e1cb056bea9ae4aab04"},{"fname":"9197","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Heigh-hoo","group_id":"fsd50k:u-f27da747f3d4","member":"FSD50K.eval_audio/9197.wav","disk":0,"offset":567845461,"compressed_bytes":1521785,"size":1665244,"method":8,"crc32":1366767746,"sha256":"74e3561708b03723d9a81ab989aa397841eaf167685071066bc995f56b54585f"},{"fname":"14828","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Heigh-hoo","group_id":"fsd50k:u-f27da747f3d4","member":"FSD50K.eval_audio/14828.wav","disk":1,"offset":2659325888,"compressed_bytes":374152,"size":478712,"method":8,"crc32":4035348967,"sha256":"e79683ceac92c1164a748d382d5173ce6b52b043d7339e8bee7bce26b9544aca"},{"fname":"23519","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Percy Duke","group_id":"fsd50k:u-3573b330b1bd","member":"FSD50K.eval_audio/23519.wav","disk":1,"offset":42453771,"compressed_bytes":874504,"size":970062,"method":8,"crc32":1019179303,"sha256":"df6abfda3252acac632971af288833f23346c2be09e4a55cddb507ea6af70904"},{"fname":"38684","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"metamorphmuses","group_id":"fsd50k:u-870facd48411","member":"FSD50K.eval_audio/38684.wav","disk":1,"offset":998020129,"compressed_bytes":101115,"size":111696,"method":8,"crc32":2531573887,"sha256":"a0adce35706fd32ac41a33ed6bcf8f980c3d5cfbb8b2a446521ed019392f5317"},{"fname":"51490","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Heigh-hoo","group_id":"fsd50k:u-f27da747f3d4","member":"FSD50K.eval_audio/51490.wav","disk":0,"offset":1569147801,"compressed_bytes":1334997,"size":1563606,"method":8,"crc32":2462310796,"sha256":"09295ffef5ceb5abafa6bdde0cabcedeca6a8085751a62ce823b251a5983400d"},{"fname":"55731","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"BoilingSand","group_id":"fsd50k:u-670093af6c4f","member":"FSD50K.eval_audio/55731.wav","disk":0,"offset":342006289,"compressed_bytes":1382191,"size":1536016,"method":8,"crc32":3401021215,"sha256":"2fdf048a8454f959f1184a55ce153b17afcdf62fd3671352cc3ac3972d46c9ec"},{"fname":"59587","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"morgantj","group_id":"fsd50k:u-362d7392b44f","member":"FSD50K.eval_audio/59587.wav","disk":1,"offset":203285295,"compressed_bytes":1730565,"size":1859406,"method":8,"crc32":1609305911,"sha256":"4e206dfef69c0c3d4e2b473cd6cfece42f6ff30545741892f336a14c8eb28950"},{"fname":"98053","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Tomlija","group_id":"fsd50k:u-868e7641dc2e","member":"FSD50K.eval_audio/98053.wav","disk":0,"offset":817511887,"compressed_bytes":1162498,"size":1330564,"method":8,"crc32":4044437795,"sha256":"f5cb9de2ee4dda1cf7dbd586056f318ce36fd498ffe3f703321f467c817d951f"},{"fname":"153578","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Thejack288","group_id":"fsd50k:u-153c381168de","member":"FSD50K.eval_audio/153578.wav","disk":1,"offset":1705398628,"compressed_bytes":1045245,"size":1912398,"method":8,"crc32":3555079127,"sha256":"7b7bd633b5f5229a3e5b17e9f4469de10682ebe22567d00102ac328e74264deb"},{"fname":"156637","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"lonemonk","group_id":"fsd50k:u-1f233999fe33","member":"FSD50K.eval_audio/156637.wav","disk":0,"offset":2040460019,"compressed_bytes":742263,"size":971118,"method":8,"crc32":510014318,"sha256":"e1f751dfa5340c1c34ca4c7ef83b55fcae610bc78c9d9131fb70bc0c10ecc8b0"},{"fname":"171447","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"hinzebeat","group_id":"fsd50k:u-96a00a32d1f5","member":"FSD50K.eval_audio/171447.wav","disk":0,"offset":251045519,"compressed_bytes":393158,"size":477130,"method":8,"crc32":3134115043,"sha256":"31e2941f8e4c8023eb95815aa93a42193369c3784c40f2c78cedd6565e0d6ca3"},{"fname":"185580","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"lonemonk","group_id":"fsd50k:u-1f233999fe33","member":"FSD50K.eval_audio/185580.wav","disk":0,"offset":990302093,"compressed_bytes":410605,"size":599744,"method":8,"crc32":2531239024,"sha256":"93a2a6930f3883ec2bf4c6d3c2711a5bfba86ee255fafa4bca27fbece062df1b"},{"fname":"188004","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Motion_S","group_id":"fsd50k:u-6978dc0f7a37","member":"FSD50K.eval_audio/188004.wav","disk":0,"offset":325616288,"compressed_bytes":985612,"size":1262670,"method":8,"crc32":2186391064,"sha256":"78ad3d01c71687e5c0fcbfa2dfef34a9663635eca01beee7ab7e2863f9e0ae87"},{"fname":"194950","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"soundmary","group_id":"fsd50k:u-51f62a00c5d6","member":"FSD50K.eval_audio/194950.wav","disk":0,"offset":281537955,"compressed_bytes":734638,"size":946288,"method":8,"crc32":3456838491,"sha256":"ba6db82fde967774116724f3dd9cf36fc4a6366b69a3a0da673f75acaf42ab1f"},{"fname":"194968","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"soundmary","group_id":"fsd50k:u-51f62a00c5d6","member":"FSD50K.eval_audio/194968.wav","disk":0,"offset":1395983046,"compressed_bytes":92157,"size":122854,"method":8,"crc32":1196903949,"sha256":"8262361c9065d7eaf99a862d8dac797ecbf3a872e8dffb7af204f2b95cda6bed"},{"fname":"196688","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"soundmary","group_id":"fsd50k:u-51f62a00c5d6","member":"FSD50K.eval_audio/196688.wav","disk":0,"offset":747986574,"compressed_bytes":1880803,"size":2444982,"method":8,"crc32":676594371,"sha256":"e571062f2fd1398502f670d3aaecd530f8b05b4aee2c86014283acb535a7cd68"},{"fname":"197106","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"musicradiocreative","group_id":"fsd50k:u-727bf8aa1885","member":"FSD50K.eval_audio/197106.wav","disk":0,"offset":3145332421,"compressed_bytes":1567075,"size":2015190,"method":8,"crc32":156773102,"sha256":"6e2b02b03f19b728056dad64ac9b11109d84fd2878c1d599812556523060a35a"},{"fname":"197346","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"nicStage","group_id":"fsd50k:u-5321096e69f0","member":"FSD50K.eval_audio/197346.wav","disk":1,"offset":622739789,"compressed_bytes":1119798,"size":1411278,"method":8,"crc32":3313005059,"sha256":"0b05fb2ef3c547c89f28d5dbef2196017e9c8b9d7bb771f772178a7eb7c5ba39"},{"fname":"202985","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Tomlija","group_id":"fsd50k:u-868e7641dc2e","member":"FSD50K.eval_audio/202985.wav","disk":1,"offset":1213560978,"compressed_bytes":638734,"size":829158,"method":8,"crc32":359777391,"sha256":"69b673d15280b558bf03e52ce397408d4543aa73ed5fd2c1be1851775b4e6bb4"},{"fname":"207119","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"ahill86","group_id":"fsd50k:u-66297ec0a7fb","member":"FSD50K.eval_audio/207119.wav","disk":0,"offset":450992741,"compressed_bytes":1531333,"size":1605906,"method":8,"crc32":3483096964,"sha256":"a93ea91ceca6ab72b4d65107ba29a21c452d3d0caf6e2c05c984b736108c1c95"},{"fname":"221665","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"daveincamas","group_id":"fsd50k:u-5b6619acc4fd","member":"FSD50K.eval_audio/221665.wav","disk":1,"offset":1453232160,"compressed_bytes":1552085,"size":1852278,"method":8,"crc32":2758011311,"sha256":"feb64efa2b6c0ee7c6d41ded708d547c1e7ec539cd137cc4e63792ec7d06983c"},{"fname":"231960","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"INS_Ribera_Baixa","group_id":"fsd50k:u-ab6afc9c87c6","member":"FSD50K.eval_audio/231960.wav","disk":0,"offset":610866055,"compressed_bytes":2188983,"size":2510152,"method":8,"crc32":1400859711,"sha256":"a379fc72384c0d2591bcb28746728e0b523cb8b04492a90bb48dc866d73a2e1d"},{"fname":"235559","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"simosco","group_id":"fsd50k:u-1a648e7cbf6e","member":"FSD50K.eval_audio/235559.wav","disk":1,"offset":1714602040,"compressed_bytes":1247189,"size":1518966,"method":8,"crc32":2350683827,"sha256":"f5b88f6b0c399077c4d6781a6342a1af24d080ac553c7787f6fbb572ff2c6bc2"},{"fname":"235957","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"mffm","group_id":"fsd50k:u-20e76c6b638a","member":"FSD50K.eval_audio/235957.wav","disk":0,"offset":1332622902,"compressed_bytes":1125482,"size":1503182,"method":8,"crc32":2432383137,"sha256":"b1983a5b802dd6d7f8d90756e0b40e4ba06bfba3a64078955d5a614ea009c062"},{"fname":"244503","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Silentwaves","group_id":"fsd50k:u-d26934e5359d","member":"FSD50K.eval_audio/244503.wav","disk":1,"offset":1099978746,"compressed_bytes":1808191,"size":2646078,"method":8,"crc32":393473177,"sha256":"dad427821e1d54c98610574a7c42bf830c273432d31f011b209f2becb858180b"},{"fname":"325844","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Daveshu88","group_id":"fsd50k:u-6e1c143caf25","member":"FSD50K.eval_audio/325844.wav","disk":1,"offset":2158140107,"compressed_bytes":854195,"size":930988,"method":8,"crc32":2061381369,"sha256":"95b7d756d67b934a7725dc64dcae06eea24cebc88bacee097c32b841ffc58a6a"},{"fname":"327900","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"cartoonrob","group_id":"fsd50k:u-af59f0f90eb2","member":"FSD50K.eval_audio/327900.wav","disk":0,"offset":1543763215,"compressed_bytes":526904,"size":712794,"method":8,"crc32":3975462472,"sha256":"e95f61cc95458f615c0f88ff961f991467a2acc081e983c93b584c7d0c74d44f"},{"fname":"348857","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"MickBoere","group_id":"fsd50k:u-39062b09a50d","member":"FSD50K.eval_audio/348857.wav","disk":0,"offset":999214780,"compressed_bytes":269286,"size":274512,"method":8,"crc32":559735929,"sha256":"1afb8b41acf2998206f050c651f45c02a90eee43f681e80d7da72ad796f8ed1e"},{"fname":"360728","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"postworkflow","group_id":"fsd50k:u-f80e23188f50","member":"FSD50K.eval_audio/360728.wav","disk":1,"offset":2582713045,"compressed_bytes":1730858,"size":2311250,"method":8,"crc32":4262391106,"sha256":"ccfe042158ca07f5e49a71c5cea6dd89bdc700604268739d34e41e2c36758198"},{"fname":"378704","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"13FPanska_Novotny_Stepan","group_id":"fsd50k:u-6eb264582204","member":"FSD50K.eval_audio/378704.wav","disk":0,"offset":709012639,"compressed_bytes":2220445,"size":2581776,"method":8,"crc32":888091780,"sha256":"393bd2eeec927a711cc927d63e8a5a44afcfb12721dfeed94fd9162e8cb4c790"},{"fname":"392258","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Viliguis","group_id":"fsd50k:u-8475c86f8463","member":"FSD50K.eval_audio/392258.wav","disk":0,"offset":1435674831,"compressed_bytes":999156,"size":1146958,"method":8,"crc32":604886601,"sha256":"4c41de63fa09487d25ea8c6506889d13596079a2892ecee601e5b7520bab6613"},{"fname":"393144","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"mikerie","group_id":"fsd50k:u-fc9f32ccc069","member":"FSD50K.eval_audio/393144.wav","disk":0,"offset":3053672912,"compressed_bytes":187947,"size":251990,"method":8,"crc32":882195722,"sha256":"e3872b6b4139da303d378b6bbf73a7fd2e21793ed54323afec4febe0361ab655"},{"fname":"398416","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"bone666138","group_id":"fsd50k:u-9c573414ebf1","member":"FSD50K.eval_audio/398416.wav","disk":1,"offset":2784930338,"compressed_bytes":644554,"size":1190892,"method":8,"crc32":1481323653,"sha256":"97767709a13e91ac0ec97d397beeecd8b674f79d3c9d9f3ec3bb565706998e63"},{"fname":"417230","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"TheLukasBanana","group_id":"fsd50k:u-2bc14d483805","member":"FSD50K.eval_audio/417230.wav","disk":1,"offset":1978443505,"compressed_bytes":298624,"size":358478,"method":8,"crc32":1680356611,"sha256":"0c097cc25e937ff840370b33a046a9a361e2c3fc5d008a7c266dcfe21559e4ef"},{"fname":"426561","split":"eval","target_label":"Engine","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"cgoulao","group_id":"fsd50k:u-b6e61b224394","member":"FSD50K.eval_audio/426561.wav","disk":0,"offset":635141438,"compressed_bytes":1174477,"size":1244444,"method":8,"crc32":1098138510,"sha256":"ab18ac653603a1ff98b7d985600e3790efa92f11e291955d82708c1fb818d4f5"},{"fname":"18603","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/18603.wav","disk":5,"offset":534490525,"compressed_bytes":1841903,"size":2205078,"method":8,"crc32":2999478129,"sha256":"0f273abf60dc5455cb46ec69df7918dab712a3b5c4b39386258e6c75a461ed5c"},{"fname":"28451","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Corsica_S","group_id":"fsd50k:u-b8d46530f98a","member":"FSD50K.dev_audio/28451.wav","disk":2,"offset":1615964224,"compressed_bytes":1098645,"size":1771290,"method":8,"crc32":1914790373,"sha256":"bf7b383e0fc8b695b8136eccf2854b9696065bc2eb38f63be9a7cbf1601feaa2"},{"fname":"28452","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Corsica_S","group_id":"fsd50k:u-b8d46530f98a","member":"FSD50K.dev_audio/28452.wav","disk":1,"offset":610071019,"compressed_bytes":1351859,"size":1996966,"method":8,"crc32":2542577346,"sha256":"15d250af19ec9f93786de1ef5475583eaaf6cfdc8b9414d2c110a493fb69cc0d"},{"fname":"28453","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Corsica_S","group_id":"fsd50k:u-b8d46530f98a","member":"FSD50K.dev_audio/28453.wav","disk":5,"offset":476218645,"compressed_bytes":1347574,"size":2007880,"method":8,"crc32":827364128,"sha256":"cadd66f3cebf976f9eeb55dcf85a9222c412029ee41ee08c81c77f3f034e6075"},{"fname":"31447","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"hello_flowers","group_id":"fsd50k:u-50ed42df4aac","member":"FSD50K.dev_audio/31447.wav","disk":5,"offset":1861273480,"compressed_bytes":1525761,"size":1601920,"method":8,"crc32":2912457983,"sha256":"fc88433d722d1f00a6388d30cacf0de3a9336f89440d653d0a828275233ed858"},{"fname":"32402","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"digifishmusic","group_id":"fsd50k:u-23878ad72831","member":"FSD50K.dev_audio/32402.wav","disk":3,"offset":485297841,"compressed_bytes":1526919,"size":1827150,"method":8,"crc32":1056200480,"sha256":"094706aacf4756eacb0dc09ab9cbcf383a6e1008838d5309045631780957fff9"},{"fname":"39984","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/39984.wav","disk":2,"offset":1248201854,"compressed_bytes":931257,"size":1194574,"method":8,"crc32":2502024634,"sha256":"e0ac57dee9c14a2012bc0e48fbbd1978a859866177ac0f427f42b58693bb19d6"},{"fname":"39985","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/39985.wav","disk":4,"offset":1390483457,"compressed_bytes":1232460,"size":1574326,"method":8,"crc32":1681065333,"sha256":"0cb5c9f80e3047b8f21de633836878e15a0819dd464e6de625fd63db3d6ea0f6"},{"fname":"57325","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"digifishmusic","group_id":"fsd50k:u-23878ad72831","member":"FSD50K.dev_audio/57325.wav","disk":4,"offset":504512767,"compressed_bytes":1821578,"size":2048334,"method":8,"crc32":133962113,"sha256":"75cc9ca0c5890b4700781785cbe11d014f7e235e58603b3fd266eb0c71d55852"},{"fname":"73288","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"NoiseCollector","group_id":"fsd50k:u-0697aba1f475","member":"FSD50K.dev_audio/73288.wav","disk":0,"offset":1522451276,"compressed_bytes":1395216,"size":1608270,"method":8,"crc32":419525099,"sha256":"4491336e632bc7430ab4ec81972e269514281463370d6f1e0dd5434875025000"},{"fname":"73290","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"NoiseCollector","group_id":"fsd50k:u-0697aba1f475","member":"FSD50K.dev_audio/73290.wav","disk":4,"offset":2012481521,"compressed_bytes":1801602,"size":2331726,"method":8,"crc32":228004508,"sha256":"2ad48a24d32a3c892fc04fcebcbadb7d35f88df9b4a116e7e607850c5c9e6c2a"},{"fname":"96210","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"CGEffex","group_id":"fsd50k:u-4f7ab92b7591","member":"FSD50K.dev_audio/96210.wav","disk":0,"offset":2110689898,"compressed_bytes":590462,"size":623800,"method":8,"crc32":3510053395,"sha256":"98c2c87df65d65ad2b24daec9057276b90e8ddb0b377cd9b08501984c3858c46"},{"fname":"100155","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"NoiseCollector","group_id":"fsd50k:u-0697aba1f475","member":"FSD50K.dev_audio/100155.wav","disk":0,"offset":1097401678,"compressed_bytes":1690452,"size":1741902,"method":8,"crc32":2381083367,"sha256":"a22399e3b475c69c490e0d30e68c1a6eb1a01159a76d310ff396f23816f9b446"},{"fname":"140589","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"LG","group_id":"fsd50k:u-b870b7809747","member":"FSD50K.dev_audio/140589.wav","disk":4,"offset":1008190172,"compressed_bytes":211015,"size":267868,"method":8,"crc32":901949800,"sha256":"d59a934e856b396a9e47e8e2d68fe97242d045388f768bfe3c9ddaf903607401"},{"fname":"140590","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"LG","group_id":"fsd50k:u-b870b7809747","member":"FSD50K.dev_audio/140590.wav","disk":5,"offset":1453105038,"compressed_bytes":1140580,"size":1356284,"method":8,"crc32":3665570871,"sha256":"2c1eb91533f47b6debbc10454b5740d59b3ba9fc3e016d6a75cb6d22b64dc699"},{"fname":"140591","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"LG","group_id":"fsd50k:u-b870b7809747","member":"FSD50K.dev_audio/140591.wav","disk":0,"offset":2895981450,"compressed_bytes":1077389,"size":1356284,"method":8,"crc32":2553414150,"sha256":"49f3d63320c6a9e3a45d5069a94887bd6b99dcdf24590f2dafaaaaf3b9c2eb5a"},{"fname":"143558","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Fight2FlyPhoto","group_id":"fsd50k:u-5f673f5a2b54","member":"FSD50K.dev_audio/143558.wav","disk":0,"offset":26949687,"compressed_bytes":875618,"size":1027164,"method":8,"crc32":2751542571,"sha256":"00dce79c0376afdf1f1f6b79fd95514223c9f58db55a4413c943d0e9eaf305d4"},{"fname":"143560","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Fight2FlyPhoto","group_id":"fsd50k:u-5f673f5a2b54","member":"FSD50K.dev_audio/143560.wav","disk":5,"offset":1930960036,"compressed_bytes":779965,"size":1021278,"method":8,"crc32":3037582993,"sha256":"d6c86595417d9bb3483e12f4c1ade30dc2aee20d25e11a04b73d8285c4fa5f55"},{"fname":"176805","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"mickmon","group_id":"fsd50k:u-e25d58d24399","member":"FSD50K.dev_audio/176805.wav","disk":2,"offset":1945313804,"compressed_bytes":197622,"size":204062,"method":8,"crc32":3500315971,"sha256":"2904ebf87cfbf57e4ef335ec8612bc3df10de8edb2915fd3459d33174172bf49"},{"fname":"187734","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"qubodup","group_id":"fsd50k:u-2afd5d535ce9","member":"FSD50K.dev_audio/187734.wav","disk":1,"offset":1899520982,"compressed_bytes":1683648,"size":1722602,"method":8,"crc32":48282544,"sha256":"e6236cf58b512ddba496d2b6d4f6875a35efda6b2bf26d4d796120208f3c3510"},{"fname":"189644","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"qubodup","group_id":"fsd50k:u-2afd5d535ce9","member":"FSD50K.dev_audio/189644.wav","disk":0,"offset":2978545309,"compressed_bytes":500923,"size":510556,"method":8,"crc32":1049809166,"sha256":"876491714ed66c45f54a8e8ad896b9d1d7bd3d602341d40654bd2f28ca617428"},{"fname":"194786","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"flcellogrl","group_id":"fsd50k:u-0498d86ff642","member":"FSD50K.dev_audio/194786.wav","disk":0,"offset":727799308,"compressed_bytes":1707303,"size":1744028,"method":8,"crc32":947549557,"sha256":"49443e7981a9878510a44a9beea1a47bc93add5bc9fcd6c735b7e99a2e2a8b9e"},{"fname":"197248","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"qubodup","group_id":"fsd50k:u-2afd5d535ce9","member":"FSD50K.dev_audio/197248.wav","disk":2,"offset":2462429552,"compressed_bytes":903534,"size":1058478,"method":8,"crc32":3251712796,"sha256":"171ac2bd76cd82c362ac6733b921efc0f0c9ff4e8d378412929757755222abf7"},{"fname":"207247","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"AurelioSons","group_id":"fsd50k:u-5d871bb2cd0e","member":"FSD50K.dev_audio/207247.wav","disk":1,"offset":2365567910,"compressed_bytes":716274,"size":817290,"method":8,"crc32":254472148,"sha256":"d330188a07df3ec3b91293e586df394995f08d2b4bedda74b023797105082968"},{"fname":"207252","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"AurelioSons","group_id":"fsd50k:u-5d871bb2cd0e","member":"FSD50K.dev_audio/207252.wav","disk":2,"offset":1053417504,"compressed_bytes":607219,"size":801390,"method":8,"crc32":2147335289,"sha256":"288ed925236a4d5305438d39d5cde4bc3ea9eb04b040bc218b90c59130b7a0ac"},{"fname":"207456","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"AurelioSons","group_id":"fsd50k:u-5d871bb2cd0e","member":"FSD50K.dev_audio/207456.wav","disk":1,"offset":1737916787,"compressed_bytes":1334040,"size":1458766,"method":8,"crc32":3007253482,"sha256":"5dec7dab6e24070d87008bec464131abde810422dc874febb3111e18e9f391d4"},{"fname":"208861","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Insestanyricarda","group_id":"fsd50k:u-568563aa5b58","member":"FSD50K.dev_audio/208861.wav","disk":2,"offset":880621943,"compressed_bytes":751022,"size":781260,"method":8,"crc32":1106876060,"sha256":"9e21efeb661c5a28c680032b7ceb2bd5895fc3116fb337088f4629f7016f1865"},{"fname":"209434","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Insestanyricarda","group_id":"fsd50k:u-568563aa5b58","member":"FSD50K.dev_audio/209434.wav","disk":3,"offset":301593849,"compressed_bytes":423820,"size":485210,"method":8,"crc32":4190893341,"sha256":"a014e84f8fb47d578dfd21ff064fd2fe2275002161e9dc93557a21289446bac6"},{"fname":"209445","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Insestanyricarda","group_id":"fsd50k:u-568563aa5b58","member":"FSD50K.dev_audio/209445.wav","disk":4,"offset":1224972243,"compressed_bytes":907787,"size":974276,"method":8,"crc32":4092396204,"sha256":"2f1aca36e8ee914954f4349af6522bc48d1574495507cfa10763cf83175a3460"},{"fname":"216150","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"RSilveira_88","group_id":"fsd50k:u-3c8e9518c05a","member":"FSD50K.dev_audio/216150.wav","disk":0,"offset":1202626985,"compressed_bytes":1635611,"size":2202318,"method":8,"crc32":2885477683,"sha256":"cc9609d0f11a8dc97fd72ac6e615960ec839c279226513607a2926fa537564b8"},{"fname":"231934","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"insdali","group_id":"fsd50k:u-fc9af2da2a46","member":"FSD50K.dev_audio/231934.wav","disk":0,"offset":3032786846,"compressed_bytes":637241,"size":793920,"method":8,"crc32":713019327,"sha256":"efa4b5c95b49c41bdfddf15a34de585024084c501388754d0a2e8f2fb150f055"},{"fname":"251957","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"cliftonmcarlson","group_id":"fsd50k:u-661d286debf5","member":"FSD50K.dev_audio/251957.wav","disk":5,"offset":1425152705,"compressed_bytes":1336047,"size":1662838,"method":8,"crc32":3379826067,"sha256":"b7251f13d734638524c4fa7bc9033cba690a8c7eb936768e5431b306a6d1f7f6"},{"fname":"251958","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"cliftonmcarlson","group_id":"fsd50k:u-661d286debf5","member":"FSD50K.dev_audio/251958.wav","disk":2,"offset":65862763,"compressed_bytes":1266173,"size":1574552,"method":8,"crc32":3869254292,"sha256":"9ee5b8b03420fdbb810fff065b3483fd1b24c7e71184a30d7e3e4f86e7f865bd"},{"fname":"251966","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"cliftonmcarlson","group_id":"fsd50k:u-661d286debf5","member":"FSD50K.dev_audio/251966.wav","disk":0,"offset":363932012,"compressed_bytes":1939584,"size":2453756,"method":8,"crc32":3578862321,"sha256":"e29791fd5865abf8e9632085d073dd4d81ed6585db9470f9007013d3e7d82018"},{"fname":"257815","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"xtrgamr","group_id":"fsd50k:u-b139e561c76e","member":"FSD50K.dev_audio/257815.wav","disk":2,"offset":1908009389,"compressed_bytes":414304,"size":433230,"method":8,"crc32":2125828662,"sha256":"35f25b9229be4a86390fc4fbb0263e0ff0e2dc440d5d45ec92ceb794317e055c"},{"fname":"262753","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"bigpickle51","group_id":"fsd50k:u-2091b0470b89","member":"FSD50K.dev_audio/262753.wav","disk":4,"offset":994782278,"compressed_bytes":1374277,"size":2469678,"method":8,"crc32":1116820377,"sha256":"d072e58f22f02dccc701bb233600246b63214a639595c322403f10832561cb51"},{"fname":"321169","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"InspectorJ","group_id":"fsd50k:u-931670491c3a","member":"FSD50K.dev_audio/321169.wav","disk":5,"offset":1600573562,"compressed_bytes":885114,"size":899412,"method":8,"crc32":3874788796,"sha256":"1d873feb31cebb83545ef42654c4b1cb039ad11241938e4946236d2fd50fd8e9"},{"fname":"324228","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Greg Baumont","group_id":"fsd50k:u-0397a1f598df","member":"FSD50K.dev_audio/324228.wav","disk":2,"offset":2389621575,"compressed_bytes":2196985,"size":2474574,"method":8,"crc32":1742705365,"sha256":"791f40357d5110abd021b16b13bf4e043c6a63df662ac8ab4a41e258b7c5ad8c"},{"fname":"324230","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Greg Baumont","group_id":"fsd50k:u-0397a1f598df","member":"FSD50K.dev_audio/324230.wav","disk":2,"offset":1612418433,"compressed_bytes":1730135,"size":2082894,"method":8,"crc32":763214286,"sha256":"0ce5575db5e30c8eb1555283db2f6099cb7785d5b08193b56415c322e67ee211"},{"fname":"341113","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Aiyumi","group_id":"fsd50k:u-416371b6a18a","member":"FSD50K.dev_audio/341113.wav","disk":0,"offset":659797305,"compressed_bytes":1726349,"size":1896758,"method":8,"crc32":3962334283,"sha256":"f529a8e554908f5da3912b23a61d0d3f0aea80df7d810333fd89b6f24d1dc4bb"},{"fname":"382302","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"goingnowhereeatingthings","group_id":"fsd50k:u-e59b87c92c32","member":"FSD50K.dev_audio/382302.wav","disk":2,"offset":2452662343,"compressed_bytes":709372,"size":1117860,"method":8,"crc32":527619785,"sha256":"6d27f1e6c7c7503e67032c4e54ceb610b4a78daf359d80776598f054d8e3e2f0"},{"fname":"414976","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"InspectorJ","group_id":"fsd50k:u-931670491c3a","member":"FSD50K.dev_audio/414976.wav","disk":0,"offset":2591302530,"compressed_bytes":1546666,"size":1822880,"method":8,"crc32":4212057674,"sha256":"01466bbeed05e9599fbfa5f788fef115213dd96fb018a5ef3a5fb3ce76b77a41"},{"fname":"418199","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"ddaedalus","group_id":"fsd50k:u-513fa32baae8","member":"FSD50K.dev_audio/418199.wav","disk":0,"offset":3108023396,"compressed_bytes":908107,"size":951630,"method":8,"crc32":298501263,"sha256":"effd4932afebe12d2f161f7da6394abc04d40a8406caa26f8ac270fc1c93385d"},{"fname":"418200","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"ddaedalus","group_id":"fsd50k:u-513fa32baae8","member":"FSD50K.dev_audio/418200.wav","disk":1,"offset":1726688670,"compressed_bytes":876047,"size":912462,"method":8,"crc32":3600950595,"sha256":"d4b97f96cf8b7ee99be8c8ccbf34aa6592daf276d867a85637389c6856d3e0db"},{"fname":"418385","split":"dev","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Mrthenoronha","group_id":"fsd50k:u-bbec41281fd3","member":"FSD50K.dev_audio/418385.wav","disk":0,"offset":1320253351,"compressed_bytes":2242674,"size":2376366,"method":8,"crc32":727949461,"sha256":"ce7b4acbc46fe75c6b8f9c75dd74ead43d902b95f8720b9fa9df99d31d91b5b9"},{"fname":"35384","split":"eval","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"UncleSigmund","group_id":"fsd50k:u-5151862954fc","member":"FSD50K.eval_audio/35384.wav","disk":1,"offset":2792791820,"compressed_bytes":2260296,"size":2513778,"method":8,"crc32":2892844544,"sha256":"cf33fb7de77f1ca2f551002b5f4177b42bc9b740ed7cd82482062c8f93aa859e"},{"fname":"51491","split":"eval","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Heigh-hoo","group_id":"fsd50k:u-f27da747f3d4","member":"FSD50K.eval_audio/51491.wav","disk":0,"offset":1954429132,"compressed_bytes":880993,"size":1067554,"method":8,"crc32":1087233249,"sha256":"daa38317b34b64e62048a78c7791b878a1dec64897450a88de67d902f4a1b953"},{"fname":"88787","split":"eval","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Josh Herman","group_id":"fsd50k:u-9543ae281a8b","member":"FSD50K.eval_audio/88787.wav","disk":0,"offset":1252430344,"compressed_bytes":1150880,"size":1381436,"method":8,"crc32":2179172329,"sha256":"5bed727ed75e3d2a291594cfef8b181a9d64562377c51ce77dde700f1d8af0c0"},{"fname":"122807","split":"eval","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"geodylabs","group_id":"fsd50k:u-c0a9f590c446","member":"FSD50K.eval_audio/122807.wav","disk":1,"offset":2020945594,"compressed_bytes":2383693,"size":2472270,"method":8,"crc32":2162075255,"sha256":"558200fbfe24c2372c069a2b1b907945b2f7f32346c6e95cddb25b67587c0f13"},{"fname":"185951","split":"eval","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"moonfisher","group_id":"fsd50k:u-ac742ba4a8d7","member":"FSD50K.eval_audio/185951.wav","disk":0,"offset":2067540140,"compressed_bytes":1348360,"size":1738010,"method":8,"crc32":4254124380,"sha256":"257793daeffd3ee62b8778f249ca10ec46c760b607accc95df87ae8f43b9b988"},{"fname":"231956","split":"eval","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"INS_Ribera_Baixa","group_id":"fsd50k:u-ab6afc9c87c6","member":"FSD50K.eval_audio/231956.wav","disk":1,"offset":2867638661,"compressed_bytes":697774,"size":719048,"method":8,"crc32":1722327884,"sha256":"202d2a6c57837dc83fedeb1cb64ce5523e7b5dd669a72c035501d8003473cd5d"},{"fname":"235365","split":"eval","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"3863","group_id":"fsd50k:u-5c3d91e81ae5","member":"FSD50K.eval_audio/235365.wav","disk":1,"offset":856773048,"compressed_bytes":1224748,"size":1391340,"method":8,"crc32":4288623691,"sha256":"db67c024ed59450093337e5a9906a334e95fa16a5f707857f23c2b340751fa0a"},{"fname":"235956","split":"eval","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"mffm","group_id":"fsd50k:u-20e76c6b638a","member":"FSD50K.eval_audio/235956.wav","disk":0,"offset":1840645114,"compressed_bytes":1261125,"size":1435470,"method":8,"crc32":3938234882,"sha256":"b6160ec4a25160f668ac552e4424bfbf303277d592b9aa74906f48563cf936ab"},{"fname":"270271","split":"eval","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"IanStarGem","group_id":"fsd50k:u-d9c5a498740e","member":"FSD50K.eval_audio/270271.wav","disk":1,"offset":2823800565,"compressed_bytes":414010,"size":438178,"method":8,"crc32":2920538836,"sha256":"723c7e3a0e95d25187d0638fd2e8e38cf24e2edabefef39bd09082ea671737c4"},{"fname":"275042","split":"eval","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Cell31_Sound_Productions","group_id":"fsd50k:u-8d8be27f5bb5","member":"FSD50K.eval_audio/275042.wav","disk":0,"offset":1717437587,"compressed_bytes":1345132,"size":1573764,"method":8,"crc32":3290739267,"sha256":"310e22eb3f3dd03c549704dec137ff161a5aca1b8473d174870f92115ec56846"},{"fname":"325845","split":"eval","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Daveshu88","group_id":"fsd50k:u-6e1c143caf25","member":"FSD50K.eval_audio/325845.wav","disk":0,"offset":2738674254,"compressed_bytes":1802455,"size":1824940,"method":8,"crc32":2460460164,"sha256":"f47260470d0738a03e6c9589136ecd72371b519807e2206dd4a80be18a2e56b5"},{"fname":"386006","split":"eval","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"THPSounds","group_id":"fsd50k:u-05eec8b3dcae","member":"FSD50K.eval_audio/386006.wav","disk":1,"offset":1548888551,"compressed_bytes":1008378,"size":1050204,"method":8,"crc32":434924422,"sha256":"ac752d7193e616053921fbced808972d39761f5872d622fcdb6083ef87f65804"},{"fname":"391516","split":"eval","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"relwin","group_id":"fsd50k:u-c826467b98f3","member":"FSD50K.eval_audio/391516.wav","disk":0,"offset":2665053863,"compressed_bytes":1876410,"size":2014542,"method":8,"crc32":3403050284,"sha256":"8f9f7c1192f0f351014eded896a4b2c5e2d4f132bc44485a64dd4b2a3a3e705e"},{"fname":"392324","split":"eval","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"rdg1421","group_id":"fsd50k:u-7c59f4697613","member":"FSD50K.eval_audio/392324.wav","disk":0,"offset":432120165,"compressed_bytes":715498,"size":1419342,"method":8,"crc32":3792020809,"sha256":"8acd3967221b60ce01fc84ec55f3925358341e68c0eec7c0c7dbb2339a218086"},{"fname":"424592","split":"eval","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Daniela-Santos","group_id":"fsd50k:u-49881ec563f1","member":"FSD50K.eval_audio/424592.wav","disk":0,"offset":1170372459,"compressed_bytes":1317720,"size":1431858,"method":8,"crc32":1452604487,"sha256":"6cd8ff8f16be456f10f471bb08d08dd43edb1446ff3d905cfec433e95b38db11"},{"fname":"424616","split":"eval","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Claudia96","group_id":"fsd50k:u-efde639b73d7","member":"FSD50K.eval_audio/424616.wav","disk":0,"offset":2020476044,"compressed_bytes":1254356,"size":1599566,"method":8,"crc32":1924213960,"sha256":"330bf10a831cfb926eced60a0363605ffc658809b6eabcc029448ccef6bd9aa0"},{"fname":"431206","split":"eval","target_label":"Fixed-wing_aircraft_and_airplane","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"oscaraudiogeek","group_id":"fsd50k:u-eee555759646","member":"FSD50K.eval_audio/431206.wav","disk":1,"offset":2449073621,"compressed_bytes":1132824,"size":1397632,"method":8,"crc32":954066046,"sha256":"d7da9eb4840a2a50e598dd90f040beacc8e9c55dfdff84f1881da54f3f66fe7a"},{"fname":"32330","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"ERH","group_id":"fsd50k:u-0dd57e59898e","member":"FSD50K.dev_audio/32330.wav","disk":4,"offset":1430158814,"compressed_bytes":834722,"size":982358,"method":8,"crc32":515642840,"sha256":"dbb697442ca1bf97eed270b0a8e0c5bd9ed7985e242beaed45ff12cdd8a44f5f"},{"fname":"38017","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"rockdoctor","group_id":"fsd50k:u-454248af3cdd","member":"FSD50K.dev_audio/38017.wav","disk":2,"offset":2092020678,"compressed_bytes":678475,"size":882960,"method":8,"crc32":2613773285,"sha256":"157b22e468cff591b3030bec653d1ea9044d1e67c0ad7aa9bbfdd107312b2fdd"},{"fname":"43778","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"digifishmusic","group_id":"fsd50k:u-23878ad72831","member":"FSD50K.dev_audio/43778.wav","disk":3,"offset":2913970998,"compressed_bytes":1395605,"size":1750262,"method":8,"crc32":2057409163,"sha256":"3c759044b463ab2e14657bd55b324d4ccd10a3e09bedb790f227b4e19144fe4e"},{"fname":"59291","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/59291.wav","disk":4,"offset":1476787414,"compressed_bytes":456479,"size":514732,"method":8,"crc32":131924718,"sha256":"dbc1398f14d6f203e5695def7b1752bffd307f95c2a0a5d99997e7e92294150d"},{"fname":"59292","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/59292.wav","disk":5,"offset":230881623,"compressed_bytes":388909,"size":431414,"method":8,"crc32":3017710674,"sha256":"9b65ccaf646691918c3a8429a968df25bb400a69a24fe41f0335cec0e64b882d"},{"fname":"59293","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/59293.wav","disk":0,"offset":2654569169,"compressed_bytes":379353,"size":441078,"method":8,"crc32":2575940348,"sha256":"f83b19d1c740f56977ee8713398325177ddcb5f99ec2b82be56ac5321de0d49c"},{"fname":"62158","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"freesound61476","group_id":"fsd50k:u-addf0a217c3d","member":"FSD50K.dev_audio/62158.wav","disk":2,"offset":1276704999,"compressed_bytes":465876,"size":494898,"method":8,"crc32":1357642257,"sha256":"8b20828aa2b1a5ee744533038830c4fb7584a4877f4bc8184710af8d7423c790"},{"fname":"62159","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"freesound61476","group_id":"fsd50k:u-addf0a217c3d","member":"FSD50K.dev_audio/62159.wav","disk":1,"offset":201773906,"compressed_bytes":432230,"size":459854,"method":8,"crc32":1251609221,"sha256":"8ceeea83e0fe17530b909419fb09c5978d9e9b6ca0a887bafd431bceba6c678b"},{"fname":"75656","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"NoiseCollector","group_id":"fsd50k:u-0697aba1f475","member":"FSD50K.dev_audio/75656.wav","disk":1,"offset":2119573117,"compressed_bytes":275257,"size":341070,"method":8,"crc32":3345401649,"sha256":"297eb47427f6e54884b408635001ff6dfb73e773fb4a8638b0a4894b33ab4626"},{"fname":"75657","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"NoiseCollector","group_id":"fsd50k:u-0697aba1f475","member":"FSD50K.dev_audio/75657.wav","disk":2,"offset":1059267849,"compressed_bytes":44502,"size":50826,"method":8,"crc32":4114657713,"sha256":"f074657e81f50eaf17a29980b94a890e97847ea5d2863899a97e42e0b3539efc"},{"fname":"75687","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"NoiseCollector","group_id":"fsd50k:u-0697aba1f475","member":"FSD50K.dev_audio/75687.wav","disk":2,"offset":1571092780,"compressed_bytes":230966,"size":368718,"method":8,"crc32":2016813463,"sha256":"7112782c5452175bfe2d6b8f002626c25e9c28f64b758c1853b5eb0e1037edf2"},{"fname":"75966","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"mhusinsky","group_id":"fsd50k:u-8c147923c681","member":"FSD50K.dev_audio/75966.wav","disk":3,"offset":2121230555,"compressed_bytes":1311718,"size":2524088,"method":8,"crc32":2447488467,"sha256":"0220724e2d5075e0d19dbcd619ac838a94370d5509bfd10380ea615002612a81"},{"fname":"75972","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"mhusinsky","group_id":"fsd50k:u-8c147923c681","member":"FSD50K.dev_audio/75972.wav","disk":5,"offset":1416593126,"compressed_bytes":1103192,"size":1542520,"method":8,"crc32":2956524898,"sha256":"05fbb5fdbdbcdf7a228c672b387443409a98d513b40661ecc39369c89b4d5b17"},{"fname":"76447","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"mhusinsky","group_id":"fsd50k:u-8c147923c681","member":"FSD50K.dev_audio/76447.wav","disk":1,"offset":242984142,"compressed_bytes":578438,"size":825134,"method":8,"crc32":2315021463,"sha256":"19a891bb654aa30f0f8f7a4538ce018b051b30cb64ab9207341d78daaeec8e05"},{"fname":"108772","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"soundbytez","group_id":"fsd50k:u-02e41892b44a","member":"FSD50K.dev_audio/108772.wav","disk":0,"offset":381958895,"compressed_bytes":1536976,"size":1891968,"method":8,"crc32":2942798132,"sha256":"53e913b68328d6da2bce81d54ccb507efc3945802338a3e38d92bc40f60c49fc"},{"fname":"111305","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"echobones","group_id":"fsd50k:u-8ea5c2eb675b","member":"FSD50K.dev_audio/111305.wav","disk":0,"offset":1958949557,"compressed_bytes":903301,"size":1181982,"method":8,"crc32":566836631,"sha256":"7d0b44ca0a7eca37a3bed63ff15db1f8c4328816ccecf909d626dc46850ae313"},{"fname":"131475","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"thecityrings","group_id":"fsd50k:u-4641341523d9","member":"FSD50K.dev_audio/131475.wav","disk":4,"offset":782202758,"compressed_bytes":142790,"size":251214,"method":8,"crc32":420812914,"sha256":"b540f74ea9007b0873e27886160633f1cfeacd95a3228ce9fb7358620cc10357"},{"fname":"144420","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Eelke","group_id":"fsd50k:u-a97dc44da275","member":"FSD50K.dev_audio/144420.wav","disk":0,"offset":1690971950,"compressed_bytes":1239901,"size":1764078,"method":8,"crc32":2163749453,"sha256":"3c38068bce7e5c6826adb105bf97fcdf4c6650bf1141106a7f4e7df85243a674"},{"fname":"166703","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Snapper4298","group_id":"fsd50k:u-831da112893d","member":"FSD50K.dev_audio/166703.wav","disk":4,"offset":938929939,"compressed_bytes":501724,"size":646414,"method":8,"crc32":397022009,"sha256":"0b085cfc7deda210fc3699368477b9e442274337844f60e2818a4c8719be0e68"},{"fname":"166709","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Snapper4298","group_id":"fsd50k:u-831da112893d","member":"FSD50K.dev_audio/166709.wav","disk":1,"offset":843973415,"compressed_bytes":1430,"size":1367054,"method":8,"crc32":528066978,"sha256":"8b67b0aea65a55d6d28f3b813aadf49ab125c7356646a8ae7847bd66eb22fa14"},{"fname":"166710","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Snapper4298","group_id":"fsd50k:u-831da112893d","member":"FSD50K.dev_audio/166710.wav","disk":1,"offset":1848411734,"compressed_bytes":676210,"size":1321566,"method":8,"crc32":3408283961,"sha256":"06cc51cd646c48e41224314ce5b715b5400dbcc23644827764298680151ac033"},{"fname":"169021","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"inchadney","group_id":"fsd50k:u-ea67aa4b603e","member":"FSD50K.dev_audio/169021.wav","disk":0,"offset":18054460,"compressed_bytes":1679166,"size":2100372,"method":8,"crc32":4131425468,"sha256":"af219f405ddbd4677f5238cde6c09f961f416b30c0761a67b514c2e5a62cf8ca"},{"fname":"174885","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"toiletrolltube","group_id":"fsd50k:u-898a3f52b9ab","member":"FSD50K.dev_audio/174885.wav","disk":2,"offset":1569254862,"compressed_bytes":1837833,"size":2053580,"method":8,"crc32":3281570176,"sha256":"a440172d30900b91ea3cfe7e86bb00eb20894783f14cf4fc98f545208b2df5e5"},{"fname":"174894","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"toiletrolltube","group_id":"fsd50k:u-898a3f52b9ab","member":"FSD50K.dev_audio/174894.wav","disk":0,"offset":1650745646,"compressed_bytes":1726379,"size":2080442,"method":8,"crc32":1242156706,"sha256":"89cc44cbb15adefa848308a1d9097ee63ee69884af8a89970ac632d64e7b939f"},{"fname":"174902","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"toiletrolltube","group_id":"fsd50k:u-898a3f52b9ab","member":"FSD50K.dev_audio/174902.wav","disk":0,"offset":709798016,"compressed_bytes":843005,"size":1009328,"method":8,"crc32":2867313318,"sha256":"ce335e4cf1ca81f043ff9d87e9c6435268526b30d65ed17b0222956044ffa9be"},{"fname":"192766","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"jemtman","group_id":"fsd50k:u-8d3be8353d14","member":"FSD50K.dev_audio/192766.wav","disk":0,"offset":3175271217,"compressed_bytes":1267492,"size":1528708,"method":8,"crc32":685762222,"sha256":"9977776d617e3562854354e29658b6cb6756d5f8ad8b21c718273f2311f01af5"},{"fname":"197743","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"beejeeb1314","group_id":"fsd50k:u-c7581e101a78","member":"FSD50K.dev_audio/197743.wav","disk":4,"offset":3051973195,"compressed_bytes":751315,"size":1221198,"method":8,"crc32":2668068403,"sha256":"a54d615dade47cd3fcf5370209c65ee2feadadd779f535be6a43ba99ccc43a51"},{"fname":"208848","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Insestanyricarda","group_id":"fsd50k:u-568563aa5b58","member":"FSD50K.dev_audio/208848.wav","disk":4,"offset":518864705,"compressed_bytes":245241,"size":293288,"method":8,"crc32":1214857636,"sha256":"1642376bbcf9188e0efdfcb97b85899f297bdf90740e8e6c03963def09fa30b1"},{"fname":"240691","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"mariiao2","group_id":"fsd50k:u-17627e7a41e3","member":"FSD50K.dev_audio/240691.wav","disk":2,"offset":850121051,"compressed_bytes":848603,"size":1030476,"method":8,"crc32":1346364235,"sha256":"f83e70394203262102a3dcdd5c6f0f4863e912bbadd30f21005eadb9981893d4"},{"fname":"263786","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"isteak","group_id":"fsd50k:u-5430ed806b34","member":"FSD50K.dev_audio/263786.wav","disk":0,"offset":647337886,"compressed_bytes":26731,"size":36558,"method":8,"crc32":1626035076,"sha256":"7135d6191d128d140049304a5a27ff7a29cb8296f12f346559a89ea68ed57796"},{"fname":"275617","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Chernoby","group_id":"fsd50k:u-1962c00e8fd3","member":"FSD50K.dev_audio/275617.wav","disk":2,"offset":109233600,"compressed_bytes":1284274,"size":1784052,"method":8,"crc32":472458217,"sha256":"5b7a6970f58357eef436e4db2e9339136e717a24bab8272bcd5eda58d1b7231e"},{"fname":"322855","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"GoodListener","group_id":"fsd50k:u-ce62d2fabaa4","member":"FSD50K.dev_audio/322855.wav","disk":4,"offset":2152666164,"compressed_bytes":742190,"size":1717104,"method":8,"crc32":3710390151,"sha256":"d3b3c2977931cd32069be97e6e5e6b00176647254e63b2aba88bcd4e8de7ede6"},{"fname":"331685","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"LouisDirector","group_id":"fsd50k:u-ff0090a85ffc","member":"FSD50K.dev_audio/331685.wav","disk":1,"offset":2395404635,"compressed_bytes":1871274,"size":2465342,"method":8,"crc32":3464696690,"sha256":"c2ddee2ebf3fa880886ccac27f8245e91cce0edcb0567387d7e6e178ef72ac62"},{"fname":"344535","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"newagesoup","group_id":"fsd50k:u-cee262677ef6","member":"FSD50K.dev_audio/344535.wav","disk":5,"offset":107551597,"compressed_bytes":769212,"size":1228770,"method":8,"crc32":1488617374,"sha256":"92775f2e85bbb5559778fc8511cec270acaa1caf241c625480b8d4f9dee7ca3e"},{"fname":"345024","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"eleonora_diana","group_id":"fsd50k:u-d3b5834b8f63","member":"FSD50K.dev_audio/345024.wav","disk":3,"offset":2095861876,"compressed_bytes":969364,"size":1510094,"method":8,"crc32":875773846,"sha256":"fe7e75b6c2da10dab5d4cbb4bfd3617ae94450a495a952fa413b588895a84439"},{"fname":"368934","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"AcousticMemory","group_id":"fsd50k:u-39182944bc0d","member":"FSD50K.dev_audio/368934.wav","disk":1,"offset":860308250,"compressed_bytes":854671,"size":1238294,"method":8,"crc32":2819271606,"sha256":"d8a6c46394cba84b3c91eed53d0d5889049c817420794bd574032e9b65544175"},{"fname":"379083","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"13FPanska_Krug_Antonin","group_id":"fsd50k:u-ef8b02632d55","member":"FSD50K.dev_audio/379083.wav","disk":2,"offset":1184325179,"compressed_bytes":1009335,"size":1386582,"method":8,"crc32":419318562,"sha256":"a0380d8088f764e03ea69e9d9ffd6b020dca171f6c35b882dfd904436ce421d8"},{"fname":"379086","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"13FPanska_Krug_Antonin","group_id":"fsd50k:u-ef8b02632d55","member":"FSD50K.dev_audio/379086.wav","disk":2,"offset":1796736277,"compressed_bytes":827302,"size":987918,"method":8,"crc32":538858544,"sha256":"ddd53050513606c26441ae7ec085257c42c6bb6098778cbbeb01b1c6f09fb9be"},{"fname":"379087","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"13FPanska_Krug_Antonin","group_id":"fsd50k:u-ef8b02632d55","member":"FSD50K.dev_audio/379087.wav","disk":4,"offset":1485438934,"compressed_bytes":1112320,"size":1361886,"method":8,"crc32":1914311087,"sha256":"c0e2a7694c238920d1a67ded3b0d45668ec78a646b900350a4195d4c50b3fe50"},{"fname":"390153","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"iut_Paris8","group_id":"fsd50k:u-e3399b16301a","member":"FSD50K.dev_audio/390153.wav","disk":4,"offset":356708909,"compressed_bytes":2077402,"size":2192846,"method":8,"crc32":4053789654,"sha256":"93444ffcd5cd7b2f7170c0e759a7c904096b962b63a0ba8c31ddc8d7521b60e9"},{"fname":"410825","split":"dev","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"univ_lyon3","group_id":"fsd50k:u-1f41785243cf","member":"FSD50K.dev_audio/410825.wav","disk":3,"offset":163435044,"compressed_bytes":1227020,"size":1415472,"method":8,"crc32":535273881,"sha256":"33cb270787888799df9ae561c488ab07f81a1a5de071daca2f627a512a2bbaee"},{"fname":"23707","split":"eval","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"hazure","group_id":"fsd50k:u-0e8e7d3feccf","member":"FSD50K.eval_audio/23707.wav","disk":1,"offset":1997939725,"compressed_bytes":610086,"size":816306,"method":8,"crc32":1767344175,"sha256":"1157625ec3a73ebd9abb3754210f33ac14539f8cdb50c6b234a7a5550527d962"},{"fname":"39052","split":"eval","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Heigh-hoo","group_id":"fsd50k:u-f27da747f3d4","member":"FSD50K.eval_audio/39052.wav","disk":1,"offset":1663736634,"compressed_bytes":1649571,"size":2280342,"method":8,"crc32":3867837504,"sha256":"889e99f8d7b9063eeb6e536cb5907fcf7d81b2f23f1bdb3fd177b495c0fcc653"},{"fname":"50335","split":"eval","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"ingsey101","group_id":"fsd50k:u-93fbea58e442","member":"FSD50K.eval_audio/50335.wav","disk":1,"offset":1125570700,"compressed_bytes":666116,"size":1312416,"method":8,"crc32":2638113064,"sha256":"3d50fb0bf373cc0ae76e00352344e1515dd740d84e26bff640fe7bdf02436697"},{"fname":"56531","split":"eval","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"juskiddink","group_id":"fsd50k:u-1add789cab59","member":"FSD50K.eval_audio/56531.wav","disk":1,"offset":310820845,"compressed_bytes":448957,"size":511062,"method":8,"crc32":3841386408,"sha256":"e0f88b6e5119f48a34411a2177d60e88b895a3478b0a31576842d907256d5ac6"},{"fname":"56532","split":"eval","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"juskiddink","group_id":"fsd50k:u-1add789cab59","member":"FSD50K.eval_audio/56532.wav","disk":0,"offset":1781044053,"compressed_bytes":2220495,"size":2553994,"method":8,"crc32":189600474,"sha256":"d4b3983b9976930e7c598b6c30634eed669081afe8192224bf3e5ea9ed16d432"},{"fname":"56533","split":"eval","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"juskiddink","group_id":"fsd50k:u-1add789cab59","member":"FSD50K.eval_audio/56533.wav","disk":0,"offset":582886986,"compressed_bytes":1922905,"size":2224928,"method":8,"crc32":1927539989,"sha256":"01463e17ffeec38311265eefc1c5e9cef5ddaad9a53167650948afa7dbf3a3f1"},{"fname":"73497","split":"eval","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"nigelcoop","group_id":"fsd50k:u-1e17b8398a2e","member":"FSD50K.eval_audio/73497.wav","disk":0,"offset":2763039102,"compressed_bytes":264071,"size":324588,"method":8,"crc32":2931020734,"sha256":"f7626be0559fdba919b50b6f99b58df6164749f7f925a01cea06ef7f1341ff2a"},{"fname":"75195","split":"eval","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"nigelcoop","group_id":"fsd50k:u-1e17b8398a2e","member":"FSD50K.eval_audio/75195.wav","disk":0,"offset":2594150940,"compressed_bytes":105016,"size":169890,"method":8,"crc32":404758063,"sha256":"443f3c4f703463bb3f2d9fb664f4690321cd03c85ea7d7ed15a478d754e9a48e"},{"fname":"149249","split":"eval","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Henrythetrain","group_id":"fsd50k:u-f3947b53a8cc","member":"FSD50K.eval_audio/149249.wav","disk":1,"offset":611314018,"compressed_bytes":273251,"size":529278,"method":8,"crc32":1792987275,"sha256":"af314bc4a809374bd648efcfe7eb1842e62ab23f8cfec8e7649b3dc8cff7f494"},{"fname":"194675","split":"eval","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"javetakami","group_id":"fsd50k:u-685103e3611d","member":"FSD50K.eval_audio/194675.wav","disk":1,"offset":2433522622,"compressed_bytes":983101,"size":1514272,"method":8,"crc32":2881900683,"sha256":"c644bdc60e54e16d3148e62642eacccea2e1f8e8052ed72fc849930a15e35bd2"},{"fname":"194940","split":"eval","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"soundmary","group_id":"fsd50k:u-51f62a00c5d6","member":"FSD50K.eval_audio/194940.wav","disk":1,"offset":2344050725,"compressed_bytes":1847156,"size":2415348,"method":8,"crc32":2978414115,"sha256":"47a7339f9368d44d6954a21903fdaa3c3a855c4a61a9d25d7465471a23ad25e9"},{"fname":"194942","split":"eval","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"soundmary","group_id":"fsd50k:u-51f62a00c5d6","member":"FSD50K.eval_audio/194942.wav","disk":1,"offset":624071966,"compressed_bytes":1017968,"size":1225704,"method":8,"crc32":2546937649,"sha256":"1a8c1cc320590ea8bef55771ac6c21b58aca46772d09f4f583ea6f2a0eac1e94"},{"fname":"207131","split":"eval","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"ahill86","group_id":"fsd50k:u-66297ec0a7fb","member":"FSD50K.eval_audio/207131.wav","disk":1,"offset":1747634144,"compressed_bytes":1522630,"size":1974310,"method":8,"crc32":1927106820,"sha256":"a2fcd67194c8878bfcc8be9d858c3659c943ec93993af1a1c8cd84c8ba531583"},{"fname":"234681","split":"eval","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"odilonmarcenaro","group_id":"fsd50k:u-dcad53546ac5","member":"FSD50K.eval_audio/234681.wav","disk":1,"offset":155024028,"compressed_bytes":755476,"size":830900,"method":8,"crc32":1519957952,"sha256":"b9adb58fd98a2881be01aa217680fac77120c5a3a12f7c7a3d6f060535b97ff9"},{"fname":"321737","split":"eval","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Skjor1","group_id":"fsd50k:u-7f76f9884fc5","member":"FSD50K.eval_audio/321737.wav","disk":1,"offset":1140277910,"compressed_bytes":323617,"size":436608,"method":8,"crc32":456051957,"sha256":"b18800e4182ebe53a69bf3705817153b29267002cac486a2c876e9425142f7e3"},{"fname":"332066","split":"eval","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"TheSoundcatcher","group_id":"fsd50k:u-3108daeeb659","member":"FSD50K.eval_audio/332066.wav","disk":0,"offset":2863514217,"compressed_bytes":1120927,"size":1524616,"method":8,"crc32":2723593626,"sha256":"e1cd11f292e51770e7e15d8a61568f54c820f98405dc892a90b91a767bc35597"},{"fname":"395472","split":"eval","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"byronabadia","group_id":"fsd50k:u-d1e9e9dbc642","member":"FSD50K.eval_audio/395472.wav","disk":1,"offset":2215987747,"compressed_bytes":1702573,"size":2131696,"method":8,"crc32":240383188,"sha256":"0a8a1e093d347e178c7bd9204a3e788b99d407acf47c3898a29e8ebeab9bff2b"},{"fname":"424870","split":"eval","target_label":"Gull_and_seagull","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"joanaseabra","group_id":"fsd50k:u-2d5b51395153","member":"FSD50K.eval_audio/424870.wav","disk":0,"offset":2905735172,"compressed_bytes":1252565,"size":1402830,"method":8,"crc32":703470405,"sha256":"d6479be299ff32a99924e6789f26a3877d9450e157d3975bf51056d4c361acf3"},{"fname":"18966","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/18966.wav","disk":0,"offset":1080944070,"compressed_bytes":1486205,"size":2074806,"method":8,"crc32":1600346683,"sha256":"4ec772bbe4d4d583441a2a6e7ec930a40c3e948ab0680e6549fbe6fb9ae9ca44"},{"fname":"20062","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"euroblues","group_id":"fsd50k:u-e42a002f8f12","member":"FSD50K.dev_audio/20062.wav","disk":5,"offset":1410760099,"compressed_bytes":1520165,"size":1900900,"method":8,"crc32":2451869304,"sha256":"7c621df34a06c3e9ab4c565c631848e4bfd207e4929cc712847a067b93a13e0f"},{"fname":"23129","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"digitil","group_id":"fsd50k:u-549a819fa367","member":"FSD50K.dev_audio/23129.wav","disk":1,"offset":716490305,"compressed_bytes":1371341,"size":1446338,"method":8,"crc32":1295047996,"sha256":"b429be946f45b603266dd3d0fac8424939dd8d3ade26e5f25debf029726e7571"},{"fname":"26133","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"WIM","group_id":"fsd50k:u-07d9d387aebf","member":"FSD50K.dev_audio/26133.wav","disk":4,"offset":2181172922,"compressed_bytes":410258,"size":486310,"method":8,"crc32":913468712,"sha256":"0b0253f0ea2427bfb4bfb6fa10fc82b259caecae5f81ef582852947b39b8e813"},{"fname":"30201","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"bondegi","group_id":"fsd50k:u-0cf1c461a413","member":"FSD50K.dev_audio/30201.wav","disk":1,"offset":1321169650,"compressed_bytes":606792,"size":657402,"method":8,"crc32":1589284647,"sha256":"5f114c373a5635d25c66932924f1f76c933333d6306bbffc6768ce73fc850c98"},{"fname":"32244","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"digifishmusic","group_id":"fsd50k:u-23878ad72831","member":"FSD50K.dev_audio/32244.wav","disk":0,"offset":631561135,"compressed_bytes":213134,"size":247008,"method":8,"crc32":1614522997,"sha256":"b8267f414651cabf426e1f8ad0bbe885dd2fdca37e679c046544dffa81a931bb"},{"fname":"34363","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/34363.wav","disk":4,"offset":2256169286,"compressed_bytes":1164154,"size":1480306,"method":8,"crc32":229433307,"sha256":"adb233ba533a87dc976d09f032a282e188b129b7a7b499264e1407edd2c37fdb"},{"fname":"35140","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"roubignolle","group_id":"fsd50k:u-bba66f09f839","member":"FSD50K.dev_audio/35140.wav","disk":5,"offset":1177891825,"compressed_bytes":57342,"size":89500,"method":8,"crc32":1629828128,"sha256":"bcc28598f169d6a8011cc7f79cdcfbee4498e013167ba97d0ec6adfb606f7e30"},{"fname":"35147","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"roubignolle","group_id":"fsd50k:u-bba66f09f839","member":"FSD50K.dev_audio/35147.wav","disk":2,"offset":1237587670,"compressed_bytes":2162234,"size":2581632,"method":8,"crc32":2710221017,"sha256":"bd9206dcf9394291fef8c69731a4a1a09df4eb9ad14ba8a3a357182b563dd241"},{"fname":"35150","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"roubignolle","group_id":"fsd50k:u-bba66f09f839","member":"FSD50K.dev_audio/35150.wav","disk":3,"offset":1843019329,"compressed_bytes":52280,"size":69696,"method":8,"crc32":2835399094,"sha256":"b5927f36e1e3ebc5cf9c26676fc67c0d45ebbcc51ed7ec4c3ec494138197b851"},{"fname":"38584","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"digifishmusic","group_id":"fsd50k:u-23878ad72831","member":"FSD50K.dev_audio/38584.wav","disk":3,"offset":2087957135,"compressed_bytes":1081379,"size":1815630,"method":8,"crc32":1243003790,"sha256":"05c382cf7c0b09a1712e17ab93ed629847485621a6e1810786edc86f66ac9dea"},{"fname":"56779","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"ekiekitabaang","group_id":"fsd50k:u-0074d0e66c07","member":"FSD50K.dev_audio/56779.wav","disk":1,"offset":405306948,"compressed_bytes":1305317,"size":1532672,"method":8,"crc32":1977389374,"sha256":"de96976277ee80d9dbab056fca0cc4a8df519ca3ddab2a049ba3a1f421d4a5ef"},{"fname":"57317","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"J - Luc","group_id":"fsd50k:u-4179230a9687","member":"FSD50K.dev_audio/57317.wav","disk":3,"offset":1972668287,"compressed_bytes":1580240,"size":1980494,"method":8,"crc32":3861561645,"sha256":"e90878780b4eb6734021043cd5fe68aad433682bedc8d54b544999d23a2863d1"},{"fname":"57789","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"reinsamba","group_id":"fsd50k:u-1d952795538e","member":"FSD50K.dev_audio/57789.wav","disk":4,"offset":478258528,"compressed_bytes":523516,"size":945408,"method":8,"crc32":1208360022,"sha256":"a8daa8b05d1c79efafd5aafc86bf7f03664a66d69abffa7b7fd7647adaccd5fa"},{"fname":"58337","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dkustic","group_id":"fsd50k:u-5ccb3817e707","member":"FSD50K.dev_audio/58337.wav","disk":3,"offset":1917643523,"compressed_bytes":283550,"size":436620,"method":8,"crc32":1641321642,"sha256":"7d7bdafb0365a839682b5dbed0085e5ca840e512c895060e14663bf633c7eea6"},{"fname":"58498","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Satoration","group_id":"fsd50k:u-1f6f609b58a8","member":"FSD50K.dev_audio/58498.wav","disk":3,"offset":798409715,"compressed_bytes":1564931,"size":1718570,"method":8,"crc32":3900976428,"sha256":"6a496f2001b5d641d2c3d5fc02c19142e2c5ec5090e02c1c906f10320c606f23"},{"fname":"66002","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"frankd","group_id":"fsd50k:u-4dc84eff23c6","member":"FSD50K.dev_audio/66002.wav","disk":3,"offset":1547021816,"compressed_bytes":978238,"size":1347904,"method":8,"crc32":552554653,"sha256":"915cc4dfd3cc394e1de1b98558c467a45763a2933b5b0d53b842e54a75d09dde"},{"fname":"76196","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dkustic","group_id":"fsd50k:u-5ccb3817e707","member":"FSD50K.dev_audio/76196.wav","disk":2,"offset":2107366271,"compressed_bytes":1320920,"size":2634318,"method":8,"crc32":492774034,"sha256":"9323371704144e21a9048aa276ee44455b2636d551fa04426e1b34b7a83198c9"},{"fname":"80783","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"NoiseCollector","group_id":"fsd50k:u-0697aba1f475","member":"FSD50K.dev_audio/80783.wav","disk":4,"offset":2148989096,"compressed_bytes":961745,"size":1077326,"method":8,"crc32":3519351441,"sha256":"58d9df506dd19618816a0b2a9db183e038c8ad63ccdfb7f81678f38e9de7c580"},{"fname":"80787","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"NoiseCollector","group_id":"fsd50k:u-0697aba1f475","member":"FSD50K.dev_audio/80787.wav","disk":5,"offset":2130442890,"compressed_bytes":198773,"size":236620,"method":8,"crc32":2585607712,"sha256":"c2bd6fff43d740b7f9fba36fa5dc7739ba45e30f5dacbcad2e98140d070fb135"},{"fname":"80842","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"sunsinger","group_id":"fsd50k:u-f7cb75ebdf08","member":"FSD50K.dev_audio/80842.wav","disk":1,"offset":942980063,"compressed_bytes":1291524,"size":1963998,"method":8,"crc32":2373846521,"sha256":"340aa2fc7c5acfb9cfdc7f1fb79a0ee8df4cc5da22969bf88a77b00ec7bf946c"},{"fname":"89354","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"zimbot","group_id":"fsd50k:u-d2262f37a09d","member":"FSD50K.dev_audio/89354.wav","disk":2,"offset":2265696789,"compressed_bytes":324907,"size":442994,"method":8,"crc32":3691460457,"sha256":"501c801a885f56ed1d8c9953be6df4f400f56caec586fb10f42334da57055275"},{"fname":"89355","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"zimbot","group_id":"fsd50k:u-d2262f37a09d","member":"FSD50K.dev_audio/89355.wav","disk":2,"offset":338405075,"compressed_bytes":27714,"size":212644,"method":8,"crc32":3334109278,"sha256":"c475d784e44f0a8059e09edd14e8e13da7fc0fe589e349083c2d3317403e1661"},{"fname":"89357","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"zimbot","group_id":"fsd50k:u-d2262f37a09d","member":"FSD50K.dev_audio/89357.wav","disk":0,"offset":129704,"compressed_bytes":144552,"size":383922,"method":8,"crc32":1422371946,"sha256":"95653b97a66ce9257a87e8fcda44bf08305392ea9031e9a1a322879ca98321e0"},{"fname":"98648","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/98648.wav","disk":5,"offset":100172931,"compressed_bytes":716765,"size":882078,"method":8,"crc32":187343634,"sha256":"c6bd17553d573b024e1b4bb00d4a386142c7d23837eebd3a2ae99bad106a8e50"},{"fname":"99956","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"CGEffex","group_id":"fsd50k:u-4f7ab92b7591","member":"FSD50K.dev_audio/99956.wav","disk":2,"offset":2876651484,"compressed_bytes":261888,"size":374040,"method":8,"crc32":2342418861,"sha256":"3bcacba73e771e1af4c48dc567ea16b31b6d51d996d4391501494651532bd335"},{"fname":"103046","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Department64","group_id":"fsd50k:u-c5f145eb5fc4","member":"FSD50K.dev_audio/103046.wav","disk":4,"offset":962360525,"compressed_bytes":43206,"size":54822,"method":8,"crc32":2640914013,"sha256":"c84d39c7b8822130508d35eac46b77a7e59e0529574ca191340523cd2fc9d278"},{"fname":"104602","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Udit Duseja","group_id":"fsd50k:u-c63bb7085b71","member":"FSD50K.dev_audio/104602.wav","disk":0,"offset":1733380868,"compressed_bytes":471178,"size":668408,"method":8,"crc32":1992614757,"sha256":"9f9b7ecfc9be7fe58cc312d211de325c3bf9889aae9a86c63e0a33f5b6eb09e6"},{"fname":"127114","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Grigore","group_id":"fsd50k:u-8ee8a5806b41","member":"FSD50K.dev_audio/127114.wav","disk":2,"offset":1600401075,"compressed_bytes":196659,"size":264678,"method":8,"crc32":2154695709,"sha256":"be6d9c07c71cc90c5e5cce43feb69051d23f64dabed51adc4acf1002ba7fe2f8"},{"fname":"130971","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"KeyKrusher","group_id":"fsd50k:u-86440ceceede","member":"FSD50K.dev_audio/130971.wav","disk":2,"offset":476653898,"compressed_bytes":1813847,"size":2028678,"method":8,"crc32":1924494170,"sha256":"867dca5acaf4b15203482c175cb0f6325837e8541be5d12d68a3b911720c3987"},{"fname":"134813","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"zedkah","group_id":"fsd50k:u-a5b07ad5f715","member":"FSD50K.dev_audio/134813.wav","disk":4,"offset":2217064613,"compressed_bytes":170487,"size":215042,"method":8,"crc32":1093732203,"sha256":"81ee3d8a565f3f7830872e461ca59f12e3b10d16f7b031f2fe0f040942a17c23"},{"fname":"151776","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"universildo","group_id":"fsd50k:u-67c00323e070","member":"FSD50K.dev_audio/151776.wav","disk":4,"offset":2525199333,"compressed_bytes":1866509,"size":2184998,"method":8,"crc32":1549078424,"sha256":"a91c90516660937193308f2b7478d3b8c7cf338fdaf9f9803b2295f0ec4c035f"},{"fname":"157870","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"gfrog","group_id":"fsd50k:u-b23cf9709684","member":"FSD50K.dev_audio/157870.wav","disk":5,"offset":1593320903,"compressed_bytes":662098,"size":762962,"method":8,"crc32":2313287083,"sha256":"c4f8f3df66be77c8daed4eb8fe656ccc381dc472478ceceedc060d4e6c3558d0"},{"fname":"160480","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"unfa","group_id":"fsd50k:u-9fdbf710204b","member":"FSD50K.dev_audio/160480.wav","disk":3,"offset":552867182,"compressed_bytes":111087,"size":229768,"method":8,"crc32":2948735962,"sha256":"64b8ba18d1c77a0272a36485ca0e6a2082d2ac499885aad969b409bbcf8975d9"},{"fname":"161720","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"lauris3722","group_id":"fsd50k:u-54e82497fecc","member":"FSD50K.dev_audio/161720.wav","disk":2,"offset":594934322,"compressed_bytes":738441,"size":917070,"method":8,"crc32":4224213436,"sha256":"da8c661f275171f5bfcf9afbcfbc1e07802b9ee948c65394cb0ef857da2a2357"},{"fname":"164482","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Adam_N","group_id":"fsd50k:u-124149624cb4","member":"FSD50K.dev_audio/164482.wav","disk":3,"offset":54895251,"compressed_bytes":31334,"size":32364,"method":8,"crc32":2248170747,"sha256":"d3cc69ef41564d546e3a568649790eb5f05ffefe5101db07400e0d91c6fe7cf5"},{"fname":"164485","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Adam_N","group_id":"fsd50k:u-124149624cb4","member":"FSD50K.dev_audio/164485.wav","disk":3,"offset":260339255,"compressed_bytes":42480,"size":43750,"method":8,"crc32":341658360,"sha256":"14d159dff7857d4941d652f15d014229d46c104d478440cb75446b6f71441eab"},{"fname":"171650","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"cvp1965","group_id":"fsd50k:u-1a4a5e3f8236","member":"FSD50K.dev_audio/171650.wav","disk":3,"offset":2744792756,"compressed_bytes":964738,"size":1390816,"method":8,"crc32":1928253523,"sha256":"0d76dc59e457773bbaf0b70d66a19dd65d695abeae022b1f6a376a82a558df43"},{"fname":"182206","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"kendel","group_id":"fsd50k:u-d66babfb7e48","member":"FSD50K.dev_audio/182206.wav","disk":5,"offset":2149869235,"compressed_bytes":1490122,"size":1652046,"method":8,"crc32":913943083,"sha256":"4c15b7bb6514b8c98a159048ddb837e38aff35daa678e8ae95751d9c5c5930a6"},{"fname":"193877","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"wjoojoo","group_id":"fsd50k:u-ed302167a923","member":"FSD50K.dev_audio/193877.wav","disk":3,"offset":2041771227,"compressed_bytes":1616815,"size":2207068,"method":8,"crc32":2291087642,"sha256":"7b73d379ee0e6035c3be11d1f65f3644521f4278b01982352360714b7f2329f4"},{"fname":"195617","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"xDimebagx","group_id":"fsd50k:u-427fa573deb4","member":"FSD50K.dev_audio/195617.wav","disk":0,"offset":2625585504,"compressed_bytes":771012,"size":889994,"method":8,"crc32":942662951,"sha256":"60acf4c298a007f7e455758624833e3a83cf04ad9d37b07b9ca4f9e7a38e55e0"},{"fname":"198244","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Guitarz1970","group_id":"fsd50k:u-c801d443721b","member":"FSD50K.dev_audio/198244.wav","disk":2,"offset":2251017272,"compressed_bytes":903415,"size":1048654,"method":8,"crc32":1031226941,"sha256":"4797f2232896699777c173dff0aa70ac19cb1912c55fc0c6dfb1f369c9cc92a6"},{"fname":"204484","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"gthall","group_id":"fsd50k:u-1c2239436502","member":"FSD50K.dev_audio/204484.wav","disk":0,"offset":1264329738,"compressed_bytes":671163,"size":1617312,"method":8,"crc32":2654449456,"sha256":"66d85a224c0be24ac185ec08f2e79808b3f04abedb289c4b5eee971543792900"},{"fname":"209329","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"ThePriest909","group_id":"fsd50k:u-0b82c000862b","member":"FSD50K.dev_audio/209329.wav","disk":3,"offset":835891393,"compressed_bytes":37130,"size":45892,"method":8,"crc32":2548550324,"sha256":"0c694dbe441bbb48536f931122db90b034bea10012d956faea7756b166b4f957"},{"fname":"209331","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"ThePriest909","group_id":"fsd50k:u-0b82c000862b","member":"FSD50K.dev_audio/209331.wav","disk":5,"offset":298301047,"compressed_bytes":203108,"size":252046,"method":8,"crc32":1110550340,"sha256":"bf93f461487cc802f288d44d9b916bb6afd58d53983db520e01023e1894d71c0"},{"fname":"216217","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"RSilveira_88","group_id":"fsd50k:u-3c8e9518c05a","member":"FSD50K.dev_audio/216217.wav","disk":5,"offset":326942048,"compressed_bytes":477729,"size":595942,"method":8,"crc32":3311185276,"sha256":"224c22c84f907328b333aed747c092c3e137ae316c0613277842be2676500480"},{"fname":"222640","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"tec studios","group_id":"fsd50k:u-be53219926a7","member":"FSD50K.dev_audio/222640.wav","disk":1,"offset":281779590,"compressed_bytes":1175475,"size":1268620,"method":8,"crc32":336223677,"sha256":"8d59f0a583d90abdef01ea20d5ab859a9bc3d4f93f7c1ee95ed0aabf45166e47"},{"fname":"233781","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"delphidebrain","group_id":"fsd50k:u-f3be5b866511","member":"FSD50K.dev_audio/233781.wav","disk":2,"offset":1470918273,"compressed_bytes":741256,"size":901602,"method":8,"crc32":3816353217,"sha256":"66466401749bc4c50bf3ff30c7f993ac84d40623602dd5907a154fb6dd7d117c"},{"fname":"233786","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"delphidebrain","group_id":"fsd50k:u-f3be5b866511","member":"FSD50K.dev_audio/233786.wav","disk":2,"offset":3016990195,"compressed_bytes":291654,"size":455744,"method":8,"crc32":3817161933,"sha256":"28a9d432373224781756e25c9a4c40ae17bcf3482de6875f64125b3133cb5f6d"},{"fname":"238147","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"M4D3R0","group_id":"fsd50k:u-e866ac82aab0","member":"FSD50K.dev_audio/238147.wav","disk":0,"offset":2017502399,"compressed_bytes":137594,"size":242486,"method":8,"crc32":2545594431,"sha256":"d45f5678d25d0e4a2dbd0cdaa0524546bed756856cb5eb4936f8c1df326a7d5f"},{"fname":"241994","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"volivieri","group_id":"fsd50k:u-af86f6485d9e","member":"FSD50K.dev_audio/241994.wav","disk":2,"offset":2871028719,"compressed_bytes":1190336,"size":1299330,"method":8,"crc32":3601933723,"sha256":"b00656dc3ba9dfe0260a04261a6dd9806ec4bef56912eade9429cec0aa97c7de"},{"fname":"244591","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Cornwallis89","group_id":"fsd50k:u-b60df451a6b1","member":"FSD50K.dev_audio/244591.wav","disk":0,"offset":1868303823,"compressed_bytes":1332613,"size":2167680,"method":8,"crc32":2400270759,"sha256":"71c423c05fd34149f0f3f82c9257c80f6ca20386ff04aa2046dbac8ddcde0afa"},{"fname":"244712","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Olichite","group_id":"fsd50k:u-7dff76c3b857","member":"FSD50K.dev_audio/244712.wav","disk":5,"offset":1830463466,"compressed_bytes":1892429,"size":2126286,"method":8,"crc32":2824686929,"sha256":"0936e8d77af0b50b5615a42469743414a834b63253cd6c2724ce343cb290c1f8"},{"fname":"248103","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"DidntGoToFilmSchool","group_id":"fsd50k:u-7bb03e4b4790","member":"FSD50K.dev_audio/248103.wav","disk":5,"offset":1783233962,"compressed_bytes":982643,"size":1719978,"method":8,"crc32":1681715570,"sha256":"623a3c3adc92e57b4079411928560061936fed0cdca4237b4afa56d6bdfd8131"},{"fname":"268956","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"harryScary","group_id":"fsd50k:u-b0dc6da02fe1","member":"FSD50K.dev_audio/268956.wav","disk":2,"offset":305935534,"compressed_bytes":1637731,"size":1988558,"method":8,"crc32":320299644,"sha256":"a2c4335f38b00f9a84304944fda025b09932ec6658c263cd12c01b1fd584ab2c"},{"fname":"329807","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Omar Alvarado","group_id":"fsd50k:u-ed3f5444b5dc","member":"FSD50K.dev_audio/329807.wav","disk":5,"offset":2227270440,"compressed_bytes":665648,"size":1069056,"method":8,"crc32":2917828709,"sha256":"54d1e3257fb86b72f35ab32ff77d82ba20634bc33218570615615ebc255123ac"},{"fname":"345793","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Christinesguitar","group_id":"fsd50k:u-280ec92f4252","member":"FSD50K.dev_audio/345793.wav","disk":1,"offset":2176568987,"compressed_bytes":679623,"size":705678,"method":8,"crc32":908817442,"sha256":"e2192be467cce43698a5e53c450c873ec052e317d6d7161dabb01deba0ad8ebf"},{"fname":"361449","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"spalena","group_id":"fsd50k:u-73c3c09989ad","member":"FSD50K.dev_audio/361449.wav","disk":2,"offset":1190959980,"compressed_bytes":243716,"size":370518,"method":8,"crc32":725692578,"sha256":"6d780871dd9f37dc4382a4b1536f987a5c4bec27693625cdc14b2d630edbc34e"},{"fname":"365247","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"AlienXXX","group_id":"fsd50k:u-0a44b5962b11","member":"FSD50K.dev_audio/365247.wav","disk":2,"offset":2339601880,"compressed_bytes":277360,"size":393274,"method":8,"crc32":2287000247,"sha256":"c69436c93deb473b73e91b1c86bb6c0ca286357334d0a1375082ea8adfa2ada5"},{"fname":"390733","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"FunWithSound","group_id":"fsd50k:u-611a5586bfd5","member":"FSD50K.dev_audio/390733.wav","disk":1,"offset":461980548,"compressed_bytes":66116,"size":89934,"method":8,"crc32":2828193235,"sha256":"aa59cf506545495c0a15e3aa31bbd44d98e7ebb9f8cf3ad0ebbe4a13869c25e0"},{"fname":"400663","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Sojan","group_id":"fsd50k:u-8a7eda57871f","member":"FSD50K.dev_audio/400663.wav","disk":2,"offset":1024800861,"compressed_bytes":61746,"size":109936,"method":8,"crc32":628841689,"sha256":"609c4ad771b47ff236089e751ac43d0c4a3633b5ca5e94fa05c97135638812d5"},{"fname":"401194","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"straget","group_id":"fsd50k:u-86fe64f5e1ad","member":"FSD50K.dev_audio/401194.wav","disk":1,"offset":1970561749,"compressed_bytes":1984561,"size":2468232,"method":8,"crc32":3033317167,"sha256":"97abc361fe9a4d237a7808ae652030f67263d12fc224c3cc70340e59e177e6ab"},{"fname":"404905","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"guyburns","group_id":"fsd50k:u-e79ba4776805","member":"FSD50K.dev_audio/404905.wav","disk":4,"offset":2904708361,"compressed_bytes":1860397,"size":2467738,"method":8,"crc32":3077924982,"sha256":"9b8c9d461a5c93d70dcb76ce2934c4a0bd0e65c5b61c8b026c5fae0d89949f6b"},{"fname":"433018","split":"dev","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Aleks41","group_id":"fsd50k:u-3df8c09c800b","member":"FSD50K.dev_audio/433018.wav","disk":0,"offset":2942044702,"compressed_bytes":83852,"size":96296,"method":8,"crc32":3452141095,"sha256":"7b049d26c2c330abc6a883fd19ec88e978960afe4bdc4e838822a05f8a14595a"},{"fname":"62468","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"bsumusictech","group_id":"fsd50k:u-b4d45cfdfd1e","member":"FSD50K.eval_audio/62468.wav","disk":0,"offset":776060772,"compressed_bytes":295588,"size":388018,"method":8,"crc32":726229759,"sha256":"ebc910b3cae40b5927c8ecfd0fa9a285159dff22ba159a059930984cd1055054"},{"fname":"69439","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"guitarguy1985","group_id":"fsd50k:u-ef77d9dba19d","member":"FSD50K.eval_audio/69439.wav","disk":1,"offset":465686212,"compressed_bytes":127686,"size":176714,"method":8,"crc32":2766042875,"sha256":"2de99af04f86180b6b97eacb9803cb6c7cc407376adb7c506dac03b4922c30b6"},{"fname":"73371","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"sofajoe","group_id":"fsd50k:u-6df5a05c5af2","member":"FSD50K.eval_audio/73371.wav","disk":0,"offset":2761690778,"compressed_bytes":1088858,"size":1353842,"method":8,"crc32":1298896135,"sha256":"cbd99a7b18d02e543b2f3ef90a7214b288f8976a08a2552e804da8a3cf5064cc"},{"fname":"106019","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Christoff45","group_id":"fsd50k:u-a42c3aa20f36","member":"FSD50K.eval_audio/106019.wav","disk":0,"offset":1242270680,"compressed_bytes":432901,"size":2030620,"method":8,"crc32":4244954020,"sha256":"cb2a86ff8936b0285bb5bb4b6b831fb9840f6a5bbb9adcaf49bb6e4b5a74e585"},{"fname":"113616","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Puniho","group_id":"fsd50k:u-f7a5688ae252","member":"FSD50K.eval_audio/113616.wav","disk":1,"offset":978210230,"compressed_bytes":902758,"size":940070,"method":8,"crc32":1018910035,"sha256":"428da40d91a5dcf5e0ce69f5accd07f67205fec8c40c96bafcfa82fe8e5e62c1"},{"fname":"117146","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"cognito perceptu","group_id":"fsd50k:u-5dc0240bad00","member":"FSD50K.eval_audio/117146.wav","disk":1,"offset":2214964101,"compressed_bytes":1023560,"size":1334774,"method":8,"crc32":1312735231,"sha256":"c4c98694fd6b5d1ba5f71c482149dcb7bab19ba85ec7a583e0068b61da2bd2c0"},{"fname":"119445","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"lmbubec","group_id":"fsd50k:u-3d05552f5c25","member":"FSD50K.eval_audio/119445.wav","disk":1,"offset":1472525960,"compressed_bytes":1011817,"size":1240876,"method":8,"crc32":3111544671,"sha256":"f71aae4fd6e96c3d2581575f333429b69f790c9f6a8452be5ece68b792e568a6"},{"fname":"155513","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"DrZoom","group_id":"fsd50k:u-ee7bb42bb04a","member":"FSD50K.eval_audio/155513.wav","disk":0,"offset":2445765765,"compressed_bytes":863921,"size":1014378,"method":8,"crc32":3268813094,"sha256":"729552e3aa71a60cf863d56cb4447f3a50a87e802891b49d4cac5a83ca93ef2d"},{"fname":"161405","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Alvinwhatup2","group_id":"fsd50k:u-cbc722866e68","member":"FSD50K.eval_audio/161405.wav","disk":0,"offset":261432923,"compressed_bytes":578703,"size":851560,"method":8,"crc32":2239124130,"sha256":"4350c2ccd9735ac26af9ff546d04ceec689491806a28c14a170a6708035dce25"},{"fname":"193798","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"yanov","group_id":"fsd50k:u-0b1f9f5bcf65","member":"FSD50K.eval_audio/193798.wav","disk":1,"offset":367475595,"compressed_bytes":1187737,"size":1515086,"method":8,"crc32":3477945090,"sha256":"b4f2fce31befcd2298910c27491af77baf928a54337dd3970ee688223acfe88f"},{"fname":"194932","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"soundmary","group_id":"fsd50k:u-51f62a00c5d6","member":"FSD50K.eval_audio/194932.wav","disk":1,"offset":263605825,"compressed_bytes":775757,"size":1020376,"method":8,"crc32":927373800,"sha256":"18744c78d5b548dc61ba07d0e99d929e7ee9274ca5efa63cd58940a04151d619"},{"fname":"215174","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"john9","group_id":"fsd50k:u-0c134b8f1971","member":"FSD50K.eval_audio/215174.wav","disk":1,"offset":581220495,"compressed_bytes":1419811,"size":1789642,"method":8,"crc32":3668943519,"sha256":"5efb409a19e6048766349e4e4af3e389d6aeffff2b56503b2a6ca5f78acce5a2"},{"fname":"235566","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"SkeetMasterFunk69","group_id":"fsd50k:u-34939b218773","member":"FSD50K.eval_audio/235566.wav","disk":1,"offset":1970336770,"compressed_bytes":2119049,"size":2502312,"method":8,"crc32":2319861382,"sha256":"feb03229730bbadfbae5d0dd00742dba29172085f0b12ed9fc046e2f39949095"},{"fname":"238074","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"cognito perceptu","group_id":"fsd50k:u-5dc0240bad00","member":"FSD50K.eval_audio/238074.wav","disk":1,"offset":797960674,"compressed_bytes":1605334,"size":1877002,"method":8,"crc32":2369190916,"sha256":"c72ceadb5cc54d291d81a6ee135cf94c25506674bbffe18c578dc46674c0d963"},{"fname":"242046","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"lezaarth","group_id":"fsd50k:u-158b0c7488e5","member":"FSD50K.eval_audio/242046.wav","disk":1,"offset":2501497077,"compressed_bytes":458551,"size":649066,"method":8,"crc32":4064259928,"sha256":"f31bae6b5ff53f49a332ae8b28a015214915d837d6f0a6e88eeecc9b5c504992"},{"fname":"244505","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Silentwaves","group_id":"fsd50k:u-d26934e5359d","member":"FSD50K.eval_audio/244505.wav","disk":0,"offset":1917009746,"compressed_bytes":1276145,"size":1674088,"method":8,"crc32":501704818,"sha256":"7537bcf8285017c310ef82f616e3c8c84234d7814d80a3199288b1dc5d84f1ad"},{"fname":"250355","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"rucisko","group_id":"fsd50k:u-a45d445c0cde","member":"FSD50K.eval_audio/250355.wav","disk":0,"offset":1506128252,"compressed_bytes":1836329,"size":2392434,"method":8,"crc32":3309490475,"sha256":"5a575efb08ebb5fb1c497f040e9faeebd2aaeb5e18cddf5b96ab5dcfda28c1dc"},{"fname":"250381","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"lostphosphene","group_id":"fsd50k:u-f278c1b8e8af","member":"FSD50K.eval_audio/250381.wav","disk":1,"offset":1425861225,"compressed_bytes":1361164,"size":2339790,"method":8,"crc32":270589722,"sha256":"7ef3085ff9e8203b6439f6cc9e3540db3cbe5d626f9b3d07fa5eb399ef9ac223"},{"fname":"264481","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"aglinder","group_id":"fsd50k:u-db33e8af09d1","member":"FSD50K.eval_audio/264481.wav","disk":1,"offset":2666290825,"compressed_bytes":1212551,"size":2170032,"method":8,"crc32":1835265428,"sha256":"c788e824fc454a5134614c3d9f5c11ecd25a75b2dcf907f58c8fdb8da252624a"},{"fname":"274151","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"darcydunes","group_id":"fsd50k:u-6aebb28bb7c3","member":"FSD50K.eval_audio/274151.wav","disk":1,"offset":518779267,"compressed_bytes":90003,"size":107096,"method":8,"crc32":3846155407,"sha256":"74055a8e8b6e64b052dd8428193acf7404a4a89829ba68860751ca480c6af57e"},{"fname":"321868","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"reg7783","group_id":"fsd50k:u-d25d7956e86f","member":"FSD50K.eval_audio/321868.wav","disk":0,"offset":987165588,"compressed_bytes":489526,"size":945298,"method":8,"crc32":808995840,"sha256":"602b0fdef77ab42b8065de601fc759a530cbcb4dc1e5fb259f777679082852c9"},{"fname":"339086","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"FullMetalJedi","group_id":"fsd50k:u-c0a11a887101","member":"FSD50K.eval_audio/339086.wav","disk":0,"offset":360047615,"compressed_bytes":30182,"size":34306,"method":8,"crc32":1432812487,"sha256":"9a1c78d218aece04624fe8e8ce90a4304a4e5d72e60b2cf0a7aea648b2980704"},{"fname":"348204","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"konstantinusz","group_id":"fsd50k:u-05765aa99b8b","member":"FSD50K.eval_audio/348204.wav","disk":1,"offset":331684587,"compressed_bytes":117371,"size":150838,"method":8,"crc32":3232352981,"sha256":"f7bd3633ef492f0ae65c05d27e71adbfb020f1eb43146cfcb8e3fc8e831d513a"},{"fname":"352613","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"macdaddyno1","group_id":"fsd50k:u-f7168755a878","member":"FSD50K.eval_audio/352613.wav","disk":1,"offset":2842021546,"compressed_bytes":716853,"size":884814,"method":8,"crc32":1217644907,"sha256":"2e0abf323fb8b5cc3ebd7726ea2b6fd3b4753aa94220386ee17f8282d28f7723"},{"fname":"376825","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"felix.blume","group_id":"fsd50k:u-d76b68397f6b","member":"FSD50K.eval_audio/376825.wav","disk":0,"offset":2689255595,"compressed_bytes":1958225,"size":2385948,"method":8,"crc32":935069545,"sha256":"60f1ef7f7da60aa9c161a20b787c62df0df613a7d7c63fd8cad2c4e762ce7b8d"},{"fname":"394181","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"saturdaysoundguy","group_id":"fsd50k:u-1061b6586207","member":"FSD50K.eval_audio/394181.wav","disk":0,"offset":665400205,"compressed_bytes":26675,"size":38652,"method":8,"crc32":3136650641,"sha256":"bdb9c7391d40a344c106bc98fc7cdbaf7163bd0b96f4f563f63d956a05cc5be3"},{"fname":"398004","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Suvo","group_id":"fsd50k:u-50c133a6c800","member":"FSD50K.eval_audio/398004.wav","disk":0,"offset":1671388411,"compressed_bytes":619607,"size":782824,"method":8,"crc32":1411423151,"sha256":"6152eb9367b628ee8f001708353ff35ce21363cb74cf3b44811252a91025bd6d"},{"fname":"398487","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Wakerone","group_id":"fsd50k:u-b0832c4125a8","member":"FSD50K.eval_audio/398487.wav","disk":1,"offset":2979094458,"compressed_bytes":86553,"size":106760,"method":8,"crc32":4086843115,"sha256":"8cd0b0fb32f524e73f0f925f8b7cfd63c20d4e85f006cf4d8c24600743956847"},{"fname":"400096","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"harrietniamh","group_id":"fsd50k:u-ad718eae1ae9","member":"FSD50K.eval_audio/400096.wav","disk":1,"offset":1257058134,"compressed_bytes":1890512,"size":2236602,"method":8,"crc32":50323016,"sha256":"9de4bda5734f1ee4861addc982662dd2aa6ca54f095130016e9797c1727d628a"},{"fname":"400169","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"chromakei","group_id":"fsd50k:u-754ee7a1b0a0","member":"FSD50K.eval_audio/400169.wav","disk":1,"offset":2120575575,"compressed_bytes":2199753,"size":2643754,"method":8,"crc32":1879815081,"sha256":"f13347fc33d2da8b31a04a5ddf169e4f85afe6016bc71fb803998819f7c50383"},{"fname":"418458","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"JavierZumer","group_id":"fsd50k:u-db4aaedb21d0","member":"FSD50K.eval_audio/418458.wav","disk":1,"offset":2472357703,"compressed_bytes":1112067,"size":1211830,"method":8,"crc32":4291544336,"sha256":"b83d1b74c87bdfeaba9cb40e1de2288505160117303595f423aa1437c011c487"},{"fname":"424871","split":"eval","target_label":"Insect","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"joanaseabra","group_id":"fsd50k:u-2d5b51395153","member":"FSD50K.eval_audio/424871.wav","disk":1,"offset":1885371094,"compressed_bytes":878894,"size":1017806,"method":8,"crc32":2831251443,"sha256":"a586814d7d9cf864382b2febb833cc87933c57157b9d8480ab00c527ab1de70d"},{"fname":"3187","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Pingel","group_id":"fsd50k:u-86243ba9e043","member":"FSD50K.dev_audio/3187.wav","disk":3,"offset":2115850250,"compressed_bytes":926899,"size":1140168,"method":8,"crc32":3053989520,"sha256":"728a8a434b23d675bc616820dfe6edad553b44fcd9651bf2c672d7e533167e1f"},{"fname":"3189","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Pingel","group_id":"fsd50k:u-86243ba9e043","member":"FSD50K.dev_audio/3189.wav","disk":2,"offset":3212083982,"compressed_bytes":1931413,"size":2235284,"method":8,"crc32":4271032813,"sha256":"480c9bb92cea39b17d36a86c385b49aa57438f1d7daad41655692a5c55e1b181"},{"fname":"3190","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Pingel","group_id":"fsd50k:u-86243ba9e043","member":"FSD50K.dev_audio/3190.wav","disk":1,"offset":748492479,"compressed_bytes":815134,"size":905294,"method":8,"crc32":1487849702,"sha256":"ee3084586b57ce2200e5898e4029d52639d116e41be983c3f24c9532ab9b46a6"},{"fname":"7917","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Anton","group_id":"fsd50k:u-42972c1b3e85","member":"FSD50K.dev_audio/7917.wav","disk":1,"offset":2042926093,"compressed_bytes":917128,"size":1122530,"method":8,"crc32":324941490,"sha256":"945dd73f58a8950ef8af3d6e26d286c8e917c8d8649bd70cf28e329f1b4a5323"},{"fname":"20198","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"stinkhorn","group_id":"fsd50k:u-dba5a6469ecc","member":"FSD50K.dev_audio/20198.wav","disk":0,"offset":1137992356,"compressed_bytes":954845,"size":1146958,"method":8,"crc32":3150983446,"sha256":"0863904923b956bb746ebf6345be175b8103362509a9cf219b29bcfe81671825"},{"fname":"23212","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"m1rk0","group_id":"fsd50k:u-71038f7956bf","member":"FSD50K.dev_audio/23212.wav","disk":1,"offset":2580851466,"compressed_bytes":1375877,"size":1539388,"method":8,"crc32":3986504182,"sha256":"b7fdc9766fbb331ecf0c3163f9d1722b70ec40bc5f63d2d6e614b42e48a4d2e5"},{"fname":"23218","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Noizemaker","group_id":"fsd50k:u-7a7f6d613b82","member":"FSD50K.dev_audio/23218.wav","disk":4,"offset":2758556530,"compressed_bytes":312054,"size":463158,"method":8,"crc32":603452836,"sha256":"c516c8e203b219b91c4d4bab1a20d209893f137d4b7ef8e05cc1be8d212c64b8"},{"fname":"53383","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/53383.wav","disk":4,"offset":3170977601,"compressed_bytes":882896,"size":1069470,"method":8,"crc32":43647529,"sha256":"ea0de35f3b6e66d112b6276d5c61ac5e8c46abe631f7576154012ecc16d38402"},{"fname":"68709","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"mikaelfernstrom","group_id":"fsd50k:u-d18d78ad1f85","member":"FSD50K.dev_audio/68709.wav","disk":2,"offset":2503789115,"compressed_bytes":736845,"size":784972,"method":8,"crc32":2545881303,"sha256":"0f4182ec07d093a80e78ca67ac2ef42690fc2694b46bb6b27c680013c73865f6"},{"fname":"74382","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"crop_circles","group_id":"fsd50k:u-177d83bbcee4","member":"FSD50K.dev_audio/74382.wav","disk":2,"offset":861587298,"compressed_bytes":823005,"size":1199094,"method":8,"crc32":1102470652,"sha256":"4388602c131cbe3aafba9f8d18bde0e1ec4a456ce4d1a71d8a200291c17a5c70"},{"fname":"74384","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"crop_circles","group_id":"fsd50k:u-177d83bbcee4","member":"FSD50K.dev_audio/74384.wav","disk":5,"offset":83211713,"compressed_bytes":582481,"size":902546,"method":8,"crc32":4263769524,"sha256":"feae4243c5649312192e1cb0a4b40a0139064ecb709bdfe405128d5f13593b82"},{"fname":"80590","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"miastodzwiekow","group_id":"fsd50k:u-999af4936c85","member":"FSD50K.dev_audio/80590.wav","disk":5,"offset":54243905,"compressed_bytes":2214224,"size":2358194,"method":8,"crc32":2652665913,"sha256":"3326ffa62b04630366ad4593bb0fea835ccb9bdcc0623506a1bf3b9e106e8dd0"},{"fname":"85365","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/85365.wav","disk":4,"offset":1130164070,"compressed_bytes":1987896,"size":2549020,"method":8,"crc32":4102905420,"sha256":"14fc0351a24749d1259ea26411d8b5571eff309db66899750807b1f73a729c07"},{"fname":"88771","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"tec studios","group_id":"fsd50k:u-be53219926a7","member":"FSD50K.dev_audio/88771.wav","disk":5,"offset":599253528,"compressed_bytes":331494,"size":352874,"method":8,"crc32":3724930196,"sha256":"2a4f61c5fe5070b0f14837ae17a152bb4e2c98d779d015e675be4cec57fa5200"},{"fname":"92992","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/92992.wav","disk":3,"offset":2534725402,"compressed_bytes":1745522,"size":1852066,"method":8,"crc32":1888746234,"sha256":"625819221eb175966d7b80a02f2620e56a5844381a7028156cff05075069edc2"},{"fname":"95864","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"paubg_pou","group_id":"fsd50k:u-8ac0f3f255b9","member":"FSD50K.dev_audio/95864.wav","disk":5,"offset":1437042450,"compressed_bytes":1661813,"size":1753298,"method":8,"crc32":2944008024,"sha256":"215c0a9f40299f654e570332d817f3e9cb3f255cdcf4f4b20911305f3b8fa6fa"},{"fname":"101855","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"thebigracket","group_id":"fsd50k:u-4871e89d2411","member":"FSD50K.dev_audio/101855.wav","disk":5,"offset":215878048,"compressed_bytes":845094,"size":976844,"method":8,"crc32":4090527118,"sha256":"e376468b3e31ad412e47f861a88e71cb9e487716b438ab6796124259e0ecb7c9"},{"fname":"101861","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"thebigracket","group_id":"fsd50k:u-4871e89d2411","member":"FSD50K.dev_audio/101861.wav","disk":3,"offset":334017864,"compressed_bytes":569283,"size":608462,"method":8,"crc32":4256178013,"sha256":"70579f926d13e23cbb047b075168b729c0b2822bef0b4825b07facdc8fef7ae1"},{"fname":"103429","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"confusion_music","group_id":"fsd50k:u-3c7d0fe58b74","member":"FSD50K.dev_audio/103429.wav","disk":3,"offset":1828351056,"compressed_bytes":280660,"size":305406,"method":8,"crc32":2034036718,"sha256":"7ed3f99e5f8ab7ae36c4cd33cc81a1a9470ffe19ee3a92f243b821efe4df250f"},{"fname":"103431","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"confusion_music","group_id":"fsd50k:u-3c7d0fe58b74","member":"FSD50K.dev_audio/103431.wav","disk":2,"offset":1594817297,"compressed_bytes":562824,"size":682310,"method":8,"crc32":2887692563,"sha256":"dbab1fa3d73eb4b75649c454486f2c61a973f2ffc24a6fa26e236c4f4b294af4"},{"fname":"145541","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"bmoreno","group_id":"fsd50k:u-136e2a534b61","member":"FSD50K.dev_audio/145541.wav","disk":1,"offset":1475636402,"compressed_bytes":109053,"size":116814,"method":8,"crc32":1554046584,"sha256":"cae6a81c269ae4436f9ed9eb782171b7d0edfb9326933d079176cc3163fdbdd4"},{"fname":"147925","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"bmoreno","group_id":"fsd50k:u-136e2a534b61","member":"FSD50K.dev_audio/147925.wav","disk":0,"offset":1068822220,"compressed_bytes":238079,"size":259790,"method":8,"crc32":366846841,"sha256":"b27766ed154115fdea52da542774e816a5a9f3fb7217bea412d2a727b6f02429"},{"fname":"156203","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Lau7","group_id":"fsd50k:u-a0c4d1ccc81d","member":"FSD50K.dev_audio/156203.wav","disk":4,"offset":1077969294,"compressed_bytes":310221,"size":330810,"method":8,"crc32":297263758,"sha256":"2a8c23c427cb613bf5da6895bccc23826d31791647642edda372bc78bec35a87"},{"fname":"156204","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Lau7","group_id":"fsd50k:u-a0c4d1ccc81d","member":"FSD50K.dev_audio/156204.wav","disk":0,"offset":2635356856,"compressed_bytes":387878,"size":433644,"method":8,"crc32":2784624319,"sha256":"35dc70e0c3105c5cf1c1cb72e20246497ef6ec776706848a1e0e6a6339026a5a"},{"fname":"162443","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"zendry423","group_id":"fsd50k:u-75845d5c8313","member":"FSD50K.dev_audio/162443.wav","disk":4,"offset":401726583,"compressed_bytes":973876,"size":1055822,"method":8,"crc32":108431521,"sha256":"701d7add5af7b065b8c693052de79cbd2740a273d1e97cbe9ed103fa4d22123d"},{"fname":"180671","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"AshFox","group_id":"fsd50k:u-cc615f4b066f","member":"FSD50K.dev_audio/180671.wav","disk":0,"offset":988513175,"compressed_bytes":403331,"size":491176,"method":8,"crc32":3948499870,"sha256":"c4b46c5c76724f61aaecfd557c6cb6ed0b42749923d11ceed06c43c78a3b4841"},{"fname":"180672","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"AshFox","group_id":"fsd50k:u-cc615f4b066f","member":"FSD50K.dev_audio/180672.wav","disk":4,"offset":781005377,"compressed_bytes":345373,"size":462952,"method":8,"crc32":1938153434,"sha256":"b72166a76f25631bb7ca5b1e7969729913c491fa17d69ba4b980defd5e42ab12"},{"fname":"180673","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"AshFox","group_id":"fsd50k:u-cc615f4b066f","member":"FSD50K.dev_audio/180673.wav","disk":3,"offset":1140746295,"compressed_bytes":336652,"size":445078,"method":8,"crc32":1004661440,"sha256":"5cf2f0e7a34cdbef72f416c32876c45686b3fda940fa14414641e2a44fbbca4b"},{"fname":"192389","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"madmini","group_id":"fsd50k:u-a20d6979495c","member":"FSD50K.dev_audio/192389.wav","disk":3,"offset":1892228705,"compressed_bytes":262958,"size":322638,"method":8,"crc32":3189743268,"sha256":"5005ff7cb127882a279958618bb1031702aa14d708dabb6f15755fcc39511daf"},{"fname":"192391","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"madmini","group_id":"fsd50k:u-a20d6979495c","member":"FSD50K.dev_audio/192391.wav","disk":3,"offset":873340962,"compressed_bytes":588586,"size":735054,"method":8,"crc32":2017049516,"sha256":"5179d667bb091b074fab5e865c088cfdd94fcd73b1a127ab605818e092081fbf"},{"fname":"192399","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"madmini","group_id":"fsd50k:u-a20d6979495c","member":"FSD50K.dev_audio/192399.wav","disk":3,"offset":3098007095,"compressed_bytes":484489,"size":659022,"method":8,"crc32":731109396,"sha256":"611d407c3e87cb8ac0266bd2110449199bd4e8188d9633234f1af6200e6856b3"},{"fname":"195655","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"absent1010","group_id":"fsd50k:u-2aa4e7b24b8f","member":"FSD50K.dev_audio/195655.wav","disk":4,"offset":3188321710,"compressed_bytes":2004535,"size":2641506,"method":8,"crc32":2310448769,"sha256":"972769e0626ec5da3ae4fae5fb163eeed5039dfd9f9dc9d1fb2dc76882d16953"},{"fname":"245694","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"TicAshfield","group_id":"fsd50k:u-f61aaf37c2fe","member":"FSD50K.dev_audio/245694.wav","disk":1,"offset":2890640752,"compressed_bytes":428887,"size":523838,"method":8,"crc32":3984052298,"sha256":"f29cc3c2e6016a8c8162360253b039715ddb55f8de004c6e30b7f43ad31f9257"},{"fname":"249920","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"hintringer","group_id":"fsd50k:u-c0a7eca6f3f6","member":"FSD50K.dev_audio/249920.wav","disk":0,"offset":1386763898,"compressed_bytes":1671598,"size":2399002,"method":8,"crc32":148464672,"sha256":"e2db09198603307612b753cc1fa4d2dea7cd1c60a922a78d82a3eed9398faa71"},{"fname":"251008","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"GildaCuevas","group_id":"fsd50k:u-a24408f77c9c","member":"FSD50K.dev_audio/251008.wav","disk":5,"offset":13419650,"compressed_bytes":263297,"size":289834,"method":8,"crc32":2511289730,"sha256":"c0d20088ac89bda2d1c1835ce93fb24ca100e626ce07eff2117634e270f615f3"},{"fname":"251969","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"cliftonmcarlson","group_id":"fsd50k:u-661d286debf5","member":"FSD50K.dev_audio/251969.wav","disk":0,"offset":1619579729,"compressed_bytes":500620,"size":710934,"method":8,"crc32":3380322815,"sha256":"815e69341e2e000e2e37052a64d39fb6260333e8d3739ccd3f8f932326315eeb"},{"fname":"255706","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"YleArkisto","group_id":"fsd50k:u-d9969ea40981","member":"FSD50K.dev_audio/255706.wav","disk":5,"offset":415024424,"compressed_bytes":1561944,"size":2210770,"method":8,"crc32":4198259219,"sha256":"a99df4052199ce5000a3781209cf6673f74a421479f45186d0243965da251add"},{"fname":"274962","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"rsn267","group_id":"fsd50k:u-40be72304602","member":"FSD50K.dev_audio/274962.wav","disk":4,"offset":2200179309,"compressed_bytes":1630645,"size":1774158,"method":8,"crc32":2538313042,"sha256":"4fb807592b96b8e8ab31ad28c4cd0ab20e36f45470d33d458bec67530128a8a3"},{"fname":"336549","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"nhaudio","group_id":"fsd50k:u-e83af7663fb0","member":"FSD50K.dev_audio/336549.wav","disk":0,"offset":879048271,"compressed_bytes":738272,"size":855736,"method":8,"crc32":871307046,"sha256":"805de84f52dd6ffdfdeb3fd8ec4f8d4c1de27c4cfada9e11e5bc79668f3ff790"},{"fname":"338125","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"akoros","group_id":"fsd50k:u-4baef153fc4c","member":"FSD50K.dev_audio/338125.wav","disk":5,"offset":945500089,"compressed_bytes":448686,"size":600310,"method":8,"crc32":4261015239,"sha256":"f24012cfbabafd81da4c7da9cbfd3d439f1809bc6224acb3dd86c50d8767285d"},{"fname":"338127","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"akoros","group_id":"fsd50k:u-4baef153fc4c","member":"FSD50K.dev_audio/338127.wav","disk":5,"offset":1775065215,"compressed_bytes":391799,"size":517518,"method":8,"crc32":372154649,"sha256":"152616915411b8220a32de2988d7adb5d7ca09a5f6ba05a893e8127a2da031ae"},{"fname":"338129","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"akoros","group_id":"fsd50k:u-4baef153fc4c","member":"FSD50K.dev_audio/338129.wav","disk":0,"offset":3067861305,"compressed_bytes":571061,"size":679208,"method":8,"crc32":1909734438,"sha256":"303e5ca87db071eff06002117d68df7d65c7f0fac750c7d2704ff0dc46b60435"},{"fname":"353881","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"cynthia7matty","group_id":"fsd50k:u-82ded0331b30","member":"FSD50K.dev_audio/353881.wav","disk":0,"offset":258478254,"compressed_bytes":1218438,"size":1577664,"method":8,"crc32":607882798,"sha256":"db3641868c6dc07398c398521f24d25aa5ac876bc73aa0fd521aa8e241fff4e9"},{"fname":"361406","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"richwise","group_id":"fsd50k:u-e4ab9481470e","member":"FSD50K.dev_audio/361406.wav","disk":5,"offset":62221774,"compressed_bytes":991487,"size":1236438,"method":8,"crc32":1827984893,"sha256":"331f4b5b5f99a9cf9e62357dafa4fe4854546a631d0d7a6ac5c7c02629497500"},{"fname":"368210","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dycmaster","group_id":"fsd50k:u-4bd06f7794ca","member":"FSD50K.dev_audio/368210.wav","disk":4,"offset":2246877215,"compressed_bytes":162054,"size":183732,"method":8,"crc32":1185516939,"sha256":"adfe0b3cc98ea9addc9616dbf8f9d03bf449726f0cddab2126e65d372a42d866"},{"fname":"368211","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dycmaster","group_id":"fsd50k:u-4bd06f7794ca","member":"FSD50K.dev_audio/368211.wav","disk":0,"offset":1312153796,"compressed_bytes":1182697,"size":1392078,"method":8,"crc32":3185722938,"sha256":"0b8cfd3c95e060db2e89e517eeb5034a4c5a096f01d0f55dd7b7ebd7aba6a94e"},{"fname":"368215","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dycmaster","group_id":"fsd50k:u-4bd06f7794ca","member":"FSD50K.dev_audio/368215.wav","disk":0,"offset":331370539,"compressed_bytes":687153,"size":814398,"method":8,"crc32":3756648605,"sha256":"671f42b39b5c1f89089f4fd0eb88bc5bf8de212322f9edebfd3c85eabfc1b6bc"},{"fname":"378340","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"13FPanska_Mikova_Adela","group_id":"fsd50k:u-f29ea6c94fe4","member":"FSD50K.dev_audio/378340.wav","disk":4,"offset":315677636,"compressed_bytes":1122619,"size":1449970,"method":8,"crc32":2310179310,"sha256":"b8308afc1ec39fc2f22642ad8f88e7e05340f98ecca22c7b7d5b8e60639b2cb2"},{"fname":"379915","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"theneedle.tv","group_id":"fsd50k:u-abcac5990df1","member":"FSD50K.dev_audio/379915.wav","disk":0,"offset":2804077180,"compressed_bytes":1795267,"size":1887558,"method":8,"crc32":1461089790,"sha256":"daa8810a5e4a77976a0e84b8cb0ca09930fb301f6e82964dbbb33554221375c2"},{"fname":"405019","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"kyles","group_id":"fsd50k:u-2bef0e7cc443","member":"FSD50K.dev_audio/405019.wav","disk":0,"offset":3130222309,"compressed_bytes":791332,"size":1236114,"method":8,"crc32":1983063365,"sha256":"32db13dc1b08b941c1888cfc180ee715d013ee36743de6a35481dba805bc261f"},{"fname":"406387","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"kyles","group_id":"fsd50k:u-2bef0e7cc443","member":"FSD50K.dev_audio/406387.wav","disk":4,"offset":597288832,"compressed_bytes":976651,"size":1147824,"method":8,"crc32":1508098054,"sha256":"99a7fb68576144e3687148ff991fcc6a725efce6b1279d36144a967cafb7813d"},{"fname":"412433","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"lenznbua","group_id":"fsd50k:u-6cea386bfd2d","member":"FSD50K.dev_audio/412433.wav","disk":1,"offset":121982393,"compressed_bytes":1442605,"size":1776462,"method":8,"crc32":2762565418,"sha256":"e3ecbb3e22aa8ec7417c930b50950622fe14be000c75b2ce9e21fae3a5b62cca"},{"fname":"412434","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"lenznbua","group_id":"fsd50k:u-6cea386bfd2d","member":"FSD50K.dev_audio/412434.wav","disk":3,"offset":262701876,"compressed_bytes":1637609,"size":1988430,"method":8,"crc32":1172814329,"sha256":"8572cb8c36c23c2427c6863f008fca5eed7df86c6d7d76e8b32a727b15860e6d"},{"fname":"412436","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"lenznbua","group_id":"fsd50k:u-6cea386bfd2d","member":"FSD50K.dev_audio/412436.wav","disk":2,"offset":1047560281,"compressed_bytes":1492674,"size":1840974,"method":8,"crc32":1254951494,"sha256":"eb46244b0575a03c1a538301e7fce3fba873b1063db5ea50f51a60081c47d288"},{"fname":"414263","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Silbad","group_id":"fsd50k:u-4a80d63691c7","member":"FSD50K.dev_audio/414263.wav","disk":5,"offset":1136812415,"compressed_bytes":1259589,"size":1337814,"method":8,"crc32":3019072778,"sha256":"58a7766cd4b059ddd2f9e167b3735ae7af4702dad9f2afe03f8e923d42e73eef"},{"fname":"429491","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"richwise","group_id":"fsd50k:u-e4ab9481470e","member":"FSD50K.dev_audio/429491.wav","disk":2,"offset":614581447,"compressed_bytes":776695,"size":2173784,"method":8,"crc32":2701913785,"sha256":"5d23f52a9220bb6ea398f7bf762efb94000e3e9b970215ca5a8f979e70e04620"},{"fname":"432804","split":"dev","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"JanIsAGuy","group_id":"fsd50k:u-16a03e89eedd","member":"FSD50K.dev_audio/432804.wav","disk":0,"offset":1847864954,"compressed_bytes":381459,"size":430054,"method":8,"crc32":3720608754,"sha256":"68518ff63346a75fc11c07879e0d81d70056b0c3528c256fe13d3ec025283521"},{"fname":"1045","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"RHumphries","group_id":"fsd50k:u-4a418faf9644","member":"FSD50K.eval_audio/1045.wav","disk":1,"offset":1835726482,"compressed_bytes":1382647,"size":2459094,"method":8,"crc32":2030735382,"sha256":"41d7b548fc67599e1ea762600ae6872482728e7a9eba9e42bf353f57eb412d97"},{"fname":"22459","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Erdie","group_id":"fsd50k:u-8e9da3e096ab","member":"FSD50K.eval_audio/22459.wav","disk":0,"offset":2846620588,"compressed_bytes":850512,"size":1334104,"method":8,"crc32":43770112,"sha256":"d4b3f2affdda8c96bd85e85995f565a3715eda47b97d3e737ea75242ab782814"},{"fname":"22460","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Erdie","group_id":"fsd50k:u-8e9da3e096ab","member":"FSD50K.eval_audio/22460.wav","disk":0,"offset":3135764341,"compressed_bytes":879004,"size":1334104,"method":8,"crc32":3474146672,"sha256":"e6433e82f5c3573fbcd830eb315c5629476048ac00fe6670a12b244b8f60cd48"},{"fname":"22615","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Erdie","group_id":"fsd50k:u-8e9da3e096ab","member":"FSD50K.eval_audio/22615.wav","disk":1,"offset":365360019,"compressed_bytes":1898580,"size":2491728,"method":8,"crc32":332658344,"sha256":"4af5f39944f1b96f71ad4b0bb969f2bb5b9e7525b2a478c8d49cdaef3917698b"},{"fname":"29579","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"man","group_id":"fsd50k:u-48b676e2b107","member":"FSD50K.eval_audio/29579.wav","disk":1,"offset":1633870480,"compressed_bytes":2016040,"size":2618020,"method":8,"crc32":3562889861,"sha256":"23789f826259ffda0689fd5c77b7563207883298b4d8feb2ee9409bcc8e03629"},{"fname":"49326","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Heigh-hoo","group_id":"fsd50k:u-f27da747f3d4","member":"FSD50K.eval_audio/49326.wav","disk":0,"offset":1649083120,"compressed_bytes":1635662,"size":1983502,"method":8,"crc32":452894670,"sha256":"e2e462e3396b065752324646ca4309d0da6153254d62029814d4fd17556641c3"},{"fname":"49328","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Heigh-hoo","group_id":"fsd50k:u-f27da747f3d4","member":"FSD50K.eval_audio/49328.wav","disk":0,"offset":659032705,"compressed_bytes":663591,"size":1131538,"method":8,"crc32":4004692822,"sha256":"755b1bcd71e17050caee87b65c067eac27056e291c136a5924a4c416cdb08372"},{"fname":"49329","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Heigh-hoo","group_id":"fsd50k:u-f27da747f3d4","member":"FSD50K.eval_audio/49329.wav","disk":1,"offset":2251952804,"compressed_bytes":779220,"size":1126690,"method":8,"crc32":682609684,"sha256":"b9b5379c1ae07949544a63fdd2dfae0625b58d57a7c03ea0da7d4293e28abaa0"},{"fname":"54271","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"jaava73","group_id":"fsd50k:u-84e01575eebc","member":"FSD50K.eval_audio/54271.wav","disk":0,"offset":313840587,"compressed_bytes":941276,"size":1081422,"method":8,"crc32":778225488,"sha256":"d0910dbc50c0b3652a5174db42092e74e46598f2e715816de6cd3848751668fc"},{"fname":"54512","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"jaava73","group_id":"fsd50k:u-84e01575eebc","member":"FSD50K.eval_audio/54512.wav","disk":0,"offset":2088045158,"compressed_bytes":121226,"size":132430,"method":8,"crc32":2457757603,"sha256":"77c7a7b03ec8c1cc7a784951faca2fc751f15bbc90654de08f90dfb9adc0f230"},{"fname":"59666","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"morgantj","group_id":"fsd50k:u-362d7392b44f","member":"FSD50K.eval_audio/59666.wav","disk":1,"offset":909719908,"compressed_bytes":788967,"size":889422,"method":8,"crc32":4067980681,"sha256":"db2d3f44fc3663f494b7b8a1e50ba07e18586aebf8d25250fc6a819e08a2a4d2"},{"fname":"59667","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"morgantj","group_id":"fsd50k:u-362d7392b44f","member":"FSD50K.eval_audio/59667.wav","disk":1,"offset":162176678,"compressed_bytes":1114679,"size":1336398,"method":8,"crc32":1265523574,"sha256":"df94da89f569a03d3d8c842c3c9c424c31fa20461f8d21652a72d6a2d9f85265"},{"fname":"62463","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"bsumusictech","group_id":"fsd50k:u-b4d45cfdfd1e","member":"FSD50K.eval_audio/62463.wav","disk":1,"offset":2958090590,"compressed_bytes":1093978,"size":1417422,"method":8,"crc32":3801980410,"sha256":"9e68e77b8a043929eacc5ebfdd33d078d81ad309bbd6dfb00c5e8abc430479ad"},{"fname":"109405","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Cmckeeman","group_id":"fsd50k:u-af7faaee71dd","member":"FSD50K.eval_audio/109405.wav","disk":1,"offset":2521804355,"compressed_bytes":849438,"size":1111632,"method":8,"crc32":1996005114,"sha256":"8626d887966b46f5944adae209831f708bf1878db6193e38ba69e7372b16c57a"},{"fname":"120145","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"kaldan","group_id":"fsd50k:u-2e5620ab7272","member":"FSD50K.eval_audio/120145.wav","disk":0,"offset":1392084037,"compressed_bytes":1397136,"size":1518752,"method":8,"crc32":3047489946,"sha256":"0fbf1721842b395df5d5ab6d7ee279b2784fd26e644c49e292decaa0e75a2030"},{"fname":"144941","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"mangowylder","group_id":"fsd50k:u-e95027abf98b","member":"FSD50K.eval_audio/144941.wav","disk":0,"offset":62289648,"compressed_bytes":1791932,"size":1882190,"method":8,"crc32":903049117,"sha256":"ae2e5f8a1d75a20be076d1b43df7bf42bda53e8049a2b6878d1477ac32f3f634"},{"fname":"148862","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"martian","group_id":"fsd50k:u-16b33ba57853","member":"FSD50K.eval_audio/148862.wav","disk":0,"offset":531446032,"compressed_bytes":941233,"size":1530092,"method":8,"crc32":750294738,"sha256":"97dfeb111a3335dda1db8f84138331f53a76c499ae67ac918706b925e4656109"},{"fname":"156095","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"marcolo91","group_id":"fsd50k:u-f0c635203aad","member":"FSD50K.eval_audio/156095.wav","disk":1,"offset":1968815933,"compressed_bytes":340816,"size":354314,"method":8,"crc32":1868748334,"sha256":"4a9ef7f5dc69e40050fee728cffaad7bf028dc4146e3daf4736da0a521f3b818"},{"fname":"186395","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"jeicob22","group_id":"fsd50k:u-8282739c1b04","member":"FSD50K.eval_audio/186395.wav","disk":0,"offset":1210181928,"compressed_bytes":1280040,"size":1572716,"method":8,"crc32":1229159176,"sha256":"79011d6ca703887227e94502157959f99b98c9dc2ebfb5458eb27be2e5a60e0e"},{"fname":"188006","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Motion_S","group_id":"fsd50k:u-6978dc0f7a37","member":"FSD50K.eval_audio/188006.wav","disk":0,"offset":778136983,"compressed_bytes":879121,"size":1058478,"method":8,"crc32":2436559785,"sha256":"c43bbc523a3c62a192cf108ed5cf8db9e28c20e61df500c55a9294a66b9325d6"},{"fname":"196694","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"soundmary","group_id":"fsd50k:u-51f62a00c5d6","member":"FSD50K.eval_audio/196694.wav","disk":0,"offset":1985190255,"compressed_bytes":463625,"size":611832,"method":8,"crc32":3126112167,"sha256":"a07d01ea4813e73604de86bc1512bd983c0968af884dbc11a88f4492e983fe20"},{"fname":"198611","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"extrafonia","group_id":"fsd50k:u-cf84aa55070a","member":"FSD50K.eval_audio/198611.wav","disk":0,"offset":10652823,"compressed_bytes":917569,"size":1181774,"method":8,"crc32":3544977759,"sha256":"d1d815d933d4f7af98167bf7c425371507cdfcd8910671e08a3c31514835f214"},{"fname":"211667","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Sclolex","group_id":"fsd50k:u-450889fb3127","member":"FSD50K.eval_audio/211667.wav","disk":0,"offset":668866881,"compressed_bytes":1665604,"size":2140870,"method":8,"crc32":387613839,"sha256":"edee39a9fedb38eebce3975cc68f0e9a3afb585b2c8c088d2ad3f4fb2ce21054"},{"fname":"380811","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Giannisk3","group_id":"fsd50k:u-8b01ccdadf78","member":"FSD50K.eval_audio/380811.wav","disk":1,"offset":1467215633,"compressed_bytes":232039,"size":386682,"method":8,"crc32":1811442280,"sha256":"0f798971d1431f42cee999959837205fc3ab4d80f8921d65f0c2c734f5805f56"},{"fname":"388080","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"monotraum","group_id":"fsd50k:u-050d6fe05972","member":"FSD50K.eval_audio/388080.wav","disk":0,"offset":760907482,"compressed_bytes":1615705,"size":1850632,"method":8,"crc32":1363225429,"sha256":"65559355acc554655abd4e4124c973ef7cf61ec89b7a219ec7a5760dbfab4bae"},{"fname":"397313","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Starscade","group_id":"fsd50k:u-2a715c389991","member":"FSD50K.eval_audio/397313.wav","disk":1,"offset":1481876873,"compressed_bytes":158395,"size":187566,"method":8,"crc32":3183762277,"sha256":"d05dbd734b76498533642065ca242d6331e41ce1c11b9ca5afcdcdd6c6bee0b1"},{"fname":"408624","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"poderarcano","group_id":"fsd50k:u-95af84b7275d","member":"FSD50K.eval_audio/408624.wav","disk":1,"offset":95080314,"compressed_bytes":482004,"size":498042,"method":8,"crc32":624335521,"sha256":"1af5c69a1fe40f7af07c279dd01ca4082d3463e86e1ac33a383669eeaa93698c"},{"fname":"424619","split":"eval","target_label":"Motorcycle","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Claudia96","group_id":"fsd50k:u-efde639b73d7","member":"FSD50K.eval_audio/424619.wav","disk":0,"offset":349271303,"compressed_bytes":1771430,"size":2637902,"method":8,"crc32":77348296,"sha256":"9219c2b476bfd17fcdf5f65de4c40dbf24cab1b62e1f86ec51e13e570f513bf4"},{"fname":"11619","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"psakion","group_id":"fsd50k:u-e8660d9ad121","member":"FSD50K.dev_audio/11619.wav","disk":4,"offset":1846451151,"compressed_bytes":1760504,"size":1866318,"method":8,"crc32":2903009521,"sha256":"e41c55c4994fbf2a12a46b373bdf9a2ce4ee901be4edd76bdc84751cb2b65ec3"},{"fname":"12653","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"mich3d","group_id":"fsd50k:u-c5022a862a85","member":"FSD50K.dev_audio/12653.wav","disk":1,"offset":1649564878,"compressed_bytes":478916,"size":733262,"method":8,"crc32":908696299,"sha256":"834b4aa0aabf5f7db4441d2839c25e3c6ee25f365bc095e1f369b0373419973b"},{"fname":"16748","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"kijjaz","group_id":"fsd50k:u-bba926a3e003","member":"FSD50K.dev_audio/16748.wav","disk":3,"offset":768466299,"compressed_bytes":38972,"size":90172,"method":8,"crc32":494803174,"sha256":"e1cba5b876f179fc679afb1b42f0a08221415963d6a3abfc9fb2b2638f445a20"},{"fname":"16751","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"kijjaz","group_id":"fsd50k:u-bba926a3e003","member":"FSD50K.dev_audio/16751.wav","disk":4,"offset":130411699,"compressed_bytes":72772,"size":181310,"method":8,"crc32":1433934980,"sha256":"745e1ce49c694275eee4826807368378808b10661c53f0da53519943a3d75d13"},{"fname":"22033","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Ionizing","group_id":"fsd50k:u-75ab56f740eb","member":"FSD50K.dev_audio/22033.wav","disk":0,"offset":1403426086,"compressed_bytes":2388958,"size":2634318,"method":8,"crc32":3221663140,"sha256":"e4bbc553ff2bf433270cccd34645ded2123f2952c253eed595940e3ed234dbe5"},{"fname":"23087","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"digifishmusic","group_id":"fsd50k:u-23878ad72831","member":"FSD50K.dev_audio/23087.wav","disk":3,"offset":2514281020,"compressed_bytes":2041824,"size":2495272,"method":8,"crc32":3123623367,"sha256":"e940ba7fc869d4eec7ee8f9f5fd8acae6294e5529c87b29a98ade1a0ac3fdeeb"},{"fname":"28650","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"gim-audio","group_id":"fsd50k:u-d2939a4a10a7","member":"FSD50K.dev_audio/28650.wav","disk":3,"offset":1093111202,"compressed_bytes":863848,"size":1144092,"method":8,"crc32":2742128805,"sha256":"cf439b86511f6f64ba8dc14198d7284f7abe1ec738111fd085b46a3d5bc5049a"},{"fname":"29675","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"artifact","group_id":"fsd50k:u-c7c5c1d70c5d","member":"FSD50K.dev_audio/29675.wav","disk":5,"offset":1459441217,"compressed_bytes":1095959,"size":1405006,"method":8,"crc32":519705515,"sha256":"61c62a944861922c538262197cd3b6c7ac82cb12ada5ee479bc446b9947073ef"},{"fname":"30341","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"junggle","group_id":"fsd50k:u-1b9a085f459e","member":"FSD50K.dev_audio/30341.wav","disk":5,"offset":575092754,"compressed_bytes":29573,"size":60600,"method":8,"crc32":516272343,"sha256":"075fa75c767227b6117f0babd7ca6939a3e5de533c11b42d0db5b443d14d4021"},{"fname":"30342","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"junggle","group_id":"fsd50k:u-1b9a085f459e","member":"FSD50K.dev_audio/30342.wav","disk":4,"offset":792112174,"compressed_bytes":39079,"size":59748,"method":8,"crc32":4125340011,"sha256":"a599b67236514bd919020746c3053c24050d550541ba24a974a643b5ca9c7366"},{"fname":"36062","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Charel Sytze","group_id":"fsd50k:u-56c76bccb0a0","member":"FSD50K.dev_audio/36062.wav","disk":1,"offset":1314279958,"compressed_bytes":1363886,"size":1503006,"method":8,"crc32":1842077915,"sha256":"06558bf3e1ad1329ccc36b221cea20c029d6965dbcd44f995a6f69df8d45bb45"},{"fname":"37355","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Charel Sytze","group_id":"fsd50k:u-56c76bccb0a0","member":"FSD50K.dev_audio/37355.wav","disk":1,"offset":2353313334,"compressed_bytes":1450362,"size":1856512,"method":8,"crc32":4089289311,"sha256":"40de542ae18c514680ccab445afb629a287e964dd563350f053f2e3ade6b6434"},{"fname":"46268","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"PhreaKsAccount","group_id":"fsd50k:u-d4dd5938ed27","member":"FSD50K.dev_audio/46268.wav","disk":3,"offset":1621097019,"compressed_bytes":36040,"size":80440,"method":8,"crc32":2044773689,"sha256":"b39045817635970d75bbc122cc9d2c4b18681caf25a0fc9f4a70a32f2c7eb38d"},{"fname":"46270","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"PhreaKsAccount","group_id":"fsd50k:u-d4dd5938ed27","member":"FSD50K.dev_audio/46270.wav","disk":5,"offset":1838769793,"compressed_bytes":38195,"size":83056,"method":8,"crc32":201158609,"sha256":"1b53d721d9a44dfdf7e88e19011e86732dd1634a86b586be80d2b77178c6601f"},{"fname":"50058","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"vibe_crc","group_id":"fsd50k:u-b7a9554c521a","member":"FSD50K.dev_audio/50058.wav","disk":4,"offset":1571353883,"compressed_bytes":1015997,"size":1191302,"method":8,"crc32":384611339,"sha256":"dba614aaf32b842d7636ec84910f3e910c2ccdd385c440eafca69b6a4aeb9c8f"},{"fname":"54964","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Northern Rebel","group_id":"fsd50k:u-52a73a953deb","member":"FSD50K.dev_audio/54964.wav","disk":3,"offset":1736535082,"compressed_bytes":883313,"size":1152558,"method":8,"crc32":1909012566,"sha256":"f425249f1f078eb29ec49267037226a58d8d306ee22eac342ed57c05097d91eb"},{"fname":"56286","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dkustic","group_id":"fsd50k:u-5ccb3817e707","member":"FSD50K.dev_audio/56286.wav","disk":4,"offset":165538296,"compressed_bytes":713673,"size":1110052,"method":8,"crc32":520728296,"sha256":"01f6f72a07fa11e3d7ef6591fdd99dbb032b1838d7d491f2893fa3a0bcbe0913"},{"fname":"60127","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"creebefu","group_id":"fsd50k:u-f1fffcdfce91","member":"FSD50K.dev_audio/60127.wav","disk":4,"offset":1191308414,"compressed_bytes":1924876,"size":2523986,"method":8,"crc32":516896376,"sha256":"e2e23b3693f5176687a33e34881ba658f5553a4160ed4087ec78d1a76cfc390f"},{"fname":"68719","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"mikaelfernstrom","group_id":"fsd50k:u-d18d78ad1f85","member":"FSD50K.dev_audio/68719.wav","disk":2,"offset":1164156899,"compressed_bytes":278127,"size":589902,"method":8,"crc32":1441573000,"sha256":"eb32ccb03dc6c43b6599d1b3749068003af65ab41ff83b6a6cefc61ec455374f"},{"fname":"74998","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"miastodzwiekow","group_id":"fsd50k:u-999af4936c85","member":"FSD50K.dev_audio/74998.wav","disk":4,"offset":1755805849,"compressed_bytes":968627,"size":1117518,"method":8,"crc32":2420236392,"sha256":"af2dff9fa17fcd22699ded989221739d851d1e89156ddfd3b6a03327695e5ce0"},{"fname":"84683","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"tomc1985","group_id":"fsd50k:u-b01752a73848","member":"FSD50K.dev_audio/84683.wav","disk":3,"offset":1204701736,"compressed_bytes":849959,"size":1328464,"method":8,"crc32":3578606137,"sha256":"b07968169d97caf06e0cf67cff4a9d01611984a6f9c5bcc1a9a9982918dc7394"},{"fname":"85142","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"becks77","group_id":"fsd50k:u-23af2542acd0","member":"FSD50K.dev_audio/85142.wav","disk":3,"offset":1707882097,"compressed_bytes":1779428,"size":2324914,"method":8,"crc32":2653188522,"sha256":"02f3283b049b8f5ddca4ae7fb145c2c89194c783289eb372a9a46f0b0646d947"},{"fname":"96728","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Ryding","group_id":"fsd50k:u-d029aa3be69b","member":"FSD50K.dev_audio/96728.wav","disk":1,"offset":1354579263,"compressed_bytes":1244617,"size":1694176,"method":8,"crc32":1365373495,"sha256":"74ad1fea8b4f7d2131d208635a5e79a16f61c49afc5e9afe028052b6406e693e"},{"fname":"106393","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"j1987","group_id":"fsd50k:u-0bd7aed9f374","member":"FSD50K.dev_audio/106393.wav","disk":1,"offset":1735171418,"compressed_bytes":1056540,"size":1159368,"method":8,"crc32":2473822761,"sha256":"7fbbc8a2e86d2f1c2e23689e9bf191f49c190c102d2e10754973042f9818031e"},{"fname":"110322","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"vinjatovix","group_id":"fsd50k:u-a89e24cca828","member":"FSD50K.dev_audio/110322.wav","disk":1,"offset":1369123313,"compressed_bytes":1109156,"size":1337136,"method":8,"crc32":2846263892,"sha256":"23b8b868a7677c1fcc3fcf2dfef57c25c50926ab004075581d1c4555c072ba33"},{"fname":"110324","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"vinjatovix","group_id":"fsd50k:u-a89e24cca828","member":"FSD50K.dev_audio/110324.wav","disk":5,"offset":674199599,"compressed_bytes":966878,"size":1194224,"method":8,"crc32":890519581,"sha256":"da3d802ac1993bd34aa59d7b75de4cc9089f52959401edc8e6835431b37e7057"},{"fname":"112741","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"iut_Paris8","group_id":"fsd50k:u-e3399b16301a","member":"FSD50K.dev_audio/112741.wav","disk":3,"offset":1550872398,"compressed_bytes":134579,"size":162894,"method":8,"crc32":175453468,"sha256":"976b62d15ca6fde287a3f283a959f6c83d69e0c94abb6fab8b7b3aeeae20d7a8"},{"fname":"120695","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"ramagochi","group_id":"fsd50k:u-eecd4e2a942d","member":"FSD50K.dev_audio/120695.wav","disk":0,"offset":1512013686,"compressed_bytes":2033185,"size":2270754,"method":8,"crc32":54016556,"sha256":"8b05426c841a7f0e0013249d44056f2618e585e3b7a22f13dd811d62e62d6520"},{"fname":"120699","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"ramagochi","group_id":"fsd50k:u-eecd4e2a942d","member":"FSD50K.dev_audio/120699.wav","disk":0,"offset":2686863790,"compressed_bytes":1128212,"size":1269454,"method":8,"crc32":1881863217,"sha256":"b104fbeeacb4918f0e19dbc3ed6f9eeefb0738e5529a3e12334edc48360f330b"},{"fname":"137022","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Jeffreys2","group_id":"fsd50k:u-a31d41c993e5","member":"FSD50K.dev_audio/137022.wav","disk":1,"offset":1307510776,"compressed_bytes":364204,"size":632748,"method":8,"crc32":1621804832,"sha256":"1687ba06f5db498819f5888f7f598b1b7e7d78f047fce8bfe584a37d35951fc9"},{"fname":"159173","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"cms4f","group_id":"fsd50k:u-c2ee8d651506","member":"FSD50K.dev_audio/159173.wav","disk":1,"offset":811183730,"compressed_bytes":1112474,"size":1512886,"method":8,"crc32":3037011822,"sha256":"9a9dd2e9242df6a2bf32238df992cb1082a1a9af625bcbdc10cc5c3486854538"},{"fname":"166317","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Adam_N","group_id":"fsd50k:u-124149624cb4","member":"FSD50K.dev_audio/166317.wav","disk":5,"offset":1894838396,"compressed_bytes":6626,"size":52538,"method":8,"crc32":2979775370,"sha256":"dedf4e1bdbcdf2f26c95ff0b2f882214d5f78b6e02339a6b21a9e9190cc37933"},{"fname":"166320","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Adam_N","group_id":"fsd50k:u-124149624cb4","member":"FSD50K.dev_audio/166320.wav","disk":2,"offset":462530930,"compressed_bytes":11019,"size":59000,"method":8,"crc32":759544732,"sha256":"feddf256a36d4968c04ee785d852f4f8af4a5f47fe3aec34a101c06eb48018d9"},{"fname":"166325","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Adam_N","group_id":"fsd50k:u-124149624cb4","member":"FSD50K.dev_audio/166325.wav","disk":4,"offset":3174983003,"compressed_bytes":11676,"size":60282,"method":8,"crc32":1726412831,"sha256":"d114156462b35b763e3a5cab14bc16544f94341cab77c84909a584e16bd82f45"},{"fname":"178571","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"copycatfilms","group_id":"fsd50k:u-21860d118e2b","member":"FSD50K.dev_audio/178571.wav","disk":3,"offset":2101883513,"compressed_bytes":1949909,"size":2338782,"method":8,"crc32":4204015022,"sha256":"8e297899f838e56b5f3150a4bc2cc09aef2aed197ef118e8f384a0a6e9970603"},{"fname":"180949","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"jc144940","group_id":"fsd50k:u-af7f8829f515","member":"FSD50K.dev_audio/180949.wav","disk":1,"offset":1180656811,"compressed_bytes":1752416,"size":2468814,"method":8,"crc32":3872346392,"sha256":"acb826a095b64903ad39726e38938f82aa1893d44a26fb98bca63f6ece40dedb"},{"fname":"197871","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Millavsb","group_id":"fsd50k:u-6afde805b54b","member":"FSD50K.dev_audio/197871.wav","disk":1,"offset":165713695,"compressed_bytes":64970,"size":190272,"method":8,"crc32":686366374,"sha256":"fe79370b342c23285132e41e9b085199013fd0988030997fe447dbb278509f71"},{"fname":"197885","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Millavsb","group_id":"fsd50k:u-6afde805b54b","member":"FSD50K.dev_audio/197885.wav","disk":3,"offset":1952669482,"compressed_bytes":45914,"size":115870,"method":8,"crc32":1582683582,"sha256":"843aae5d48199069e63260175136e1316ea2ff3c00f438d454ce6c687820aee9"},{"fname":"197900","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Millavsb","group_id":"fsd50k:u-6afde805b54b","member":"FSD50K.dev_audio/197900.wav","disk":4,"offset":846917558,"compressed_bytes":29898,"size":78946,"method":8,"crc32":415601112,"sha256":"658137d0a0f629a4476e2510e6ccc2ddb189bf76b30197eb08404f817bdef107"},{"fname":"201810","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"J_Bond","group_id":"fsd50k:u-58c48576b6fc","member":"FSD50K.dev_audio/201810.wav","disk":5,"offset":1665346784,"compressed_bytes":1211044,"size":1834062,"method":8,"crc32":2978836648,"sha256":"825a6a06977a2d026fdf691631db42bf0ba1a723e88b1be0702524f7de5cace8"},{"fname":"234877","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"gonzalocolosia","group_id":"fsd50k:u-c7506f68d98e","member":"FSD50K.dev_audio/234877.wav","disk":4,"offset":2387328625,"compressed_bytes":314857,"size":1154670,"method":8,"crc32":252344133,"sha256":"315ac8c3027d21b20139fc3c92fe3686352ef88f00ba54fcfb77189bbfcab846"},{"fname":"248110","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Kostas17","group_id":"fsd50k:u-8ff108727a25","member":"FSD50K.dev_audio/248110.wav","disk":3,"offset":26122706,"compressed_bytes":1629283,"size":2061244,"method":8,"crc32":2550837675,"sha256":"748aabc8cee197bc0ac6c28f7b683a6c0c07503001c8f1f8ab7e071091fec602"},{"fname":"251635","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"bmlake","group_id":"fsd50k:u-d6fb0f45caa8","member":"FSD50K.dev_audio/251635.wav","disk":2,"offset":2600147730,"compressed_bytes":615503,"size":880668,"method":8,"crc32":3336180001,"sha256":"6fa55055c77de6ea466e3fd774814edb1f54e4fa41ffde40a4a12d5c772b112e"},{"fname":"253763","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"ullsokk","group_id":"fsd50k:u-21283baa8f14","member":"FSD50K.dev_audio/253763.wav","disk":4,"offset":1281854183,"compressed_bytes":1731293,"size":2212134,"method":8,"crc32":652103847,"sha256":"56f7883d5c2c5c878f30becbfc44e29c101039fc9d52840c707b1acb57fa4390"},{"fname":"259978","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"heyim89yearsold","group_id":"fsd50k:u-ab903856cfdb","member":"FSD50K.dev_audio/259978.wav","disk":3,"offset":327924382,"compressed_bytes":1201068,"size":1679994,"method":8,"crc32":946251632,"sha256":"62b1ba42900f21e0c74413cd4c29694b11bc9ef403ef7639702454f188bbd4b5"},{"fname":"267221","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"gkillhour","group_id":"fsd50k:u-717fbd7f5dfc","member":"FSD50K.dev_audio/267221.wav","disk":4,"offset":2746744534,"compressed_bytes":27455,"size":44110,"method":8,"crc32":1235231579,"sha256":"3afefe16e55307eb2dc78688765ef8b6d1a62e70f1632f1f4ffe5c849ffc5943"},{"fname":"270918","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Sevin7","group_id":"fsd50k:u-3ada7d296da4","member":"FSD50K.dev_audio/270918.wav","disk":5,"offset":1607386141,"compressed_bytes":553567,"size":588416,"method":8,"crc32":401101613,"sha256":"232dff5411149130e7338af0429c58ed554796b2ccea188da2d113f86e66317a"},{"fname":"273870","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"beskhu","group_id":"fsd50k:u-c60b78bd374f","member":"FSD50K.dev_audio/273870.wav","disk":3,"offset":2244897685,"compressed_bytes":22141,"size":33806,"method":8,"crc32":3435521751,"sha256":"ca1a03273ea3c37da316f982916dbd75486769d1dda6d686b2e299494f24d004"},{"fname":"324417","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Rajaraman","group_id":"fsd50k:u-c45af6bd34de","member":"FSD50K.dev_audio/324417.wav","disk":4,"offset":718230701,"compressed_bytes":1040812,"size":1205726,"method":8,"crc32":2746201898,"sha256":"79717670b7366d1c1f4446eadf0ae4682034c8559b14cdacdc183358f2f28190"},{"fname":"328995","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"davenatale","group_id":"fsd50k:u-ab82a9d375b9","member":"FSD50K.dev_audio/328995.wav","disk":3,"offset":1721773537,"compressed_bytes":2091038,"size":2525262,"method":8,"crc32":2077561453,"sha256":"5a6a5961e24b27d6357984618df6712724526f8a2b24c55911aa15fabe3f65a6"},{"fname":"332180","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Trollarch2","group_id":"fsd50k:u-4e473e8de534","member":"FSD50K.dev_audio/332180.wav","disk":4,"offset":217555733,"compressed_bytes":870872,"size":973806,"method":8,"crc32":4104761872,"sha256":"a759955f5a0790701ba776aa296682c30acabc4a8ac523b25fe6c746bb3b72e7"},{"fname":"334149","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"svampen","group_id":"fsd50k:u-b44dacb0c7bf","member":"FSD50K.dev_audio/334149.wav","disk":0,"offset":1505894142,"compressed_bytes":495089,"size":567746,"method":8,"crc32":2203456863,"sha256":"c3463945555de908d42027f86fb79b5a0deb4cc0b1bb9a118851f1b0f80f8f10"},{"fname":"336404","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"moai15","group_id":"fsd50k:u-9a8eb1b26ed3","member":"FSD50K.dev_audio/336404.wav","disk":0,"offset":1988022792,"compressed_bytes":50254,"size":129520,"method":8,"crc32":1232552072,"sha256":"e37f3180e862d579fc94853e20ccfdfb754627465e33bed7cbd199f54476f983"},{"fname":"336406","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"moai15","group_id":"fsd50k:u-9a8eb1b26ed3","member":"FSD50K.dev_audio/336406.wav","disk":5,"offset":109560866,"compressed_bytes":44853,"size":115448,"method":8,"crc32":1082018690,"sha256":"187ab7aaeb3640999cc92b7ccdd2add9173e230d707afb3631bf4efcb10a6aa4"},{"fname":"336421","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"moai15","group_id":"fsd50k:u-9a8eb1b26ed3","member":"FSD50K.dev_audio/336421.wav","disk":1,"offset":827417384,"compressed_bytes":50455,"size":129520,"method":8,"crc32":3839098541,"sha256":"aba5c8af971b31fd18fb81d6325a5a8790eebfffc26ec7620c9543a347eeddad"},{"fname":"339900","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/339900.wav","disk":4,"offset":2385869515,"compressed_bytes":1459025,"size":1655462,"method":8,"crc32":1126103635,"sha256":"4bd6e0f80ea91e3b92e307beb919b45eba700dd466c22c040ce220c35a748926"},{"fname":"345032","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"iut_Paris8","group_id":"fsd50k:u-e3399b16301a","member":"FSD50K.dev_audio/345032.wav","disk":0,"offset":2508925037,"compressed_bytes":685928,"size":737606,"method":8,"crc32":3095345060,"sha256":"ce15cce94286140a9eec062a35623602ff999fb7e277711b246be92849993c39"},{"fname":"350812","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Natty23","group_id":"fsd50k:u-caffa130c132","member":"FSD50K.dev_audio/350812.wav","disk":0,"offset":2990512528,"compressed_bytes":102686,"size":105544,"method":8,"crc32":1993433196,"sha256":"93d14da780137c1f49d12c8d3984bded5ddf9b167d27ff1acc29dd2d3ea977b5"},{"fname":"360340","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"ivolipa","group_id":"fsd50k:u-a7a9c9359b5e","member":"FSD50K.dev_audio/360340.wav","disk":4,"offset":1753631403,"compressed_bytes":1400800,"size":1770344,"method":8,"crc32":105278306,"sha256":"0462fe676f1a5231418efaeeb82ea308ed0682639b2004b945c576c70a638db0"},{"fname":"363868","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Juanma1008","group_id":"fsd50k:u-46d311ff34b5","member":"FSD50K.dev_audio/363868.wav","disk":5,"offset":485260897,"compressed_bytes":542402,"size":745044,"method":8,"crc32":1766779266,"sha256":"2b92f71ef70a751f57251647cdb4abb4bd3504d08e7db5b1bee899e315ed2c33"},{"fname":"365007","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"SAUL92","group_id":"fsd50k:u-e77c5107dac3","member":"FSD50K.dev_audio/365007.wav","disk":4,"offset":2183842841,"compressed_bytes":374286,"size":684982,"method":8,"crc32":1564707460,"sha256":"de71c0533f9aff1d677c94c7c2683fce2351bc34695e1491bc473e91f2dc695c"},{"fname":"379755","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"MicrosoftSam","group_id":"fsd50k:u-f2b4f768a902","member":"FSD50K.dev_audio/379755.wav","disk":3,"offset":183218907,"compressed_bytes":1261088,"size":1888334,"method":8,"crc32":2608361050,"sha256":"62b0ae3758a5c39c722e4d8c54a04ceeb3433816e2321b5c5022b2daf84f1485"},{"fname":"382752","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"audrey13rose","group_id":"fsd50k:u-23bf41889171","member":"FSD50K.dev_audio/382752.wav","disk":3,"offset":1061911214,"compressed_bytes":944251,"size":1247310,"method":8,"crc32":1582785844,"sha256":"6e697d909ae512756e10b2b1a1dec1a50ce5b21835baedc3f24fcd0e37758434"},{"fname":"391687","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"JPolito","group_id":"fsd50k:u-478fcfb3df4b","member":"FSD50K.dev_audio/391687.wav","disk":1,"offset":2346563505,"compressed_bytes":980814,"size":1115506,"method":8,"crc32":2408403410,"sha256":"9fa64187b93d42d43be2842deea495b07bd1ef77a69257722e5c5ce55e225140"},{"fname":"395415","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"mosvane","group_id":"fsd50k:u-d8032b7b1819","member":"FSD50K.dev_audio/395415.wav","disk":3,"offset":1205551779,"compressed_bytes":850147,"size":1473840,"method":8,"crc32":2227816483,"sha256":"9b2f7d9016e5e6b12fafeb8796538db188aae7762e2ef8aa357035dea5d2cb5b"},{"fname":"398742","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Anthousai","group_id":"fsd50k:u-c1ff1994cff5","member":"FSD50K.dev_audio/398742.wav","disk":5,"offset":595609404,"compressed_bytes":1743603,"size":2294646,"method":8,"crc32":42507536,"sha256":"5dd812e65ffa7b1286a582ef36891ef04937a763181f8417db9351f5b859904f"},{"fname":"401013","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Mexhe","group_id":"fsd50k:u-ddf559947f3c","member":"FSD50K.dev_audio/401013.wav","disk":0,"offset":2578090782,"compressed_bytes":665855,"size":708174,"method":8,"crc32":1215449794,"sha256":"b46ba84c4440c7a2161e76f9a06b44417399685e7108231eb65808b2d773400c"},{"fname":"406763","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"kretopi","group_id":"fsd50k:u-00cb39572631","member":"FSD50K.dev_audio/406763.wav","disk":0,"offset":1304080635,"compressed_bytes":32504,"size":70872,"method":8,"crc32":1621166172,"sha256":"1dc27e3d4c07e35d76457eea19accc84d51388e5bcdb1f67fa17d7591860d029"},{"fname":"418099","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"KleiwLinaraki","group_id":"fsd50k:u-7f9c8743880a","member":"FSD50K.dev_audio/418099.wav","disk":5,"offset":1520956300,"compressed_bytes":522285,"size":793878,"method":8,"crc32":2771862564,"sha256":"5ad9f67f776672a596ea75fbc33616bc8669d29f940a532108a6d62291fa2392"},{"fname":"423964","split":"dev","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"ritasofio","group_id":"fsd50k:u-e91f96d7854b","member":"FSD50K.dev_audio/423964.wav","disk":0,"offset":54154253,"compressed_bytes":611923,"size":1013700,"method":8,"crc32":1836399590,"sha256":"e4c010fa461476bd208c5984bc7908bfaa3a33afbefe9a8481881695074c62b7"},{"fname":"2531","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"RHumphries","group_id":"fsd50k:u-4a418faf9644","member":"FSD50K.eval_audio/2531.wav","disk":0,"offset":2072840310,"compressed_bytes":1917209,"size":2408798,"method":8,"crc32":1231361411,"sha256":"22eb079b40eb71740064ad7b263d910a5f1e1c25cf9c263bdec36188bc393d02"},{"fname":"35367","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"fresco","group_id":"fsd50k:u-a12de35e2537","member":"FSD50K.eval_audio/35367.wav","disk":1,"offset":2417074563,"compressed_bytes":523372,"size":590690,"method":8,"crc32":1736958265,"sha256":"6bf8f027e8d87eee9f9def1be9cf5ef19f6fc8e39a3fe564f1728bb81a02f3c2"},{"fname":"42978","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"jaava73","group_id":"fsd50k:u-84e01575eebc","member":"FSD50K.eval_audio/42978.wav","disk":1,"offset":134432097,"compressed_bytes":280810,"size":526414,"method":8,"crc32":2050706923,"sha256":"4b10bc4a6274ce874093cc9196c139e58c8f4c2426da27d60484ff603f514d48"},{"fname":"47050","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"gelo_papas","group_id":"fsd50k:u-565060dab64c","member":"FSD50K.eval_audio/47050.wav","disk":0,"offset":2427776861,"compressed_bytes":48775,"size":104216,"method":8,"crc32":784362063,"sha256":"5193f620447adebb110eef57322dde36de645fe256b8c071565239d7307aed28"},{"fname":"56306","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Q.K.","group_id":"fsd50k:u-dcf5bb796d42","member":"FSD50K.eval_audio/56306.wav","disk":0,"offset":2097387550,"compressed_bytes":1734998,"size":2272254,"method":8,"crc32":516374837,"sha256":"1cac245f928a9fa50228d5c5857425d549b09a0e2097e476aa63004fe6817b51"},{"fname":"83742","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"bsumusictech","group_id":"fsd50k:u-b4d45cfdfd1e","member":"FSD50K.eval_audio/83742.wav","disk":0,"offset":2530355970,"compressed_bytes":395251,"size":681482,"method":8,"crc32":918043874,"sha256":"ad694022fe1ce034ebd838c844bff518b9e0c55137c947daae90d54808441d35"},{"fname":"86344","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"surrey_film","group_id":"fsd50k:u-39e2363e0a8d","member":"FSD50K.eval_audio/86344.wav","disk":0,"offset":807422824,"compressed_bytes":1043983,"size":1982962,"method":8,"crc32":1920706225,"sha256":"99a27db4320d7adfaa4a1e33f95484dabe6000e6ed09ce8f24591b0b361e13cc"},{"fname":"114521","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"juskiddink","group_id":"fsd50k:u-1add789cab59","member":"FSD50K.eval_audio/114521.wav","disk":0,"offset":2807685724,"compressed_bytes":2038035,"size":2541996,"method":8,"crc32":3307231351,"sha256":"17e4a031cf97b0933cadc3e96136fac49d2b744c99636738ce14b2d83d979e13"},{"fname":"115254","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"matucha","group_id":"fsd50k:u-7bd6247eaccd","member":"FSD50K.eval_audio/115254.wav","disk":1,"offset":1328905375,"compressed_bytes":351909,"size":561230,"method":8,"crc32":3266105000,"sha256":"46f308e5c924a5941005226ab1e42a31463148dbdd4043a9c8aceb0e36e19744"},{"fname":"119500","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"ingudios","group_id":"fsd50k:u-900c3e34ff7d","member":"FSD50K.eval_audio/119500.wav","disk":1,"offset":2673184332,"compressed_bytes":943029,"size":1345804,"method":8,"crc32":2206581272,"sha256":"276bcaed88c68a584ebdc41517d63d9c9af854b349c273372a88c50fd68a1f34"},{"fname":"122808","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"geodylabs","group_id":"fsd50k:u-c0a9f590c446","member":"FSD50K.eval_audio/122808.wav","disk":1,"offset":1681696997,"compressed_bytes":977479,"size":1359438,"method":8,"crc32":2906300726,"sha256":"99e22c327aa33bf933c44358b3db5269eab0e3e85ce49aeba457bd6fd692e8ad"},{"fname":"196669","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"soundmary","group_id":"fsd50k:u-51f62a00c5d6","member":"FSD50K.eval_audio/196669.wav","disk":0,"offset":2265757060,"compressed_bytes":266928,"size":376870,"method":8,"crc32":3080741627,"sha256":"6476723424eada7e81a55c29d2ef329d6d4df6975c551004cec02ca9e7d375ac"},{"fname":"197738","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"ragamuffin","group_id":"fsd50k:u-1501f210951b","member":"FSD50K.eval_audio/197738.wav","disk":0,"offset":2179946606,"compressed_bytes":1876373,"size":2583470,"method":8,"crc32":3140686274,"sha256":"a0af891c09b8e42aede6ee9919bb85a4c8424fd412c70cc28f425e1cb1722d64"},{"fname":"199926","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"thedapperdan","group_id":"fsd50k:u-7ecbbdce8ef9","member":"FSD50K.eval_audio/199926.wav","disk":0,"offset":2859110934,"compressed_bytes":1999263,"size":2449458,"method":8,"crc32":3373869885,"sha256":"c5185c50ce392b5320151bf4e14bf1064f03a30a11eacf57e8b41c2a81504526"},{"fname":"337791","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"miradeshazer","group_id":"fsd50k:u-47d819570bde","member":"FSD50K.eval_audio/337791.wav","disk":0,"offset":1839259852,"compressed_bytes":1385176,"size":1552114,"method":8,"crc32":1858819678,"sha256":"93c8da7e8abc54eac6f0b08fe11808285e77afa4adcb4a6da18cc5f154b3d1b7"},{"fname":"347853","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"cognito perceptu","group_id":"fsd50k:u-5dc0240bad00","member":"FSD50K.eval_audio/347853.wav","disk":0,"offset":2203141686,"compressed_bytes":1247402,"size":1678330,"method":8,"crc32":1924083597,"sha256":"78b40b22d0f670b37c8419094aa19426c263beef5121986aa6eb7b2202a141d4"},{"fname":"369989","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"northern87","group_id":"fsd50k:u-8a4685d77507","member":"FSD50K.eval_audio/369989.wav","disk":0,"offset":1063346662,"compressed_bytes":769825,"size":1190778,"method":8,"crc32":2839824274,"sha256":"38842a953de73ffe4521d74e4320fdee60c122c5b55781bca9900d0268945461"},{"fname":"387904","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"bajko","group_id":"fsd50k:u-63b1e01daac0","member":"FSD50K.eval_audio/387904.wav","disk":1,"offset":643601440,"compressed_bytes":113620,"size":412658,"method":8,"crc32":3633460056,"sha256":"82ec04b9ad0b7239514b482240273e325cba750b3053a8caf3c0dbf45866988c"},{"fname":"400076","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"harrietniamh","group_id":"fsd50k:u-ad718eae1ae9","member":"FSD50K.eval_audio/400076.wav","disk":0,"offset":1579825217,"compressed_bytes":1134304,"size":1454184,"method":8,"crc32":3402321539,"sha256":"4b2eebf23aa7b66847877cce67ed1aafa78290f3009f000f263127dc8583a38f"},{"fname":"400077","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"harrietniamh","group_id":"fsd50k:u-ad718eae1ae9","member":"FSD50K.eval_audio/400077.wav","disk":1,"offset":1611488572,"compressed_bytes":979372,"size":1380812,"method":8,"crc32":4154380261,"sha256":"c6f107d811ab66d738b9edc4810b43ed158641c3f4228e54927e624813e9037e"},{"fname":"402549","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"DrMaysta","group_id":"fsd50k:u-e1f5d66365cd","member":"FSD50K.eval_audio/402549.wav","disk":0,"offset":2928754591,"compressed_bytes":936424,"size":1668900,"method":8,"crc32":1521390000,"sha256":"faa52671d556f8cacf4b7bae69cda15e39344ba3245b9992bc4dcd382f9a2505"},{"fname":"427752","split":"eval","target_label":"Rain","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"simongray","group_id":"fsd50k:u-b67742ff26e8","member":"FSD50K.eval_audio/427752.wav","disk":0,"offset":1656814200,"compressed_bytes":1772248,"size":2040408,"method":8,"crc32":317325230,"sha256":"1e45304b1c89bffc26de6b4f57dfd6708a5b8405c76963d4b80974590ef697ed"},{"fname":"2215","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Andrew Duke","group_id":"fsd50k:u-a07c88d9e23e","member":"FSD50K.dev_audio/2215.wav","disk":1,"offset":1832390959,"compressed_bytes":242186,"size":313422,"method":8,"crc32":2765472188,"sha256":"3177aa0292c5317272e28d969dd6dc251013a91cf0807168802c2637edbfefd0"},{"fname":"15547","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"LG","group_id":"fsd50k:u-b870b7809747","member":"FSD50K.dev_audio/15547.wav","disk":1,"offset":2424804998,"compressed_bytes":262611,"size":308674,"method":8,"crc32":1356423781,"sha256":"edc1aa7d64abd10208ff7672878f6b135560cca1437c7f3f5eb60bad0cff986b"},{"fname":"15548","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"LG","group_id":"fsd50k:u-b870b7809747","member":"FSD50K.dev_audio/15548.wav","disk":4,"offset":890241493,"compressed_bytes":387060,"size":431444,"method":8,"crc32":3248154828,"sha256":"fe344dae60410b25c2612f34fdd1a022c34e7c032a40a0eaa0e51238d14d2bba"},{"fname":"17510","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/17510.wav","disk":1,"offset":1995799652,"compressed_bytes":1351680,"size":1518684,"method":8,"crc32":2207484134,"sha256":"d1023ecbfa08a4f671092913055365962b590663d6647aeb1aa72ade2d6787e7"},{"fname":"51034","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"dobroide","group_id":"fsd50k:u-d79289fa9122","member":"FSD50K.dev_audio/51034.wav","disk":2,"offset":3151594366,"compressed_bytes":1031190,"size":1310112,"method":8,"crc32":2048616064,"sha256":"108ad1cac164e22299d8022c644d7dffa2683052d7f54f1f1093c395fa2d4092"},{"fname":"68749","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"mikaelfernstrom","group_id":"fsd50k:u-d18d78ad1f85","member":"FSD50K.dev_audio/68749.wav","disk":3,"offset":2729918667,"compressed_bytes":464736,"size":696854,"method":8,"crc32":3541045469,"sha256":"9867b1ae6c5deeffb32ae792dbb7703cbcd5dd57748ee31e455d7616bde45527"},{"fname":"69509","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Zixem","group_id":"fsd50k:u-9b46c2251544","member":"FSD50K.dev_audio/69509.wav","disk":0,"offset":883998471,"compressed_bytes":506741,"size":872526,"method":8,"crc32":3223118591,"sha256":"d1da12c4b61b7cf30094a746daa256f02214874a2af08e622f5c747a5cad0114"},{"fname":"75646","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"NoiseCollector","group_id":"fsd50k:u-0697aba1f475","member":"FSD50K.dev_audio/75646.wav","disk":1,"offset":1769986047,"compressed_bytes":84799,"size":104526,"method":8,"crc32":3255017522,"sha256":"daefd2dffb9f1a8c9a4607f8a3b6bb49514f0f794c073af60625fc8c30f2f6f4"},{"fname":"78430","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Benboncan","group_id":"fsd50k:u-05af64f71b1e","member":"FSD50K.dev_audio/78430.wav","disk":5,"offset":548641796,"compressed_bytes":2421939,"size":2628408,"method":8,"crc32":2633931,"sha256":"43f36d3b0a8e016128ba4c328cba9e53ecbaa68ff061720991989671b7f271d7"},{"fname":"79253","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"RoofDog","group_id":"fsd50k:u-1154f857b6b1","member":"FSD50K.dev_audio/79253.wav","disk":1,"offset":835976740,"compressed_bytes":298498,"size":726940,"method":8,"crc32":1084691550,"sha256":"729327caecca2acfca12c3a1f06b92df8400349c9970a96eab00227f0ff878ef"},{"fname":"85570","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"nmscher","group_id":"fsd50k:u-fdc7fc81c6d4","member":"FSD50K.dev_audio/85570.wav","disk":5,"offset":1540545334,"compressed_bytes":689267,"size":1141404,"method":8,"crc32":4279782602,"sha256":"6c4d546c28c021ec8e3508c002556e7e40462e08dfcc659d28fbcae1e38b9963"},{"fname":"100632","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"edbles","group_id":"fsd50k:u-af66ce56fb05","member":"FSD50K.dev_audio/100632.wav","disk":2,"offset":441965900,"compressed_bytes":429358,"size":444750,"method":8,"crc32":3703453380,"sha256":"dbfdcafe97fba2129133601d4f8b67096812a68dfca6650064bdfc1ccb6e2f3d"},{"fname":"100643","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"edbles","group_id":"fsd50k:u-af66ce56fb05","member":"FSD50K.dev_audio/100643.wav","disk":3,"offset":155696449,"compressed_bytes":563549,"size":589902,"method":8,"crc32":2194066686,"sha256":"6e474958c9ae858d8696fb823b1b7162ed1820e4b5851d29821c2e90d4cbb07c"},{"fname":"109291","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"vinjatovix","group_id":"fsd50k:u-a89e24cca828","member":"FSD50K.dev_audio/109291.wav","disk":0,"offset":1987485707,"compressed_bytes":537000,"size":1042840,"method":8,"crc32":1749999264,"sha256":"0d891ab0e6239cb263a80f4a4965c263fd9b71c780456301fa17c9ad8b2816b8"},{"fname":"145230","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"alanmckinney","group_id":"fsd50k:u-7d28da2859ac","member":"FSD50K.dev_audio/145230.wav","disk":5,"offset":1941273592,"compressed_bytes":2387762,"size":2538922,"method":8,"crc32":1132856190,"sha256":"5aecc9225acb54754bd85a285a5c79ac5e1a2c4346b9737338220fb2d390e0bc"},{"fname":"151853","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"carroll27","group_id":"fsd50k:u-1c221a451210","member":"FSD50K.dev_audio/151853.wav","disk":2,"offset":223703284,"compressed_bytes":336292,"size":761538,"method":8,"crc32":3970621729,"sha256":"bb7891b53c6389764b5257b85ee413b8191fc759a4ce0ca83842f8126cfe2b63"},{"fname":"152980","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"dianser72","group_id":"fsd50k:u-6ef3f17d82db","member":"FSD50K.dev_audio/152980.wav","disk":3,"offset":355814146,"compressed_bytes":449159,"size":487612,"method":8,"crc32":3397810668,"sha256":"a5aaf952c3343c4162ab56876d7befc892588d35876f9dda2c928a970afc655b"},{"fname":"166707","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Snapper4298","group_id":"fsd50k:u-831da112893d","member":"FSD50K.dev_audio/166707.wav","disk":3,"offset":2239038898,"compressed_bytes":680,"size":547310,"method":8,"crc32":634281820,"sha256":"6a0f89e403fb8ae8a7ef05e105467ba129879b7ed452a7433c9c93b48096fd81"},{"fname":"177059","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"ajkerekes","group_id":"fsd50k:u-1e1fc2e3422c","member":"FSD50K.dev_audio/177059.wav","disk":2,"offset":834431561,"compressed_bytes":459092,"size":664372,"method":8,"crc32":1642477177,"sha256":"c6b844c4e6fe50d809328a789fbfa13da2ad2e7d10ac954d482cb293f7e857c7"},{"fname":"178084","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"ajkerekes","group_id":"fsd50k:u-1e1fc2e3422c","member":"FSD50K.dev_audio/178084.wav","disk":4,"offset":963612439,"compressed_bytes":844321,"size":1225278,"method":8,"crc32":1790593299,"sha256":"caf5877196cd7a1dd3175d4a415829961f0b32a98a87c923727866bef4cdba6c"},{"fname":"179899","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"timsc","group_id":"fsd50k:u-defe87587941","member":"FSD50K.dev_audio/179899.wav","disk":2,"offset":736845736,"compressed_bytes":1236559,"size":1393608,"method":8,"crc32":405893436,"sha256":"0f3b5e5ff228d072ad2ffad0ba75f53c618928e684afab74288e31dabc2a2e5d"},{"fname":"179903","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"timsc","group_id":"fsd50k:u-defe87587941","member":"FSD50K.dev_audio/179903.wav","disk":1,"offset":2520743489,"compressed_bytes":463404,"size":535674,"method":8,"crc32":436175092,"sha256":"971f0b6584b8947e74f16f95102358d95521757a242d4fc5428d9aab3ef2e1f4"},{"fname":"179907","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"timsc","group_id":"fsd50k:u-defe87587941","member":"FSD50K.dev_audio/179907.wav","disk":4,"offset":158292029,"compressed_bytes":422628,"size":469236,"method":8,"crc32":2149452898,"sha256":"f6091c4fd0d653c0fb007e24517cdc6dc39a1958df5d22b80cf692b755062c41"},{"fname":"182906","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"toiletrolltube","group_id":"fsd50k:u-898a3f52b9ab","member":"FSD50K.dev_audio/182906.wav","disk":4,"offset":2605475453,"compressed_bytes":474859,"size":606084,"method":8,"crc32":808177187,"sha256":"7a8e01eaf4fe5d815c949fcbcb93ca735303ed3fff5ca26f2bdbb720838367a3"},{"fname":"182907","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"toiletrolltube","group_id":"fsd50k:u-898a3f52b9ab","member":"FSD50K.dev_audio/182907.wav","disk":5,"offset":1441918389,"compressed_bytes":303060,"size":423858,"method":8,"crc32":641944804,"sha256":"9cb7529ab7608917ad943fa574fd3da02a339c7f96e52cdabca5b7ee46917f8e"},{"fname":"182923","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"toiletrolltube","group_id":"fsd50k:u-898a3f52b9ab","member":"FSD50K.dev_audio/182923.wav","disk":4,"offset":2166151419,"compressed_bytes":306785,"size":384246,"method":8,"crc32":4049004332,"sha256":"7cead651b62b2607b168fdd90930c17731da815150c4205427262d7f9699157c"},{"fname":"190675","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"davidferoli","group_id":"fsd50k:u-ce280a33a753","member":"FSD50K.dev_audio/190675.wav","disk":2,"offset":416518636,"compressed_bytes":1385579,"size":1590594,"method":8,"crc32":3125188910,"sha256":"5dac87248d5a7c11462e8da6c4360b730933695d38be125dc0d5a99251869df3"},{"fname":"194842","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"xspackets","group_id":"fsd50k:u-e53db5bd3fac","member":"FSD50K.dev_audio/194842.wav","disk":3,"offset":241522215,"compressed_bytes":1134744,"size":1622094,"method":8,"crc32":318521207,"sha256":"248c6647471be59088e6f66e8df9bce6931e41dc5dab1873f69df1e07331b7aa"},{"fname":"204531","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"peridactyloptrix","group_id":"fsd50k:u-8694697bee1c","member":"FSD50K.dev_audio/204531.wav","disk":3,"offset":913003561,"compressed_bytes":411811,"size":872822,"method":8,"crc32":1365406769,"sha256":"50027a2d91c0ea76b01e80e2a200d09c271af73a7b327c07c519b708959a50dd"},{"fname":"208849","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Insestanyricarda","group_id":"fsd50k:u-568563aa5b58","member":"FSD50K.dev_audio/208849.wav","disk":4,"offset":1493730178,"compressed_bytes":530008,"size":583310,"method":8,"crc32":861388349,"sha256":"3408bfa3e727da99585b64f3cf14e355d870f7428708c33d2faa7905e173bcda"},{"fname":"209044","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Insestanyricarda","group_id":"fsd50k:u-568563aa5b58","member":"FSD50K.dev_audio/209044.wav","disk":0,"offset":1178034448,"compressed_bytes":1345096,"size":1499470,"method":8,"crc32":3767305809,"sha256":"8298dd36317196587c425e5d68fc1ea46ea24f4152d79a71fbf93d8a3a42ac67"},{"fname":"209205","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"whirlproductions","group_id":"fsd50k:u-512ff8a4915a","member":"FSD50K.dev_audio/209205.wav","disk":1,"offset":1293997777,"compressed_bytes":959892,"size":1308526,"method":8,"crc32":3408671227,"sha256":"4343c579d104cd8dcfe5a94181551c50d4f89c5ab470065f31a96cead0e58f11"},{"fname":"213037","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Taira Komori","group_id":"fsd50k:u-d05198fcfdb9","member":"FSD50K.dev_audio/213037.wav","disk":3,"offset":2948705422,"compressed_bytes":2177744,"size":2352156,"method":8,"crc32":26106539,"sha256":"5f435706e2e8fe5786b6d478bf684ec5e055be84f96a94a0e275f86b91899830"},{"fname":"216936","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Zabuhailo","group_id":"fsd50k:u-3a68bf7506a2","member":"FSD50K.dev_audio/216936.wav","disk":1,"offset":338664677,"compressed_bytes":479502,"size":770374,"method":8,"crc32":1205094613,"sha256":"d8411e7dd178dba0d8cd548c6b578dfaa533ff0422bc64e9894fa0cc568706f2"},{"fname":"237324","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"nmarciniegasm","group_id":"fsd50k:u-f1d8b13c2fe6","member":"FSD50K.dev_audio/237324.wav","disk":4,"offset":2378494458,"compressed_bytes":1403509,"size":1886382,"method":8,"crc32":987333014,"sha256":"d21c5d6c1690a7b4d78f2f04911eba724a95dd35954012dcb68ee167bacfac61"},{"fname":"238797","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"OneSoundToRuleThemAll","group_id":"fsd50k:u-c80aa311f077","member":"FSD50K.dev_audio/238797.wav","disk":3,"offset":381391469,"compressed_bytes":194910,"size":269824,"method":8,"crc32":4292886255,"sha256":"361da30839efda41f881f6e2aaebc31bc046e7e63fc0710d430dfd2d58d76975"},{"fname":"238798","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"OneSoundToRuleThemAll","group_id":"fsd50k:u-c80aa311f077","member":"FSD50K.dev_audio/238798.wav","disk":4,"offset":2172599992,"compressed_bytes":282784,"size":389284,"method":8,"crc32":1738985882,"sha256":"c429fed836353f117f45f00e903351b1761a65f9ba5a305d764e4107514bcc6a"},{"fname":"238803","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"OneSoundToRuleThemAll","group_id":"fsd50k:u-c80aa311f077","member":"FSD50K.dev_audio/238803.wav","disk":0,"offset":2234145971,"compressed_bytes":562665,"size":653250,"method":8,"crc32":3280062227,"sha256":"ee6cffe8775e9eda28f1cc164a67505e235793ebb08b4dfbd44651aa55dd3700"},{"fname":"244296","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"sherandu","group_id":"fsd50k:u-2b37f2031482","member":"FSD50K.dev_audio/244296.wav","disk":1,"offset":1447716493,"compressed_bytes":855267,"size":1131764,"method":8,"crc32":641252703,"sha256":"33e502bde5b991c0526030d71648790cb5d9712d3eae05fc710e976ba0d53d9e"},{"fname":"250653","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"pieltain","group_id":"fsd50k:u-2676ae3e067c","member":"FSD50K.dev_audio/250653.wav","disk":1,"offset":575659660,"compressed_bytes":1035449,"size":1158862,"method":8,"crc32":2503819005,"sha256":"62c3baeb5786d284dac9dec5c4af1973fe2396b6a7f3f0d7b9f32a3f995e5f8e"},{"fname":"250725","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"univ_lyon3","group_id":"fsd50k:u-1f41785243cf","member":"FSD50K.dev_audio/250725.wav","disk":2,"offset":1442945065,"compressed_bytes":443481,"size":495694,"method":8,"crc32":3323059726,"sha256":"fc5f8064da1f3eca9a2fb7631e18a2aef4479576614971cf026563b0b1cbf4ea"},{"fname":"252276","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Reitanna","group_id":"fsd50k:u-abc3334f2efa","member":"FSD50K.dev_audio/252276.wav","disk":4,"offset":1930164071,"compressed_bytes":384653,"size":427086,"method":8,"crc32":3411111190,"sha256":"ce03044fd13259fe4965781df2acfc8e7a76cd425ece31241a0500f228f416b2"},{"fname":"253612","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"xtrgamr","group_id":"fsd50k:u-b139e561c76e","member":"FSD50K.dev_audio/253612.wav","disk":2,"offset":1677809491,"compressed_bytes":136228,"size":167880,"method":8,"crc32":2586087384,"sha256":"8864eca479ca13bd0e95e3ec19e84a31152505d5e2d23cbf16a0d83b17b541cc"},{"fname":"256862","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Kodack","group_id":"fsd50k:u-d98c11f59e95","member":"FSD50K.dev_audio/256862.wav","disk":4,"offset":1606843362,"compressed_bytes":2091534,"size":2246838,"method":8,"crc32":2149372377,"sha256":"3b4515cfa06a5e0ef6fac76ceeadd6afa2bc02e0c354a9ccd12705848b0485ec"},{"fname":"275640","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Sevin7","group_id":"fsd50k:u-3ada7d296da4","member":"FSD50K.dev_audio/275640.wav","disk":4,"offset":150894729,"compressed_bytes":279761,"size":318016,"method":8,"crc32":3243566601,"sha256":"681951d3105337b37a2f7278b48202d7ed94d0165aa9019e1abef022a7ab59c1"},{"fname":"327771","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Jeffreys2","group_id":"fsd50k:u-a31d41c993e5","member":"FSD50K.dev_audio/327771.wav","disk":1,"offset":2787033943,"compressed_bytes":1605076,"size":2198094,"method":8,"crc32":3397789435,"sha256":"5a1aa014da4919b88d7f8a040ee8f1e7581a5c4c7a1b314293e98466b09424fa"},{"fname":"334903","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"equiloud","group_id":"fsd50k:u-75f12e66a239","member":"FSD50K.dev_audio/334903.wav","disk":4,"offset":1635086864,"compressed_bytes":1564724,"size":1675878,"method":8,"crc32":147635337,"sha256":"1f27c24a6795cce0fae8560e46aa233ec38ca5bc3a3b8be3d8f9cbe5eaf9fd99"},{"fname":"336652","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"chrillz3r","group_id":"fsd50k:u-bad1ccdfccf3","member":"FSD50K.dev_audio/336652.wav","disk":2,"offset":579153862,"compressed_bytes":657293,"size":901702,"method":8,"crc32":2491764939,"sha256":"3c5eda7923424f8a165ae5001b2b55d6c82ea6befdf1acdb30942ef56dd80687"},{"fname":"340218","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"kingof_thelab","group_id":"fsd50k:u-9a8e940fc86b","member":"FSD50K.dev_audio/340218.wav","disk":3,"offset":2809209726,"compressed_bytes":1305838,"size":1643074,"method":8,"crc32":3209185714,"sha256":"7fa723d344dfa2d46549021a6107ec3ced5b0fb0f65476fd82145e9b6433499e"},{"fname":"341874","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Baldirideltasona","group_id":"fsd50k:u-74856a351950","member":"FSD50K.dev_audio/341874.wav","disk":1,"offset":1413603207,"compressed_bytes":726606,"size":828528,"method":8,"crc32":1621788516,"sha256":"b883504c89ea691c3484d1aba8238c3c13aecfd37a103c5c5b2710b7ecadeaf8"},{"fname":"350979","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"AlienXXX","group_id":"fsd50k:u-0a44b5962b11","member":"FSD50K.dev_audio/350979.wav","disk":3,"offset":996392495,"compressed_bytes":1110398,"size":1632814,"method":8,"crc32":3870275640,"sha256":"b5c009aefa8e737ef7f007b12a6fdda6f0e347f3179025a7e8e52acd2aa76fb6"},{"fname":"351243","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"AlienXXX","group_id":"fsd50k:u-0a44b5962b11","member":"FSD50K.dev_audio/351243.wav","disk":2,"offset":928166629,"compressed_bytes":136117,"size":228190,"method":8,"crc32":357528182,"sha256":"5268db62f7c6e0f0039eab694b79869c9918ecef07482d616c57f6e851f922eb"},{"fname":"351503","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"elliotrambach","group_id":"fsd50k:u-4676d846b28c","member":"FSD50K.dev_audio/351503.wav","disk":1,"offset":1474478711,"compressed_bytes":638323,"size":812962,"method":8,"crc32":26894228,"sha256":"85c88bb71c08c3c7c60e886ba328d0ddb5c1d6d0184945af69025fe8fc676a00"},{"fname":"363935","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"lordpoque25","group_id":"fsd50k:u-17d3cce68918","member":"FSD50K.dev_audio/363935.wav","disk":5,"offset":768075636,"compressed_bytes":183820,"size":248450,"method":8,"crc32":2322255704,"sha256":"aa3378d0a499d253dbe7e9034c598f76bb01d9817b28c5f812fc07fb7ee88c1b"},{"fname":"363937","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"lordpoque25","group_id":"fsd50k:u-17d3cce68918","member":"FSD50K.dev_audio/363937.wav","disk":4,"offset":3098901990,"compressed_bytes":45551,"size":73462,"method":8,"crc32":2881122878,"sha256":"084efe2d4551920337a5d4c1416d5adaeb69fbaf806d423d04544d5501fddb52"},{"fname":"371717","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Iamgiorgio","group_id":"fsd50k:u-362875bf10eb","member":"FSD50K.dev_audio/371717.wav","disk":4,"offset":127338421,"compressed_bytes":885133,"size":1164318,"method":8,"crc32":2400510667,"sha256":"06658de34d5146b6b5dd464e719199a29f68b7d26060c94000da8eee35ba8b50"},{"fname":"379468","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"vandale","group_id":"fsd50k:u-d8651eb69c37","member":"FSD50K.dev_audio/379468.wav","disk":4,"offset":1758215434,"compressed_bytes":273069,"size":349330,"method":8,"crc32":1512131557,"sha256":"40c44069d114312c21e82111bed5255c786d859b5253f715e591359fc4fc198f"},{"fname":"402801","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Lisework","group_id":"fsd50k:u-d6d1d9fd71ce","member":"FSD50K.dev_audio/402801.wav","disk":4,"offset":2963319085,"compressed_bytes":785911,"size":1080270,"method":8,"crc32":3860828673,"sha256":"f0dc83c29d4e12091cc3e2018e2ad4cb1ca37818a1e8a3b5d1f5a889c5cabb0c"},{"fname":"407010","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"erothebest","group_id":"fsd50k:u-578f02a8dfac","member":"FSD50K.dev_audio/407010.wav","disk":1,"offset":1340993004,"compressed_bytes":918288,"size":1292850,"method":8,"crc32":1324784620,"sha256":"9233c4f4569b874f3c3539bce53dd4c2e1749573c359a103ba1bfd5377c506a2"},{"fname":"407012","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"erothebest","group_id":"fsd50k:u-578f02a8dfac","member":"FSD50K.dev_audio/407012.wav","disk":4,"offset":2834267755,"compressed_bytes":753458,"size":1213418,"method":8,"crc32":2846934683,"sha256":"934e694865105f9665c2ef962ff32830890a2e7a560f192936d1f14ad67a32ed"},{"fname":"407014","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"erothebest","group_id":"fsd50k:u-578f02a8dfac","member":"FSD50K.dev_audio/407014.wav","disk":5,"offset":1235630978,"compressed_bytes":577924,"size":856568,"method":8,"crc32":1482034033,"sha256":"cd8b1d2330557ab11003c3fcda7f29f8b4b9b79629e230be4b48f6b08bcf6ab2"},{"fname":"414303","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"RadioOAF","group_id":"fsd50k:u-7fbcfc2fe1de","member":"FSD50K.dev_audio/414303.wav","disk":4,"offset":3187769832,"compressed_bytes":551793,"size":770766,"method":8,"crc32":3602991291,"sha256":"60554bd0c49e42d1193480133a84c1392fa69e98c5a9df9474607f0529ea31dc"},{"fname":"420297","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"tonik-95","group_id":"fsd50k:u-fe2cb1ee98b9","member":"FSD50K.dev_audio/420297.wav","disk":4,"offset":52125305,"compressed_bytes":174397,"size":191470,"method":8,"crc32":447838719,"sha256":"c6cea29ffd680e035cd00223255cc86a4cd4a422cc8b089dce03d2931e8a1ee3"},{"fname":"420305","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"tonik-95","group_id":"fsd50k:u-fe2cb1ee98b9","member":"FSD50K.dev_audio/420305.wav","disk":4,"offset":1510288549,"compressed_bytes":116942,"size":121338,"method":8,"crc32":4144580697,"sha256":"c54bbcbf14b10f7aae100e7b3f58e3acdf442af09d98a1242beaaa6b96f30e4b"},{"fname":"420307","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"tonik-95","group_id":"fsd50k:u-fe2cb1ee98b9","member":"FSD50K.dev_audio/420307.wav","disk":3,"offset":1217061791,"compressed_bytes":74935,"size":95452,"method":8,"crc32":2464886950,"sha256":"fd626bf9d839705383b4a2d9f9fc8870dbd9811d3ca50449a635a32b3039b45e"},{"fname":"422865","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"ciaranm758","group_id":"fsd50k:u-6ff979d6f451","member":"FSD50K.dev_audio/422865.wav","disk":0,"offset":2681026100,"compressed_bytes":256196,"size":397588,"method":8,"crc32":480704396,"sha256":"462570f620c9cd41f2d9f9712ab2ad4caeed47f3dc985a744296993a9c87e936"},{"fname":"427245","split":"dev","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"GET_Accel","group_id":"fsd50k:u-49d56feba9ee","member":"FSD50K.dev_audio/427245.wav","disk":2,"offset":2119579577,"compressed_bytes":87871,"size":116814,"method":8,"crc32":916038135,"sha256":"6e3c5dceaedbcdff55a84740aa739dd0443f106e27ab9787385aca907eaeb7b3"},{"fname":"17923","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"cognito perceptu","group_id":"fsd50k:u-5dc0240bad00","member":"FSD50K.eval_audio/17923.wav","disk":0,"offset":2140402732,"compressed_bytes":446122,"size":546238,"method":8,"crc32":1880953861,"sha256":"34b0e2cce954ecb143eee754f384590d47fd12a7e19475bb06e236edee6387fb"},{"fname":"22682","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Erdie","group_id":"fsd50k:u-8e9da3e096ab","member":"FSD50K.eval_audio/22682.wav","disk":0,"offset":1369924490,"compressed_bytes":1930618,"size":2229578,"method":8,"crc32":1886857974,"sha256":"12263dc9f4c57307ac1d2d36b7768a17bff118f37a0fefba1a6a53b01a3f6f1b"},{"fname":"22719","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Erdie","group_id":"fsd50k:u-8e9da3e096ab","member":"FSD50K.eval_audio/22719.wav","disk":1,"offset":1389509296,"compressed_bytes":1611768,"size":2149952,"method":8,"crc32":4180318334,"sha256":"aff967d530a261c1c45b7528dda83375c20d717769f77e01f91960c160437b98"},{"fname":"26213","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Walter_Odington","group_id":"fsd50k:u-43f441a4e7af","member":"FSD50K.eval_audio/26213.wav","disk":0,"offset":2937753628,"compressed_bytes":1257590,"size":1483342,"method":8,"crc32":3844997896,"sha256":"d0ed74c46c41844431f77c39bc32b0d3081aa041449d8359fe4562607e5489e7"},{"fname":"29530","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"eliasheuninck","group_id":"fsd50k:u-863406f3fa1e","member":"FSD50K.eval_audio/29530.wav","disk":1,"offset":1915323453,"compressed_bytes":922521,"size":1167228,"method":8,"crc32":1549254851,"sha256":"e612a3b4d5c5d96e78f529f6f18745b38b1f2a2b5780755bde9d45316e7c870f"},{"fname":"30216","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Erdie","group_id":"fsd50k:u-8e9da3e096ab","member":"FSD50K.eval_audio/30216.wav","disk":1,"offset":2539048068,"compressed_bytes":1943144,"size":2537676,"method":8,"crc32":120762658,"sha256":"8709a63dc2b248b1b1d9aa14371be32c17067783daf701e4da5e398e42e22ba9"},{"fname":"68891","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"simongray","group_id":"fsd50k:u-b67742ff26e8","member":"FSD50K.eval_audio/68891.wav","disk":0,"offset":2637624768,"compressed_bytes":876285,"size":1002766,"method":8,"crc32":1874887190,"sha256":"cc956240fb35ba56092e4558ae0d9f42810752570428cc099f648b2c8481e65a"},{"fname":"87963","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"guitarguy1985","group_id":"fsd50k:u-ef77d9dba19d","member":"FSD50K.eval_audio/87963.wav","disk":0,"offset":2060217370,"compressed_bytes":1312257,"size":1597518,"method":8,"crc32":2884660215,"sha256":"d15f9ef832803cf714de14079f369912f7d3be357f22963ce4bf44d6ef3e5217"},{"fname":"91444","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"uair01","group_id":"fsd50k:u-65e698efb727","member":"FSD50K.eval_audio/91444.wav","disk":0,"offset":3126097237,"compressed_bytes":755463,"size":876622,"method":8,"crc32":3849199845,"sha256":"9d77fa4fcd8c2fbb1e89967ef5ce41f48f5c466b6e018edef7607c26e6183ae1"},{"fname":"113175","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Fasolt","group_id":"fsd50k:u-24a0521614ee","member":"FSD50K.eval_audio/113175.wav","disk":0,"offset":3164788873,"compressed_bytes":595914,"size":694856,"method":8,"crc32":1620253546,"sha256":"0616eab1b3297100171dacb5acecbfdc9af8cd1af05d8d45f9e7822539158f7c"},{"fname":"121943","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"hantorio","group_id":"fsd50k:u-a06a88904be1","member":"FSD50K.eval_audio/121943.wav","disk":1,"offset":959050928,"compressed_bytes":1317997,"size":2066496,"method":8,"crc32":2443786606,"sha256":"a5d80f32b4efeaf82a978675a8a68ad02972af9b2d1548d61c72455f22239224"},{"fname":"127127","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"westernsynthetics","group_id":"fsd50k:u-f0013200da0a","member":"FSD50K.eval_audio/127127.wav","disk":0,"offset":3195250602,"compressed_bytes":623585,"size":773666,"method":8,"crc32":1960357495,"sha256":"c120bfb79c8f1c46a8d29f69abbc404a5fd4551671a131fcb55a2700d0022ead"},{"fname":"135034","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"MrLindstrom","group_id":"fsd50k:u-f0649ffdd932","member":"FSD50K.eval_audio/135034.wav","disk":1,"offset":2356271658,"compressed_bytes":492628,"size":529278,"method":8,"crc32":507197113,"sha256":"683e11e6a68cd2b9795cc1fccee5e671efc3415679feca84794a2218abd9ee06"},{"fname":"135429","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"OTBTechno","group_id":"fsd50k:u-deef7e992b5b","member":"FSD50K.eval_audio/135429.wav","disk":1,"offset":2675447841,"compressed_bytes":1756571,"size":2099022,"method":8,"crc32":768944315,"sha256":"2c92b23ce4970dd012c055ade133d46e6b39a4a2a46d5ed1159cb891c3890c49"},{"fname":"185457","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"mshahen","group_id":"fsd50k:u-6f4eb55606c5","member":"FSD50K.eval_audio/185457.wav","disk":0,"offset":799773138,"compressed_bytes":1526567,"size":2051768,"method":8,"crc32":3372934692,"sha256":"68455a1ea74ca28fdd15f1d7d724d5fc142d935cee0db5d51890ca2039550368"},{"fname":"198857","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"bone666138","group_id":"fsd50k:u-9c573414ebf1","member":"FSD50K.eval_audio/198857.wav","disk":1,"offset":2280063452,"compressed_bytes":1193436,"size":1856090,"method":8,"crc32":3143377161,"sha256":"9390f34c2f72e6a75e3ea1b630a5979247da39cb735bcf907674b7767128e21d"},{"fname":"217800","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"minian89","group_id":"fsd50k:u-d66a8df8543c","member":"FSD50K.eval_audio/217800.wav","disk":0,"offset":1971212225,"compressed_bytes":2013081,"size":2266742,"method":8,"crc32":1161666619,"sha256":"f5c7fb12f01e39213c5869c0fbd4d982fed06c207ac1a48131d7fa522e1f95b6"},{"fname":"218386","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Motion_S","group_id":"fsd50k:u-6978dc0f7a37","member":"FSD50K.eval_audio/218386.wav","disk":0,"offset":525200980,"compressed_bytes":150173,"size":177488,"method":8,"crc32":1839303850,"sha256":"a63cfb11533e1821f563364a8ad99cee2b59ed976d40782f9228673a55dca068"},{"fname":"219636","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"andrescompovigo","group_id":"fsd50k:u-9959b1835963","member":"FSD50K.eval_audio/219636.wav","disk":1,"offset":796573086,"compressed_bytes":767307,"size":882078,"method":8,"crc32":1895979622,"sha256":"8611326478919739996ef96c7ce5f7ce975bdd38b3659edaf57656ca1f258f75"},{"fname":"238522","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"ggc9","group_id":"fsd50k:u-93b5d97b22c8","member":"FSD50K.eval_audio/238522.wav","disk":0,"offset":436424913,"compressed_bytes":312109,"size":348842,"method":8,"crc32":4110791914,"sha256":"e6d934d96cfde587bfbec43abf25cb21c5365077b07e9f0d1202a4b0d99012c2"},{"fname":"275466","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"adharca","group_id":"fsd50k:u-be4f96b72473","member":"FSD50K.eval_audio/275466.wav","disk":1,"offset":47140455,"compressed_bytes":584303,"size":898308,"method":8,"crc32":2634687432,"sha256":"cc234858ecb2871f363e63b41544fbe15ab73ab121799a122a54ff294dc2fb46"},{"fname":"319700","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/licenses/by/3.0/","uploader":"Peacewaves","group_id":"fsd50k:u-7d6827158205","member":"FSD50K.eval_audio/319700.wav","disk":0,"offset":410801013,"compressed_bytes":999777,"size":1234878,"method":8,"crc32":3773341288,"sha256":"9cb2d871ca127f63f2495f56f8ad8dd0262664e1bc264d94e79e6f903a6aa989"},{"fname":"344887","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"thearxx08","group_id":"fsd50k:u-de910c1ffd56","member":"FSD50K.eval_audio/344887.wav","disk":1,"offset":91643346,"compressed_bytes":1632961,"size":1741902,"method":8,"crc32":4135520499,"sha256":"caeeb421f0fe39e97b591fb8d113918522b01acbc28a3825cf3d9c5f203c9921"},{"fname":"347557","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Kinoton","group_id":"fsd50k:u-3a502bca35df","member":"FSD50K.eval_audio/347557.wav","disk":0,"offset":3184749304,"compressed_bytes":1327915,"size":1668682,"method":8,"crc32":2713458533,"sha256":"29eee8ff36ddeebdccb472adddb980c87c89de7767f8cc59bd2b486eeaa242cf"},{"fname":"380810","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Giannisk3","group_id":"fsd50k:u-8b01ccdadf78","member":"FSD50K.eval_audio/380810.wav","disk":0,"offset":294276757,"compressed_bytes":444110,"size":646018,"method":8,"crc32":878935551,"sha256":"7c465bd5e82df07b4d08cf972d1393ec859bc54ce83ec6953a0b865796edc706"},{"fname":"382591","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"macdaddyno1","group_id":"fsd50k:u-f7168755a878","member":"FSD50K.eval_audio/382591.wav","disk":0,"offset":2791604752,"compressed_bytes":1252538,"size":1352526,"method":8,"crc32":3224442413,"sha256":"df446bfe1a2a7013cd1b8284a1220c9752729156bca460e2a6f26d98d096ff47"},{"fname":"396995","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"macdaddyno1","group_id":"fsd50k:u-f7168755a878","member":"FSD50K.eval_audio/396995.wav","disk":1,"offset":2053701452,"compressed_bytes":2144959,"size":2513742,"method":8,"crc32":2650796245,"sha256":"ff5cdd852d4ef81f8c7b830e4a506d450c6c9f45d46d1fdc29db021c212ce878"},{"fname":"417998","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"JavierZumer","group_id":"fsd50k:u-db4aaedb21d0","member":"FSD50K.eval_audio/417998.wav","disk":0,"offset":2285916188,"compressed_bytes":1142857,"size":1332252,"method":8,"crc32":1348119980,"sha256":"339144471fe58b576e54fae179ba8209359a41cdaa934a17e74a250fa442ccf3"},{"fname":"424740","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"Joaofmms","group_id":"fsd50k:u-0ecde5fc5e4b","member":"FSD50K.eval_audio/424740.wav","disk":1,"offset":531083649,"compressed_bytes":1301339,"size":1554510,"method":8,"crc32":542490324,"sha256":"67abaf7f29a4baeacfd9413953e04b2f68f181ecb5134c3f85e7ec188ef5ee56"},{"fname":"424792","split":"eval","target_label":"Wind","license_url":"http://creativecommons.org/publicdomain/zero/1.0/","uploader":"bolkmar","group_id":"fsd50k:u-d6cc4e0f3dff","member":"FSD50K.eval_audio/424792.wav","disk":1,"offset":654972919,"compressed_bytes":864183,"size":1019670,"method":8,"crc32":1721855893,"sha256":"e48c9eb5705d861de73f84ce3d4271ca38bab94679bed8c1d5ca9d1aa56fc0c9"}]},"aris":{"record":"22682339","archive":{"key":"itu-aris-lab-acoustic-drone-dataset-v1.0.0.zip","bytes":169094149,"md5":"64aed06e8dac417d0c3c7db0e3ac340b","sha256":"11d1b1208ef63515d3fe62de3047b58977d00959cc938f49d17913e428400511"},"inner_prefix":"itu-aris-lab-acoustic-drone-dataset/","inner_files_sha256":{"LICENSE":"b79e2b6ded8d3c4cee0a64ab6562864d210fb33449095ad4bf330e64d8058c11","README.md":"fae2b49ab71e6cd01a0a1b7ef531f10e4681a7dbbf3d7a54a8cdc5c7216151f1","metadata.json":"45a14bb6d54fb9550e9584ed36f9c841309b6cd8d3a6805c82b1340043ea13d5","groups.json":"a79cd37058795b15ad355c156934493f546bd83acad9ba3e961add270d20d837","SHA256SUMS":"d4064353ac803be2c9615f2631f83902c9eb50ba742d0b4e07e3bffe3730f620","splits/split_interval_disjoint.json":"1492a1984c014adcd5daeecf34c17c174e8eaec38e05de0922b6befd4a6d873f","splits/split_temporal.json":"a086485805694cdab62cd18e051b098d483d0efaf25316d924e44cba22d8adc5"},"split_used":"splits/split_interval_disjoint.json","segment_count":5491},"prohibited":["DADS","Al-Emadi","UrbanSound8K","ESC-50 fuera de ESC-10","Sonniss","AeroSonicDB"],"fraunhofer_record":"https://zenodo.org/api/records/17525811","dads_card":"https://huggingface.co/datasets/geronimobasso/drone-audio-detection-samples","extra":{"salford":{"api":"https://api.figshare.com/v2/articles/22133411","files":[{"path":"46212081_Ed_3p_10_H00_N_C_nw_ev1_M5.wav","bytes":4000080,"md5":"d4daa88ba9ca6dcf544130b0a1e887de","url":"https://ndownloader.figshare.com/files/46212081","figshare_file_id":46212081,"suas_code":"3p","event_id":"Ed_3p_10_H00_N_C_nw_ev1","group_id":"salford:Ed_3p_10_H00_N_C_nw_ev1"},{"path":"Ed_3p_10_F15_N_W_dw_ev1_M1.wav","bytes":4000080,"md5":"e86f4084d16597d5e4467a4fd4a4797c","url":"https://ndownloader.figshare.com/files/41804634","figshare_file_id":41804634,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev1","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev1"},{"path":"Ed_3p_10_F15_N_W_dw_ev1_M2.wav","bytes":4000080,"md5":"94322ba83a0a181a833ccfe05db3da7b","url":"https://ndownloader.figshare.com/files/41804679","figshare_file_id":41804679,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev1","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev1"},{"path":"Ed_3p_10_F15_N_W_dw_ev1_M3.wav","bytes":4000080,"md5":"a203c2bf4679ceb5fd5ce33190a3aae5","url":"https://ndownloader.figshare.com/files/41804721","figshare_file_id":41804721,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev1","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev1"},{"path":"Ed_3p_10_F15_N_W_dw_ev1_M4.wav","bytes":4000080,"md5":"8c99146522122e8cf6f3593b31760350","url":"https://ndownloader.figshare.com/files/41804805","figshare_file_id":41804805,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev1","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev1"},{"path":"Ed_3p_10_F15_N_W_dw_ev1_M5.wav","bytes":4000080,"md5":"6f66936b12262c9070d4c47ae89b8aa4","url":"https://ndownloader.figshare.com/files/41804832","figshare_file_id":41804832,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev1","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev1"},{"path":"Ed_3p_10_F15_N_W_dw_ev1_M6.wav","bytes":4000080,"md5":"f3302fdec9fd1e5174fd0e1a93a24c67","url":"https://ndownloader.figshare.com/files/41804862","figshare_file_id":41804862,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev1","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev1"},{"path":"Ed_3p_10_F15_N_W_dw_ev1_M7.wav","bytes":4000080,"md5":"58cf6d1e4a00397c5bcab87694ad2604","url":"https://ndownloader.figshare.com/files/41804907","figshare_file_id":41804907,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev1","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev1"},{"path":"Ed_3p_10_F15_N_W_dw_ev1_M8.wav","bytes":4000080,"md5":"95a08fb5a7c5b93489e474080adb69ba","url":"https://ndownloader.figshare.com/files/41804997","figshare_file_id":41804997,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev1","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev1"},{"path":"Ed_3p_10_F15_N_W_dw_ev1_M9.wav","bytes":4000080,"md5":"1b39ff9a318912fba272cb54af5485fc","url":"https://ndownloader.figshare.com/files/41805006","figshare_file_id":41805006,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev1","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev1"},{"path":"Ed_3p_10_F15_N_W_dw_ev2_M1.wav","bytes":4000080,"md5":"414eca8b2e31992402e05de298d7fe0c","url":"https://ndownloader.figshare.com/files/41804637","figshare_file_id":41804637,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev2","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev2"},{"path":"Ed_3p_10_F15_N_W_dw_ev2_M2.wav","bytes":4000080,"md5":"812359c4145a8bf0d0496f51c6f5ba43","url":"https://ndownloader.figshare.com/files/41804682","figshare_file_id":41804682,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev2","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev2"},{"path":"Ed_3p_10_F15_N_W_dw_ev2_M3.wav","bytes":4000080,"md5":"653011a728519a497e899395cfa86de1","url":"https://ndownloader.figshare.com/files/41804724","figshare_file_id":41804724,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev2","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev2"},{"path":"Ed_3p_10_F15_N_W_dw_ev2_M4.wav","bytes":4000080,"md5":"1a879e84e96e3294b8199d5908ad5f78","url":"https://ndownloader.figshare.com/files/41804802","figshare_file_id":41804802,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev2","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev2"},{"path":"Ed_3p_10_F15_N_W_dw_ev2_M5.wav","bytes":4000080,"md5":"3e22ac58c54a2289f00ac4071d53f326","url":"https://ndownloader.figshare.com/files/41804835","figshare_file_id":41804835,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev2","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev2"},{"path":"Ed_3p_10_F15_N_W_dw_ev2_M6.wav","bytes":4000080,"md5":"a5af2720eea0b236cdbb4da087d4591c","url":"https://ndownloader.figshare.com/files/41804865","figshare_file_id":41804865,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev2","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev2"},{"path":"Ed_3p_10_F15_N_W_dw_ev2_M7.wav","bytes":4000080,"md5":"1962f2ed8310a1344707084d888fe276","url":"https://ndownloader.figshare.com/files/41804910","figshare_file_id":41804910,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev2","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev2"},{"path":"Ed_3p_10_F15_N_W_dw_ev2_M8.wav","bytes":4000080,"md5":"5ec00942207f857acc1ce76a142c3ff6","url":"https://ndownloader.figshare.com/files/41804994","figshare_file_id":41804994,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev2","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev2"},{"path":"Ed_3p_10_F15_N_W_dw_ev2_M9.wav","bytes":4000080,"md5":"53fcab42d747caa29da60a389ba1a646","url":"https://ndownloader.figshare.com/files/41805009","figshare_file_id":41805009,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev2","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev2"},{"path":"Ed_3p_10_F15_N_W_dw_ev3_M1.wav","bytes":4000080,"md5":"7f6f4cc68b7975a058ded285acf02906","url":"https://ndownloader.figshare.com/files/41804640","figshare_file_id":41804640,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev3","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev3"},{"path":"Ed_3p_10_F15_N_W_dw_ev3_M2.wav","bytes":4000080,"md5":"3ad596f96e66d3ab1e9c330aad090941","url":"https://ndownloader.figshare.com/files/41804685","figshare_file_id":41804685,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev3","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev3"},{"path":"Ed_3p_10_F15_N_W_dw_ev3_M3.wav","bytes":4000080,"md5":"9129876ea4c2adf217dd7aa9dc067ee2","url":"https://ndownloader.figshare.com/files/41804727","figshare_file_id":41804727,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev3","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev3"},{"path":"Ed_3p_10_F15_N_W_dw_ev3_M4.wav","bytes":4000080,"md5":"98561816f1c8aca6fec82eb9387f420d","url":"https://ndownloader.figshare.com/files/41804799","figshare_file_id":41804799,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev3","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev3"},{"path":"Ed_3p_10_F15_N_W_dw_ev3_M5.wav","bytes":4000080,"md5":"a41f81afaece8cc3fb5b01b6b01bf718","url":"https://ndownloader.figshare.com/files/41804838","figshare_file_id":41804838,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev3","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev3"},{"path":"Ed_3p_10_F15_N_W_dw_ev3_M6.wav","bytes":4000080,"md5":"304ca5df0b302720fc49a1989ae94817","url":"https://ndownloader.figshare.com/files/41804868","figshare_file_id":41804868,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev3","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev3"},{"path":"Ed_3p_10_F15_N_W_dw_ev3_M7.wav","bytes":4000080,"md5":"3a3366aa34517e53d0a57c8683e0f33d","url":"https://ndownloader.figshare.com/files/41804913","figshare_file_id":41804913,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev3","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev3"},{"path":"Ed_3p_10_F15_N_W_dw_ev3_M8.wav","bytes":4000080,"md5":"5d44c7ae212bec5d7046cc24282a2d3a","url":"https://ndownloader.figshare.com/files/41804991","figshare_file_id":41804991,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev3","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev3"},{"path":"Ed_3p_10_F15_N_W_dw_ev3_M9.wav","bytes":4000080,"md5":"22e028927892bb281139743a63363403","url":"https://ndownloader.figshare.com/files/41805012","figshare_file_id":41805012,"suas_code":"3p","event_id":"Ed_3p_10_F15_N_W_dw_ev3","group_id":"salford:Ed_3p_10_F15_N_W_dw_ev3"},{"path":"Ed_3p_10_H00_N_C_nw_ev1_M1.wav","bytes":4000080,"md5":"d19739c7b7ce2c6cd3fc1f3fa927e1fa","url":"https://ndownloader.figshare.com/files/46212183","figshare_file_id":46212183,"suas_code":"3p","event_id":"Ed_3p_10_H00_N_C_nw_ev1","group_id":"salford:Ed_3p_10_H00_N_C_nw_ev1"},{"path":"Ed_3p_10_H00_N_C_nw_ev1_M2.wav","bytes":4000080,"md5":"49bab1cb491ed838fe715dec6073d0f4","url":"https://ndownloader.figshare.com/files/46212186","figshare_file_id":46212186,"suas_code":"3p","event_id":"Ed_3p_10_H00_N_C_nw_ev1","group_id":"salford:Ed_3p_10_H00_N_C_nw_ev1"},{"path":"Ed_3p_10_H00_N_C_nw_ev1_M3.wav","bytes":4000080,"md5":"468899ce3ec86f718b6f616e85da0a8e","url":"https://ndownloader.figshare.com/files/46212189","figshare_file_id":46212189,"suas_code":"3p","event_id":"Ed_3p_10_H00_N_C_nw_ev1","group_id":"salford:Ed_3p_10_H00_N_C_nw_ev1"},{"path":"Ed_3p_10_H00_N_C_nw_ev1_M4.wav","bytes":4000080,"md5":"fa425608c323ab3338eeb7e1766b1e10","url":"https://ndownloader.figshare.com/files/46212192","figshare_file_id":46212192,"suas_code":"3p","event_id":"Ed_3p_10_H00_N_C_nw_ev1","group_id":"salford:Ed_3p_10_H00_N_C_nw_ev1"},{"path":"Ed_3p_10_H00_N_C_nw_ev1_M6.wav","bytes":4000080,"md5":"eae03d7d50b737fbe8d4b9547e4152cd","url":"https://ndownloader.figshare.com/files/46212198","figshare_file_id":46212198,"suas_code":"3p","event_id":"Ed_3p_10_H00_N_C_nw_ev1","group_id":"salford:Ed_3p_10_H00_N_C_nw_ev1"},{"path":"Ed_3p_10_H00_N_C_nw_ev1_M7.wav","bytes":4000080,"md5":"70a04b033f1ad8ef9e910ad545ddf4a1","url":"https://ndownloader.figshare.com/files/46212201","figshare_file_id":46212201,"suas_code":"3p","event_id":"Ed_3p_10_H00_N_C_nw_ev1","group_id":"salford:Ed_3p_10_H00_N_C_nw_ev1"},{"path":"Ed_3p_10_H00_N_C_nw_ev1_M8.wav","bytes":4000080,"md5":"224f36abcb0cb7e19c742165d1cb26ed","url":"https://ndownloader.figshare.com/files/46212204","figshare_file_id":46212204,"suas_code":"3p","event_id":"Ed_3p_10_H00_N_C_nw_ev1","group_id":"salford:Ed_3p_10_H00_N_C_nw_ev1"},{"path":"Ed_3p_10_H00_N_C_nw_ev1_M9.wav","bytes":4000080,"md5":"3a9997ee41e751f2f52ac8c9fda7fe82","url":"https://ndownloader.figshare.com/files/46212207","figshare_file_id":46212207,"suas_code":"3p","event_id":"Ed_3p_10_H00_N_C_nw_ev1","group_id":"salford:Ed_3p_10_H00_N_C_nw_ev1"},{"path":"Ed_3p_10_H00_N_C_nw_ev2_M5.wav","bytes":4000080,"md5":"af36e6d14fcf6e12a0962adea0ca097c","url":"https://ndownloader.figshare.com/files/46212084","figshare_file_id":46212084,"suas_code":"3p","event_id":"Ed_3p_10_H00_N_C_nw_ev2","group_id":"salford:Ed_3p_10_H00_N_C_nw_ev2"},{"path":"Ed_3p_10_H00_N_C_nw_ev3_M5.wav","bytes":4000080,"md5":"851d1f509abd0089e2a004f8469511c8","url":"https://ndownloader.figshare.com/files/46212087","figshare_file_id":46212087,"suas_code":"3p","event_id":"Ed_3p_10_H00_N_C_nw_ev3","group_id":"salford:Ed_3p_10_H00_N_C_nw_ev3"},{"path":"Ed_Fp_10_F15_N_W_dw_ev1_M1.wav","bytes":4000080,"md5":"893c6e6f00b09db0810e7e416f3e8d4d","url":"https://ndownloader.figshare.com/files/41804643","figshare_file_id":41804643,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev1","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev1"},{"path":"Ed_Fp_10_F15_N_W_dw_ev1_M2.wav","bytes":4000080,"md5":"f27a96f333ee414ca07b0bb8d9a8ea67","url":"https://ndownloader.figshare.com/files/41804688","figshare_file_id":41804688,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev1","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev1"},{"path":"Ed_Fp_10_F15_N_W_dw_ev1_M3.wav","bytes":4000080,"md5":"41cdf100ca7b2fd0c179970207459d8e","url":"https://ndownloader.figshare.com/files/41804733","figshare_file_id":41804733,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev1","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev1"},{"path":"Ed_Fp_10_F15_N_W_dw_ev1_M4.wav","bytes":4000080,"md5":"2a9b13764fb6c8c5b6feb3633956e5a6","url":"https://ndownloader.figshare.com/files/41804796","figshare_file_id":41804796,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev1","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev1"},{"path":"Ed_Fp_10_F15_N_W_dw_ev1_M5.wav","bytes":4000080,"md5":"9c5d2de2e50c5933ebd29fa1c2b7fdc4","url":"https://ndownloader.figshare.com/files/41804841","figshare_file_id":41804841,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev1","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev1"},{"path":"Ed_Fp_10_F15_N_W_dw_ev1_M6.wav","bytes":4000080,"md5":"c0231e59f67862c5355c324f751aa441","url":"https://ndownloader.figshare.com/files/41804871","figshare_file_id":41804871,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev1","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev1"},{"path":"Ed_Fp_10_F15_N_W_dw_ev1_M7.wav","bytes":4000080,"md5":"c030a2fee1b649b16c530dbe2c4b6786","url":"https://ndownloader.figshare.com/files/41804916","figshare_file_id":41804916,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev1","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev1"},{"path":"Ed_Fp_10_F15_N_W_dw_ev1_M8.wav","bytes":4000080,"md5":"b3aa87c3e14b2ea05afaec75f0e1a85a","url":"https://ndownloader.figshare.com/files/41804988","figshare_file_id":41804988,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev1","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev1"},{"path":"Ed_Fp_10_F15_N_W_dw_ev1_M9.wav","bytes":4000080,"md5":"52bd784ba66ed60ab016bc2d86d077ee","url":"https://ndownloader.figshare.com/files/41805015","figshare_file_id":41805015,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev1","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev1"},{"path":"Ed_Fp_10_F15_N_W_dw_ev2_M1.wav","bytes":4000080,"md5":"86b87ab3f5b93fd9c22d8dca18303113","url":"https://ndownloader.figshare.com/files/41804646","figshare_file_id":41804646,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev2","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev2"},{"path":"Ed_Fp_10_F15_N_W_dw_ev2_M2.wav","bytes":4000080,"md5":"53cdcb7a1d98f91d8678492707b31694","url":"https://ndownloader.figshare.com/files/41804691","figshare_file_id":41804691,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev2","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev2"},{"path":"Ed_Fp_10_F15_N_W_dw_ev2_M3.wav","bytes":4000080,"md5":"3477dd78ea3b5c5cf1e71fe57f2bca12","url":"https://ndownloader.figshare.com/files/41804736","figshare_file_id":41804736,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev2","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev2"},{"path":"Ed_Fp_10_F15_N_W_dw_ev2_M4.wav","bytes":4000080,"md5":"7f97de3ac696f9cb634e9d15d95d5546","url":"https://ndownloader.figshare.com/files/41804793","figshare_file_id":41804793,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev2","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev2"},{"path":"Ed_Fp_10_F15_N_W_dw_ev2_M5.wav","bytes":4000080,"md5":"b9aa1532a3b87eb7f629153abc1f570a","url":"https://ndownloader.figshare.com/files/41804844","figshare_file_id":41804844,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev2","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev2"},{"path":"Ed_Fp_10_F15_N_W_dw_ev2_M6.wav","bytes":4000080,"md5":"276c17db9dad3b8aecb2775ab6c9582f","url":"https://ndownloader.figshare.com/files/41804874","figshare_file_id":41804874,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev2","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev2"},{"path":"Ed_Fp_10_F15_N_W_dw_ev2_M7.wav","bytes":4000080,"md5":"bf839172d9233996d9c6d0a05d719948","url":"https://ndownloader.figshare.com/files/41804919","figshare_file_id":41804919,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev2","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev2"},{"path":"Ed_Fp_10_F15_N_W_dw_ev2_M8.wav","bytes":4000080,"md5":"09e1754ac4d20ba4f9a9c00d5b1c0ea2","url":"https://ndownloader.figshare.com/files/41804985","figshare_file_id":41804985,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev2","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev2"},{"path":"Ed_Fp_10_F15_N_W_dw_ev2_M9.wav","bytes":4000080,"md5":"760a9bdaa0a0657b08f3c39aa793ab91","url":"https://ndownloader.figshare.com/files/41805018","figshare_file_id":41805018,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev2","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev2"},{"path":"Ed_Fp_10_F15_N_W_dw_ev3_M1.wav","bytes":4000080,"md5":"55e3d53853353c3324f5fe60c238b6bc","url":"https://ndownloader.figshare.com/files/41804649","figshare_file_id":41804649,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev3","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev3"},{"path":"Ed_Fp_10_F15_N_W_dw_ev3_M2.wav","bytes":4000080,"md5":"6f4a17802a1b8105dd2176578475cb49","url":"https://ndownloader.figshare.com/files/41804694","figshare_file_id":41804694,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev3","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev3"},{"path":"Ed_Fp_10_F15_N_W_dw_ev3_M3.wav","bytes":4000080,"md5":"ffe5eac07c8a7e9a6f24c0f1f762dbd1","url":"https://ndownloader.figshare.com/files/41804730","figshare_file_id":41804730,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev3","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev3"},{"path":"Ed_Fp_10_F15_N_W_dw_ev3_M4.wav","bytes":4000080,"md5":"4f71eb5794ba6bf26dd07882898f9f04","url":"https://ndownloader.figshare.com/files/41804790","figshare_file_id":41804790,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev3","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev3"},{"path":"Ed_Fp_10_F15_N_W_dw_ev3_M5.wav","bytes":4000080,"md5":"8ede18785d9c1ebaef6fbacec167a665","url":"https://ndownloader.figshare.com/files/41804847","figshare_file_id":41804847,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev3","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev3"},{"path":"Ed_Fp_10_F15_N_W_dw_ev3_M6.wav","bytes":4000080,"md5":"edbf0472511f5075aa01c51696dbf349","url":"https://ndownloader.figshare.com/files/41804877","figshare_file_id":41804877,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev3","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev3"},{"path":"Ed_Fp_10_F15_N_W_dw_ev3_M7.wav","bytes":4000080,"md5":"3df8045b515179d09ed413fca8abc4ae","url":"https://ndownloader.figshare.com/files/41804922","figshare_file_id":41804922,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev3","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev3"},{"path":"Ed_Fp_10_F15_N_W_dw_ev3_M8.wav","bytes":4000080,"md5":"501c16471938a10400de3ad1066cda9f","url":"https://ndownloader.figshare.com/files/41804982","figshare_file_id":41804982,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev3","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev3"},{"path":"Ed_Fp_10_F15_N_W_dw_ev3_M9.wav","bytes":4000080,"md5":"110811b533fee94500b409f3724f1bc0","url":"https://ndownloader.figshare.com/files/41805021","figshare_file_id":41805021,"suas_code":"Fp","event_id":"Ed_Fp_10_F15_N_W_dw_ev3","group_id":"salford:Ed_Fp_10_F15_N_W_dw_ev3"},{"path":"Ed_M3_10_F15_N_E_dw_ev1_M1.wav","bytes":4000080,"md5":"a2c87b03d735c56879f8d40e27192dbc","url":"https://ndownloader.figshare.com/files/41804652","figshare_file_id":41804652,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev1","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev1"},{"path":"Ed_M3_10_F15_N_E_dw_ev1_M2.wav","bytes":4000080,"md5":"d5541238118a3411d3fd95eee2e2e100","url":"https://ndownloader.figshare.com/files/41804697","figshare_file_id":41804697,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev1","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev1"},{"path":"Ed_M3_10_F15_N_E_dw_ev1_M3.wav","bytes":4000080,"md5":"0f2b30c33047dbc7cc9c2bd9a8af04d1","url":"https://ndownloader.figshare.com/files/41804757","figshare_file_id":41804757,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev1","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev1"},{"path":"Ed_M3_10_F15_N_E_dw_ev1_M4.wav","bytes":4000080,"md5":"a835c3f359f4521b439898421cd90ced","url":"https://ndownloader.figshare.com/files/41804787","figshare_file_id":41804787,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev1","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev1"},{"path":"Ed_M3_10_F15_N_E_dw_ev1_M5.wav","bytes":4000080,"md5":"a13c86d106e8c17696e2c2112932abbb","url":"https://ndownloader.figshare.com/files/41804850","figshare_file_id":41804850,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev1","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev1"},{"path":"Ed_M3_10_F15_N_E_dw_ev1_M6.wav","bytes":4000080,"md5":"3e51bd6713430e1d7eca5425154a2bc1","url":"https://ndownloader.figshare.com/files/41804880","figshare_file_id":41804880,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev1","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev1"},{"path":"Ed_M3_10_F15_N_E_dw_ev1_M7.wav","bytes":4000080,"md5":"224037cbd929d52d09c8f0da614fb17d","url":"https://ndownloader.figshare.com/files/41804925","figshare_file_id":41804925,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev1","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev1"},{"path":"Ed_M3_10_F15_N_E_dw_ev1_M8.wav","bytes":4000080,"md5":"2c9c17807b6561e0fa2fde28a08f4c33","url":"https://ndownloader.figshare.com/files/41804979","figshare_file_id":41804979,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev1","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev1"},{"path":"Ed_M3_10_F15_N_E_dw_ev1_M9.wav","bytes":4000080,"md5":"91c06cb10514bd1841c648d533a88b7e","url":"https://ndownloader.figshare.com/files/41805024","figshare_file_id":41805024,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev1","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev1"},{"path":"Ed_M3_10_F15_N_E_dw_ev2_M1.wav","bytes":4000080,"md5":"cddb53bc03c992ecca520800ef39c697","url":"https://ndownloader.figshare.com/files/41804655","figshare_file_id":41804655,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev2","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev2"},{"path":"Ed_M3_10_F15_N_E_dw_ev2_M2.wav","bytes":4000080,"md5":"d3328d09bf0605354a3c9472c0690a74","url":"https://ndownloader.figshare.com/files/41804700","figshare_file_id":41804700,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev2","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev2"},{"path":"Ed_M3_10_F15_N_E_dw_ev2_M3.wav","bytes":4000080,"md5":"f2a282084a1bf62accdd00276e5a8029","url":"https://ndownloader.figshare.com/files/41804760","figshare_file_id":41804760,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev2","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev2"},{"path":"Ed_M3_10_F15_N_E_dw_ev2_M4.wav","bytes":4000080,"md5":"3e8259763ff679177f158a4c487b53db","url":"https://ndownloader.figshare.com/files/41804784","figshare_file_id":41804784,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev2","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev2"},{"path":"Ed_M3_10_F15_N_E_dw_ev2_M5.wav","bytes":4000080,"md5":"49fb751dc2162e23451c1b38af975ef6","url":"https://ndownloader.figshare.com/files/41804853","figshare_file_id":41804853,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev2","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev2"},{"path":"Ed_M3_10_F15_N_E_dw_ev2_M6.wav","bytes":4000080,"md5":"770869cfab3c93b0465355966cf15e2d","url":"https://ndownloader.figshare.com/files/41804883","figshare_file_id":41804883,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev2","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev2"},{"path":"Ed_M3_10_F15_N_E_dw_ev2_M7.wav","bytes":4000080,"md5":"f5b74bbb6b1be2c3e6217c1e2c360bf6","url":"https://ndownloader.figshare.com/files/41804928","figshare_file_id":41804928,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev2","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev2"},{"path":"Ed_M3_10_F15_N_E_dw_ev2_M8.wav","bytes":4000080,"md5":"71f2ebe1319aec26df696e4b2e34246b","url":"https://ndownloader.figshare.com/files/41804976","figshare_file_id":41804976,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev2","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev2"},{"path":"Ed_M3_10_F15_N_E_dw_ev2_M9.wav","bytes":4000080,"md5":"4ff5c07cb63e838191e2db2cacf4bf9c","url":"https://ndownloader.figshare.com/files/41805027","figshare_file_id":41805027,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev2","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev2"},{"path":"Ed_M3_10_F15_N_E_dw_ev3_M1.wav","bytes":4000080,"md5":"0bebc607834056dbb0d64624e967457c","url":"https://ndownloader.figshare.com/files/41804658","figshare_file_id":41804658,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev3","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev3"},{"path":"Ed_M3_10_F15_N_E_dw_ev3_M2.wav","bytes":4000080,"md5":"c7245ffaf39ed5ac3a61c08c08719b8c","url":"https://ndownloader.figshare.com/files/41804703","figshare_file_id":41804703,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev3","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev3"},{"path":"Ed_M3_10_F15_N_E_dw_ev3_M3.wav","bytes":4000080,"md5":"8346914e5f1f0fd3a2724c3534ed0607","url":"https://ndownloader.figshare.com/files/41804748","figshare_file_id":41804748,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev3","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev3"},{"path":"Ed_M3_10_F15_N_E_dw_ev3_M4.wav","bytes":4000080,"md5":"a4b3b3da848966912b5253dfc2778df7","url":"https://ndownloader.figshare.com/files/41804781","figshare_file_id":41804781,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev3","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev3"},{"path":"Ed_M3_10_F15_N_E_dw_ev3_M5.wav","bytes":4000080,"md5":"4f7512a25a17c7e2aea9a49746576c7b","url":"https://ndownloader.figshare.com/files/41804811","figshare_file_id":41804811,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev3","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev3"},{"path":"Ed_M3_10_F15_N_E_dw_ev3_M6.wav","bytes":4000080,"md5":"06dc10a554093fb64acf7c077198dfd0","url":"https://ndownloader.figshare.com/files/41804886","figshare_file_id":41804886,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev3","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev3"},{"path":"Ed_M3_10_F15_N_E_dw_ev3_M7.wav","bytes":4000080,"md5":"57ffd9d1fdf614a8e96ae01adade48e5","url":"https://ndownloader.figshare.com/files/41804931","figshare_file_id":41804931,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev3","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev3"},{"path":"Ed_M3_10_F15_N_E_dw_ev3_M8.wav","bytes":4000080,"md5":"abe50565b0f769765c4bc6be8713dab7","url":"https://ndownloader.figshare.com/files/41804973","figshare_file_id":41804973,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev3","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev3"},{"path":"Ed_M3_10_F15_N_E_dw_ev3_M9.wav","bytes":4000080,"md5":"2ff9f22acc58b67383fbf55c6ad575c4","url":"https://ndownloader.figshare.com/files/41805030","figshare_file_id":41805030,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev3","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev3"},{"path":"Ed_M3_10_F15_N_E_dw_ev4_M1.wav","bytes":4000080,"md5":"44289a63085836bea94878d226625bf1","url":"https://ndownloader.figshare.com/files/41804661","figshare_file_id":41804661,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev4","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev4"},{"path":"Ed_M3_10_F15_N_E_dw_ev4_M2.wav","bytes":4000080,"md5":"5ab82da85bec1d4c1d15c9bc77d6282e","url":"https://ndownloader.figshare.com/files/41804706","figshare_file_id":41804706,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev4","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev4"},{"path":"Ed_M3_10_F15_N_E_dw_ev4_M3.wav","bytes":4000080,"md5":"2279ff27193a6f159f9491b9621e5ded","url":"https://ndownloader.figshare.com/files/41804751","figshare_file_id":41804751,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev4","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev4"},{"path":"Ed_M3_10_F15_N_E_dw_ev4_M4.wav","bytes":4000080,"md5":"5a20e16f81dcfc486dc313c0d9ba744b","url":"https://ndownloader.figshare.com/files/41804778","figshare_file_id":41804778,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev4","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev4"},{"path":"Ed_M3_10_F15_N_E_dw_ev4_M5.wav","bytes":4000080,"md5":"e17b9bdb0c0151b62fdda4f8e519b84e","url":"https://ndownloader.figshare.com/files/41804814","figshare_file_id":41804814,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev4","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev4"},{"path":"Ed_M3_10_F15_N_E_dw_ev4_M6.wav","bytes":4000080,"md5":"35d06fa79ba5095eb32386add608abd4","url":"https://ndownloader.figshare.com/files/41804889","figshare_file_id":41804889,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev4","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev4"},{"path":"Ed_M3_10_F15_N_E_dw_ev4_M7.wav","bytes":4000080,"md5":"fa55474787e1502b2b25ebe04969c9ae","url":"https://ndownloader.figshare.com/files/41804934","figshare_file_id":41804934,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev4","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev4"},{"path":"Ed_M3_10_F15_N_E_dw_ev4_M8.wav","bytes":4000080,"md5":"289dfae46ca9f7b487f779dec9a6b83d","url":"https://ndownloader.figshare.com/files/41804970","figshare_file_id":41804970,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev4","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev4"},{"path":"Ed_M3_10_F15_N_E_dw_ev4_M9.wav","bytes":4000080,"md5":"272f5c9b27e079ff754dc5e5d8c1808c","url":"https://ndownloader.figshare.com/files/41805033","figshare_file_id":41805033,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev4","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev4"},{"path":"Ed_M3_10_F15_N_E_dw_ev5_M1.wav","bytes":4000080,"md5":"d97f565ef5a45d4e9585ff509145fe78","url":"https://ndownloader.figshare.com/files/41804664","figshare_file_id":41804664,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev5","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev5"},{"path":"Ed_M3_10_F15_N_E_dw_ev5_M2.wav","bytes":4000080,"md5":"5295af7b85bc703c9dd42a67d570349e","url":"https://ndownloader.figshare.com/files/41804709","figshare_file_id":41804709,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev5","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev5"},{"path":"Ed_M3_10_F15_N_E_dw_ev5_M3.wav","bytes":4000080,"md5":"00d6d819c2f8a990949a34cb1406b8e0","url":"https://ndownloader.figshare.com/files/41804754","figshare_file_id":41804754,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev5","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev5"},{"path":"Ed_M3_10_F15_N_E_dw_ev5_M4.wav","bytes":4000080,"md5":"85dfa3f69cadd7c76057c70e8aa391da","url":"https://ndownloader.figshare.com/files/41804775","figshare_file_id":41804775,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev5","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev5"},{"path":"Ed_M3_10_F15_N_E_dw_ev5_M5.wav","bytes":4000080,"md5":"6b9829948ae4f878560ce30ca9f0e45f","url":"https://ndownloader.figshare.com/files/41804817","figshare_file_id":41804817,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev5","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev5"},{"path":"Ed_M3_10_F15_N_E_dw_ev5_M6.wav","bytes":4000080,"md5":"57eebc34973b445e18020e66aa6e1a74","url":"https://ndownloader.figshare.com/files/41804892","figshare_file_id":41804892,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev5","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev5"},{"path":"Ed_M3_10_F15_N_E_dw_ev5_M7.wav","bytes":4000080,"md5":"34f0a7bc8ee782d1706b3b8a67a648dd","url":"https://ndownloader.figshare.com/files/41804937","figshare_file_id":41804937,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev5","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev5"},{"path":"Ed_M3_10_F15_N_E_dw_ev5_M8.wav","bytes":4000080,"md5":"7dceba2681c9a23e0f5d3106d2c9e6c8","url":"https://ndownloader.figshare.com/files/41804967","figshare_file_id":41804967,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev5","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev5"},{"path":"Ed_M3_10_F15_N_E_dw_ev5_M9.wav","bytes":4000080,"md5":"a9cc6685487af89add90846c2828c7a2","url":"https://ndownloader.figshare.com/files/41805036","figshare_file_id":41805036,"suas_code":"M3","event_id":"Ed_M3_10_F15_N_E_dw_ev5","group_id":"salford:Ed_M3_10_F15_N_E_dw_ev5"},{"path":"Ed_M3_10_H00_N_C_nw_ev1_M1.wav","bytes":4000080,"md5":"f394aaea131482c9cb6ec8d1d7d48bf3","url":"https://ndownloader.figshare.com/files/46212096","figshare_file_id":46212096,"suas_code":"M3","event_id":"Ed_M3_10_H00_N_C_nw_ev1","group_id":"salford:Ed_M3_10_H00_N_C_nw_ev1"},{"path":"Ed_M3_10_H00_N_C_nw_ev1_M2.wav","bytes":4000080,"md5":"1a7650e8387615b755252fe458c638f3","url":"https://ndownloader.figshare.com/files/46212099","figshare_file_id":46212099,"suas_code":"M3","event_id":"Ed_M3_10_H00_N_C_nw_ev1","group_id":"salford:Ed_M3_10_H00_N_C_nw_ev1"},{"path":"Ed_M3_10_H00_N_C_nw_ev1_M3.wav","bytes":4000080,"md5":"24c9083ba9abada8aa6098b90d7e4134","url":"https://ndownloader.figshare.com/files/46212102","figshare_file_id":46212102,"suas_code":"M3","event_id":"Ed_M3_10_H00_N_C_nw_ev1","group_id":"salford:Ed_M3_10_H00_N_C_nw_ev1"},{"path":"Ed_M3_10_H00_N_C_nw_ev1_M4.wav","bytes":4000080,"md5":"feabd8d22c5f6e199e9a17a7e365288e","url":"https://ndownloader.figshare.com/files/46212105","figshare_file_id":46212105,"suas_code":"M3","event_id":"Ed_M3_10_H00_N_C_nw_ev1","group_id":"salford:Ed_M3_10_H00_N_C_nw_ev1"},{"path":"Ed_M3_10_H00_N_C_nw_ev1_M5.wav","bytes":4000080,"md5":"01489c7ab2caf1997d2f5fb79c68648a","url":"https://ndownloader.figshare.com/files/46212108","figshare_file_id":46212108,"suas_code":"M3","event_id":"Ed_M3_10_H00_N_C_nw_ev1","group_id":"salford:Ed_M3_10_H00_N_C_nw_ev1"},{"path":"Ed_M3_10_H00_N_C_nw_ev1_M6.wav","bytes":4000080,"md5":"6f0c8fdb439004236e4b6aa01e09f953","url":"https://ndownloader.figshare.com/files/46212111","figshare_file_id":46212111,"suas_code":"M3","event_id":"Ed_M3_10_H00_N_C_nw_ev1","group_id":"salford:Ed_M3_10_H00_N_C_nw_ev1"},{"path":"Ed_M3_10_H00_N_C_nw_ev1_M7.wav","bytes":4000080,"md5":"f2013e61ca1ce6ba9599a188cbcaac39","url":"https://ndownloader.figshare.com/files/46212114","figshare_file_id":46212114,"suas_code":"M3","event_id":"Ed_M3_10_H00_N_C_nw_ev1","group_id":"salford:Ed_M3_10_H00_N_C_nw_ev1"},{"path":"Ed_M3_10_H00_N_C_nw_ev1_M8.wav","bytes":4000080,"md5":"487435bc942a36c623f3ca67c0791c78","url":"https://ndownloader.figshare.com/files/46212117","figshare_file_id":46212117,"suas_code":"M3","event_id":"Ed_M3_10_H00_N_C_nw_ev1","group_id":"salford:Ed_M3_10_H00_N_C_nw_ev1"},{"path":"Ed_M3_10_H00_N_C_nw_ev1_M9.wav","bytes":4000080,"md5":"87b97e5bc7fe1236b7dadd4af5ba9880","url":"https://ndownloader.figshare.com/files/46212120","figshare_file_id":46212120,"suas_code":"M3","event_id":"Ed_M3_10_H00_N_C_nw_ev1","group_id":"salford:Ed_M3_10_H00_N_C_nw_ev1"},{"path":"Ed_M3_10_H00_Y_C_nw_ev1_M1.wav","bytes":4000080,"md5":"99319b1ce9f92626a74f2694e10170c0","url":"https://ndownloader.figshare.com/files/46212123","figshare_file_id":46212123,"suas_code":"M3","event_id":"Ed_M3_10_H00_Y_C_nw_ev1","group_id":"salford:Ed_M3_10_H00_Y_C_nw_ev1"},{"path":"Ed_M3_10_H00_Y_C_nw_ev1_M2.wav","bytes":4000080,"md5":"19059176e755d034065d36b43ff27652","url":"https://ndownloader.figshare.com/files/46212126","figshare_file_id":46212126,"suas_code":"M3","event_id":"Ed_M3_10_H00_Y_C_nw_ev1","group_id":"salford:Ed_M3_10_H00_Y_C_nw_ev1"},{"path":"Ed_M3_10_H00_Y_C_nw_ev1_M3.wav","bytes":4000080,"md5":"dbe4e93b623c7f0a2dca92b5e5a71d79","url":"https://ndownloader.figshare.com/files/46212129","figshare_file_id":46212129,"suas_code":"M3","event_id":"Ed_M3_10_H00_Y_C_nw_ev1","group_id":"salford:Ed_M3_10_H00_Y_C_nw_ev1"},{"path":"Ed_M3_10_H00_Y_C_nw_ev1_M4.wav","bytes":4000080,"md5":"16575996a0f2e4690510f28e8ca8657d","url":"https://ndownloader.figshare.com/files/46212132","figshare_file_id":46212132,"suas_code":"M3","event_id":"Ed_M3_10_H00_Y_C_nw_ev1","group_id":"salford:Ed_M3_10_H00_Y_C_nw_ev1"},{"path":"Ed_M3_10_H00_Y_C_nw_ev1_M5.wav","bytes":4000080,"md5":"c1a64fc03f33dc77a09989f36f3181bc","url":"https://ndownloader.figshare.com/files/46212135","figshare_file_id":46212135,"suas_code":"M3","event_id":"Ed_M3_10_H00_Y_C_nw_ev1","group_id":"salford:Ed_M3_10_H00_Y_C_nw_ev1"},{"path":"Ed_M3_10_H00_Y_C_nw_ev1_M6.wav","bytes":4000080,"md5":"8002b6606f44c464f1ac1be72d4cf509","url":"https://ndownloader.figshare.com/files/46212138","figshare_file_id":46212138,"suas_code":"M3","event_id":"Ed_M3_10_H00_Y_C_nw_ev1","group_id":"salford:Ed_M3_10_H00_Y_C_nw_ev1"},{"path":"Ed_M3_10_H00_Y_C_nw_ev1_M7.wav","bytes":4000080,"md5":"ba304b3eb3cfe7a75904840a3c948177","url":"https://ndownloader.figshare.com/files/46212141","figshare_file_id":46212141,"suas_code":"M3","event_id":"Ed_M3_10_H00_Y_C_nw_ev1","group_id":"salford:Ed_M3_10_H00_Y_C_nw_ev1"},{"path":"Ed_M3_10_H00_Y_C_nw_ev1_M8.wav","bytes":4000080,"md5":"8bddf3ccf9f76839d417d25380fd663f","url":"https://ndownloader.figshare.com/files/46212144","figshare_file_id":46212144,"suas_code":"M3","event_id":"Ed_M3_10_H00_Y_C_nw_ev1","group_id":"salford:Ed_M3_10_H00_Y_C_nw_ev1"},{"path":"Ed_M3_10_H00_Y_C_nw_ev1_M9.wav","bytes":4000080,"md5":"f7e926273cc873ec9f70b0b0c39d4cd5","url":"https://ndownloader.figshare.com/files/46212147","figshare_file_id":46212147,"suas_code":"M3","event_id":"Ed_M3_10_H00_Y_C_nw_ev1","group_id":"salford:Ed_M3_10_H00_Y_C_nw_ev1"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev1_M1.wav","bytes":6000080,"md5":"c2a8964cb9bf8eabdac342c5284ccfa5","url":"https://ndownloader.figshare.com/files/41804667","figshare_file_id":41804667,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev1","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev1"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev1_M2.wav","bytes":6000080,"md5":"d7c90113a00519cc78dd3076ddfe90f3","url":"https://ndownloader.figshare.com/files/41804712","figshare_file_id":41804712,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev1","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev1"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev1_M3.wav","bytes":6000080,"md5":"53414dbe9aa82db074a9157eeadb4847","url":"https://ndownloader.figshare.com/files/41804742","figshare_file_id":41804742,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev1","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev1"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev1_M4.wav","bytes":6000080,"md5":"31967930023a9778f35b1c5b367e6cd9","url":"https://ndownloader.figshare.com/files/41804772","figshare_file_id":41804772,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev1","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev1"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev1_M5.wav","bytes":6000080,"md5":"3aa0775a47d77aa1f721471f52eba421","url":"https://ndownloader.figshare.com/files/41804820","figshare_file_id":41804820,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev1","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev1"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev1_M6.wav","bytes":6000080,"md5":"e8fe9ea0b68c5843e83d3ce0323ba3cb","url":"https://ndownloader.figshare.com/files/41804895","figshare_file_id":41804895,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev1","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev1"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev1_M7.wav","bytes":6000080,"md5":"65bf4b0a3be439749a162e7864659d1a","url":"https://ndownloader.figshare.com/files/41804940","figshare_file_id":41804940,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev1","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev1"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev1_M8.wav","bytes":6000080,"md5":"d0501a5f99cd241d6b9b5277c2c4fbfb","url":"https://ndownloader.figshare.com/files/41804964","figshare_file_id":41804964,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev1","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev1"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev1_M9.wav","bytes":6000080,"md5":"d0e6cf6cde70855ffdfb83b79f67c414","url":"https://ndownloader.figshare.com/files/41805039","figshare_file_id":41805039,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev1","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev1"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev2_M1.wav","bytes":6000080,"md5":"2d24dae45264c3134e274e99d15e6b8e","url":"https://ndownloader.figshare.com/files/41804670","figshare_file_id":41804670,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev2","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev2"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev2_M2.wav","bytes":6000080,"md5":"a0974b94e411bae45c4d67fcbc974349","url":"https://ndownloader.figshare.com/files/41804715","figshare_file_id":41804715,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev2","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev2"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev2_M3.wav","bytes":6000080,"md5":"9eb5dfb6785c35bb6a5610ab250e27fc","url":"https://ndownloader.figshare.com/files/41804745","figshare_file_id":41804745,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev2","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev2"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev2_M4.wav","bytes":6000080,"md5":"64e4dc995a5557077346b6f947280224","url":"https://ndownloader.figshare.com/files/41804769","figshare_file_id":41804769,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev2","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev2"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev2_M5.wav","bytes":6000080,"md5":"bdc01878d4271ce42858b34d2bd1c705","url":"https://ndownloader.figshare.com/files/41804823","figshare_file_id":41804823,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev2","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev2"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev2_M6.wav","bytes":6000080,"md5":"e9d2e8a25ba464576414abc9f4f9f18d","url":"https://ndownloader.figshare.com/files/41804898","figshare_file_id":41804898,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev2","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev2"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev2_M7.wav","bytes":6000080,"md5":"e2fdebf2edeaf3d407847142685f7123","url":"https://ndownloader.figshare.com/files/41804943","figshare_file_id":41804943,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev2","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev2"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev2_M8.wav","bytes":6000080,"md5":"38f316fd8ca5b811ff5c50016cc3bab1","url":"https://ndownloader.figshare.com/files/41804961","figshare_file_id":41804961,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev2","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev2"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev2_M9.wav","bytes":6000080,"md5":"98a403a8a0568a94c8fef97610c0b59e","url":"https://ndownloader.figshare.com/files/41805042","figshare_file_id":41805042,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev2","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev2"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev3_M1.wav","bytes":6000080,"md5":"4ba10556da68e3cfe5042addf13a6ddf","url":"https://ndownloader.figshare.com/files/41804673","figshare_file_id":41804673,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev3","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev3"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev3_M2.wav","bytes":6000080,"md5":"22330f0c14a60d5fd409f1e5763b5df6","url":"https://ndownloader.figshare.com/files/41804718","figshare_file_id":41804718,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev3","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev3"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev3_M3.wav","bytes":6000080,"md5":"10e50d541f94c01495a0ec0ab5ff2cc3","url":"https://ndownloader.figshare.com/files/41804739","figshare_file_id":41804739,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev3","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev3"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev3_M4.wav","bytes":6000080,"md5":"348de7c847ceaaac4279f0e41565f545","url":"https://ndownloader.figshare.com/files/41804766","figshare_file_id":41804766,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev3","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev3"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev3_M5.wav","bytes":6000080,"md5":"f06af53faa5f6ef8726daba913fca7dc","url":"https://ndownloader.figshare.com/files/41804826","figshare_file_id":41804826,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev3","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev3"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev3_M6.wav","bytes":6000080,"md5":"8fd5a042d3e89b2e2be603c85da4e039","url":"https://ndownloader.figshare.com/files/41804901","figshare_file_id":41804901,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev3","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev3"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev3_M7.wav","bytes":6000080,"md5":"c1646959f8ec5b49f4d479699322918b","url":"https://ndownloader.figshare.com/files/41804946","figshare_file_id":41804946,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev3","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev3"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev3_M8.wav","bytes":6000080,"md5":"b0b962ab27988d56e537174979ef70aa","url":"https://ndownloader.figshare.com/files/41804958","figshare_file_id":41804958,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev3","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev3"},{"path":"Ed_Yn_10_F15_Y_W_dw_ev3_M9.wav","bytes":6000080,"md5":"6794845baf6bdf3fa49f436a98ac95ea","url":"https://ndownloader.figshare.com/files/41805045","figshare_file_id":41805045,"suas_code":"Yn","event_id":"Ed_Yn_10_F15_Y_W_dw_ev3","group_id":"salford:Ed_Yn_10_F15_Y_W_dw_ev3"},{"path":"Ed_Yn_10_H00_Y_C_nw_ev1_M1.wav","bytes":4000080,"md5":"89b5eb7ee3000e81dfdf8a8936ded15e","url":"https://ndownloader.figshare.com/files/46212156","figshare_file_id":46212156,"suas_code":"Yn","event_id":"Ed_Yn_10_H00_Y_C_nw_ev1","group_id":"salford:Ed_Yn_10_H00_Y_C_nw_ev1"},{"path":"Ed_Yn_10_H00_Y_C_nw_ev1_M2.wav","bytes":4000080,"md5":"4b3a70fda00b4c5de1c8b1711b9d0c37","url":"https://ndownloader.figshare.com/files/46212159","figshare_file_id":46212159,"suas_code":"Yn","event_id":"Ed_Yn_10_H00_Y_C_nw_ev1","group_id":"salford:Ed_Yn_10_H00_Y_C_nw_ev1"},{"path":"Ed_Yn_10_H00_Y_C_nw_ev1_M3.wav","bytes":4000080,"md5":"70d14887a223db6daec3344b45bd34f7","url":"https://ndownloader.figshare.com/files/46212162","figshare_file_id":46212162,"suas_code":"Yn","event_id":"Ed_Yn_10_H00_Y_C_nw_ev1","group_id":"salford:Ed_Yn_10_H00_Y_C_nw_ev1"},{"path":"Ed_Yn_10_H00_Y_C_nw_ev1_M4.wav","bytes":4000080,"md5":"ffd5e0713c27d4cddd910933cbe38a20","url":"https://ndownloader.figshare.com/files/46212165","figshare_file_id":46212165,"suas_code":"Yn","event_id":"Ed_Yn_10_H00_Y_C_nw_ev1","group_id":"salford:Ed_Yn_10_H00_Y_C_nw_ev1"},{"path":"Ed_Yn_10_H00_Y_C_nw_ev1_M5.wav","bytes":4000080,"md5":"f2249d1472647a6d4916e6d7660a04b2","url":"https://ndownloader.figshare.com/files/46212168","figshare_file_id":46212168,"suas_code":"Yn","event_id":"Ed_Yn_10_H00_Y_C_nw_ev1","group_id":"salford:Ed_Yn_10_H00_Y_C_nw_ev1"},{"path":"Ed_Yn_10_H00_Y_C_nw_ev1_M6.wav","bytes":4000080,"md5":"8841c27c0678c73b9a40cd923bce5d46","url":"https://ndownloader.figshare.com/files/46212171","figshare_file_id":46212171,"suas_code":"Yn","event_id":"Ed_Yn_10_H00_Y_C_nw_ev1","group_id":"salford:Ed_Yn_10_H00_Y_C_nw_ev1"},{"path":"Ed_Yn_10_H00_Y_C_nw_ev1_M7.wav","bytes":4000080,"md5":"ffc3871dd82e2fda54fd05da264c77b5","url":"https://ndownloader.figshare.com/files/46212174","figshare_file_id":46212174,"suas_code":"Yn","event_id":"Ed_Yn_10_H00_Y_C_nw_ev1","group_id":"salford:Ed_Yn_10_H00_Y_C_nw_ev1"},{"path":"Ed_Yn_10_H00_Y_C_nw_ev1_M8.wav","bytes":4000080,"md5":"a7eaeb718bb7771f537627a36c8212b0","url":"https://ndownloader.figshare.com/files/46212177","figshare_file_id":46212177,"suas_code":"Yn","event_id":"Ed_Yn_10_H00_Y_C_nw_ev1","group_id":"salford:Ed_Yn_10_H00_Y_C_nw_ev1"},{"path":"Ed_Yn_10_H00_Y_C_nw_ev1_M9.wav","bytes":4000080,"md5":"bb9cd0ba1002be60e0cd66f9628eb8bf","url":"https://ndownloader.figshare.com/files/46212180","figshare_file_id":46212180,"suas_code":"Yn","event_id":"Ed_Yn_10_H00_Y_C_nw_ev1","group_id":"salford:Ed_Yn_10_H00_Y_C_nw_ev1"}]},"droneaudioset":{"info_url":"https://huggingface.co/api/datasets/ahlab-drone-project/DroneAudioSet/revision/e794007062e2d2ad29262a5795b60d09f2b345c4","revision":"e794007062e2d2ad29262a5795b60d09f2b345c4","files":[{"path":"drone-only/train_003-00000-of-00001.parquet","bytes":134875787,"sha256":"38d7bb50001c32c0c50173b9e50c3982e2fd9b38395358dbd74857cb97d9acce","url":"https://huggingface.co/datasets/ahlab-drone-project/DroneAudioSet/resolve/e794007062e2d2ad29262a5795b60d09f2b345c4/drone-only/train_003-00000-of-00001.parquet","rows":[{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-50cm/throttle-low/mic1_soundskrit-File5.wav","drone":"drone2","take":"File5","group_id":"droneaudioset:drone2/mic-dist-50cm/throttle-low","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-50cm/throttle-low/mic1_soundskrit-File6.wav","drone":"drone2","take":"File6","group_id":"droneaudioset:drone2/mic-dist-50cm/throttle-low","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-50cm/throttle-low/mic2_8array-down-File1.wav","drone":"drone2","take":"File1","group_id":"droneaudioset:drone2/mic-dist-50cm/throttle-low","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-50cm/throttle-low/mic2_8array-down-File4.wav","drone":"drone2","take":"File4","group_id":"droneaudioset:drone2/mic-dist-50cm/throttle-low","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-50cm/throttle-low/mic2_8array-down-File6.wav","drone":"drone2","take":"File6","group_id":"droneaudioset:drone2/mic-dist-50cm/throttle-low","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-50cm/throttle-low/mic3_8array-up-File6.wav","drone":"drone2","take":"File6","group_id":"droneaudioset:drone2/mic-dist-50cm/throttle-low","label_status":"drone_only_upstream"}]},{"path":"drone-only/train_004-00000-of-00001.parquet","bytes":90797224,"sha256":"e1cb6a8fcbc54d8a12484bff94dde7ef1d9f2c2eef3bf77d98c3f7abc66d0776","url":"https://huggingface.co/datasets/ahlab-drone-project/DroneAudioSet/resolve/e794007062e2d2ad29262a5795b60d09f2b345c4/drone-only/train_004-00000-of-00001.parquet","rows":[{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-50cm/throttle-high/mic1_soundskrit-File3.wav","drone":"drone2","take":"File3","group_id":"droneaudioset:drone2/mic-dist-50cm/throttle-high","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-50cm/throttle-high/mic2_8array-down-File3.wav","drone":"drone2","take":"File3","group_id":"droneaudioset:drone2/mic-dist-50cm/throttle-high","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-50cm/throttle-low/mic1_soundskrit-File4.wav","drone":"drone2","take":"File4","group_id":"droneaudioset:drone2/mic-dist-50cm/throttle-low","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-50cm/throttle-low/mic3_8array-up-File3.wav","drone":"drone2","take":"File3","group_id":"droneaudioset:drone2/mic-dist-50cm/throttle-low","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-50cm/throttle-low/mic3_8array-up-File5.wav","drone":"drone2","take":"File5","group_id":"droneaudioset:drone2/mic-dist-50cm/throttle-low","label_status":"drone_only_upstream"}]},{"path":"drone-only/train_011-00000-of-00001.parquet","bytes":93740184,"sha256":"f671268d70b6c0943fce4a50cdd56ef20de4525d9a4c377038b585684f216532","url":"https://huggingface.co/datasets/ahlab-drone-project/DroneAudioSet/resolve/e794007062e2d2ad29262a5795b60d09f2b345c4/drone-only/train_011-00000-of-00001.parquet","rows":[{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-25cm/throttle-high/mic1_soundskrit-File3.wav","drone":"drone2","take":"File3","group_id":"droneaudioset:drone2/mic-dist-25cm/throttle-high","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-25cm/throttle-high/mic2_8array-down-File3.wav","drone":"drone2","take":"File3","group_id":"droneaudioset:drone2/mic-dist-25cm/throttle-high","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-25cm/throttle-low/mic1_soundskrit-File4.wav","drone":"drone2","take":"File4","group_id":"droneaudioset:drone2/mic-dist-25cm/throttle-low","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-25cm/throttle-low/mic3_8array-up-File3.wav","drone":"drone2","take":"File3","group_id":"droneaudioset:drone2/mic-dist-25cm/throttle-low","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-25cm/throttle-low/mic3_8array-up-File5.wav","drone":"drone2","take":"File5","group_id":"droneaudioset:drone2/mic-dist-25cm/throttle-low","label_status":"drone_only_upstream"}]},{"path":"drone-only/train_014-00000-of-00001.parquet","bytes":148459974,"sha256":"f82980995fe297f02885ebab5ba6be56972996e88154073b3852fcadd3ccec26","url":"https://huggingface.co/datasets/ahlab-drone-project/DroneAudioSet/resolve/e794007062e2d2ad29262a5795b60d09f2b345c4/drone-only/train_014-00000-of-00001.parquet","rows":[{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-25cm/throttle-high/mic1_soundskrit-File4.wav","drone":"drone2","take":"File4","group_id":"droneaudioset:drone2/mic-dist-25cm/throttle-high","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-25cm/throttle-high/mic1_soundskrit-File5.wav","drone":"drone2","take":"File5","group_id":"droneaudioset:drone2/mic-dist-25cm/throttle-high","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-25cm/throttle-high/mic2_8array-down-File1.wav","drone":"drone2","take":"File1","group_id":"droneaudioset:drone2/mic-dist-25cm/throttle-high","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-25cm/throttle-high/mic2_8array-down-File6.wav","drone":"drone2","take":"File6","group_id":"droneaudioset:drone2/mic-dist-25cm/throttle-high","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-25cm/throttle-high/mic3_8array-up-File3.wav","drone":"drone2","take":"File3","group_id":"droneaudioset:drone2/mic-dist-25cm/throttle-high","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone2-only/mic-dist-25cm/throttle-high/mic3_8array-up-File5.wav","drone":"drone2","take":"File5","group_id":"droneaudioset:drone2/mic-dist-25cm/throttle-high","label_status":"drone_only_upstream"}]},{"path":"drone-only/train_018-00000-of-00001.parquet","bytes":91005420,"sha256":"3071673d930b656495d1a74d7ccb5ca0f599685107734b9170259e0b118ddab5","url":"https://huggingface.co/datasets/ahlab-drone-project/DroneAudioSet/resolve/e794007062e2d2ad29262a5795b60d09f2b345c4/drone-only/train_018-00000-of-00001.parquet","rows":[{"upstream_file_path":"drone-only-recordings/drone1-only/mic-dist-50cm/throttle-high/mic1_soundskrit-File3.wav","drone":"drone1","take":"File3","group_id":"droneaudioset:drone1/mic-dist-50cm/throttle-high","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone1-only/mic-dist-50cm/throttle-high/mic2_8array-down-File3.wav","drone":"drone1","take":"File3","group_id":"droneaudioset:drone1/mic-dist-50cm/throttle-high","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone1-only/mic-dist-50cm/throttle-low/mic1_soundskrit-File4.wav","drone":"drone1","take":"File4","group_id":"droneaudioset:drone1/mic-dist-50cm/throttle-low","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone1-only/mic-dist-50cm/throttle-low/mic3_8array-up-File3.wav","drone":"drone1","take":"File3","group_id":"droneaudioset:drone1/mic-dist-50cm/throttle-low","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone1-only/mic-dist-50cm/throttle-low/mic3_8array-up-File5.wav","drone":"drone1","take":"File5","group_id":"droneaudioset:drone1/mic-dist-50cm/throttle-low","label_status":"drone_only_upstream"}]},{"path":"drone-only/train_023-00000-of-00001.parquet","bytes":129135320,"sha256":"1e47809eae62e2232d2f959c9d72f81a6a9eea7e528f9fe3ce60cdb9f1f4df37","url":"https://huggingface.co/datasets/ahlab-drone-project/DroneAudioSet/resolve/e794007062e2d2ad29262a5795b60d09f2b345c4/drone-only/train_023-00000-of-00001.parquet","rows":[{"upstream_file_path":"drone-only-recordings/drone1-only/mic-dist-25cm/throttle-low/mic1_soundskrit-File1.wav","drone":"drone1","take":"File1","group_id":"droneaudioset:drone1/mic-dist-25cm/throttle-low","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone1-only/mic-dist-25cm/throttle-low/mic1_soundskrit-File2.wav","drone":"drone1","take":"File2","group_id":"droneaudioset:drone1/mic-dist-25cm/throttle-low","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone1-only/mic-dist-25cm/throttle-low/mic2_8array-down-File2.wav","drone":"drone1","take":"File2","group_id":"droneaudioset:drone1/mic-dist-25cm/throttle-low","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone1-only/mic-dist-25cm/throttle-low/mic2_8array-down-File5.wav","drone":"drone1","take":"File5","group_id":"droneaudioset:drone1/mic-dist-25cm/throttle-low","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone1-only/mic-dist-25cm/throttle-low/mic3_8array-up-File2.wav","drone":"drone1","take":"File2","group_id":"droneaudioset:drone1/mic-dist-25cm/throttle-low","label_status":"drone_only_upstream"}]},{"path":"drone-only/train_028-00000-of-00001.parquet","bytes":148788207,"sha256":"d5a103325d975a78f42bd328be8fe3cb51186f7679740fd7a117475f495448bf","url":"https://huggingface.co/datasets/ahlab-drone-project/DroneAudioSet/resolve/e794007062e2d2ad29262a5795b60d09f2b345c4/drone-only/train_028-00000-of-00001.parquet","rows":[{"upstream_file_path":"drone-only-recordings/drone1-only/mic-dist-25cm/throttle-high/mic1_soundskrit-File4.wav","drone":"drone1","take":"File4","group_id":"droneaudioset:drone1/mic-dist-25cm/throttle-high","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone1-only/mic-dist-25cm/throttle-high/mic1_soundskrit-File5.wav","drone":"drone1","take":"File5","group_id":"droneaudioset:drone1/mic-dist-25cm/throttle-high","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone1-only/mic-dist-25cm/throttle-high/mic2_8array-down-File1.wav","drone":"drone1","take":"File1","group_id":"droneaudioset:drone1/mic-dist-25cm/throttle-high","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone1-only/mic-dist-25cm/throttle-high/mic2_8array-down-File6.wav","drone":"drone1","take":"File6","group_id":"droneaudioset:drone1/mic-dist-25cm/throttle-high","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone1-only/mic-dist-25cm/throttle-high/mic3_8array-up-File3.wav","drone":"drone1","take":"File3","group_id":"droneaudioset:drone1/mic-dist-25cm/throttle-high","label_status":"drone_only_upstream"},{"upstream_file_path":"drone-only-recordings/drone1-only/mic-dist-25cm/throttle-high/mic3_8array-up-File5.wav","drone":"drone1","take":"File5","group_id":"droneaudioset:drone1/mic-dist-25cm/throttle-high","label_status":"drone_only_upstream"}]}]},"yi_kaist":{"api":"https://zenodo.org/api/records/7779574","file":{"path":"drone_A.tar","bytes":2554408960,"md5":"14a6fd11add6aedb6e53404df7811147","url":"https://zenodo.org/api/records/7779574/files/drone_A.tar/content"},"n_pairs":1200}}}''')
SEED = 20261111
N_PER_CLASS = 30
TARGET_RECALL = 0.90  # SOLO para la línea base de la iteración 5 (su regla de umbral)
N_FOLDS = 5
# Iteración 6 (valores fijados ANTES de ejecutar; no se ajustan mirando LOSO ni test):
MAIN_VARIANT = 'iter6 · completo (canal + cruzadas + armónico + INLP + DRO)'
BASELINE_VARIANT = 'iter5 · YAMNet media (línea base)'
MAX_WINDOWS_CLIP = 24     # ventanas de 0,96 s por clip (equiespaciadas, deterministas)
MAX_WINDOWS_AUG = 4       # ventanas por copia aumentada o mezcla cruzada
MAX_FRAMES_TRAIN = 4      # ventanas por clip en el entrenamiento
N_CROSS_MIXES = 1500      # mezclas cruzadas dron/fondo (fuentes distintas), solo entrenamiento
N_CHANNEL_AUG = 2000      # copias con canal aleatorio (ambas clases), solo entrenamiento
MAX_FPR_ON = 0.05         # umbral por peor fuente: FPR de validación <= 5 % en CADA fuente
MAX_FPR_OFF = 0.15        # fin de alerta (histéresis)
INLP_ROUNDS = 2
DRO_ROUNDS = 3
DRO_ETA = 1.0
LR_C = 0.5
CAL_GROUPS = 5            # variante de sitio (D-7): grupos negativos de la fuente retenida
ONNX_OPSET = 17
PARITY_TOL = 1e-4
PARITY_WINDOWS = 60
CODE_VERSION = "iteracion-6"
NOTEBOOK_CODE_SHA256 = "f913eb344d218c2a9e4670055849fe4200773f42e5fd33e4df861e770a6ad401"  # SHA-256 de las celdas (con este marcador), fijado al generar
EXTRA_AUTHORIZATION = {'salford': 'usuario, 2026-11-16: «Sí, las 3 (Recommended)»', 'droneaudioset': 'usuario, 2026-11-16: «Sí, las 3 (Recommended)»', 'yi_kaist': 'usuario, 2026-11-16: «Sí, las 3 (Recommended)»'}  # fuente -> cita de DECISIONES-USUARIO.md (None = desactivada)
BASE = Path("/content/atalaya_piloto")
BASE.mkdir(parents=True, exist_ok=True)
CHECKPOINTS = BASE / "checkpoints"
CHECKPOINTS.mkdir(exist_ok=True)
COMMIT = PILOT["source_commit"]
PROHIBITED = PILOT["prohibited"]
T0 = time.time()
def step(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)
def load_audio(path):
    return librosa.load(str(path), sr=None, mono=False)
step(f"Iteración 6 · modelo principal preinscrito: {MAIN_VARIANT}")
print("Fuentes prohibidas (no se descargan):", ", ".join(PROHIBITED))


In [ ]:
# Partición por grupos, umbral elegido en validación y métricas con incertidumbre.
# Autocontenido (stdlib, numpy, scikit-learn): build_colab.py lo incrusta tal cual en el
# cuaderno de Colab, de modo que el código probado es el que se ejecuta allí.
import math
import random
from collections.abc import Sequence

import numpy as np
from sklearn.metrics import average_precision_score

PROVISIONAL_LABEL = "prueba técnica, contaminación entre particiones no descartada"


def split_indices_by_group(
    labels: Sequence[int], groups: Sequence[str], *, seed: int
) -> tuple[tuple[int, ...], tuple[int, ...], tuple[int, ...]]:
    """Train/validación/test por grupos: ningún grupo aparece en dos particiones."""
    if len(labels) != len(groups):
        raise ValueError("labels y groups deben tener la misma longitud")
    unique = sorted(set(groups))
    if len(unique) < 6:
        raise ValueError("se necesitan al menos seis grupos independientes")
    n_test = max(2, round(len(unique) * 0.2))
    n_validation = max(2, round(len(unique) * 0.2))
    if len(unique) - n_test - n_validation < 2:
        raise ValueError("grupos insuficientes para train/validation/test")
    for attempt in range(1000):
        shuffled = unique.copy()
        random.Random(seed + attempt).shuffle(shuffled)
        chosen = (
            set(shuffled[n_test + n_validation :]),
            set(shuffled[n_test : n_test + n_validation]),
            set(shuffled[:n_test]),
        )
        parts = tuple(
            tuple(i for i, group in enumerate(groups) if group in selected) for selected in chosen
        )
        if all({labels[i] for i in part} == {0, 1} for part in parts):
            train, validation, test = parts
            seen = [{groups[i] for i in part} for part in parts]
            if seen[0] & seen[1] or seen[0] & seen[2] or seen[1] & seen[2]:
                raise ValueError("un grupo aparece en varias particiones")
            return train, validation, test
    raise ValueError("no hay partición por grupos con ambas clases en los tres conjuntos")


def choose_threshold(y_validation: np.ndarray, p_validation: np.ndarray) -> float:
    """Umbral que maximiza F1 en validación; en empate, el más alto (menos alarmas)."""
    y = np.asarray(y_validation).astype(int)
    p = np.asarray(p_validation, dtype=float)
    if set(np.unique(y).tolist()) != {0, 1}:
        raise ValueError("la validación necesita ambas clases para fijar el umbral")
    best_threshold, best_f1 = math.nan, -1.0
    for threshold in np.unique(p):
        pred = p >= threshold
        tp = int(np.sum(pred & (y == 1)))
        fp = int(np.sum(pred & (y == 0)))
        fn = int(np.sum(~pred & (y == 1)))
        f1 = 2 * tp / (2 * tp + fp + fn) if tp else 0.0
        if f1 >= best_f1:
            best_threshold, best_f1 = float(threshold), f1
    return best_threshold


def binary_metrics(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    """Matriz de confusión, precision, recall y PR-AUC (precisión media)."""
    y = np.asarray(y_true).astype(int)
    p = np.asarray(scores, dtype=float)
    pred = p >= threshold
    tp = int(np.sum(pred & (y == 1)))
    fp = int(np.sum(pred & (y == 0)))
    fn = int(np.sum(~pred & (y == 1)))
    tn = int(np.sum(~pred & (y == 0)))
    both = set(np.unique(y).tolist()) == {0, 1}
    return {
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "precision": tp / (tp + fp) if tp + fp else math.nan,
        "recall": tp / (tp + fn) if tp + fn else math.nan,
        "pr_auc": float(average_precision_score(y, p)) if both else math.nan,
        # Tasa de falsos positivos por clip: la única cifra útil en fuentes solo negativas.
        "fpr": fp / (fp + tn) if fp + tn else math.nan,
    }


def group_bootstrap(
    y_true: np.ndarray,
    scores: np.ndarray,
    groups: Sequence[str],
    threshold: float,
    *,
    n_boot: int = 2000,
    seed: int = 20261111,
    min_groups: int = 10,
) -> dict:
    """IC 95 % por percentiles remuestreando grupos completos, no clips sueltos."""
    y = np.asarray(y_true).astype(int)
    p = np.asarray(scores, dtype=float)
    unique = sorted(set(groups))
    if len(unique) < min_groups:
        return {"status": "INSUFICIENTE", "n_groups": len(unique), "min_groups": min_groups}
    members = {g: np.array([i for i, h in enumerate(groups) if h == g]) for g in unique}
    rng = np.random.default_rng(seed)
    samples: dict[str, list[float]] = {"precision": [], "recall": [], "pr_auc": [], "fpr": []}
    for _ in range(n_boot):
        drawn = rng.choice(len(unique), size=len(unique), replace=True)
        idx = np.concatenate([members[unique[k]] for k in drawn])
        metrics = binary_metrics(y[idx], p[idx], threshold)
        for name in samples:
            samples[name].append(metrics[name])
    # Por métrica: None si menos de la mitad de las réplicas dan un valor definido (p. ej.
    # recall en una fuente sin positivos). Nunca se rellena con un valor inventado.
    intervals: dict[str, tuple[float, float] | None] = {}
    for name, values in samples.items():
        finite = np.asarray(values, dtype=float)
        finite = finite[np.isfinite(finite)]
        intervals[name] = (
            None
            if finite.size < n_boot // 2
            else (float(np.percentile(finite, 2.5)), float(np.percentile(finite, 97.5)))
        )
    return {"status": "OK", "n_groups": len(unique), "n_boot": n_boot, "intervals": intervals}


def false_alarms_per_hour(
    n_false_alarms: int, *, continuous_negative_seconds: float, min_hours: float
) -> dict:
    """Solo hay cifra con duración negativa continua conocida y suficiente."""
    hours = continuous_negative_seconds / 3600
    if hours < min_hours:
        return {"status": "NO VERIFICADA", "value": None, "hours": hours, "min_hours": min_hours}
    return {"status": "OK", "value": n_false_alarms / hours, "hours": hours, "min_hours": min_hours}


def metric_caption(*, provisional: bool) -> str:
    """Rótulo que acompaña a cada métrica según la fiabilidad de los grupos."""
    if provisional:
        return f"[{PROVISIONAL_LABEL}]"
    return "[grupos verificados; generalización limitada a estos grupos]"


def split_with_preassigned(
    labels: Sequence[int],
    groups: Sequence[str],
    preassigned: dict[int, str],
    *,
    seed: int,
) -> tuple[tuple[int, ...], tuple[int, ...], tuple[int, ...]]:
    """Partición única: los índices preasignados ('train'/'val'/'test', p. ej. el split de los
    autores de ITU ARIS) conservan su parte; el resto se reparte por grupos. Un grupo nunca
    cruza particiones ni mezcla índices preasignados y libres."""
    if len(labels) != len(groups):
        raise ValueError("labels y groups deben tener la misma longitud")
    names = ("train", "val", "test")
    fixed_part: dict[str, str] = {}
    for index, part in preassigned.items():
        if part not in names:
            raise ValueError(f"partición desconocida: {part}")
        if fixed_part.setdefault(groups[index], part) != part:
            raise ValueError(f"el grupo {groups[index]} cruza particiones")
    free = [i for i in range(len(labels)) if i not in preassigned]
    if any(groups[i] in fixed_part for i in free):
        raise ValueError("un grupo mezcla segmentos preasignados y libres")
    parts: list[list[int]] = [[], [], []]
    for index, part in preassigned.items():
        parts[names.index(part)].append(index)
    if free:
        sub = split_indices_by_group(
            [labels[i] for i in free], [groups[i] for i in free], seed=seed
        )
        for k, chunk in enumerate(sub):
            parts[k].extend(free[j] for j in chunk)
    train, validation, test = (tuple(sorted(p)) for p in parts)
    seen = [{groups[i] for i in part} for part in (train, validation, test)]
    if seen[0] & seen[1] or seen[0] & seen[2] or seen[1] & seen[2]:
        raise ValueError("un grupo aparece en varias particiones")
    if any({labels[i] for i in part} != {0, 1} for part in (train, validation, test)):
        raise ValueError("cada partición necesita ambas clases")
    return train, validation, test


def group_folds(
    labels: Sequence[int], groups: Sequence[str], *, k: int = 5, seed: int
) -> tuple[int, ...]:
    """Asigna cada grupo entero a uno de k folds (devuelve el fold de cada índice). Los grupos
    se recorren de mayor a menor (desempate barajado con semilla) y cada uno va al fold con
    menos ejemplos de su clase mayoritaria. Todos los folds deben tener ambas clases."""
    if len(labels) != len(groups):
        raise ValueError("labels y groups deben tener la misma longitud")
    if k < 3:
        raise ValueError("se necesitan al menos tres folds")
    members: dict[str, list[int]] = {}
    for i, g in enumerate(groups):
        members.setdefault(g, []).append(i)
    if len(members) < k:
        raise ValueError("menos grupos que folds")
    keys = sorted(members)
    random.Random(seed).shuffle(keys)
    keys.sort(key=lambda g: -len(members[g]))
    load = [[0, 0] for _ in range(k)]  # [negativos, positivos] por fold
    fold_of_group = {}
    for g in keys:
        pos = sum(int(labels[i]) for i in members[g])
        neg = len(members[g]) - pos
        cls = int(pos >= neg)
        target = min(range(k), key=lambda f: (load[f][cls], load[f][0] + load[f][1], f))
        fold_of_group[g] = target
        load[target][0] += neg
        load[target][1] += pos
    if any(n == 0 or p == 0 for n, p in load):
        raise ValueError("algún fold no tiene ambas clases")
    return tuple(fold_of_group[g] for g in groups)


def rotating_folds(
    folds: Sequence[int], k: int
) -> list[tuple[tuple[int, ...], tuple[int, ...], tuple[int, ...]]]:
    """Validación cruzada por grupos: en la vuelta f, test = fold f, validación = fold f+1
    (módulo k) solo para fijar el umbral, entrenamiento = el resto."""
    if set(folds) != set(range(k)):
        raise ValueError("los folds deben ser exactamente 0..k-1")
    out = []
    for f in range(k):
        v = (f + 1) % k
        test = tuple(i for i, x in enumerate(folds) if x == f)
        validation = tuple(i for i, x in enumerate(folds) if x == v)
        train = tuple(i for i, x in enumerate(folds) if x not in (f, v))
        out.append((train, validation, test))
    return out


def choose_threshold_for_recall(
    y_validation: np.ndarray, p_validation: np.ndarray, *, target_recall: float = 0.9
) -> dict:
    """Umbral MÁS ALTO cuyo recall en validación es >= target_recall: no perder drones con el
    mínimo de alarmas a esa exigencia. Se elige solo en validación, nunca en test."""
    if not 0.0 < target_recall <= 1.0:
        raise ValueError("target_recall debe estar en (0, 1]")
    y = np.asarray(y_validation).astype(int)
    p = np.asarray(p_validation, dtype=float)
    if set(np.unique(y).tolist()) != {0, 1}:
        raise ValueError("la validación necesita ambas clases para fijar el umbral")
    positives = int(np.sum(y == 1))
    for threshold in sorted(np.unique(p).tolist(), reverse=True):
        pred = p >= threshold
        tp = int(np.sum(pred & (y == 1)))
        if tp / positives >= target_recall:
            fp = int(np.sum(pred & (y == 0)))
            return {
                "threshold": float(threshold),
                "target_recall": float(target_recall),
                "validation_recall": tp / positives,
                "validation_precision": tp / (tp + fp),
                "criterion": f"umbral más alto con recall de validación >= {target_recall:.2f}",
            }
    raise ValueError("puntuaciones no válidas")  # el umbral mínimo siempre da recall 1


def metrics_by_source(
    y_true: np.ndarray,
    scores: np.ndarray,
    sources: Sequence[str],
    groups: Sequence[str],
    threshold: float,
    *,
    seed: int = 20261111,
    n_boot: int = 2000,
    min_groups: int = 10,
) -> list[dict]:
    """Métricas por fuente y agregadas (fila TOTAL) con IC 95 % remuestreando grupos."""
    if not len(y_true) == len(scores) == len(sources) == len(groups):
        raise ValueError("longitudes distintas")
    y = np.asarray(y_true).astype(int)
    p = np.asarray(scores, dtype=float)
    rows = []
    for name in [*sorted(set(sources)), "TOTAL"]:
        idx = np.array([i for i, s in enumerate(sources) if name in ("TOTAL", s)], dtype=int)
        metrics = binary_metrics(y[idx], p[idx], threshold)
        boot = group_bootstrap(
            y[idx],
            p[idx],
            [groups[i] for i in idx],
            threshold,
            n_boot=n_boot,
            seed=seed,
            min_groups=min_groups,
        )
        rows.append(
            {
                "fuente": name,
                "n": int(idx.size),
                "positivos": int(y[idx].sum()),
                "grupos": len({groups[i] for i in idx}),
                **metrics,
                "ic95": boot,
            }
        )
    return rows


In [ ]:
# Puerta automática de privacidad: excluye clips con posible voz según YAMNet.
# Autocontenido (solo stdlib): build_colab.py lo incrusta tal cual en el cuaderno de Colab,
# de modo que el código probado es el que se ejecuta allí.
import math
from collections.abc import Mapping, Sequence

# Máxima puntuación YAMNet de Speech/Conversation/Narration/Child speech en cualquier ventana.
# Revisión AUTOMÁTICA y conservadora para la prueba técnica; NO sustituye a la escucha humana.
SPEECH_EXCLUSION_THRESHOLD = 0.30
SPEECH_GATE_NOTE = (
    "Exclusión automática por puntuación de voz de YAMNet (max_speech_score >= umbral); "
    "revisión automática que no sustituye a la escucha humana"
)


def apply_speech_gate(
    names: Sequence[str],
    max_speech_scores: Sequence[float],
    *,
    threshold: float = SPEECH_EXCLUSION_THRESHOLD,
    human_cleared: Mapping[str, str] | None = None,
) -> dict:
    """Separa los clips aptos de los excluidos por posible voz (función pura, fallo cerrado).

    Se excluye todo clip con puntuación >= umbral y todo clip sin puntuación finita, salvo
    que una persona lo haya escuchado y declarado sin voz (human_cleared: clip -> revisión
    con revisor, fecha y nota). La revisión humana nunca rescata una puntuación no válida.
    """
    cleared = dict(human_cleared or {})
    if any(not str(v).strip() for v in cleared.values()):
        raise ValueError("toda revisión humana debe indicar revisor, fecha y nota")
    if not (isinstance(threshold, (int, float)) and 0.0 < threshold <= 1.0):
        raise ValueError("el umbral de voz debe estar en (0, 1]")
    if len(names) != len(max_speech_scores):
        raise ValueError("names y max_speech_scores deben tener la misma longitud")
    if len(set(names)) != len(names):
        raise ValueError("nombres de clip duplicados")
    kept, excluded, rescued = [], [], []
    for index, (name, score) in enumerate(zip(names, max_speech_scores, strict=True)):
        value = float(score)
        if not math.isfinite(value):
            excluded.append(
                {"clip": str(name), "max_speech_score": None, "motivo": "puntuacion_no_valida"}
            )
        elif value >= threshold and str(name) in cleared:
            rescued.append(
                {"clip": str(name), "max_speech_score": value, "revision": cleared[str(name)]}
            )
            kept.append(index)
        elif value >= threshold:
            excluded.append(
                {"clip": str(name), "max_speech_score": value, "motivo": "voz_automatica"}
            )
        else:
            kept.append(index)
    return {
        "threshold": float(threshold),
        "note": SPEECH_GATE_NOTE,
        "kept_indices": tuple(kept),
        "excluded": excluded,
        "human_cleared": rescued,
    }


In [ ]:
# Descargas HTTPS restringidas: solo hosts permitidos (también tras redirecciones), rangos
# opcionales, tope de tamaño y reintentos ante 429/503 respetando Retry-After (Zenodo limita
# a ~133 peticiones/minuto). Autocontenido (solo stdlib): build_colab.py lo incrusta tal cual.
import time
import urllib.error
import urllib.request
from urllib.parse import urlparse

USER_AGENT = "Atalaya-Colab-Pilot/1"


def check_url(url: str, allowed_hosts: set[str] | frozenset[str]) -> None:
    parsed = urlparse(url)
    if parsed.scheme != "https" or parsed.hostname not in allowed_hosts:
        raise ValueError(f"origen no permitido: {url}")


def make_opener(allowed_hosts: set[str] | frozenset[str]) -> urllib.request.OpenerDirector:
    """Opener que comprueba cada redirección ANTES de seguirla."""
    hosts = frozenset(allowed_hosts)

    class CheckedRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, req, fp, code, msg, headers, newurl):  # type: ignore[no-untyped-def]
            check_url(newurl, hosts)
            return super().redirect_request(req, fp, code, msg, headers, newurl)

    return urllib.request.build_opener(CheckedRedirect)


def read_https(
    url: str,
    allowed_hosts: set[str] | frozenset[str],
    *,
    byte_range: tuple[int, int] | None = None,
    limit: int = 1 << 31,
    timeout: float = 120,
    retries: int = 6,
    opener: object | None = None,
    sleep=time.sleep,  # type: ignore[no-untyped-def]
) -> bytes:
    """GET con host permitido. byte_range = (inicio, fin_exclusivo) exige respuesta 206 con
    exactamente esos bytes. Falla si la respuesta supera ``limit``."""
    check_url(url, allowed_hosts)
    headers = {"User-Agent": USER_AGENT}
    if byte_range is not None:
        start, end = byte_range
        if not 0 <= start < end:
            raise ValueError("rango no válido")
        headers["Range"] = f"bytes={start}-{end - 1}"
        limit = min(limit, end - start)
    client = opener or make_opener(allowed_hosts)
    for attempt in range(retries + 1):
        try:
            with client.open(urllib.request.Request(url, headers=headers), timeout=timeout) as r:  # type: ignore[attr-defined]
                check_url(r.geturl(), allowed_hosts)
                status = getattr(r, "status", 200)
                if byte_range is not None and status != 206:
                    raise ValueError(f"el servidor no respetó el rango (HTTP {status})")
                data = r.read(limit + 1)
            if len(data) > limit:
                raise ValueError(f"respuesta demasiado grande: {url}")
            if byte_range is not None and len(data) != byte_range[1] - byte_range[0]:
                raise ConnectionError("rango incompleto")
            return data
        except urllib.error.HTTPError as error:
            if error.code not in (429, 502, 503, 504) or attempt == retries:
                raise
            wait = error.headers.get("Retry-After") if error.headers else None
            sleep(min(120.0, float(wait)) if wait and wait.isdigit() else 2.0 * (attempt + 1))
        except (ConnectionError, TimeoutError, urllib.error.URLError):
            # Transferencia cortada o red caída: se reintenta; los datos se verifican después.
            if attempt == retries:
                raise
            sleep(2.0 * (attempt + 1))
    raise ConnectionError("sin respuesta")  # inalcanzable: el último intento devuelve o lanza


In [ ]:
# Lectura parcial de ZIP (también divididos en volúmenes .z01…/.zip) mediante rangos HTTP.
# Autocontenido (solo stdlib): build_colab.py lo incrusta tal cual en el cuaderno de Colab.
# Permite descargar solo los clips seleccionados de un archivo grande (FSD50K) sin bajarlo
# entero; cada clip se verifica después por tamaño, CRC-32 y SHA-256 fijado.
import struct
import zlib
from collections.abc import Sequence

EOCD_SIG = 0x06054B50
ZIP64_LOCATOR_SIG = 0x07064B50
ZIP64_EOCD_SIG = 0x06064B50
CENTRAL_SIG = 0x02014B50
LOCAL_SIG = 0x04034B50
LOCAL_HEADER_LEN = 30
_U32 = 0xFFFFFFFF
_U16 = 0xFFFF


def parse_eocd(tail: bytes) -> dict:
    """Busca el registro final (EOCD) en la cola del último volumen y, si existe, el
    localizador ZIP64 que lo precede. Las posiciones son relativas al inicio de ``tail``."""
    pos = tail.rfind(struct.pack("<I", EOCD_SIG))
    if pos < 0 or len(tail) - pos < 22:
        raise ValueError("no se encuentra el registro final del ZIP")
    (_, disk, cd_disk, _, n_entries, cd_size, cd_offset, comment_len) = struct.unpack(
        "<IHHHHIIH", tail[pos : pos + 22]
    )
    if pos + 22 + comment_len != len(tail):
        raise ValueError("registro final del ZIP truncado o con datos detrás")
    result = {
        "disk": disk,
        "cd_disk": cd_disk,
        "n_entries": n_entries,
        "cd_size": cd_size,
        "cd_offset": cd_offset,
        "zip64_locator": None,
    }
    loc = pos - 20
    if loc >= 0 and struct.unpack("<I", tail[loc : loc + 4])[0] == ZIP64_LOCATOR_SIG:
        _, z_disk, z_offset, total_disks = struct.unpack("<IIQI", tail[loc : loc + 20])
        result["zip64_locator"] = {"disk": z_disk, "offset": z_offset, "total_disks": total_disks}
    return result


def parse_zip64_eocd(record: bytes) -> dict:
    """Registro final ZIP64 (56 bytes fijos)."""
    if len(record) < 56 or struct.unpack("<I", record[:4])[0] != ZIP64_EOCD_SIG:
        raise ValueError("registro final ZIP64 no válido")
    (_, _, _, _, disk, cd_disk, _, n_entries, cd_size, cd_offset) = struct.unpack(
        "<IQHHIIQQQQ", record[:56]
    )
    return {
        "disk": disk,
        "cd_disk": cd_disk,
        "n_entries": n_entries,
        "cd_size": cd_size,
        "cd_offset": cd_offset,
    }


def _zip64_extra(extra: bytes, need: Sequence[str]) -> dict:
    values: dict = {}
    i = 0
    while i + 4 <= len(extra):
        tag, size = struct.unpack("<HH", extra[i : i + 4])
        body = extra[i + 4 : i + 4 + size]
        if tag == 0x0001:
            j = 0
            for name in need:
                width = 4 if name == "disk" else 8
                if j + width > len(body):
                    raise ValueError("campo ZIP64 truncado")
                values[name] = int.from_bytes(body[j : j + width], "little")
                j += width
            return values
        i += 4 + size
    if need:
        raise ValueError("faltan campos ZIP64")
    return values


def parse_central_directory(data: bytes, n_entries: int) -> list[dict]:
    """Entradas del directorio central: nombre, volumen, desplazamiento local, tamaños,
    método y CRC-32. Falla si el número de entradas no coincide."""
    entries, i = [], 0
    while i < len(data) and len(entries) < n_entries:
        if struct.unpack("<I", data[i : i + 4])[0] != CENTRAL_SIG:
            raise ValueError(f"firma de directorio central no válida en {i}")
        (
            _,
            _,
            _,
            flags,
            method,
            _,
            _,
            crc,
            csize,
            usize,
            nlen,
            xlen,
            clen,
            disk,
            _,
            _,
            offset,
        ) = struct.unpack("<IHHHHHHIIIHHHHHII", data[i : i + 46])
        name = data[i + 46 : i + 46 + nlen].decode("utf-8" if flags & 0x800 else "cp437")
        extra = data[i + 46 + nlen : i + 46 + nlen + xlen]
        need = [
            n
            for n, v, top in (
                ("usize", usize, _U32),
                ("csize", csize, _U32),
                ("offset", offset, _U32),
                ("disk", disk, _U16),
            )
            if v == top
        ]
        big = _zip64_extra(extra, need)
        entries.append(
            {
                "name": name,
                "disk": big.get("disk", disk),
                "offset": big.get("offset", offset),
                "compressed_size": big.get("csize", csize),
                "size": big.get("usize", usize),
                "method": method,
                "crc32": crc,
                "encrypted": bool(flags & 0x1),
            }
        )
        i += 46 + nlen + xlen + clen
    if len(entries) != n_entries:
        raise ValueError("el directorio central no tiene el número de entradas declarado")
    return entries


def local_header_length(header: bytes, expected_name: str) -> int:
    """Longitud de la cabecera local (30 + nombre + extra); comprueba firma y nombre."""
    if len(header) < LOCAL_HEADER_LEN or struct.unpack("<I", header[:4])[0] != LOCAL_SIG:
        raise ValueError("cabecera local no válida")
    nlen, xlen = struct.unpack("<HH", header[26:30])
    name = header[30 : 30 + nlen].decode("utf-8", errors="replace")
    if name != expected_name:
        raise ValueError(f"cabecera local de otro fichero: {name!r}")
    return LOCAL_HEADER_LEN + nlen + xlen


def volume_ranges(
    disk: int, offset: int, length: int, volume_sizes: Sequence[int]
) -> list[tuple[int, int, int]]:
    """Trocea un intervalo lógico que empieza en (disk, offset) en rangos por volumen:
    (volumen, inicio, fin_exclusivo). Un miembro puede cruzar al volumen siguiente."""
    if length < 0 or not 0 <= disk < len(volume_sizes):
        raise ValueError("volumen o longitud fuera de rango")
    if not 0 <= offset < volume_sizes[disk]:
        raise ValueError("desplazamiento fuera del volumen")
    ranges, remaining = [], length
    while remaining:
        if disk >= len(volume_sizes):
            raise ValueError("el intervalo sobrepasa el último volumen")
        take = min(remaining, volume_sizes[disk] - offset)
        ranges.append((disk, offset, offset + take))
        remaining -= take
        disk, offset = disk + 1, 0
    return ranges


def decode_member(payload: bytes, *, method: int, size: int, crc32: int) -> bytes:
    """Descomprime (almacenado o deflate) y verifica tamaño y CRC-32 del miembro."""
    if method == 0:
        data = payload
    elif method == 8:
        data = zlib.decompressobj(-15).decompress(payload)
    else:
        raise ValueError(f"método de compresión no admitido: {method}")
    if len(data) != size or (zlib.crc32(data) & _U32) != crc32:
        raise ValueError("tamaño o CRC-32 del miembro no coinciden")
    return data


In [ ]:
# Filtros de licencia por clip, selección reproducible y agrupación por fuente/sesión.
# Autocontenido (solo stdlib): build_colab.py lo incrusta tal cual en el cuaderno de Colab, y
# build_sources.py lo usa para fijar los manifiestos; el código probado es el que se ejecuta.
import csv
import hashlib
import io
import re
from collections.abc import Iterable, Mapping, Sequence

# Licencias por clip admitidas (URL exacta de Freesound/FSD50K). Todo lo demás se rechaza:
# CC BY-NC (no comercial), CC Sampling+ (dudosa) y cualquier valor desconocido.
ALLOWED_CLIP_LICENSES = {
    "http://creativecommons.org/publicdomain/zero/1.0/": "CC0-1.0",
    "http://creativecommons.org/licenses/by/3.0/": "CC-BY-3.0",
    "http://creativecommons.org/licenses/by/4.0/": "CC-BY-4.0",
}

# Etiquetas FSD50K de sonidos humanos (voz, cuerpo, grupos) y música: siempre se excluyen.
HUMAN_OR_MUSIC_LABELS = frozenset(
    {
        "Applause",
        "Breathing",
        "Burping_and_eructation",
        "Chatter",
        "Cheering",
        "Chewing_and_mastication",
        "Child_speech_and_kid_speaking",
        "Chuckle_and_chortle",
        "Clapping",
        "Conversation",
        "Cough",
        "Crowd",
        "Crying_and_sobbing",
        "Fart",
        "Female_singing",
        "Female_speech_and_woman_speaking",
        "Finger_snapping",
        "Gasp",
        "Giggle",
        "Hands",
        "Human_group_actions",
        "Human_voice",
        "Laughter",
        "Male_singing",
        "Male_speech_and_man_speaking",
        "Respiratory_sounds",
        "Run",
        "Screaming",
        "Shout",
        "Sigh",
        "Singing",
        "Sneeze",
        "Speech",
        "Speech_synthesizer",
        "Walk_and_footsteps",
        "Whispering",
        "Yell",
        "Music",
    }
)

# Confusores pedidos: motores, motos, barcos, aviones, insectos, viento, lluvia y aves.
FSD50K_TARGET_LABELS = (
    "Engine",
    "Motorcycle",
    "Boat_and_Water_vehicle",
    "Fixed-wing_aircraft_and_airplane",
    "Aircraft",
    "Insect",
    "Wind",
    "Rain",
    "Gull_and_seagull",
    "Bird",
)

# Un dron no puede entrar como negativo: se descarta cualquier mención (con límite de palabra;
# «dji» no coincide con «zildjian»). Conservador: también excluye «drone» musical.
DRONE_PATTERN = re.compile(
    r"\b(drones?|quadcopters?|quadrocopters?|multicopters?|multirotors?|hexacopters?|"
    r"octocopters?|uavs?|uas|dji|mavic)\b",
    re.IGNORECASE,
)


def clip_license_allowed(url: str) -> bool:
    return str(url).strip() in ALLOWED_CLIP_LICENSES


def mentions_drone(info: Mapping) -> bool:
    text = " ".join(
        [str(info.get("title", "")), str(info.get("description", ""))]
        + [str(t) for t in info.get("tags", []) or []]
    )
    return bool(DRONE_PATTERN.search(text.replace("_", " ").replace("-", " ")))


def uploader_group(source: str, uploader: str) -> str:
    """Grupo = fuente + autor (seudonimizado). Mismo autor => mismo grupo (fuga de equipo)."""
    if not str(uploader).strip():
        raise ValueError("uploader vacío: no se puede agrupar")
    digest = hashlib.sha256(str(uploader).strip().encode("utf-8")).hexdigest()[:12]
    return f"{source}:u-{digest}"


def read_fsd50k_ground_truth(text: str, split: str) -> dict[str, list[str]]:
    rows = {}
    for row in csv.DictReader(io.StringIO(text)):
        fname = row["fname"].strip()
        if not fname.isdigit() or fname in rows:
            raise ValueError(f"fname no válido o repetido en {split}: {fname!r}")
        rows[fname] = [label for label in row["labels"].split(",") if label]
    return rows


def fsd50k_rejection(fname: str, labels: Sequence[str], info: Mapping | None) -> str | None:
    """Motivo de rechazo de un clip, o None si es admisible como negativo."""
    if info is None:
        return "sin_metadatos"
    if not clip_license_allowed(info.get("license", "")):
        return "licencia_no_admitida"
    if set(labels) & HUMAN_OR_MUSIC_LABELS:
        return "etiqueta_humana_o_musica"
    if mentions_drone(info):
        return "menciona_dron"
    if not str(info.get("uploader", "")).strip():
        return "sin_autor"
    return None


def _rank(seed: int, key: str) -> str:
    return hashlib.sha256(f"{seed}:{key}".encode()).hexdigest()


def select_fsd50k(
    ground_truth: Mapping[str, Mapping[str, Sequence[str]]],
    clips_info: Mapping[str, Mapping[str, Mapping]],
    compressed_sizes: Mapping[str, Mapping[str, int]],
    *,
    target_labels: Sequence[str] = FSD50K_TARGET_LABELS,
    per_label: int,
    per_uploader_per_label: int,
    max_compressed_bytes: int,
    excluded_ids: Iterable[str] = (),
    seed: int,
) -> list[dict]:
    """Selección determinista: por etiqueta objetivo (en orden), clips admisibles ordenados
    por hash con semilla, con tope por autor y presupuesto de bytes repartido por etiqueta.
    Cada clip se asigna a la primera etiqueta objetivo que lo reclama (sin repeticiones)."""
    if per_label < 1 or per_uploader_per_label < 1 or max_compressed_bytes < 1:
        raise ValueError("topes de selección no válidos")
    excluded = {str(x) for x in excluded_ids}
    budget = max_compressed_bytes // len(target_labels)
    chosen: dict[str, dict] = {}
    for label in target_labels:
        candidates = []
        for split in sorted(ground_truth):
            for fname, labels in ground_truth[split].items():
                if label not in labels or fname in excluded or fname in chosen:
                    continue
                info = clips_info.get(split, {}).get(fname)
                if fsd50k_rejection(fname, labels, info) is not None:
                    continue
                size = compressed_sizes.get(split, {}).get(fname)
                if size is None:
                    continue
                candidates.append((_rank(seed, f"{label}:{fname}"), split, fname, size))
        used, per_uploader, n = 0, {}, 0
        for _, split, fname, size in sorted(candidates):
            if n >= per_label:
                break
            info = clips_info[split][fname]
            uploader = str(info["uploader"]).strip()
            if per_uploader.get(uploader, 0) >= per_uploader_per_label or used + size > budget:
                continue
            per_uploader[uploader] = per_uploader.get(uploader, 0) + 1
            used += size
            n += 1
            chosen[fname] = {
                "fname": fname,
                "split": split,
                "target_label": label,
                "labels": list(ground_truth[split][fname]),
                "license": ALLOWED_CLIP_LICENSES[str(info["license"]).strip()],
                "license_url": str(info["license"]).strip(),
                "uploader": uploader,
                "title": str(info.get("title", "")),
                "group_id": uploader_group("fsd50k", uploader),
                "compressed_bytes": size,
            }
    return sorted(chosen.values(), key=lambda c: (c["target_label"], c["split"], int(c["fname"])))


def verify_fsd50k_pins(
    pinned: Sequence[Mapping],
    ground_truth: Mapping[str, Mapping[str, Sequence[str]]],
    clips_info: Mapping[str, Mapping[str, Mapping]],
) -> list[str]:
    """Revalida en los metadatos de la fuente cada clip fijado: licencia por clip admitida e
    igual a la fijada, sin etiquetas humanas, sin mención a drones y mismo autor. Devuelve
    la lista de errores (vacía si todo cuadra)."""
    errors = []
    seen = set()
    for pin in pinned:
        split, fname = pin["split"], str(pin["fname"])
        if (split, fname) in seen:
            errors.append(f"{fname}: repetido")
        seen.add((split, fname))
        labels = ground_truth.get(split, {}).get(fname)
        info = clips_info.get(split, {}).get(fname)
        if labels is None:
            errors.append(f"{fname}: no está en ground truth de {split}")
            continue
        reason = fsd50k_rejection(fname, labels, info)
        if reason is not None:
            errors.append(f"{fname}: {reason}")
            continue
        if str(info["license"]).strip() != pin["license_url"]:
            errors.append(f"{fname}: licencia distinta de la fijada")
        if str(info["uploader"]).strip() != pin["uploader"]:
            errors.append(f"{fname}: autor distinto del fijado")
        if pin["target_label"] not in labels:
            errors.append(f"{fname}: ya no tiene la etiqueta {pin['target_label']}")
    return errors


def aris_segments(groups_doc: Mapping, split_doc: Mapping) -> list[dict]:
    """Segmentos de ITU ARIS con su grupo (intervalo de dron o bloque de fondo) y la
    partición de los autores. Falla si un grupo cruza particiones, si la etiqueta no
    coincide con la ruta o si falta algún segmento: nunca se parte al azar."""
    assignment = split_doc["assignment"]
    segments = groups_doc["segments"]
    files = [s["file"] for s in segments]
    if len(set(files)) != len(files):
        raise ValueError("segmento repetido en groups.json")
    if set(files) != set(assignment):
        raise ValueError("groups.json y el split no cubren los mismos segmentos")
    if any(v not in {"train", "val", "test"} for v in assignment.values()):
        raise ValueError("partición desconocida en el split")
    out, part_of_group = [], {}
    for seg in segments:
        path, label, group = seg["file"], seg["label"], str(seg["group"])
        folder = path.split("/")[1] if path.count("/") >= 2 else ""
        if label not in {"drone", "background"} or folder != label:
            raise ValueError(f"etiqueta y ruta no coinciden: {path}")
        if not re.fullmatch(r"audio/(drone|background)/\1_\d{4}\.wav", path):
            raise ValueError(f"ruta no permitida: {path}")
        part = assignment[path]
        if part_of_group.setdefault(group, part) != part:
            raise ValueError(f"el grupo {group} cruza particiones")
        out.append(
            {
                "path": path,
                "label": int(label == "drone"),
                "group_id": f"aris:{group}",
                "split": part,
            }
        )
    return out


def parse_sha256sums(text: str) -> dict[str, str]:
    """Fichero SHA256SUMS de la fuente: «<hex>  <ruta>» por línea."""
    sums = {}
    for line in text.splitlines():
        if not line.strip():
            continue
        digest, _, path = line.partition("  ")
        if len(digest) != 64 or any(c not in "0123456789abcdef" for c in digest) or not path:
            raise ValueError(f"línea SHA256SUMS no válida: {line[:80]!r}")
        if path in sums:
            raise ValueError(f"ruta repetida en SHA256SUMS: {path}")
        sums[path] = digest
    return sums


In [ ]:
# Exportación de predicciones a CSV determinista con SHA-256 (ADR 0016, pendiente: métricas
# recalculadas por script desde un fichero de predicciones, no copiadas de la salida).
# Autocontenido (solo stdlib): build_colab.py lo incrusta tal cual en el cuaderno de Colab.
# El recálculo independiente está en pilots/acoustic/recompute_metrics.py.
import csv
import hashlib
import io
import math
from collections.abc import Iterable, Mapping

PREDICTION_COLUMNS = (
    "modelo",
    "esquema",
    "particion",
    "fuente",
    "clip",
    "grupo",
    "dron",
    "puntuacion",
    "umbral",
)


# Iteración 5: tipo de negativo y puntuaciones sin calibrar e isotónica (la columna
# «puntuacion» es la calibrada con Platt, a la que se aplica «umbral»).
EXTENDED_COLUMNS = (
    "modelo",
    "esquema",
    "particion",
    "fuente",
    "clip",
    "grupo",
    "tipo",
    "dron",
    "puntuacion",
    "puntuacion_cruda",
    "puntuacion_isotonica",
    "umbral",
)
_SCORE_COLUMNS = ("puntuacion", "puntuacion_cruda", "puntuacion_isotonica", "umbral")


def predictions_csv(
    rows: Iterable[Mapping], columns: tuple[str, ...] = PREDICTION_COLUMNS
) -> tuple[bytes, str]:
    """CSV UTF-8 con columnas fijas, filas ordenadas y números con 9 decimales (formato base)
    o con precisión de ida y vuelta (.17g, formato ampliado: las probabilidades calibradas
    saturan cerca de 1 y 9 decimales convertirían en empates decisiones distintas); devuelve
    (bytes, sha256). Mismas predicciones => mismos bytes => mismo hash."""
    if columns not in (PREDICTION_COLUMNS, EXTENDED_COLUMNS):
        raise ValueError("formato de columnas desconocido")
    clean = []
    for row in rows:
        if set(row) != set(columns):
            raise ValueError(f"columnas inesperadas: {sorted(set(row) ^ set(columns))}")
        label = int(row["dron"])
        numbers = {k: float(row[k]) for k in _SCORE_COLUMNS if k in columns}
        if label not in (0, 1) or not all(math.isfinite(v) for v in numbers.values()):
            raise ValueError("etiqueta o puntuación no válidas")
        text = {k: str(row[k]) for k in columns}
        if any(c in v for v in text.values() for c in "\r\n"):
            raise ValueError("salto de línea en un campo")
        fmt = ".17g" if columns == EXTENDED_COLUMNS else ".9f"
        text.update(dron=str(label), **{k: format(v, fmt) for k, v in numbers.items()})
        clean.append(text)
    keys = [tuple(r[k] for k in ("modelo", "esquema", "particion", "clip")) for r in clean]
    if len(set(keys)) != len(keys):
        raise ValueError("predicción repetida para el mismo modelo, esquema, partición y clip")
    clean.sort(key=lambda r: tuple(r[k] for k in columns))
    buffer = io.StringIO()
    writer = csv.DictWriter(buffer, fieldnames=columns, lineterminator="\n")
    writer.writeheader()
    writer.writerows(clean)
    data = buffer.getvalue().encode("utf-8")
    return data, hashlib.sha256(data).hexdigest()


In [ ]:
# Calidad de datos (iteración 5): formato, clips defectuosos, sonoridad y duplicados.
# Autocontenido (stdlib, numpy, scipy): build_colab.py lo incrusta tal cual en el cuaderno.
import hashlib
import math
from collections.abc import Sequence

import numpy as np
from scipy.signal import lfilter, resample_poly

TARGET_SR = 16000
# Umbrales documentados (prueba técnica; ver docs/evidence/piloto-acustico):
SILENCE_RMS_DBFS = -60.0  # RMS por debajo => clip silencioso (sin información útil)
SATURATION_LEVEL = 0.999  # |x| >= este valor cuenta como muestra recortada
SATURATION_MAX_FRACTION = 0.01  # > 1 % de muestras recortadas => clip saturado
MIN_SECONDS = 0.5  # más corto => corrupto (YAMNet necesita 0,975 s; se rellena hasta ahí)
# Casi duplicados: coseno de la media de embeddings YAMNet >= 0,995. YAMNet da embeddings no
# negativos (ReLU) y clips distintos de una misma escena llegan a ~0,95-0,98; 0,995 solo une
# copias recortadas, re-codificadas o con ganancia distinta. Heurística: se informa siempre.
NEAR_DUPLICATE_COSINE = 0.995


def normalize_format(signal: np.ndarray, sr: int, *, target_sr: int = TARGET_SR) -> np.ndarray:
    """Mono (media de canales), target_sr Hz y float32. Acepta (n,), (canales, n) o (n, canales)
    y PCM entero (se escala a [-1, 1])."""
    if sr <= 0 or target_sr <= 0:
        raise ValueError("frecuencia de muestreo no válida")
    x = np.asarray(signal)
    if np.issubdtype(x.dtype, np.integer):
        x = x.astype(np.float64) / float(np.iinfo(x.dtype).max)
    x = x.astype(np.float64)
    if x.ndim == 2:
        x = x.mean(axis=0) if x.shape[0] <= x.shape[1] else x.mean(axis=1)
    elif x.ndim != 1:
        raise ValueError("se esperaba señal 1D o 2D")
    if sr != target_sr and x.size:
        g = math.gcd(int(sr), int(target_sr))
        x = resample_poly(x, target_sr // g, int(sr) // g)
    return x.astype(np.float32)


def clip_quality(signal: np.ndarray, sr: int) -> dict:
    """Estado del clip ANTES del remuestreo: 'ok', 'silencioso', 'saturado' o 'corrupto'.
    La saturación se mide en cada canal original (la media mono puede ocultarla)."""
    x = np.asarray(signal, dtype=np.float64)
    channels = x if x.ndim == 2 and x.shape[0] <= x.shape[1] else (x.T if x.ndim == 2 else x[None])
    n = channels.shape[-1] if channels.size else 0
    out = {
        "status": "ok",
        "seconds": n / sr if sr > 0 else 0.0,
        "rms_dbfs": None,
        "clipped_fraction": None,
    }
    if n == 0 or sr <= 0 or not np.isfinite(channels).all():
        out["status"] = "corrupto"
        return out
    if n / sr < MIN_SECONDS:
        out["status"] = "corrupto"
        return out
    mono = channels.mean(axis=0)
    rms = float(np.sqrt(np.mean(mono**2)))
    out["rms_dbfs"] = 20 * math.log10(rms) if rms > 0 else -120.0
    clipped = float(np.max(np.mean(np.abs(channels) >= SATURATION_LEVEL, axis=1)))
    out["clipped_fraction"] = clipped
    channel_rms = float(np.max(np.sqrt(np.mean(channels**2, axis=1))))
    if clipped > SATURATION_MAX_FRACTION:
        out["status"] = "saturado"
    elif channel_rms <= 0 or 20 * math.log10(channel_rms) < SILENCE_RMS_DBFS:
        out["status"] = "silencioso"
    return out


def k_weighting_coefficients(sr: int) -> tuple[tuple[np.ndarray, np.ndarray], ...]:
    """Filtros K de ITU-R BS.1770 (estante alto + paso alto) para cualquier sr, a partir de los
    parámetros analógicos de la norma (formulación de De Man, 2014)."""
    gain_db, q_s, fc_s = 3.99984385397, 0.7071752369554193, 1681.9744509555319
    k = math.tan(math.pi * fc_s / sr)
    vh = 10 ** (gain_db / 20)
    vb = vh**0.499666774155
    a0 = 1 + k / q_s + k * k
    shelf_b = (
        np.array([(vh + vb * k / q_s + k * k), 2 * (k * k - vh), (vh - vb * k / q_s + k * k)]) / a0
    )
    shelf_a = np.array([1.0, 2 * (k * k - 1) / a0, (1 - k / q_s + k * k) / a0])
    q_h, fc_h = 0.5003270373253953, 38.13547087613982
    k = math.tan(math.pi * fc_h / sr)
    a0 = 1 + k / q_h + k * k
    hp_b = np.array([1.0, -2.0, 1.0])
    hp_a = np.array([1.0, 2 * (k * k - 1) / a0, (1 - k / q_h + k * k) / a0])
    return (shelf_b, shelf_a), (hp_b, hp_a)


def approx_lufs(signal: np.ndarray, sr: int) -> float:
    """Sonoridad integrada APROXIMADA (BS.1770 con ponderación K, mono, sin compuertas)."""
    x = np.asarray(signal, dtype=np.float64)
    for b, a in k_weighting_coefficients(sr):
        x = lfilter(b, a, x)
    power = float(np.mean(x**2)) if x.size else 0.0
    return -0.691 + 10 * math.log10(power) if power > 0 else -120.0


def loudness_features(signal: np.ndarray, sr: int = TARGET_SR) -> dict:
    """Rasgos de sonoridad SOLO como característica: la señal que recibe YAMNet no se modifica."""
    x = np.asarray(signal, dtype=np.float64)
    rms = float(np.sqrt(np.mean(x**2))) if x.size else 0.0
    peak = float(np.max(np.abs(x))) if x.size else 0.0
    rms_db = 20 * math.log10(rms) if rms > 0 else -120.0
    peak_db = 20 * math.log10(peak) if peak > 0 else -120.0
    return {
        "rms_dbfs": rms_db,
        "lufs_aprox": approx_lufs(x, sr),
        "pico_dbfs": peak_db,
        "factor_cresta_db": peak_db - rms_db,
    }


def pcm_sha256(signal: np.ndarray) -> str:
    """SHA-256 del audio decodificado y normalizado (float32): detecta la misma señal aunque
    venga en contenedores o codificaciones de fichero distintas."""
    return hashlib.sha256(np.ascontiguousarray(signal, dtype="<f4").tobytes()).hexdigest()


def exact_duplicates(hashes: Sequence[str], names: Sequence[str]) -> dict[int, dict]:
    """Duplicados exactos: por hash se conserva el clip de nombre menor; devuelve
    {índice descartado: {'kept': índice conservado, 'hash': hash}}."""
    if len(hashes) != len(names):
        raise ValueError("longitudes distintas")
    by_hash: dict[str, list[int]] = {}
    for i, h in enumerate(hashes):
        by_hash.setdefault(h, []).append(i)
    dropped = {}
    for h, members in by_hash.items():
        if len(members) > 1:
            keep = min(members, key=lambda i: (names[i], i))
            dropped.update({i: {"kept": keep, "hash": h} for i in members if i != keep})
    return dropped


def near_duplicate_pairs(
    embeddings: np.ndarray,
    *,
    threshold: float = NEAR_DUPLICATE_COSINE,
    block: int = 512,
    groups: Sequence[object] | None = None,
) -> list[tuple[int, int, float]]:
    """Pares (i<j) con similitud coseno >= threshold, por bloques para acotar memoria. Con
    groups, solo pares de grupos (o particiones) distintos: los únicos que pueden filtrar."""
    e = np.asarray(embeddings, dtype=np.float32)
    norms = np.linalg.norm(e, axis=1, keepdims=True)
    e = e / np.where(norms > 0, norms, 1.0)
    codes = None
    if groups is not None:
        if len(groups) != len(e):
            raise ValueError("longitudes distintas")
        codes = np.unique(np.asarray([str(g) for g in groups]), return_inverse=True)[1]
    pairs = []
    for start in range(0, len(e), block):
        sims = e[start : start + block] @ e.T
        mask = sims >= threshold
        if codes is not None:
            mask &= codes[start : start + block, None] != codes[None, :]
        rows, cols = np.nonzero(mask)
        for r, c in zip(rows.tolist(), cols.tolist(), strict=True):
            i = start + r
            if i < c:
                pairs.append((i, c, float(sims[r, c])))
    return sorted(pairs)


def resolve_near_duplicates(
    pairs: Sequence[tuple[int, int, float]], groups: Sequence[str], names: Sequence[str]
) -> dict:
    """Componentes conexas de casi duplicados. Si un componente abarca varios grupos, se
    conservan solo los miembros del grupo que más aporta (desempate por nombre de grupo) y se
    descartan los demás: ningún par casi duplicado queda entre grupos distintos, y por tanto
    tampoco a ambos lados de una partición por grupos."""
    parent: dict[int, int] = {}

    def find(i: int) -> int:
        parent.setdefault(i, i)
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for i, j, _ in pairs:
        parent[find(i)] = find(j)
    components: dict[int, list[int]] = {}
    for i in list(parent):
        components.setdefault(find(i), []).append(i)
    dropped: dict[int, str] = {}
    cross = 0
    for members in components.values():
        count: dict[str, int] = {}
        for i in members:
            count[groups[i]] = count.get(groups[i], 0) + 1
        if len(count) > 1:
            cross += 1
            rep = min(count, key=lambda g: (-count[g], g))
            dropped.update(
                {i: rep for i in sorted(members, key=lambda i: names[i]) if groups[i] != rep}
            )
    return {
        "dropped": dict(sorted(dropped.items())),
        "clusters": len(components),
        "cross_group_clusters": cross,
    }


In [ ]:
# Control de fugas y desplazamiento de dominio (iteración 5): StratifiedGroupKFold por grupo y
# leave-one-source-out. Autocontenido: build_colab.py lo incrusta tal cual en el cuaderno.
from collections.abc import Sequence

import numpy as np
from sklearn.model_selection import StratifiedGroupKFold


def stratified_group_folds(
    labels: Sequence[int], groups: Sequence[str], *, k: int = 5, seed: int
) -> tuple[int, ...]:
    """Fold de cada índice con StratifiedGroupKFold: grupos enteros, clases estratificadas.
    Falla (no corrige) si algún fold queda sin una de las dos clases."""
    if len(labels) != len(groups):
        raise ValueError("labels y groups deben tener la misma longitud")
    if len(set(groups)) < k:
        raise ValueError("menos grupos que folds")
    y = np.asarray(labels, dtype=int)
    folds = np.full(len(y), -1, dtype=int)
    splitter = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
    for f, (_, test) in enumerate(splitter.split(np.zeros(len(y)), y, np.asarray(groups))):
        folds[test] = f
    for f in range(k):
        if set(y[folds == f].tolist()) != {0, 1}:
            raise ValueError(f"el fold {f} no tiene ambas clases")
    return tuple(int(f) for f in folds)


def leave_one_source_out(
    labels: Sequence[int],
    groups: Sequence[str],
    sources: Sequence[str],
    *,
    seed: int,
    k_inner: int = 5,
) -> list[dict]:
    """Por cada fuente: test = esa fuente entera (nunca vista); con el resto, un fold de
    StratifiedGroupKFold es validación (umbral y calibración) y los demás, entrenamiento."""
    if not len(labels) == len(groups) == len(sources):
        raise ValueError("longitudes distintas")
    out = []
    for source in sorted(set(sources)):
        rest = [i for i, s in enumerate(sources) if s != source]
        test = tuple(i for i, s in enumerate(sources) if s == source)
        inner = stratified_group_folds(
            [labels[i] for i in rest], [groups[i] for i in rest], k=k_inner, seed=seed
        )
        validation = tuple(i for i, f in zip(rest, inner, strict=True) if f == 0)
        train = tuple(i for i, f in zip(rest, inner, strict=True) if f != 0)
        seen = [{groups[i] for i in p} for p in (train, validation, test)]
        if seen[0] & seen[1] or seen[0] & seen[2] or seen[1] & seen[2]:
            raise ValueError("un grupo aparece en varias particiones")
        out.append({"fuente": source, "train": train, "validation": validation, "test": test})
    return out


def assert_no_cross_partition_pairs(
    pairs: Sequence[tuple[int, int, float]], assignment: Sequence[object]
) -> None:
    """Falla si algún par casi duplicado (i, j, sim) cae en particiones distintas."""
    bad = [(i, j, s) for i, j, s in pairs if assignment[i] != assignment[j]]
    if bad:
        raise ValueError(f"casi duplicado a ambos lados de una partición: {bad[:5]}")


In [ ]:
# Desequilibrio (iteración 5): pesos por clase y por fuente y muestreo balanceado.
# Autocontenido (numpy): build_colab.py lo incrusta tal cual en el cuaderno.
from collections.abc import Sequence

import numpy as np


def class_source_weights(labels: Sequence[int], sources: Sequence[str]) -> np.ndarray:
    """Peso por muestra: cada clase suma N/2 y, dentro de una clase, cada fuente que la aporta
    suma lo mismo. Así ninguna fuente (p. ej. ITU ARIS, ~5.500 segmentos) domina la pérdida."""
    y = np.asarray(labels, dtype=int)
    s = np.asarray(sources)
    if len(y) != len(s):
        raise ValueError("longitudes distintas")
    classes = sorted(set(y.tolist()))
    w = np.zeros(len(y), dtype=float)
    for cls in classes:
        cells = sorted(set(s[y == cls].tolist()))
        for src in cells:
            mask = (y == cls) & (s == src)
            w[mask] = len(y) / (len(classes) * len(cells) * mask.sum())
    return w


def balanced_sample(
    labels: Sequence[int], sources: Sequence[str], n: int, *, seed: int
) -> list[int]:
    """n índices con reemplazo y probabilidad proporcional a class_source_weights."""
    w = class_source_weights(labels, sources)
    rng = np.random.default_rng(seed)
    return rng.choice(len(w), size=n, replace=True, p=w / w.sum()).tolist()


In [ ]:
# Aumento de datos en forma de onda (iteración 5), SOLO para entrenamiento: drones mezclados
# sobre fondos negativos a SNR controlada, ganancia y desplazamiento temporal.
# Autocontenido (numpy): build_colab.py lo incrusta tal cual en el cuaderno.
from collections.abc import Sequence

import numpy as np

MIX_SNR_RANGE_DB = (-10.0, 20.0)
MIX_GAIN_RANGE_DB = (-6.0, 6.0)
MIX_MAX_SECONDS = 5.0
MIX_PEAK = 0.99
SYNTHETIC_NOTE = (
    "Mezclas sintéticas (synthetic=true): dron grabado + fondo negativo a SNR nominal; solo "
    "entran en entrenamiento, nunca en validación ni test, y no cuentan como rendimiento real"
)


def _power(x: np.ndarray) -> float:
    return float(np.mean(np.asarray(x, dtype=np.float64) ** 2))


def mix_at_snr(
    drone: np.ndarray, background: np.ndarray, snr_db: float
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Escala el fondo para que 10·log10(P_dron/P_fondo) = snr_db y suma; si el pico pasa de
    MIX_PEAK se escalan ambos por igual (la SNR se conserva). Devuelve (mezcla, dron, fondo).
    La SNR es NOMINAL: el clip de dron ya contiene su propio ruido de fondo."""
    d = np.asarray(drone, dtype=np.float64)
    b = np.asarray(background, dtype=np.float64)[: len(d)]
    if len(b) != len(d) or _power(d) <= 0 or _power(b) <= 0:
        raise ValueError("dron y fondo deben tener la misma longitud y energía no nula")
    b = b * np.sqrt(_power(d) / (_power(b) * 10 ** (snr_db / 10)))
    mix = d + b
    peak = float(np.max(np.abs(mix)))
    if peak > MIX_PEAK:
        scale = MIX_PEAK / peak
        mix, d, b = mix * scale, d * scale, b * scale
    return mix.astype(np.float32), d.astype(np.float32), b.astype(np.float32)


def random_gain(signal: np.ndarray, gain_db: float) -> np.ndarray:
    return (np.asarray(signal, dtype=np.float64) * 10 ** (gain_db / 20)).astype(np.float32)


def time_shift(signal: np.ndarray, samples: int) -> np.ndarray:
    """Desplazamiento circular (el contenido no se pierde)."""
    return np.roll(np.asarray(signal), int(samples))


def plan_mixes(
    positives: Sequence[int],
    negatives: Sequence[int],
    sources: Sequence[str],
    seconds: Sequence[float],
    *,
    n: int,
    seed: int,
) -> list[dict]:
    """Plan determinista de n mezclas. Drones y fondos se eligen equilibrando fuentes (cada
    fuente con la misma probabilidad y, dentro de ella, uniforme)."""
    if not positives or not negatives:
        raise ValueError("hacen falta drones y fondos")
    rng = np.random.default_rng(seed)

    def pick(pool: Sequence[int]) -> int:
        by_src: dict[str, list[int]] = {}
        for i in pool:
            by_src.setdefault(sources[i], []).append(i)
        names = sorted(by_src)
        members = by_src[names[int(rng.integers(len(names)))]]
        return int(members[int(rng.integers(len(members)))])

    plan = []
    for k in range(n):
        d, b = pick(positives), pick(negatives)
        plan.append(
            {
                "id": f"mezcla{k:05d}",
                "drone": d,
                "background": b,
                "snr_db": float(rng.uniform(*MIX_SNR_RANGE_DB)),
                "gain_db": float(rng.uniform(*MIX_GAIN_RANGE_DB)),
                "shift_fraction": float(rng.uniform()),
                "drone_offset": float(rng.uniform()),
                "background_offset": float(rng.uniform()),
                "seconds": float(min(seconds[d], seconds[b], MIX_MAX_SECONDS)),
                "synthetic": True,
                "dron": 1,
            }
        )
    return plan


def plan_cross_mixes(
    positives: Sequence[int],
    negatives: Sequence[int],
    sources: Sequence[str],
    seconds: Sequence[float],
    *,
    n: int,
    seed: int,
) -> list[dict]:
    """Hito 2 (H2.3): mezclas CRUZADAS, el dron de la fuente A sobre fondo de una fuente B != A,
    para que el canal deje de delatar la clase. Fuente de dron y fuente de fondo se eligen
    equilibradas (cada fuente con la misma probabilidad); nunca dos partes de la misma fuente.
    Mismo formato que plan_mixes (se generan con render_mix) más cross/drone_source/
    background_source."""
    by_src: dict[str, dict[str, list[int]]] = {"pos": {}, "neg": {}}
    for kind, pool in (("pos", positives), ("neg", negatives)):
        for i in pool:
            by_src[kind].setdefault(sources[i], []).append(int(i))
    pairs = [(a, b) for a in sorted(by_src["pos"]) for b in sorted(by_src["neg"]) if a != b]
    if not pairs:
        raise ValueError("hacen falta drones y fondos de fuentes distintas")
    drone_sources = sorted({a for a, _ in pairs})
    rng = np.random.default_rng(seed)
    plan = []
    for k in range(n):
        drone_src = drone_sources[int(rng.integers(len(drone_sources)))]
        bg_options = sorted(b for a, b in pairs if a == drone_src)
        bg_src = bg_options[int(rng.integers(len(bg_options)))]
        d = by_src["pos"][drone_src][int(rng.integers(len(by_src["pos"][drone_src])))]
        b = by_src["neg"][bg_src][int(rng.integers(len(by_src["neg"][bg_src])))]
        plan.append(
            {
                "id": f"cruzada{k:05d}",
                "drone": d,
                "background": b,
                "drone_source": drone_src,
                "background_source": bg_src,
                "snr_db": float(rng.uniform(*MIX_SNR_RANGE_DB)),
                "gain_db": float(rng.uniform(*MIX_GAIN_RANGE_DB)),
                "shift_fraction": float(rng.uniform()),
                "drone_offset": float(rng.uniform()),
                "background_offset": float(rng.uniform()),
                "seconds": float(min(seconds[d], seconds[b], MIX_MAX_SECONDS)),
                "synthetic": True,
                "cross": True,
                "dron": 1,
            }
        )
    return plan


def assert_cross_mixes_allowed(
    plan: Sequence[dict], used: Sequence[int], train: Sequence[int], *, held_out: str | None
) -> None:
    """Fallo cerrado: dron y fondo en entrenamiento, fuentes distintas y ninguna es la retenida."""
    allowed = set(train)
    for k in used:
        m = plan[k]
        if m["drone"] not in allowed or m["background"] not in allowed:
            raise ValueError(f"mezcla {m['id']} con audio fuera de entrenamiento")
        if m["drone_source"] == m["background_source"]:
            raise ValueError(f"mezcla {m['id']} con ambas partes de la misma fuente")
        if held_out is not None and held_out in (m["drone_source"], m["background_source"]):
            raise ValueError(f"mezcla {m['id']} usa la fuente retenida {held_out}")


def cross_mixes_for_fold(
    plan: Sequence[dict], train: Sequence[int], *, held_out: str | None
) -> list[int]:
    """Mezclas cruzadas utilizables en un pliegue (LOSO: held_out = fuente retenida)."""
    ids = [
        k
        for k in mixes_for_training(plan, train)
        if held_out not in (plan[k]["drone_source"], plan[k]["background_source"])
    ]
    assert_cross_mixes_allowed(plan, ids, train, held_out=held_out)
    return ids


def _crop(x: np.ndarray, length: int, offset_fraction: float) -> np.ndarray:
    start = int((len(x) - length) * offset_fraction) if len(x) > length else 0
    return np.asarray(x)[start : start + length]


class SilentMixError(ValueError):
    """El dron o el fondo no tienen energía en ninguna ventana: la mezcla se omite."""


def _energetic_crop(x: np.ndarray, length: int, offset_fraction: float) -> np.ndarray:
    """Recorte pedido; si es silencio (p. ej. relleno de ESC-10), la ventana de más energía."""
    crop = _crop(x, length, offset_fraction)
    if _power(crop) > 0:
        return crop
    arr = np.asarray(x, dtype=np.float64)
    if len(arr) <= length or _power(arr) <= 0:
        raise SilentMixError("clip sin energía: mezcla omitida")
    energy = np.concatenate(([0.0], np.cumsum(arr**2)))
    window = energy[length:] - energy[:-length]
    start = int(np.argmax(window))
    return np.asarray(x)[start : start + length]


def render_mix(spec: dict, drone: np.ndarray, background: np.ndarray, sr: int) -> np.ndarray:
    """Genera la mezcla de un plan: recorte, desplazamiento del dron, SNR y ganancia.
    Los recortes evitan el silencio; un clip sin energía lanza SilentMixError (se omite)."""
    length = min(int(spec["seconds"] * sr), len(drone), len(background))
    d = _energetic_crop(drone, length, spec["drone_offset"])
    d = time_shift(d, int(spec["shift_fraction"] * length))
    if _power(d) <= 0:
        raise SilentMixError("dron sin energía tras el desplazamiento: mezcla omitida")
    b = _energetic_crop(background, length, spec["background_offset"])
    mix, _, _ = mix_at_snr(d, b, spec["snr_db"])
    out = random_gain(mix, spec["gain_db"])
    peak = float(np.max(np.abs(out)))
    return (out * (MIX_PEAK / peak) if peak > MIX_PEAK else out).astype(np.float32)


def mixes_for_training(plan: Sequence[dict], train: Sequence[int]) -> list[int]:
    """Posiciones del plan utilizables en un fold: dron Y fondo deben estar en entrenamiento,
    de modo que ningún audio de validación o test entra por una mezcla."""
    allowed = set(train)
    return [k for k, m in enumerate(plan) if m["drone"] in allowed and m["background"] in allowed]


In [ ]:
# Modelado por ventanas (iteración 5): rasgos por ventana YAMNet (0,96 s, salto 0,48 s),
# agregación por clip y suavizado temporal. Autocontenido (numpy): se incrusta en el cuaderno.
from collections.abc import Sequence

import numpy as np

YAMNET_HOP_S = 0.48
SMOOTH_WINDOWS = 3  # ~1,4 s: un pico de una sola ventana no basta para disparar
ATTENTION_TEMPERATURE = 0.1  # agrupación «atención ligera» (softmax sobre la propia puntuación)
AGGREGATIONS = ("media", "max", "top3", "max_suavizado", "atencion")


def pack_frames(blocks: Sequence[np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    """Concatena las ventanas de cada clip (float32) y devuelve (datos, offsets) para cachearlas
    en un único .npz; las ventanas del clip i son datos[offsets[i]:offsets[i+1]]."""
    sizes = [len(b) for b in blocks]
    offsets = np.concatenate([[0], np.cumsum(sizes)]).astype(np.int64)
    data = np.concatenate([np.asarray(b, dtype=np.float32) for b in blocks], axis=0)
    return data, offsets


def unpack_frames(data: np.ndarray, offsets: np.ndarray, i: int) -> np.ndarray:
    return data[offsets[i] : offsets[i + 1]]


def smooth_scores(scores: np.ndarray, k: int = SMOOTH_WINDOWS) -> np.ndarray:
    """Media móvil centrada de k ventanas; en los bordes promedia solo las disponibles."""
    p = np.asarray(scores, dtype=float)
    if k <= 1 or p.size == 0:
        return p.copy()
    half = k // 2
    csum = np.concatenate([[0.0], np.cumsum(p)])
    idx = np.arange(p.size)
    lo = np.maximum(idx - half, 0)
    hi = np.minimum(idx + half + 1, p.size)
    return (csum[hi] - csum[lo]) / (hi - lo)


def aggregate_window_scores(scores: np.ndarray, method: str) -> float:
    """Probabilidad por clip a partir de las probabilidades por ventana."""
    p = np.asarray(scores, dtype=float)
    if p.size == 0:
        raise ValueError("clip sin ventanas")
    if method == "media":
        return float(p.mean())
    if method == "max":
        return float(p.max())
    if method == "top3":
        return float(np.sort(p)[-3:].mean())
    if method == "max_suavizado":
        return float(smooth_scores(p).max())
    if method == "atencion":
        w = np.exp((p - p.max()) / ATTENTION_TEMPERATURE)
        return float(np.sum(w * p) / np.sum(w))
    raise ValueError(f"agregación desconocida: {method}")


def frame_training_set(
    data: np.ndarray,
    offsets: np.ndarray,
    labels: Sequence[int],
    indices: Sequence[int],
    *,
    max_frames: int = 8,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Ventanas de entrenamiento (etiqueta débil: la del clip). Como máximo max_frames por
    clip, equiespaciadas y deterministas, para que los clips largos no dominen."""
    xs, ys, owners = [], [], []
    for i in indices:
        frames = unpack_frames(data, offsets, i)
        if len(frames) > max_frames:
            pick = np.unique(np.linspace(0, len(frames) - 1, max_frames).round().astype(int))
            frames = frames[pick]
        xs.append(frames)
        ys.append(np.full(len(frames), int(labels[i])))
        owners.append(np.full(len(frames), int(i)))
    return np.concatenate(xs), np.concatenate(ys), np.concatenate(owners)


In [ ]:
# Calibración y curvas (iteración 5): Platt o isotónica ajustadas SOLO en validación, ECE,
# diagrama de fiabilidad, DET y falsas alarmas por hora ESTIMADAS (no verificadas).
# Autocontenido (numpy, scikit-learn): build_colab.py lo incrusta tal cual en el cuaderno.
from collections.abc import Sequence

import numpy as np
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import det_curve

ECE_BINS = 15
FA_ESTIMATE_STATUS = "ESTIMACIÓN, NO VERIFICADA como métrica de campo"
FA_ESTIMATE_NOTE = (
    "Alarmas contadas sobre la concatenación de puntuaciones por ventana de clips negativos de "
    "test (fuera de fold); no es una grabación continua de un emplazamiento: no sustituye a la "
    "medida de falsas alarmas por hora en campo"
)


def _logit(p: np.ndarray) -> np.ndarray:
    q = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(q / (1 - q))


def fit_calibrator(y: np.ndarray, p: np.ndarray, method: str) -> dict:
    """Ajusta en VALIDACIÓN: 'platt' (logística sobre el logit) o 'isotonica'."""
    y = np.asarray(y, dtype=int)
    if set(np.unique(y).tolist()) != {0, 1}:
        raise ValueError("la calibración necesita ambas clases en validación")
    if method == "platt":
        lr = LogisticRegression(C=1e4, max_iter=1000).fit(_logit(p)[:, None], y)
        return {"method": "platt", "a": float(lr.coef_[0, 0]), "b": float(lr.intercept_[0])}
    if method == "isotonica":
        iso = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
        iso.fit(np.asarray(p, dtype=float), y)
        return {
            "method": "isotonica",
            "x": iso.X_thresholds_.tolist(),
            "y": iso.y_thresholds_.tolist(),
        }
    raise ValueError(f"método de calibración desconocido: {method}")


def apply_calibration(cal: dict, p: np.ndarray) -> np.ndarray:
    q = np.asarray(p, dtype=float)
    if cal["method"] == "platt":
        return 1 / (1 + np.exp(-(cal["a"] * _logit(q) + cal["b"])))
    if cal["method"] == "isotonica":
        return np.interp(q, cal["x"], cal["y"])
    raise ValueError("calibrador no válido")


def reliability_bins(y: np.ndarray, p: np.ndarray, *, n_bins: int = ECE_BINS) -> list[dict]:
    """Bins de igual anchura en [0, 1] (el último incluye 1,0); solo los no vacíos."""
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    idx = np.minimum((p * n_bins).astype(int), n_bins - 1)
    out = []
    for b in range(n_bins):
        mask = idx == b
        if mask.any():
            out.append(
                {
                    "bin": b,
                    "n": int(mask.sum()),
                    "confianza_media": float(p[mask].mean()),
                    "fraccion_positivos": float(y[mask].mean()),
                }
            )
    return out


def expected_calibration_error(y: np.ndarray, p: np.ndarray, *, n_bins: int = ECE_BINS) -> float:
    bins = reliability_bins(y, p, n_bins=n_bins)
    total = sum(b["n"] for b in bins)
    return sum(b["n"] * abs(b["fraccion_positivos"] - b["confianza_media"]) for b in bins) / total


def brier_score(y: np.ndarray, p: np.ndarray) -> float:
    return float(np.mean((np.asarray(p, dtype=float) - np.asarray(y, dtype=float)) ** 2))


def det_points(y: np.ndarray, p: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Curva DET (tasa de falsas alarmas frente a tasa de fallos) por umbral."""
    return det_curve(np.asarray(y, dtype=int), np.asarray(p, dtype=float))


def alarm_events(scores: np.ndarray, threshold: float, *, hop_s: float, refractory_s: float) -> int:
    """Alarmas = flancos de subida sobre el umbral. Un flanco a menos de refractory_s del fin
    de la alarma anterior se considera la misma alarma."""
    above = np.asarray(scores, dtype=float) >= threshold
    events, last_end = 0, None
    for t in range(len(above)):
        rising = above[t] and (t == 0 or not above[t - 1])
        if rising and (last_end is None or (t - last_end) * hop_s >= refractory_s):
            events += 1
        if above[t]:
            last_end = t
    return events


def false_alarms_per_hour_estimate(
    streams: Sequence[np.ndarray], threshold: float, *, hop_s: float, refractory_s: float
) -> dict:
    """FA/h sobre concatenaciones de negativos: SIEMPRE marcada como estimación no verificada."""
    events = sum(
        alarm_events(s, threshold, hop_s=hop_s, refractory_s=refractory_s) for s in streams
    )
    hours = sum(len(s) for s in streams) * hop_s / 3600
    return {
        "status": FA_ESTIMATE_STATUS,
        "nota": FA_ESTIMATE_NOTE,
        "eventos": int(events),
        "horas": hours,
        "valor": events / hours if hours > 0 else None,
    }


In [ ]:
# Negativos difíciles (iteración 5): tipo de cada negativo, matriz de confusión por tipo y
# reponderación de los negativos de entrenamiento con mayor puntuación. Orienta qué grabar en
# Ceuta. Autocontenido (numpy): build_colab.py lo incrusta tal cual en el cuaderno.
import math
from collections.abc import Sequence

import numpy as np

NEGATIVE_TYPES = {
    "Motorcycle": "motos",
    "Boat_and_Water_vehicle": "barcos",
    "Aircraft": "aviones",
    "Fixed-wing_aircraft_and_airplane": "aviones",
    "HELICOPTER": "helicópteros",
    "helicopter": "helicópteros",
    "Insect": "insectos",
    "Wind": "viento",
    "Rain": "lluvia",
    "rain": "lluvia",
    "sea_waves": "mar y oleaje",
    "Engine": "motores y maquinaria",
    "chainsaw": "motores y maquinaria",
    "Bird": "aves",
    "Gull_and_seagull": "aves",
    "rooster": "aves",
    "dog": "perros",
    "crackling_fire": "fuego",
    "clock_tick": "tictac",
    "BACKGROUND": "fondo de campo",
    "ARIS_BACKGROUND": "fondo de campo",
}


def negative_type(source: str, clase: str) -> str:
    """Tipo de negativo legible; 'otros' si la clase no está en la tabla (nunca falla)."""
    return NEGATIVE_TYPES.get(clase, "otros")


def confusion_by_negative_type(
    types: Sequence[str | None],
    labels: Sequence[int],
    scores: Sequence[float],
    thresholds: float | Sequence[float],
) -> list[dict]:
    """Falsas alarmas por tipo de negativo (las filas positivas se ignoran). El umbral puede
    ser único o uno por fila (p. ej. el de cada fold). Orden: FPR desc., n desc., nombre."""
    thr = np.broadcast_to(np.asarray(thresholds, dtype=float), (len(labels),))
    rows: dict[str, list[float]] = {}
    alarms: dict[str, int] = {}
    for t, y, s, h in zip(types, labels, scores, thr, strict=True):
        if int(y) == 1:
            continue
        rows.setdefault(str(t), []).append(float(s))
        alarms[str(t)] = alarms.get(str(t), 0) + int(float(s) >= h)
    table = []
    for t, values in rows.items():
        v = np.asarray(values)
        table.append(
            {
                "tipo": t,
                "n": int(v.size),
                "fp": alarms[t],
                "fpr": alarms[t] / v.size,
                "puntuacion_media": float(v.mean()),
                "puntuacion_p95": float(np.percentile(v, 95)),
            }
        )
    return sorted(table, key=lambda r: (-r["fpr"], -r["n"], r["tipo"]))


def recording_priorities(table: Sequence[dict], *, top: int = 5) -> list[str]:
    """Tipos con más falsas alarmas (fp > 0): qué confusores grabar primero en campo."""
    return [r["tipo"] for r in table if r["fp"] > 0][:top]


def hard_negative_weights(
    weights: np.ndarray,
    labels: Sequence[int],
    train_scores: Sequence[float],
    *,
    fraction: float = 0.1,
    factor: float = 3.0,
) -> tuple[np.ndarray, list[int]]:
    """Multiplica por factor el peso del `fraction` de negativos de ENTRENAMIENTO con mayor
    puntuación (minería de negativos difíciles, segunda ronda). Nunca toca validación ni test."""
    y = np.asarray(labels, dtype=int)
    s = np.asarray(train_scores, dtype=float)
    negatives = np.flatnonzero(y == 0)
    k = math.ceil(len(negatives) * fraction)
    order = negatives[np.lexsort((negatives, -s[negatives]))]
    hard = sorted(int(i) for i in order[:k])
    new = np.array(weights, dtype=float, copy=True)
    new[hard] *= factor
    return new, hard


In [ ]:
# Registro del experimento (iteración 5): semillas, configuración, hashes de datos y del
# código del cuaderno, versiones de dependencias y hash de las predicciones exportadas.
# Autocontenido (stdlib): build_colab.py lo incrusta tal cual en el cuaderno.
import hashlib
import importlib.metadata
import json
from collections.abc import Iterable, Sequence


def canonical_json(obj: object) -> tuple[bytes, str]:
    """JSON canónico (claves ordenadas, UTF-8, sin espacios) y su SHA-256."""
    data = json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode()
    return data, hashlib.sha256(data).hexdigest()


def inventory_sha256(rows: Iterable[tuple[str, str]]) -> str:
    """Hash del inventario de datos: líneas «sha256  clip» ordenadas."""
    lines = sorted(f"{digest}  {clip}\n" for clip, digest in rows)
    return hashlib.sha256("".join(lines).encode()).hexdigest()


def dependency_versions(names: Sequence[str]) -> dict[str, str]:
    out = {}
    for name in names:
        try:
            out[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            out[name] = "no instalado"
    return out


def embedding_cache_key(inventory: str, model_url: str, sr: int, code_version: str) -> str:
    """Clave de la caché de embeddings: cambia si cambian los datos, el modelo, sr o el código."""
    return hashlib.sha256(f"{inventory}|{model_url}|{sr}|{code_version}".encode()).hexdigest()[:16]


def experiment_record(
    *,
    config: dict,
    data: dict,
    dependencies: dict,
    predictions_sha256: str,
    notebook_code_sha256: str,
    extra: dict | None = None,
) -> dict:
    """Registro con hash propio (registro_sha256 sobre el JSON canónico del resto)."""
    record = {
        "config": config,
        "datos": data,
        "dependencias": dependencies,
        "predicciones_csv_sha256": predictions_sha256,
        "codigo_cuaderno_sha256": notebook_code_sha256,
        **(extra or {}),
    }
    record["registro_sha256"] = canonical_json(record)[1]
    return record


def verify_record(record: dict) -> bool:
    body = {k: v for k, v in record.items() if k != "registro_sha256"}
    return canonical_json(body)[1] == record.get("registro_sha256")


In [ ]:
# Puntos de control y descargas concurrentes (iteración 5). En Colab sin Drive todo vive en
# /content: sobrevive a un reinicio del kernel o a una reconexión a la MISMA máquina, no a que
# Colab recicle la máquina virtual. Ficheros verificados por SHA-256 antes de reutilizarse,
# fragmentos .npz con hash de entrada y escritura atómica, peticiones concurrentes con un límite
# global por minuto (Zenodo admite ~133) y progreso visible por fuente.
# Autocontenido (stdlib y numpy): build_colab.py lo incrusta tal cual en el cuaderno.
import hashlib
import io
import json
import os
import threading
import time
from collections.abc import Callable, Sequence
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np


def write_atomic(path: Path, data: bytes) -> None:
    """Escribe en un temporal y renombra: una desconexión nunca deja un fichero a medias."""
    path = Path(path)
    tmp = path.with_name(f"{path.name}.tmp{os.getpid()}-{threading.get_ident()}")
    tmp.write_bytes(data)
    os.replace(tmp, path)


def verified_cached_bytes(path: Path, sha256: str) -> bytes | None:
    """Contenido del fichero en caché solo si su SHA-256 coincide; si no, None (se descarga)."""
    path = Path(path)
    if not path.is_file():
        return None
    data = path.read_bytes()
    return data if hashlib.sha256(data).hexdigest() == sha256 else None


def shard_ranges(n: int, size: int) -> list[tuple[int, int]]:
    if size <= 0:
        raise ValueError("tamaño de fragmento no válido")
    return [(a, min(a + size, n)) for a in range(0, n, size)]


def shard_key(items: Sequence[tuple[str, str]], context: str) -> str:
    """Hash de entrada de un fragmento: (clip, sha256) en orden + contexto (modelo, código)."""
    h = hashlib.sha256(context.encode())
    for clip, digest in items:
        h.update(f"\n{clip}\t{digest}".encode())
    return h.hexdigest()


def save_shard(path: Path, key: str, arrays: dict, meta: dict) -> None:
    buffer = io.BytesIO()
    np.savez_compressed(
        buffer, **arrays, _key=np.array(key), _meta=np.array(json.dumps(meta, ensure_ascii=False))
    )
    write_atomic(Path(path), buffer.getvalue())


def load_shard(path: Path, key: str) -> tuple[dict, dict] | None:
    """(arrays, meta) si el fragmento existe, se lee entero y su hash de entrada coincide."""
    path = Path(path)
    if not path.is_file():
        return None
    try:
        with np.load(path, allow_pickle=False) as z:
            if str(z["_key"]) != key:
                return None
            arrays = {k: z[k] for k in z.files if k not in ("_key", "_meta")}
            return arrays, json.loads(str(z["_meta"]))
    except Exception:  # fichero truncado o corrupto: se recalcula, nunca se usa
        return None


class RateLimiter:
    """Límite global de peticiones por minuto compartido entre hilos (espaciado uniforme)."""

    def __init__(self, per_minute: float, *, clock=time.monotonic, sleep=time.sleep) -> None:  # type: ignore[no-untyped-def]
        if per_minute <= 0:
            raise ValueError("per_minute debe ser positivo")
        self.interval = 60.0 / per_minute
        self.clock, self.sleep = clock, sleep
        self.lock = threading.Lock()
        self.next_slot: float | None = None

    def acquire(self) -> None:
        with self.lock:
            now = self.clock()
            slot = now if self.next_slot is None else max(now, self.next_slot)
            self.next_slot = slot + self.interval
        if slot > now:
            self.sleep(slot - now)


class Progress:
    """Imprime «fuente: hechos/total (%) · velocidad» cada `every` elementos y al final."""

    def __init__(self, label: str, total: int, *, every: int = 50, clock=time.monotonic) -> None:  # type: ignore[no-untyped-def]
        self.label, self.total, self.every, self.clock = label, total, max(1, every), clock
        self.start = clock()

    def __call__(self, done: int, total: int) -> None:
        if done % self.every == 0 or done == total:
            rate = done / max(self.clock() - self.start, 1e-9)
            share = 100 * done // max(total, 1)
            print(f"  {self.label}: {done}/{total} ({share} %) · {rate:.1f} por s")


def run_concurrent(
    items: Sequence,
    fn: Callable,
    *,
    workers: int = 8,
    on_done: Callable[[int, int], None] | None = None,
) -> list:
    """Aplica fn a cada elemento con un pool de hilos; resultados en el orden de entrada. Si
    alguno falla (p. ej. hash distinto), se cancela lo pendiente y se relanza el error."""
    results: list = [None] * len(items)
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(fn, item): k for k, item in enumerate(items)}
        try:
            for done, future in enumerate(as_completed(futures), 1):
                results[futures[future]] = future.result()
                if on_done:
                    on_done(done, len(items))
        except BaseException:
            for f in futures:
                f.cancel()
            raise
    return results


In [ ]:
# Normalización de canal compartida (hito 2, H2.1): EL MISMO código se incrusta en el cuaderno de
# Colab (build_colab.py) y se importa en el Mac (acoustic_pilot.live). Autocontenido: numpy, scipy
# y stdlib. Cadena: mono -> 16 kHz (resample_poly) -> paso banda 100-7500 Hz (Butterworth SOS, fase
# cero) -> sonoridad por ventana a un RMS objetivo con ganancia limitada; y mel con PCEN para la
# rama armónica. Cualquier cambio de parámetros cambia norm_config_sha256() y rompe los vectores
# dorados del artefacto (prueba de que Colab y el Mac normalizan igual).
import hashlib
import json
import math

import numpy as np
from scipy.signal import butter, lfilter, resample_poly, sosfiltfilt

NORM_SR = 16000
# Ventana YAMNet: 0,96 s de parche + 15 ms de la STFT = 15 600 muestras a 16 kHz (una ventana).
WINDOW_SAMPLES = 15600
HOP_SAMPLES = 7680  # 0,48 s
# 11 025 Hz y 8 kHz no llegan a 7,5 kHz de banda: se rechazan (fallo cerrado).
SUPPORTED_RATES = (16000, 22050, 24000, 32000, 44100, 48000, 88200, 96000)
PCEN_DEFAULTS = {"gain": 0.98, "bias": 2.0, "power": 0.5, "time_constant_s": 0.4, "eps": 1e-6}
NORM_CONFIG = {
    "version": "canal-1",
    "sr": NORM_SR,
    "ventana_muestras": WINDOW_SAMPLES,
    "salto_muestras": HOP_SAMPLES,
    # Butterworth de orden 4 por flanco, fase cero: -0,5 dB a 100 Hz y ~-59 dB a 30 Hz.
    "banda_hz": [70.0, 7900.0],
    "banda_orden": 4,
    "rms_objetivo_dbfs": -25.0,
    "ganancia_maxima_db": 30.0,
    "suelo_ruido_dbfs": -90.0,  # por debajo: silencio, no se amplifica
    "pico_maximo": 0.99,
    "mel": {"n_fft": 512, "win": 400, "hop": 160, "n_mels": 64, "fmin": 50.0, "fmax": 7500.0},
    "pcen": PCEN_DEFAULTS,
    "pcen_escala": 2.0**31,  # mel de audio en coma flotante escalado como PCM de 32 bits
}


def norm_config_sha256(config: dict | None = None) -> str:
    """SHA-256 del JSON canónico de la configuración de normalización."""
    data = json.dumps(config or NORM_CONFIG, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(data).hexdigest()


def _finite_float64(signal: np.ndarray) -> np.ndarray:
    x = np.asarray(signal)
    if np.issubdtype(x.dtype, np.integer):
        x = x.astype(np.float64) / float(np.iinfo(x.dtype).max)
    x = x.astype(np.float64)
    if x.size == 0:
        raise ValueError("señal vacía")
    if not np.isfinite(x).all():
        raise ValueError("señal con valores no finitos (NaN o infinito)")
    return x


def to_mono_16k(signal: np.ndarray, sr: int) -> np.ndarray:
    """Mono (media de canales; (n,), (canales, n) o (n, canales)) y 16 kHz con resample_poly."""
    if int(sr) not in SUPPORTED_RATES:
        raise ValueError(f"frecuencia de muestreo no admitida: {sr}")
    x = _finite_float64(signal)
    if x.ndim == 2:
        x = x.mean(axis=0) if x.shape[0] <= x.shape[1] else x.mean(axis=1)
    elif x.ndim != 1:
        raise ValueError("se esperaba señal 1D o 2D")
    if int(sr) != NORM_SR:
        g = math.gcd(int(sr), NORM_SR)
        x = resample_poly(x, NORM_SR // g, int(sr) // g)
    return x.astype(np.float32)


def bandpass_sos(config: dict | None = None) -> np.ndarray:
    cfg = config or NORM_CONFIG
    return butter(cfg["banda_orden"], cfg["banda_hz"], btype="bandpass", fs=cfg["sr"], output="sos")


def bandpass(window: np.ndarray, config: dict | None = None) -> np.ndarray:
    """Paso banda de fase cero (sosfiltfilt) sobre una ventana de 16 kHz."""
    x = _finite_float64(window)
    if x.ndim != 1:
        raise ValueError("se esperaba una ventana 1D")
    sos = bandpass_sos(config)
    if x.size <= 3 * (2 * sos.shape[0] + 1):
        raise ValueError("ventana demasiado corta para el filtro")
    return sosfiltfilt(sos, x).astype(np.float32)


def loudness_normalize(
    window: np.ndarray, config: dict | None = None, *, return_gain: bool = False
) -> np.ndarray | tuple[np.ndarray, float]:
    """Ganancia por ventana hasta el RMS objetivo, limitada a ±ganancia_maxima_db; por debajo del
    suelo de ruido no se aplica ganancia (el silencio no se amplifica); limitador de pico."""
    cfg = config or NORM_CONFIG
    x = _finite_float64(window)
    rms = float(np.sqrt(np.mean(x**2)))
    level = 20 * math.log10(rms) if rms > 0 else -math.inf
    if level < cfg["suelo_ruido_dbfs"]:
        gain_db = 0.0
    else:
        limit = cfg["ganancia_maxima_db"]
        gain_db = float(min(max(cfg["rms_objetivo_dbfs"] - level, -limit), limit))
    y = x * 10 ** (gain_db / 20)
    peak = float(np.max(np.abs(y)))
    if peak > cfg["pico_maximo"]:
        y = y * (cfg["pico_maximo"] / peak)
    out = y.astype(np.float32)
    return (out, gain_db) if return_gain else out


def normalize_window(window: np.ndarray, config: dict | None = None) -> np.ndarray:
    """Ventana de 16 kHz ya en mono -> paso banda -> sonoridad. Determinista."""
    x = np.asarray(window)
    if x.ndim != 1 or x.size == 0:
        raise ValueError("se esperaba una ventana 1D no vacía a 16 kHz")
    return loudness_normalize(bandpass(x, config), config)


def frame_starts(n_samples: int, config: dict | None = None) -> list[int]:
    """Inicios de ventana (salto 0,48 s); un clip más corto que una ventana da una sola ventana
    (se rellena con ceros al extraerla)."""
    cfg = config or NORM_CONFIG
    win, hop = cfg["ventana_muestras"], cfg["salto_muestras"]
    if n_samples <= win:
        return [0]
    return list(range(0, n_samples - win + 1, hop))


def extract_window(signal16k: np.ndarray, start: int, config: dict | None = None) -> np.ndarray:
    cfg = config or NORM_CONFIG
    win = cfg["ventana_muestras"]
    x = np.asarray(signal16k, dtype=np.float32)[start : start + win]
    return np.pad(x, (0, win - len(x))) if len(x) < win else x


def _hz_to_mel(f: np.ndarray) -> np.ndarray:
    return 1127.0 * np.log1p(np.asarray(f, dtype=np.float64) / 700.0)


def _mel_to_hz(m: np.ndarray) -> np.ndarray:
    return 700.0 * np.expm1(np.asarray(m, dtype=np.float64) / 1127.0)


def mel_filterbank(config: dict | None = None) -> np.ndarray:
    """Banco mel triangular (escala HTK), (n_mels, n_fft // 2 + 1)."""
    cfg = (config or NORM_CONFIG)["mel"]
    n_bins = cfg["n_fft"] // 2 + 1
    freqs = np.linspace(0, NORM_SR / 2, n_bins)
    edges = _mel_to_hz(
        np.linspace(_hz_to_mel(cfg["fmin"]), _hz_to_mel(cfg["fmax"]), cfg["n_mels"] + 2)
    )
    fb = np.zeros((cfg["n_mels"], n_bins))
    for m in range(cfg["n_mels"]):
        lo, mid, hi = edges[m], edges[m + 1], edges[m + 2]
        up = (freqs - lo) / (mid - lo)
        down = (hi - freqs) / (hi - mid)
        fb[m] = np.maximum(0.0, np.minimum(up, down))
        if fb[m].sum() == 0:  # banda más estrecha que un bin: el bin más cercano
            fb[m, int(np.argmin(np.abs(freqs - mid)))] = 1.0
    return fb


def mel_power(window: np.ndarray, config: dict | None = None) -> np.ndarray:
    """Espectrograma mel de potencia (n_mels, tramas) con ventana de Hann periódica."""
    cfg = (config or NORM_CONFIG)["mel"]
    x = _finite_float64(window)
    win, hop, n_fft = cfg["win"], cfg["hop"], cfg["n_fft"]
    if x.size < win:
        x = np.pad(x, (0, win - x.size))
    n_frames = 1 + (x.size - win) // hop
    idx = np.arange(win)[None, :] + hop * np.arange(n_frames)[:, None]
    hann = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(win) / win)
    spec = np.abs(np.fft.rfft(x[idx] * hann, n_fft, axis=1)) ** 2
    return (mel_filterbank(config) @ spec.T).astype(np.float64)


def pcen(mel: np.ndarray, config: dict | None = None) -> np.ndarray:
    """Per-Channel Energy Normalization (Wang et al., 2017), con el suavizador en estado
    estacionario sobre la primera trama: atenúa el fondo estacionario y realza los cambios."""
    cfg = config or NORM_CONFIG
    p = cfg["pcen"]
    e = np.asarray(mel, dtype=np.float64) * cfg["pcen_escala"]
    if e.ndim != 2 or not np.isfinite(e).all():
        raise ValueError("mel no válido")
    t_frames = p["time_constant_s"] * NORM_SR / cfg["mel"]["hop"]
    b = (math.sqrt(1 + 4 * t_frames**2) - 1) / (2 * t_frames**2)
    zi = (1 - b) * e[:, :1]
    m, _ = lfilter([b], [1, b - 1], e, axis=1, zi=zi)
    smooth = np.exp(-p["gain"] * (np.log(p["eps"]) + np.log1p(m / p["eps"])))
    return (e * smooth + p["bias"]) ** p["power"] - p["bias"] ** p["power"]


In [ ]:
# Aumento de CANAL (hito 2, H2.2), SOLO para entrenamiento en Colab: EQ aleatoria, respuesta de
# micrófono, respuesta de altavoz pequeño, reverberación con respuesta al impulso sintética, códec
# (mu-law de 8 bits y recorte de banda) y ruido de baja amplitud. Objetivo: que el canal de cada
# dataset deje de delatar la clase (atajo de dominio de la iteración 5). Todo se aplica igual a
# drones y a fondos. Sin dependencias nuevas (numpy, scipy); los códecs con pérdidas vía ffmpeg no
# se usan (licencia y versión sin registrar). Autocontenido: build_colab.py lo incrusta tal cual.
from collections.abc import Sequence

import numpy as np
from scipy.signal import butter, fftconvolve, sosfilt

AUG_SR = 16000
AUG_PEAK = 0.99
EQ_GAIN_RANGE_DB = (-12.0, 12.0)
RT60_RANGE_S = (0.2, 0.8)
CHANNEL_SYNTHETIC_NOTE = (
    "Aumento de canal (synthetic=true): EQ, micrófono, altavoz pequeño, reverberación sintética, "
    "códec y ruido; solo en entrenamiento, nunca en validación ni test, y no cuenta como "
    "rendimiento real"
)
# Probabilidad de cada etapa en random_channel (documentadas; no se ajustan mirando test).
CHANNEL_STAGE_PROB = {
    "altavoz": 0.35,
    "eq": 0.8,
    "reverb": 0.5,
    "microfono": 0.7,
    "codec": 0.3,
    "ruido": 0.7,
}


def _check_signal(x: np.ndarray, sr: int) -> np.ndarray:
    if int(sr) != AUG_SR:
        raise ValueError(f"el aumento de canal trabaja a {AUG_SR} Hz, no a {sr}")
    arr = np.asarray(x, dtype=np.float64)
    if arr.ndim != 1 or arr.size == 0 or not np.isfinite(arr).all():
        raise ValueError("se esperaba una señal 1D finita y no vacía")
    return arr


def _biquad(kind: str, f: float, gain_db: float, q: float, sr: int) -> np.ndarray:
    """Biquads del «Audio EQ Cookbook» (R. Bristow-Johnson) como una sección SOS."""
    if not EQ_GAIN_RANGE_DB[0] <= gain_db <= EQ_GAIN_RANGE_DB[1]:
        raise ValueError(f"ganancia fuera de rango: {gain_db} dB")
    if not 0 < f < sr / 2 or q <= 0:
        raise ValueError("frecuencia o Q no válidas")
    a = 10 ** (gain_db / 40)
    w0 = 2 * np.pi * f / sr
    cw, alpha = np.cos(w0), np.sin(w0) / (2 * q)
    if kind == "pico":
        b = [1 + alpha * a, -2 * cw, 1 - alpha * a]
        den = [1 + alpha / a, -2 * cw, 1 - alpha / a]
    elif kind in ("grave", "agudo"):
        s = 1 if kind == "grave" else -1
        ra = 2 * np.sqrt(a) * alpha
        b = [
            a * ((a + 1) - s * (a - 1) * cw + ra),
            s * 2 * a * ((a - 1) - s * (a + 1) * cw),
            a * ((a + 1) - s * (a - 1) * cw - ra),
        ]
        den = [
            (a + 1) + s * (a - 1) * cw + ra,
            -s * 2 * ((a - 1) + s * (a + 1) * cw),
            (a + 1) + s * (a - 1) * cw - ra,
        ]
    else:
        raise ValueError(f"tipo de filtro desconocido: {kind}")
    return np.array([[*(np.array(b) / den[0]), 1.0, den[1] / den[0], den[2] / den[0]]])


def eq_sos(bands: Sequence[dict], sr: int = AUG_SR) -> np.ndarray:
    return np.vstack([_biquad(b["tipo"], b["f"], b["ganancia_db"], b["q"], sr) for b in bands])


def apply_eq(x: np.ndarray, bands: Sequence[dict], sr: int = AUG_SR) -> np.ndarray:
    arr = _check_signal(x, sr)
    if not bands:
        return arr.astype(np.float32)
    return sosfilt(eq_sos(bands, sr), arr).astype(np.float32)


def mic_response(
    x: np.ndarray, *, low_hz: float, high_hz: float, order: int, sr: int = AUG_SR
) -> np.ndarray:
    """Micrófono barato: paso alto (80-300 Hz) y paso bajo (6-8 kHz) de pendiente variable."""
    arr = _check_signal(x, sr)
    sos = np.vstack(
        [
            butter(order, low_hz, btype="highpass", fs=sr, output="sos"),
            butter(order, high_hz, btype="lowpass", fs=sr, output="sos"),
        ]
    )
    return sosfilt(sos, arr).astype(np.float32)


def speaker_response(
    x: np.ndarray,
    *,
    low_hz: float,
    resonances: Sequence[tuple[float, float, float]],
    sr: int = AUG_SR,
) -> np.ndarray:
    """Altavoz pequeño (móvil, portátil): sin graves por debajo de low_hz (150-400 Hz, orden 4)
    y resonancias (frecuencia, ganancia dB, Q)."""
    arr = _check_signal(x, sr)
    sos = [butter(4, low_hz, btype="highpass", fs=sr, output="sos")]
    sos += [_biquad("pico", f, g, q, sr) for f, g, q in resonances]
    return sosfilt(np.vstack(sos), arr).astype(np.float32)


def synthetic_rir(rt60: float, rng: np.random.Generator, sr: int = AUG_SR) -> np.ndarray:
    """Respuesta al impulso sintética: ruido gaussiano con caída exponencial de 60 dB en rt60
    segundos (sin datasets de RIR, que necesitarían intake). Energía unitaria."""
    if not rt60 > 0:
        raise ValueError("RT60 debe ser positivo")
    n = int(1.5 * rt60 * sr) + 1
    t = np.arange(n) / sr
    ir = rng.standard_normal(n) * np.exp(-3 * np.log(10) * t / rt60)
    return ir / np.sqrt(np.sum(ir**2))


def estimate_rt60(ir: np.ndarray, sr: int = AUG_SR) -> float:
    """RT60 por integración inversa de Schroeder y ajuste lineal entre -5 y -25 dB (T20 x 3)."""
    energy = np.cumsum(np.asarray(ir, dtype=np.float64)[::-1] ** 2)[::-1]
    edc = 10 * np.log10(energy / energy[0])
    sel = np.flatnonzero((edc <= -5) & (edc >= -25))
    if sel.size < 2:
        raise ValueError("respuesta al impulso demasiado corta para estimar RT60")
    slope = np.polyfit(sel / sr, edc[sel], 1)[0]
    return float(-60 / slope)


def apply_reverb(
    x: np.ndarray, *, rt60: float, wet: float, rir_seed: int, sr: int = AUG_SR
) -> np.ndarray:
    arr = _check_signal(x, sr)
    ir = synthetic_rir(rt60, np.random.default_rng(rir_seed), sr)
    reverberant = fftconvolve(arr, ir)[: len(arr)]
    return ((1 - wet) * arr + wet * reverberant).astype(np.float32)


def mulaw_codec(x: np.ndarray, bits: int = 8) -> np.ndarray:
    """Codifica y decodifica en mu-law de `bits` bits (cuantificación de telefonía)."""
    mu = 2**bits - 1
    arr = np.clip(np.asarray(x, dtype=np.float64), -1, 1)
    y = np.sign(arr) * np.log1p(mu * np.abs(arr)) / np.log1p(mu)
    q = np.round((y + 1) / 2 * mu) / mu * 2 - 1
    return (np.sign(q) * np.expm1(np.abs(q) * np.log1p(mu)) / mu).astype(np.float32)


def add_low_noise(x: np.ndarray, *, snr_db: float, noise_seed: int) -> np.ndarray:
    arr = np.asarray(x, dtype=np.float64)
    power = float(np.mean(arr**2))
    if power <= 0:
        return arr.astype(np.float32)
    noise = np.random.default_rng(noise_seed).standard_normal(arr.size)
    return (arr + noise * np.sqrt(power / 10 ** (snr_db / 10))).astype(np.float32)


def random_channel(seed: int) -> dict:
    """Especificación determinista (por semilla) de un canal aleatorio."""
    rng = np.random.default_rng(seed)
    p = {k: bool(rng.uniform() < v) for k, v in CHANNEL_STAGE_PROB.items()}
    spec: dict = {"seed": int(seed), "synthetic": True}
    if p["altavoz"]:
        spec["altavoz"] = {
            "low_hz": float(rng.uniform(150, 400)),
            "resonancias": [
                [float(rng.uniform(500, 4000)), float(rng.uniform(3, 10)), float(rng.uniform(2, 5))]
                for _ in range(int(rng.integers(1, 3)))
            ],
        }
    if p["eq"]:
        bands = []
        for _ in range(int(rng.integers(3, 6))):
            kind = ("pico", "grave", "agudo")[int(rng.integers(3))]
            bands.append(
                {
                    "tipo": kind,
                    "f": float(np.exp(rng.uniform(np.log(80), np.log(6000)))),
                    "ganancia_db": float(rng.uniform(*EQ_GAIN_RANGE_DB)),
                    "q": 0.7 if kind != "pico" else float(rng.uniform(0.5, 4)),
                }
            )
        spec["eq"] = bands
    if p["reverb"]:
        spec["reverb"] = {
            "rt60": float(rng.uniform(*RT60_RANGE_S)),
            "wet": float(rng.uniform(0.2, 0.7)),
            "rir_seed": int(rng.integers(2**31)),
        }
    if p["microfono"]:
        spec["microfono"] = {
            "low_hz": float(rng.uniform(80, 300)),
            "high_hz": float(rng.uniform(6000, 7800)),
            "order": int(rng.integers(1, 5)),
        }
    if p["codec"]:
        spec["codec"] = {"bits": 8, "corte_hz": float(rng.uniform(3400, 7000))}
    if p["ruido"]:
        spec["ruido"] = {
            "snr_db": float(rng.uniform(25, 50)),
            "noise_seed": int(rng.integers(2**31)),
        }
    return spec


def apply_channel(x: np.ndarray, spec: dict, sr: int = AUG_SR) -> np.ndarray:
    """Altavoz -> EQ -> sala -> micrófono -> códec -> ruido -> limitador. Conserva la longitud."""
    y = _check_signal(x, sr).astype(np.float32)
    if "altavoz" in spec:
        s = spec["altavoz"]
        y = speaker_response(y, low_hz=s["low_hz"], resonances=[tuple(r) for r in s["resonancias"]])
    if "eq" in spec:
        y = apply_eq(y, spec["eq"])
    if "reverb" in spec:
        y = apply_reverb(y, **spec["reverb"])
    if "microfono" in spec:
        y = mic_response(y, **spec["microfono"])
    if "codec" in spec:
        c = spec["codec"]
        peak = float(np.max(np.abs(y)))
        scale = AUG_PEAK / peak if peak > AUG_PEAK else 1.0
        y = mulaw_codec(y * scale, c["bits"])
        y = sosfilt(butter(6, c["corte_hz"], btype="lowpass", fs=sr, output="sos"), y)
    if "ruido" in spec:
        y = add_low_noise(y, **spec["ruido"])
    y = np.asarray(y, dtype=np.float64)
    peak = float(np.max(np.abs(y)))
    if peak > AUG_PEAK:
        y = y * (AUG_PEAK / peak)
    return y.astype(np.float32)


def plan_channel_augment(
    indices: Sequence[int],
    sources: Sequence[str],
    *,
    n: int,
    seed: int,
    labels: Sequence[int] | None = None,
) -> list[dict]:
    """n copias con canal aleatorio; el clip se elige equilibrando celdas (fuente, clase) para
    que el canal aleatorio no se correlacione con la clase. sources/labels se indexan por clip."""
    if not indices:
        raise ValueError("no hay clips que aumentar")
    rng = np.random.default_rng(seed)
    cells: dict[tuple, list[int]] = {}
    for i in indices:
        cells.setdefault((sources[i], None if labels is None else int(labels[i])), []).append(i)
    names = sorted(cells, key=str)
    plan = []
    for k in range(n):
        members = cells[names[int(rng.integers(len(names)))]]
        clip = int(members[int(rng.integers(len(members)))])
        plan.append(
            {
                "id": f"canal{k:05d}",
                "clip": clip,
                "seed": int(rng.integers(2**31)),
                "offset": float(rng.uniform()),
                "synthetic": True,
            }
        )
    return plan


def channel_augs_for_training(plan: Sequence[dict], train: Sequence[int]) -> list[int]:
    """Posiciones del plan cuyo clip de origen está en entrenamiento (las únicas utilizables)."""
    allowed = set(train)
    return [k for k, p in enumerate(plan) if p["clip"] in allowed]


def assert_channel_augs_training_only(
    plan: Sequence[dict], used: Sequence[int], train: Sequence[int]
) -> None:
    """Fallo cerrado: ninguna copia aumentada puede venir de un clip de validación o test."""
    allowed = set(train)
    bad = [plan[k]["id"] for k in used if plan[k]["clip"] not in allowed]
    if bad:
        raise ValueError(f"aumento de canal fuera de entrenamiento: {bad[:5]}")


In [ ]:
# Rasgo armónico de hélices (hito 2, H2.4) sobre una ventana YA normalizada (channel_norm) de
# 16 kHz: fundamental del paso de palas (50-1000 Hz) por suma ponderada de armónicos sobre la
# prominencia espectral, armonicidad (fracción de energía en los armónicos detectados), número de
# armónicos, prominencia media, estabilidad de la fundamental dentro de la ventana (4 subtramas),
# modulación de amplitud (profundidad y frecuencia) y flujo del mel con PCEN. Sin estado entre
# ventanas: el Mac y Colab calculan exactamente lo mismo por ventana. Vector de HARMONIC_DIM
# valores en [0, 1] que se concatena al embedding YAMNet. Autocontenido (numpy, scipy): el
# cuaderno lo incrusta después de channel_norm (build_colab.py quita la importación relativa, que
# allí ya está definida) y usa sus mel_power y pcen.
import numpy as np
from scipy.ndimage import maximum_filter1d


HARMONIC_SR = 16000
HARMONIC_NAMES = (
    "f0_log",
    "armonicidad",
    "n_armonicos",
    "prominencia",
    "estabilidad_f0",
    "modulacion_am",
    "frecuencia_am",
    "flujo_pcen",
)
HARMONIC_DIM = len(HARMONIC_NAMES)
HARMONIC_CONFIG = {
    "version": "armonico-1",
    "f0_rango_hz": [50.0, 1000.0],
    "f0_paso_hz": 0.25,
    "banda_max_hz": 7500.0,
    "max_armonicos": 20,
    "peso_armonico": 0.84,  # Hermes (1988): peso 0,84^(h-1)
    "n_fft": 32768,
    "tolerancia_hz": 3.0,
    "prominencia_min_db": 10.0,
    "bloque_suelo": 64,
    "rango_dinamico_db": 80.0,  # bins por bloque para el suelo espectral (mediana)
    "banda_energia_hz": 4.0,
    "subtramas": 4,
    "n_fft_subtrama": 8192,
    "envolvente_muestras": 160,  # 10 ms
    "am_rango_hz": [2.0, 40.0],
    "propela_min_armonicos": 3,
    "propela_min_armonicidad": 0.4,
}


def _check_window(window: np.ndarray) -> np.ndarray:
    x = np.asarray(window, dtype=np.float64)
    if x.ndim != 1 or x.size == 0:
        raise ValueError("se esperaba una ventana 1D no vacía")
    if not np.isfinite(x).all():
        raise ValueError("ventana con valores no finitos")
    return x


def _spectrum(x: np.ndarray, n_fft: int) -> tuple[np.ndarray, np.ndarray]:
    n = min(len(x), n_fft)
    mag = np.abs(np.fft.rfft(x[:n] * np.hanning(n), n_fft))
    return mag, np.fft.rfftfreq(n_fft, 1 / HARMONIC_SR)


def _prominence_db(mag: np.ndarray) -> np.ndarray:
    """dB sobre el suelo espectral local (mediana por bloques, interpolada) o sobre el pico menos
    el rango dinámico, lo que sea mayor."""
    level = 20 * np.log10(mag + 1e-12)
    block = HARMONIC_CONFIG["bloque_suelo"]
    n_blocks = len(level) // block
    medians = np.median(level[: n_blocks * block].reshape(n_blocks, block), axis=1)
    centres = np.arange(n_blocks) * block + block / 2
    floor = np.interp(np.arange(len(level)), centres, medians)
    # Rango dinámico útil limitado: lo que está a más de rango_dinamico_db por debajo del pico es
    # ruido numérico (p. ej. la cuantificación float32 de un tono puro genera «armónicos»).
    floor = np.maximum(floor, level.max() - HARMONIC_CONFIG["rango_dinamico_db"])
    return np.maximum(0.0, level - floor)


def _refine(mag: np.ndarray, freqs: np.ndarray, f0: float, harmonics: list[int]) -> float:
    """f0 por mínimos cuadrados (recta por el origen) sobre los picos interpolados."""
    df = freqs[1] - freqs[0]
    num = den = 0.0
    for h in harmonics:
        lo = int(max(1, (h * f0 * 0.97) / df))
        hi = int(min(len(mag) - 2, (h * f0 * 1.03) / df + 1))
        if hi <= lo:
            continue
        k = lo + int(np.argmax(mag[lo : hi + 1]))
        a, b, c = np.log(mag[k - 1 : k + 2] + 1e-12)
        denom = a - 2 * b + c
        offset = 0.5 * (a - c) / denom if denom < 0 else 0.0
        num += h * (k + offset) * df
        den += h * h
    return num / den if den else f0


def harmonic_summary(window: np.ndarray) -> dict:
    """Resumen interpretable de la ventana (f0 en Hz, armonicidad, número de armónicos…)."""
    cfg = HARMONIC_CONFIG
    x = _check_window(window)
    empty = {
        "f0_hz": 0.0,
        "armonicidad": 0.0,
        "n_armonicos": 0,
        "prominencia_db": 0.0,
        "estabilidad": 0.0,
        "am_profundidad": 0.0,
        "am_frecuencia_hz": 0.0,
        "flujo_pcen": 0.0,
    }
    if not np.any(x):
        return empty
    mag, freqs = _spectrum(x, cfg["n_fft"])
    df = freqs[1] - freqs[0]
    prom = _prominence_db(mag)
    peaky = maximum_filter1d(prom, size=2 * int(cfg["tolerancia_hz"] / df) + 1)
    peaky = np.where(peaky >= cfg["prominencia_min_db"], peaky, 0.0)
    lo, hi = cfg["f0_rango_hz"]
    cands = np.arange(lo, hi + cfg["f0_paso_hz"] / 2, cfg["f0_paso_hz"])
    h = np.arange(1, cfg["max_armonicos"] + 1)
    grid = cands[:, None] * h[None, :]
    valid = grid <= cfg["banda_max_hz"]
    idx = np.minimum(np.round(grid / df).astype(int), len(peaky) - 1)
    weights = cfg["peso_armonico"] ** (h - 1)
    salience = np.sum(np.where(valid, peaky[idx], 0.0) * weights, axis=1)
    if salience.max() <= 0:
        return {**empty, **_modulation(x)}
    best = int(np.argmax(salience))
    detected = [int(k) for k in h[valid[best] & (peaky[idx[best]] > 0)]]
    f0 = _refine(mag, freqs, float(cands[best]), detected)
    power = mag**2
    band = (freqs >= lo) & (freqs <= cfg["banda_max_hz"])
    near = np.zeros_like(band)
    for k in detected:
        near |= np.abs(freqs - k * f0) <= cfg["banda_energia_hz"]
    harmonicity = float(power[near & band].sum() / max(power[band].sum(), 1e-30))
    prominence = float(np.mean([peaky[int(round(k * f0 / df))] for k in detected]))
    return {
        "f0_hz": float(f0),
        "armonicidad": harmonicity,
        "n_armonicos": len(detected),
        "prominencia_db": prominence,
        "estabilidad": _stability(x, f0, detected),
        **_modulation(x),
    }


def _stability(x: np.ndarray, f0: float, detected: list[int]) -> float:
    """1 si la f0 refinada en cada subtrama no se aparta (desviación típica) más de un 0 % de
    f0; 0 si se aparta un 5 % o más."""
    cfg = HARMONIC_CONFIG
    parts = np.array_split(x, cfg["subtramas"])
    estimates = []
    for part in parts:
        if not np.any(part):
            continue
        mag, freqs = _spectrum(part, cfg["n_fft_subtrama"])
        estimates.append(_refine(mag, freqs, f0, detected))
    if len(estimates) < 2 or f0 <= 0:
        return 0.0
    return float(np.clip(1 - np.std(estimates) / (0.05 * f0), 0.0, 1.0))


def _modulation(x: np.ndarray) -> dict:
    """Profundidad (coeficiente de variación de la envolvente RMS de 10 ms) y frecuencia
    dominante de modulación de amplitud (2-40 Hz); flujo medio del mel con PCEN."""
    cfg = HARMONIC_CONFIG
    hop = cfg["envolvente_muestras"]
    n = len(x) // hop
    if n < 8:
        return {"am_profundidad": 0.0, "am_frecuencia_hz": 0.0, "flujo_pcen": 0.0}
    env = np.sqrt(np.mean(x[: n * hop].reshape(n, hop) ** 2, axis=1))
    depth = float(env.std() / env.mean()) if env.mean() > 0 else 0.0
    spec = np.abs(np.fft.rfft((env - env.mean()) * np.hanning(n), 1024))
    fr = np.fft.rfftfreq(1024, hop / HARMONIC_SR)
    sel = (fr >= cfg["am_rango_hz"][0]) & (fr <= cfg["am_rango_hz"][1])
    rate = float(fr[sel][int(np.argmax(spec[sel]))]) if depth >= 0.05 else 0.0
    p = pcen(mel_power(x.astype(np.float32)))
    flux = float(np.mean(np.abs(np.diff(p, axis=1)))) if p.shape[1] > 1 else 0.0
    return {"am_profundidad": depth, "am_frecuencia_hz": rate, "flujo_pcen": flux}


def harmonic_features(window: np.ndarray) -> np.ndarray:
    """Vector de HARMONIC_DIM rasgos en [0, 1] (orden de HARMONIC_NAMES)."""
    s = harmonic_summary(window)
    lo, hi = HARMONIC_CONFIG["f0_rango_hz"]
    f0_log = np.log(s["f0_hz"] / lo) / np.log(hi / lo) if s["n_armonicos"] else 0.0
    am_hi = HARMONIC_CONFIG["am_rango_hz"][1]
    v = [
        f0_log,
        s["armonicidad"],
        min(s["n_armonicos"], HARMONIC_CONFIG["max_armonicos"]) / HARMONIC_CONFIG["max_armonicos"],
        s["prominencia_db"] / 40.0,
        s["estabilidad"],
        s["am_profundidad"],
        s["am_frecuencia_hz"] / am_hi,
        s["flujo_pcen"] / (s["flujo_pcen"] + 1.0),
    ]
    return np.clip(np.asarray(v, dtype=np.float64), 0.0, 1.0).astype(np.float32)


def estimate_f0(window: np.ndarray) -> float:
    """Fundamental estimada en Hz (0,0 si no hay armónicos)."""
    return harmonic_summary(window)["f0_hz"]


def is_propeller_harmonic(summary: dict) -> bool:
    """Firma de hélice: >= 3 armónicos de una f0 en 50-1000 Hz con armonicidad suficiente
    (un tono puro aislado o un ruido no la cumplen)."""
    cfg = HARMONIC_CONFIG
    lo, hi = cfg["f0_rango_hz"]
    return bool(
        summary["n_armonicos"] >= cfg["propela_min_armonicos"]
        and summary["armonicidad"] >= cfg["propela_min_armonicidad"]
        and lo <= summary["f0_hz"] <= hi
    )


In [ ]:
# Lógica de alerta con persistencia e histéresis (hito 2): máquina de estados pura, sin E/S,
# compartida por la sección «Prueba tu audio» del cuaderno y por el Mac (H2.11 puede importarla).
# ESCUCHANDO -> POSIBLE -> ALERTA -> ESCUCHANDO, más VOZ (ventana descartada por la puerta de voz:
# no cuenta ni a favor ni en contra) y DEGRADADO (sin dato). ALERTA si >= k de las últimas n
# ventanas válidas superan theta_on; fin tras fin_ventanas seguidas por debajo de theta_off <
# theta_on; refractario entre el fin de una alerta y el inicio de la siguiente. Los umbrales los
# fija el artefacto (validación de Colab), no el código. Autocontenido (stdlib).
import math
from collections import deque
from collections.abc import Sequence

ALERT_STATES = ("ESCUCHANDO", "POSIBLE", "ALERTA", "VOZ", "DEGRADADO")
ALERT_DEFAULTS = {"k": 3, "n": 5, "fin_ventanas": 4, "refractario_s": 5.0, "salto_s": 0.48}


def validate_alert_config(cfg: dict) -> dict:
    """Comprueba la configuración (fallo cerrado) y devuelve una copia completa."""
    out = {**ALERT_DEFAULTS, **cfg}
    unknown = set(out) - set(ALERT_DEFAULTS) - {"theta_on", "theta_off"}
    if unknown:
        raise ValueError(f"campos desconocidos en la alerta: {sorted(unknown)}")
    for key in ("theta_on", "theta_off"):
        v = out.get(key)
        if not isinstance(v, int | float) or not math.isfinite(v) or not 0 < v < 1:
            raise ValueError(f"{key} debe estar en (0, 1)")
    if out["theta_off"] >= out["theta_on"]:
        raise ValueError("theta_off debe ser menor que theta_on (histéresis)")
    for key in ("k", "n", "fin_ventanas"):
        if not isinstance(out[key], int) or out[key] < 1:
            raise ValueError(f"{key} debe ser un entero >= 1")
    if out["k"] > out["n"]:
        raise ValueError("k no puede ser mayor que n")
    if out["refractario_s"] < 0 or out["salto_s"] <= 0:
        raise ValueError("tiempos no válidos")
    return out


class AlertMachine:
    """Una llamada a step() por ventana (cada salto_s segundos)."""

    def __init__(self, cfg: dict) -> None:
        self.cfg = validate_alert_config(cfg)
        self.recent: deque[bool] = deque(maxlen=self.cfg["n"])
        self.alerting = False
        self.below = 0
        self.t = 0
        self.last_end: float | None = None
        self.events = 0

    def _underlying(self) -> str:
        if self.alerting:
            return "ALERTA"
        return "POSIBLE" if any(self.recent) else "ESCUCHANDO"

    def step(self, prob: float | None, *, speech: bool = False) -> str:
        now = self.t * self.cfg["salto_s"]
        self.t += 1
        if prob is None or not math.isfinite(prob):
            return "DEGRADADO"
        if speech:
            return "VOZ"
        self.recent.append(prob >= self.cfg["theta_on"])
        if self.alerting:
            self.below = self.below + 1 if prob < self.cfg["theta_off"] else 0
            if self.below >= self.cfg["fin_ventanas"]:
                self.alerting, self.below, self.last_end = False, 0, now
                self.recent.clear()
        elif sum(self.recent) >= self.cfg["k"]:
            refractory_ok = (
                self.last_end is None or now - self.last_end >= self.cfg["refractario_s"]
            )
            if refractory_ok:
                self.alerting, self.below = True, 0
                self.events += 1
        return self._underlying()


def run_alert(
    probs: Sequence[float | None], speech: Sequence[bool] | None, cfg: dict
) -> tuple[list[str], int]:
    """Estados por ventana y número de alertas de una secuencia completa."""
    machine = AlertMachine(cfg)
    flags = list(speech) if speech is not None else [False] * len(probs)
    if len(flags) != len(probs):
        raise ValueError("probs y speech deben tener la misma longitud")
    states = [machine.step(p, speech=s) for p, s in zip(probs, flags, strict=True)]
    return states, machine.events


def persistence_score(probs: Sequence[float], k: int = 3, n: int = 5) -> float:
    """Puntuación de clip equivalente a «>= k de n ventanas sobre el umbral»: máximo, sobre los
    tramos de n ventanas, del k-ésimo valor más alto. Clip con m < n ventanas: k_m = ceil(k·m/n)
    (un clip corto no puede acumular k ventanas; se informa como limitación)."""
    p = [float(v) for v in probs]
    if not p:
        raise ValueError("clip sin ventanas")
    if any(not math.isfinite(v) for v in p):
        raise ValueError("probabilidades no finitas")
    if k < 1 or k > n:
        raise ValueError("k debe estar en [1, n]")
    if len(p) < n:
        k_m = max(1, math.ceil(k * len(p) / n))
        return sorted(p, reverse=True)[k_m - 1]
    return max(sorted(p[i : i + n], reverse=True)[k - 1] for i in range(len(p) - n + 1))


In [ ]:
# Técnicas contra el atajo de dominio (hito 2, iteración 6), ligeras y sobre embeddings:
# - modelo lineal con estandarización PLEGADA en (coef, intercept) del espacio original: el Mac
#   solo necesita un producto escalar (artefacto pequeño, JSON);
# - INLP (Ravfogel et al., 2020, «Iterative Null-space Projection»): se entrenan clasificadores
#   lineales de FUENTE dentro de cada clase y se proyecta fuera de sus direcciones, de modo que lo
#   que distingue una fuente de otra (canal, micrófono) deja de estar disponible para el
#   clasificador de dron; alternativa lineal y estable al entrenamiento adversarial de dominio;
# - group-DRO (Sagawa et al., 2020) por pesos exponenciados sobre grupos fuente x clase: minimiza
#   la pérdida del peor grupo en vez de la media;
# - umbral robusto por peor fuente: el más bajo que deja la FPR de CADA fuente de validación por
#   debajo del máximo (y su umbral de fin de alerta, con histéresis);
# - centrado por fuente con negativos de calibración (variante de sitio, D-7, siempre aparte).
# Autocontenido (numpy, scikit-learn): build_colab.py lo incrusta tal cual.
import math
from collections.abc import Sequence

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


def _sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.clip(z, -60, 60)))


def _orthonormal(columns: np.ndarray) -> np.ndarray:
    if columns.size == 0:
        return columns
    q, r = np.linalg.qr(columns)
    keep = np.abs(np.diag(r)) > 1e-8
    return q[:, keep]


def inlp_directions(
    z: np.ndarray,
    labels: Sequence[int],
    domains: Sequence[str],
    *,
    rounds: int,
    seed: int,
    c: float = 0.1,
) -> np.ndarray:
    """Base ortonormal (d x k) de las direcciones que predicen la fuente DENTRO de cada clase
    (así no se borra la información de clase). z debe estar estandarizada."""
    y = np.asarray(labels, dtype=int)
    dom = np.asarray(domains)
    basis = np.zeros((z.shape[1], 0))
    for _ in range(rounds):
        found = []
        proj = z - (z @ basis) @ basis.T if basis.size else z
        for cls in sorted(set(y.tolist())):
            mask = y == cls
            if len(set(dom[mask].tolist())) < 2:
                continue
            clf = LogisticRegression(C=c, max_iter=1000, random_state=seed)
            clf.fit(proj[mask], dom[mask])
            found.append(clf.coef_.T)
        if not found:
            break
        new = np.hstack(found)
        new = new - basis @ (basis.T @ new) if basis.size else new
        basis = _orthonormal(np.hstack([basis, new]))
    return basis


def fit_linear(
    x: np.ndarray,
    labels: Sequence[int],
    weights: Sequence[float],
    *,
    seed: int,
    c: float = 1.0,
    domains: Sequence[str] | None = None,
    inlp_rounds: int = 0,
    max_iter: int = 3000,
) -> dict:
    """Regresión logística sobre rasgos estandarizados (y, si inlp_rounds > 0, proyectados fuera
    de las direcciones de fuente); devuelve coef/intercept en el espacio ORIGINAL."""
    prep = _prepare(x, labels, domains, inlp_rounds=inlp_rounds, seed=seed)
    return _fit_prepared(prep, labels, weights, seed=seed, c=c, max_iter=max_iter)


def _prepare(
    x: np.ndarray,
    labels: Sequence[int],
    domains: Sequence[str] | None,
    *,
    inlp_rounds: int,
    seed: int,
) -> dict:
    """Estandarización y base INLP, calculadas una sola vez por ajuste (también para DRO)."""
    x = np.asarray(x, dtype=np.float64)
    mu = x.mean(axis=0)
    sd = np.maximum(x.std(axis=0), 1e-6)
    z = (x - mu) / sd
    basis = np.zeros((x.shape[1], 0))
    if inlp_rounds > 0:
        if domains is None:
            raise ValueError("INLP necesita la fuente de cada muestra")
        basis = inlp_directions(z, labels, domains, rounds=inlp_rounds, seed=seed)
        z = z - (z @ basis) @ basis.T
    return {"mu": mu, "sd": sd, "basis": basis, "z": z}


def _fit_prepared(
    prep: dict,
    labels: Sequence[int],
    weights: Sequence[float],
    *,
    seed: int,
    c: float,
    max_iter: int = 3000,
) -> dict:
    lr = LogisticRegression(C=c, max_iter=max_iter, random_state=seed)
    lr.fit(prep["z"], np.asarray(labels, dtype=int), sample_weight=np.asarray(weights, float))
    w = lr.coef_[0]
    basis, mu, sd = prep["basis"], prep["mu"], prep["sd"]
    if basis.size:
        w = w - basis @ (basis.T @ w)
    return {
        "coef": w / sd,
        "intercept": float(lr.intercept_[0] - np.sum(w * mu / sd)),
        "inlp_direcciones": int(basis.shape[1]),
    }


def linear_logit(model: dict, x: np.ndarray) -> np.ndarray:
    return np.asarray(x, dtype=np.float64) @ np.asarray(model["coef"]) + model["intercept"]


def linear_proba(model: dict, x: np.ndarray) -> np.ndarray:
    return _sigmoid(linear_logit(model, x))


def group_dro_fit(
    x: np.ndarray,
    labels: Sequence[int],
    groups: Sequence[str],
    *,
    seed: int,
    rounds: int = 4,
    eta: float = 1.0,
    c: float = 1.0,
    domains: Sequence[str] | None = None,
    inlp_rounds: int = 0,
) -> tuple[dict, list[dict]]:
    """Group-DRO por pesos exponenciados: cada grupo (p. ej. «fuente|clase») pesa q_g en total;
    tras cada ajuste, q_g <- q_g · exp(eta · pérdida_g) y se renormaliza."""
    y = np.asarray(labels, dtype=int)
    g = np.asarray(groups)
    names = sorted(set(g.tolist()))
    q = np.full(len(names), 1.0 / len(names))
    index = {name: i for i, name in enumerate(names)}
    gi = np.array([index[v] for v in g.tolist()])
    counts = np.bincount(gi, minlength=len(names)).astype(float)
    history = []
    model = None
    prep = _prepare(x, y, domains, inlp_rounds=inlp_rounds, seed=seed)
    for _ in range(rounds + 1):
        w = q[gi] / counts[gi] * len(y)
        model = _fit_prepared(prep, y, w, seed=seed, c=c)
        p = np.clip(linear_proba(model, x), 1e-7, 1 - 1e-7)
        loss_i = -(y * np.log(p) + (1 - y) * np.log(1 - p))
        losses = np.bincount(gi, weights=loss_i, minlength=len(names)) / counts
        history.append(
            {
                "q": dict(zip(names, q.round(6).tolist(), strict=True)),
                "perdida": dict(zip(names, losses.round(6).tolist(), strict=True)),
            }
        )
        q = q * np.exp(eta * losses)
        q = q / q.sum()
    return model, history


def worst_source_threshold(
    labels: Sequence[int], scores: Sequence[float], sources: Sequence[str], *, max_fpr: float
) -> dict:
    """Umbral más bajo con FPR <= max_fpr en CADA fuente con negativos (la peor fuente manda).
    Devuelve también recall de validación por fuente con dron y la FPR por fuente."""
    y = np.asarray(labels, dtype=int)
    p = np.asarray(scores, dtype=np.float64)
    s = np.asarray(sources)
    if not 0 <= max_fpr < 1:
        raise ValueError("max_fpr debe estar en [0, 1)")
    neg_sources = sorted(set(s[y == 0].tolist()))
    if not neg_sources:
        raise ValueError("no hay negativos en validación")
    per = {}
    for src in neg_sources:
        neg = np.sort(p[(y == 0) & (s == src)])[::-1]
        allowed = int(math.floor(max_fpr * len(neg)))
        per[src] = float(np.nextafter(neg[allowed], np.inf)) if allowed < len(neg) else 0.0
    threshold = max(per.values())
    return {
        "threshold": threshold,
        "fuente_limitante": max(per, key=per.get),
        "umbral_por_fuente": per,
        "fpr_validacion": {
            src: float(np.mean(p[(y == 0) & (s == src)] >= threshold)) for src in neg_sources
        },
        "recall_validacion": {
            src: float(np.mean(p[(y == 1) & (s == src)] >= threshold))
            for src in sorted(set(s[y == 1].tolist()))
        },
    }


def hysteresis_thresholds(
    labels: Sequence[int],
    scores: Sequence[float],
    sources: Sequence[str],
    *,
    max_fpr_on: float,
    max_fpr_off: float,
) -> dict:
    """theta_on con max_fpr_on por peor fuente; theta_off (fin de alerta) con max_fpr_off >
    max_fpr_on, siempre estrictamente por debajo de theta_on y dentro de (0, 1)."""
    if max_fpr_off <= max_fpr_on:
        raise ValueError("max_fpr_off debe ser mayor que max_fpr_on")
    on = worst_source_threshold(labels, scores, sources, max_fpr=max_fpr_on)
    off = worst_source_threshold(labels, scores, sources, max_fpr=max_fpr_off)
    theta_on = float(min(max(on["threshold"], 1e-4), 1 - 1e-6))
    theta_off = float(min(max(off["threshold"], 5e-5), theta_on * 0.999))
    return {"theta_on": theta_on, "theta_off": theta_off, "detalle_on": on}


def calibration_centers(
    x: np.ndarray, sources: Sequence[str], mask: Sequence[bool]
) -> dict[str, np.ndarray]:
    """Media por fuente de las filas de calibración (negativos del sitio, D-7)."""
    x = np.asarray(x, dtype=np.float64)
    s = np.asarray(sources)
    m = np.asarray(mask, dtype=bool)
    out = {}
    for src in sorted(set(s.tolist())):
        sel = m & (s == src)
        if not sel.any():
            raise ValueError(f"la fuente {src} no tiene filas de calibración")
        out[src] = x[sel].mean(axis=0)
    return out


def apply_centers(
    x: np.ndarray, sources: Sequence[str], centers: dict[str, np.ndarray]
) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    return x - np.vstack([centers[v] for v in sources])


def safe_roc_auc(labels: Sequence[int], scores: Sequence[float]) -> float | None:
    y = np.asarray(labels, dtype=int)
    if len(set(y.tolist())) < 2:
        return None
    return float(roc_auc_score(y, np.asarray(scores, dtype=np.float64)))


In [ ]:
# Ruta de inferencia por ventana y artefacto del modelo (hito 2, H2.5). La MISMA función
# window_features/score_features se usa al entrenar en Colab, en «Prueba tu audio» y en el Mac:
# ventana de 16 kHz -> normalización de canal (channel_norm) -> YAMNet (TF en Colab, ONNX en el
# Mac; se inyecta como función) -> rasgo armónico (harmonic) -> lineal -> Platt. El artefacto
# modelo_hito2.json es solo JSON (nunca pickle), pequeño, con hash propio y vectores dorados: tres
# entradas sintéticas regenerables por semilla con su salida esperada, para comprobar en el Mac
# que normalización, rasgos y pesos son los de Colab. Autocontenido salvo las importaciones
# relativas, que build_colab.py quita porque en el cuaderno ya están definidas.
import hashlib
import json
import math
from collections.abc import Callable, Sequence

import numpy as np


ARTIFACT_FORMAT = "atalaya-hito2-modelo/1"
EMBEDDING_DIM = 1024
YAMNET_CLASSES = 521
GOLDEN_KINDS = ("peine", "ruido_rosa", "tono_y_ruido")
GOLDEN_SR = 48000
GOLDEN_TOLERANCE = 1e-4
ARTIFACT_LABEL = "Reproducción en sala, no vuelo real"
ARTIFACT_KEYS = {
    "formato",
    "iteracion",
    "rotulo",
    "modelo",
    "calibracion",
    "alerta",
    "normalizacion",
    "armonico",
    "yamnet",
    "voz",
    "vectores_dorados",
    "criterios",
    "predicciones_csv_sha256",
    "registro_experimento_sha256",
    "artefacto_sha256",
}
# Una función YAMNet recibe una ventana float32 de 16 kHz y devuelve (scores (n, 521),
# embeddings (n, 1024)); con WINDOW_SAMPLES muestras n = 1.
YamnetFn = Callable[[np.ndarray], tuple[np.ndarray, np.ndarray]]


def _sigm(z: np.ndarray | float) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.clip(np.asarray(z, dtype=np.float64), -60, 60)))


def platt_from_logit(logit: np.ndarray | float, a: float, b: float) -> np.ndarray:
    """Platt sobre la probabilidad cruda, con el mismo recorte (1e-6) que calibration.py."""
    q = np.clip(_sigm(logit), 1e-6, 1 - 1e-6)
    return _sigm(a * np.log(q / (1 - q)) + b)


def window_features(window16k: np.ndarray, yamnet_fn: YamnetFn, speech_indices: Sequence[int]):
    """Ventana de 16 kHz (WINDOW_SAMPLES; se rellena si es más corta) -> rasgos."""
    x = np.asarray(window16k, dtype=np.float32)
    if x.ndim != 1 or x.size == 0 or x.size > WINDOW_SAMPLES:
        raise ValueError("se esperaba una ventana 1D de hasta WINDOW_SAMPLES muestras")
    if x.size < WINDOW_SAMPLES:
        x = np.pad(x, (0, WINDOW_SAMPLES - x.size))
    norm = normalize_window(x)
    scores, embeddings = yamnet_fn(norm)
    scores = np.asarray(scores, dtype=np.float64).reshape(-1, YAMNET_CLASSES)[0]
    emb = np.asarray(embeddings, dtype=np.float64).reshape(-1, EMBEDDING_DIM)[0]
    return {
        "normalizada": norm,
        "scores": scores,
        "embedding": emb,
        "armonico": harmonic_features(norm).astype(np.float64),
        "voz": float(scores[list(speech_indices)].max()) if len(speech_indices) else 0.0,
    }


def feature_vector(embedding: np.ndarray, harmonic: np.ndarray) -> np.ndarray:
    v = np.concatenate([np.asarray(embedding, np.float64), np.asarray(harmonic, np.float64)])
    if v.shape != (EMBEDDING_DIM + HARMONIC_DIM,):
        raise ValueError("dimensión de rasgos inesperada")
    return v


def score_features(artifact: dict, embedding: np.ndarray, harmonic: np.ndarray) -> dict:
    """Logit lineal, probabilidad cruda y calibrada (Platt) de una ventana."""
    m, cal = artifact["modelo"], artifact["calibracion"]
    logit = float(feature_vector(embedding, harmonic) @ np.asarray(m["coef"]) + m["intercept"])
    return {
        "logit": logit,
        "prob": float(_sigm(logit)),
        "prob_cal": float(platt_from_logit(logit, cal["a"], cal["b"])),
    }


def golden_input(seed: int, kind: str, sr: int = GOLDEN_SR) -> np.ndarray:
    """Entrada sintética determinista (PCG64 de numpy, estable entre plataformas) de una ventana
    a la frecuencia sr; se pasa por to_mono_16k como el audio del micrófono."""
    if kind not in GOLDEN_KINDS:
        raise ValueError(f"tipo dorado desconocido: {kind}")
    rng = np.random.default_rng(seed)
    n = int(math.ceil(WINDOW_SAMPLES * sr / NORM_SR))
    t = np.arange(n) / sr
    if kind == "peine":
        f0 = rng.uniform(100, 300)
        x = sum(np.sin(2 * np.pi * h * f0 * t + rng.uniform(0, 6.28)) / h for h in range(1, 13))
        x = x + 0.05 * rng.standard_normal(n)
    elif kind == "ruido_rosa":
        spec = np.fft.rfft(rng.standard_normal(n))
        f = np.fft.rfftfreq(n, 1 / sr)
        spec[1:] /= np.sqrt(f[1:])
        spec[0] = 0
        x = np.fft.irfft(spec, n)
    else:
        x = np.sin(2 * np.pi * rng.uniform(500, 2000) * t) + 0.3 * rng.standard_normal(n)
    x = 0.1 * x / np.std(x)
    return x.astype(np.float32)


def golden_record(
    artifact: dict, yamnet_fn: YamnetFn, seed: int, kind: str, sr: int = GOLDEN_SR
) -> dict:
    window = to_mono_16k(golden_input(seed, kind, sr), sr)
    feats = window_features(window, yamnet_fn, artifact["voz"]["indices"])
    s = score_features(artifact, feats["embedding"], feats["armonico"])
    return {
        "semilla": int(seed),
        "tipo": kind,
        "sr": int(sr),
        "esperado": {
            "normalizada_rms": float(
                np.sqrt(np.mean(feats["normalizada"].astype(np.float64) ** 2))
            ),
            "normalizada_primeras": [float(v) for v in feats["normalizada"][:8]],
            "armonico": [float(v) for v in feats["armonico"]],
            "embedding_l2": float(np.linalg.norm(feats["embedding"])),
            "embedding_primeros": [float(v) for v in feats["embedding"][:8]],
            "voz": feats["voz"],
            **s,
        },
    }


def _flatten(expected: dict) -> list[float]:
    out = []
    for key in sorted(expected):
        v = expected[key]
        out += [float(x) for x in v] if isinstance(v, list) else [float(v)]
    return out


def check_golden(artifact: dict, yamnet_fn: YamnetFn, *, tol: float = GOLDEN_TOLERANCE) -> float:
    """Reproduce los vectores dorados; devuelve la máxima diferencia y falla si supera tol."""
    worst = 0.0
    for g in artifact["vectores_dorados"]:
        got = golden_record(artifact, yamnet_fn, g["semilla"], g["tipo"], g["sr"])["esperado"]
        if sorted(got) != sorted(g["esperado"]):
            raise ValueError("campos distintos en un vector dorado")
        diff = np.max(np.abs(np.array(_flatten(got)) - np.array(_flatten(g["esperado"]))))
        worst = max(worst, float(diff))
    if not worst <= tol:
        raise ValueError(f"los vectores dorados no se reproducen: diferencia {worst:.3g} > {tol}")
    return worst


def artifact_sha256(artifact: dict) -> str:
    body = {k: v for k, v in artifact.items() if k != "artefacto_sha256"}
    data = json.dumps(body, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(data).hexdigest()


def build_artifact(
    *,
    model: dict,
    platt: dict,
    alert: dict,
    yamnet_info: dict,
    speech_indices: Sequence[int],
    speech_names: Sequence[str],
    speech_threshold: float,
    criteria: dict,
    predictions_sha256: str,
    record_sha256: str,
    yamnet_fn: YamnetFn,
    golden_seeds: Sequence[int] = (101, 202, 303),
) -> dict:
    """Arma, firma (hash propio) y valida el artefacto; calcula los vectores dorados con
    yamnet_fn (en Colab, la sesión ONNX que usará el Mac)."""
    artifact = {
        "formato": ARTIFACT_FORMAT,
        "iteracion": 6,
        "rotulo": ARTIFACT_LABEL,
        "modelo": {
            "nombre": str(model["nombre"]),
            "rasgos": {"embedding_dim": EMBEDDING_DIM, "armonico": list(HARMONIC_NAMES)},
            "coef": [float(v) for v in model["coef"]],
            "intercept": float(model["intercept"]),
            "inlp_direcciones": int(model.get("inlp_direcciones", 0)),
        },
        "calibracion": {"metodo": "platt", "a": float(platt["a"]), "b": float(platt["b"])},
        "alerta": validate_alert_config(alert),
        "normalizacion": {"sha256": norm_config_sha256()},
        "armonico": {"config": HARMONIC_CONFIG, "nombres": list(HARMONIC_NAMES)},
        "yamnet": dict(yamnet_info),
        "voz": {
            "indices": [int(i) for i in speech_indices],
            "nombres": list(speech_names),
            "umbral": float(speech_threshold),
        },
        "vectores_dorados": [],
        "criterios": criteria,
        "predicciones_csv_sha256": predictions_sha256,
        "registro_experimento_sha256": record_sha256,
    }
    artifact["vectores_dorados"] = [
        golden_record(artifact, yamnet_fn, seed, kind)
        for seed, kind in zip(golden_seeds, GOLDEN_KINDS, strict=True)
    ]
    artifact["artefacto_sha256"] = artifact_sha256(artifact)
    validate_artifact(artifact)
    return artifact


def validate_artifact(artifact: dict) -> None:
    """Esquema, dimensiones, números finitos, coherencia con el código local y hash propio."""
    if set(artifact) != ARTIFACT_KEYS:
        raise ValueError(f"campos inesperados o ausentes: {sorted(set(artifact) ^ ARTIFACT_KEYS)}")
    if artifact["formato"] != ARTIFACT_FORMAT:
        raise ValueError("formato de artefacto desconocido")
    m = artifact["modelo"]
    if len(m["coef"]) != EMBEDDING_DIM + HARMONIC_DIM:
        raise ValueError("los coeficientes no cuadran con embedding + rasgo armónico")
    if m["rasgos"]["armonico"] != list(HARMONIC_NAMES) or m["rasgos"]["embedding_dim"] != 1024:
        raise ValueError("orden o nombres de rasgos distintos de los del código")
    numbers = [
        *m["coef"],
        m["intercept"],
        artifact["calibracion"]["a"],
        artifact["calibracion"]["b"],
    ]
    if not all(isinstance(v, int | float) and math.isfinite(v) for v in numbers):
        raise ValueError("coeficientes no finitos")
    if artifact["calibracion"]["a"] <= 0:
        raise ValueError("calibración de Platt no creciente")
    if artifact["normalizacion"]["sha256"] != norm_config_sha256():
        raise ValueError("la normalización del artefacto no es la del código local")
    if artifact["armonico"]["config"] != HARMONIC_CONFIG:
        raise ValueError("la configuración del rasgo armónico no es la del código local")
    validate_alert_config(artifact["alerta"])
    voice = artifact["voz"]
    if not voice["indices"] or any(not 0 <= i < YAMNET_CLASSES for i in voice["indices"]):
        raise ValueError("índices de voz no válidos")
    if len(artifact["vectores_dorados"]) != len(GOLDEN_KINDS):
        raise ValueError("faltan vectores dorados")
    if artifact_sha256(artifact) != artifact["artefacto_sha256"]:
        raise ValueError("el hash propio del artefacto no cuadra")


In [ ]:
# «Prueba tu audio» (hito 2, cuaderno de la iteración 6): el usuario sube SU fichero y se analiza
# por la MISMA ruta que el modelo final y el Mac: remuestreo a 16 kHz -> ventanas de 0,96 s con
# salto de 0,48 s -> normalización de canal -> YAMNet -> puerta de voz (ventanas con voz >= umbral
# descartadas y contadas) -> rasgo armónico -> lineal -> Platt -> alerta 3 de 5 con histéresis.
# Nada se guarda: el cuaderno borra el fichero subido tras inferir. Autocontenido salvo las
# importaciones relativas (en el cuaderno ya están definidas y build_colab.py las quita).
import io
import math
import wave
from pathlib import Path

import numpy as np
from scipy.signal import resample_poly


USER_AUDIO_NOTE = (
    "Sube solo audio propio o con una licencia que permita este uso; no descargues audio de "
    "YouTube ni de otras plataformas. El fichero se analiza en esta sesión de Colab y se borra al "
    "terminar; no se guarda en Drive ni en el repositorio."
)
USER_AUDIO_LABEL = "Reproducción o grabación de prueba, no vuelo real verificado"
USER_AUDIO_SUFFIXES = (".wav", ".mp3", ".m4a", ".ogg", ".flac")
USER_MIN_SR = 8000
USER_MAX_SR = 192000
USER_SILENCE_DBFS = -60.0
USER_CLIP_LEVEL = 0.999
USER_CLIP_FRACTION = 0.01
UNCERTAINTY_MARGIN = 0.05
VERDICT_YES = "DRON DETECTADO"
VERDICT_NO = "NO DETECTADO"
VERDICT_NONE = "SIN DECISIÓN"


class AudioDecodeError(ValueError):
    """El fichero no se puede leer como audio."""


def decode_wav_bytes(data: bytes) -> tuple[np.ndarray, int]:
    """WAV PCM de 8/16/24/32 bits con la biblioteca estándar -> (float32 (n, canales), sr)."""
    try:
        with wave.open(io.BytesIO(data), "rb") as wav:
            sr, ch, width = wav.getframerate(), wav.getnchannels(), wav.getsampwidth()
            raw = wav.readframes(wav.getnframes())
    except (wave.Error, EOFError, ValueError) as exc:
        raise AudioDecodeError(f"WAV ilegible: {exc}") from exc
    if not raw or ch < 1:
        raise AudioDecodeError("WAV vacío")
    if width == 1:
        x = (np.frombuffer(raw, np.uint8).astype(np.float64) - 128) / 128
    elif width == 2:
        x = np.frombuffer(raw, "<i2").astype(np.float64) / 32768
    elif width == 3:
        b = np.frombuffer(raw, np.uint8).reshape(-1, 3).astype(np.int32)
        v = b[:, 0] | (b[:, 1] << 8) | (b[:, 2] << 16)
        x = np.where(v >= 1 << 23, v - (1 << 24), v).astype(np.float64) / (1 << 23)
    elif width == 4:
        x = np.frombuffer(raw, "<i4").astype(np.float64) / 2**31
    else:
        raise AudioDecodeError(f"ancho de muestra no admitido: {width}")
    if x.size % ch:
        raise AudioDecodeError("WAV truncado")
    return x.reshape(-1, ch).astype(np.float32), int(sr)


def load_user_audio(path: str | Path, *, fallback_loader=None) -> tuple[np.ndarray, int]:
    """WAV con la biblioteca estándar; otros formatos (mp3, m4a, ogg, flac) con fallback_loader
    (en Colab, librosa.load(path, sr=None, mono=False), que usa soundfile o ffmpeg del sistema)."""
    p = Path(path)
    if p.suffix.lower() not in USER_AUDIO_SUFFIXES:
        raise AudioDecodeError(f"extensión no admitida: {p.suffix or '(ninguna)'}")
    if p.suffix.lower() == ".wav":
        try:
            return decode_wav_bytes(p.read_bytes())
        except AudioDecodeError:
            if fallback_loader is None:
                raise
    if fallback_loader is None:
        raise AudioDecodeError(f"no hay decodificador para {p.suffix}")
    try:
        signal, sr = fallback_loader(str(p))
    except Exception as exc:  # cualquier fallo del decodificador = fichero ilegible
        raise AudioDecodeError(f"no se pudo decodificar: {type(exc).__name__}") from exc
    x = np.asarray(signal, dtype=np.float32)
    if x.size == 0:
        raise AudioDecodeError("audio vacío")
    return (x.T if x.ndim == 2 else x), int(sr)


def to_16k_any(signal: np.ndarray, sr: int) -> np.ndarray:
    """Mono y 16 kHz desde cualquier frecuencia entera de 8 a 192 kHz (8 kHz: banda estrecha)."""
    if not USER_MIN_SR <= int(sr) <= USER_MAX_SR:
        raise ValueError(f"frecuencia de muestreo no admitida: {sr}")
    x = np.asarray(signal, dtype=np.float64)
    if x.size == 0 or not np.isfinite(x).all():
        raise ValueError("audio vacío o con valores no finitos")
    if x.ndim == 2:
        x = x.mean(axis=1) if x.shape[0] >= x.shape[1] else x.mean(axis=0)
    if int(sr) != NORM_SR:
        g = math.gcd(int(sr), NORM_SR)
        x = resample_poly(x, NORM_SR // g, int(sr) // g)
    return x.astype(np.float32)


def audio_warnings(signal: np.ndarray, sr: int, alert: dict) -> list[str]:
    x = np.asarray(signal, dtype=np.float64)
    out = []
    seconds = (x.shape[0] if x.ndim else 0) / sr
    min_s = 0.96 + (alert["n"] - 1) * alert["salto_s"]
    if seconds < min_s:
        out.append(
            f"Audio muy corto ({seconds:.1f} s): la regla {alert['k']} de {alert['n']} necesita "
            f"al menos {min_s:.1f} s; el veredicto no es fiable."
        )
    rms = float(np.sqrt(np.mean(x**2))) if x.size else 0.0
    if rms <= 0 or 20 * math.log10(rms) < USER_SILENCE_DBFS:
        out.append("Audio casi silencioso (< −60 dBFS): no hay señal suficiente.")
    if x.size and float(np.mean(np.abs(x) >= USER_CLIP_LEVEL)) > USER_CLIP_FRACTION:
        out.append("Audio saturado (> 1 % de muestras recortadas): el resultado puede fallar.")
    if sr < NORM_SR:
        out.append(
            f"Banda estrecha ({sr} Hz): no hay información por encima de {sr // 2} Hz; el modelo "
            "se entrenó con 16 kHz."
        )
    return out


def analyze_audio(signal: np.ndarray, sr: int, *, artifact: dict, yamnet_fn) -> dict:
    """Informe por fichero: veredicto, probabilidades por ventana, estados y avisos."""
    alert = artifact["alerta"]
    voice = artifact["voz"]
    mono = to_16k_any(signal, sr)
    starts = frame_starts(len(mono))
    probs, speech_flags, speech_scores = [], [], []
    for start in starts:
        feats = window_features(extract_window(mono, start), yamnet_fn, voice["indices"])
        is_speech = feats["voz"] >= voice["umbral"]
        speech_flags.append(bool(is_speech))
        speech_scores.append(feats["voz"])
        if is_speech:
            probs.append(None)  # descartada antes del clasificador
        else:
            probs.append(
                score_features(artifact, feats["embedding"], feats["armonico"])["prob_cal"]
            )
    states, events = run_alert([0.0 if p is None else p for p in probs], speech_flags, alert)
    valid = [p for p in probs if p is not None]
    warnings = audio_warnings(signal, sr, alert)
    if not valid:
        verdict = VERDICT_NONE
        warnings.append("Todas las ventanas tenían voz y se descartaron: no hay decisión.")
        persistence = None
    else:
        verdict = VERDICT_YES if events > 0 else VERDICT_NO
        persistence = persistence_score(valid, alert["k"], alert["n"])
        near_on = abs(persistence - alert["theta_on"]) <= UNCERTAINTY_MARGIN
        if alert["theta_off"] <= persistence < alert["theta_on"] or near_on:
            warnings.append(
                "Puntuación en la zona de incertidumbre (cerca del umbral): trátalo como dudoso."
            )
    return {
        "veredicto": verdict,
        "rotulo": USER_AUDIO_LABEL,
        "segundos": len(mono) / NORM_SR,
        "ventanas": len(starts),
        "ventanas_voz_descartadas": int(sum(speech_flags)),
        "tiempos_s": [s / NORM_SR for s in starts],
        "prob_por_ventana": probs,
        "voz_por_ventana": speech_scores,
        "estados": states,
        "alertas": int(events),
        "prob_max": max(valid) if valid else None,
        "prob_media": float(np.mean(valid)) if valid else None,
        "fraccion_sobre_umbral": float(np.mean([p >= alert["theta_on"] for p in valid]))
        if valid
        else None,
        "persistencia": persistence,
        "theta_on": alert["theta_on"],
        "theta_off": alert["theta_off"],
        "avisos": warnings,
    }


In [ ]:
# Fuentes de DRON adicionales para la iteración 6 (Salford DroneNoise CC BY 4.0, DroneAudioSet
# solo-dron MIT, Yi-KAIST CC BY 4.0): selección determinista, agrupación sin fugas y recortes.
# Sus manifiestos siguen con intake_approved=false: el cuaderno solo las usa si build_colab.py
# lleva una autorización explícita del usuario (texto de DECISIONES-USUARIO.md) para cada una.
# Autocontenido (numpy, stdlib) salvo la importación relativa de remote (en el cuaderno ya está
# definida y build_colab.py la quita).
import hashlib
import os
import re
import urllib.request
from collections.abc import Mapping, Sequence
from pathlib import Path

import numpy as np


EXTRA_SOURCES = ("salford", "droneaudioset", "yi_kaist")
EXTRA_CROP_SECONDS = 8.0  # Salford: tramo de más energía (paso del dron) de cada sobrevuelo
EXTRA_CHUNK_SECONDS = 5.0  # DroneAudioSet: tomas largas en trozos de 5 s
EXTRA_MAX_CHUNKS = 6
YI_NAME = re.compile(
    r"(?P<model>[ABC])_(?P<direction>CC|[FBRLC])_(?P<fault>N|MF[1-4]|PC[1-4])_(?P<drone>[^_]+)_"
    r"(?P<background>[^_]+)_(?P<bg_index>[^_]+)_(?P<snr>[^_.]+)"
)


def enabled_extra_sources(authorizations: Mapping[str, str | None]) -> list[str]:
    """Fuentes extra con autorización explícita (texto no vacío); fallo cerrado ante nombres
    desconocidos."""
    unknown = set(authorizations) - set(EXTRA_SOURCES)
    if unknown:
        raise ValueError(f"fuentes extra desconocidas: {sorted(unknown)}")
    return [s for s in EXTRA_SOURCES if (authorizations.get(s) or "").strip()]


def stream_download(
    url: str,
    hosts: set[str] | frozenset[str],
    dest: Path,
    *,
    size: int,
    md5: str | None = None,
    sha256: str | None = None,
    opener=None,
    chunk: int = 1 << 20,
) -> str:
    """Descarga a disco por trozos (ficheros de GB sin llenar la RAM) con host comprobado también
    tras redirecciones, tamaño exacto y MD5 y/o SHA-256; reutiliza un fichero ya verificado.
    Escritura atómica (.part + rename). Devuelve el SHA-256."""
    if md5 is None and sha256 is None:
        raise ValueError("hace falta al menos un hash de referencia")
    dest = Path(dest)

    def digests(path: Path) -> tuple[str, str]:
        h_md5, h_sha = hashlib.md5(usedforsecurity=False), hashlib.sha256()
        with path.open("rb") as fh:
            for block in iter(lambda: fh.read(chunk), b""):
                h_md5.update(block)
                h_sha.update(block)
        return h_md5.hexdigest(), h_sha.hexdigest()

    def ok(path: Path) -> str | None:
        if not path.is_file() or path.stat().st_size != size:
            return None
        got_md5, got_sha = digests(path)
        if (md5 is None or got_md5 == md5) and (sha256 is None or got_sha == sha256):
            return got_sha
        return None

    cached = ok(dest)
    if cached:
        return cached
    check_url(url, hosts)
    client = opener or make_opener(hosts)
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_name(dest.name + ".part")
    total = 0
    request = urllib.request.Request(url, headers={"User-Agent": "Atalaya-Colab-Pilot/1"})
    with client.open(request, timeout=300) as response, tmp.open("wb") as out:
        check_url(response.geturl(), hosts)
        for block in iter(lambda: response.read(chunk), b""):
            total += len(block)
            if total > size:
                raise ValueError(f"tamaño excedido: {dest.name}")
            out.write(block)
    got = ok(tmp)
    if total != size or got is None:
        tmp.unlink(missing_ok=True)
        raise ValueError(f"verificación fallida: {dest.name} ({total} B)")
    os.replace(tmp, dest)
    return got


def salford_selection(files: Sequence[dict]) -> list[dict]:
    """Solo grabaciones de vuelo no excluidas; grupo = evento (los 9 micrófonos de un evento
    graban a la vez y nunca se separan)."""
    out = []
    for f in files:
        if f.get("kind") != "flight" or f.get("training_excluded") or f.get("label") != 1:
            continue
        out.append({**f, "group_id": "salford:" + f["event_id"]})
    return sorted(out, key=lambda f: f["path"])


def energy_crop(signal: np.ndarray, sr: int, seconds: float = EXTRA_CROP_SECONDS) -> np.ndarray:
    """Tramo de `seconds` centrado en el segundo de más energía (el paso del dron)."""
    x = np.asarray(signal, dtype=np.float32)
    n = int(seconds * sr)
    if len(x) <= n:
        return x
    hop = max(1, sr // 4)
    energy = np.array(
        [np.mean(x[i : i + sr].astype(np.float64) ** 2) for i in range(0, len(x) - sr + 1, hop)]
    )
    centre = int(np.argmax(energy)) * hop + sr // 2
    start = min(max(0, centre - n // 2), len(x) - n)
    return x[start : start + n]


def chunk_starts(
    n_samples: int,
    sr: int,
    *,
    seconds: float = EXTRA_CHUNK_SECONDS,
    max_chunks: int = EXTRA_MAX_CHUNKS,
) -> list[int]:
    """Inicios equiespaciados y deterministas de hasta max_chunks trozos sin solape."""
    size = int(seconds * sr)
    if n_samples < size:
        return [0] if n_samples > 0 else []
    fits = n_samples // size
    count = min(fits, max_chunks)
    return [int(v) for v in np.linspace(0, n_samples - size, count).round()]


def droneaudioset_selection(files: Sequence[dict], *, seed: int) -> list[dict]:
    """Un parquet por sesión (dron x distancia x acelerador), elegido con semilla; filas sin las
    tomas «silence» (etiqueta pendiente de escucha). Grupo = sesión."""
    rng = np.random.default_rng(seed)
    by_session: dict[str, list[dict]] = {}
    for f in files:
        sessions = sorted({r["group_id"] for r in f["rows"]})
        for s in sessions:
            by_session.setdefault(s, []).append(f)
    chosen: dict[str, dict] = {}
    for session in sorted(by_session):
        candidates = sorted(by_session[session], key=lambda f: f["path"])
        pick = candidates[int(rng.integers(len(candidates)))]
        chosen[pick["path"]] = pick
    out = []
    for path in sorted(chosen):
        f = chosen[path]
        rows = [
            {**r, "group_id": "droneaudioset:" + r["group_id"]}
            for r in f["rows"]
            if r["take"] != "silence" and r["label_status"] == "drone_only_upstream"
        ]
        if rows:
            out.append({**f, "rows": rows})
    return out


def parse_yi_name(name: str) -> dict | None:
    m = YI_NAME.match(name.rsplit("/", 1)[-1])
    return m.groupdict() if m else None


def yi_selection(names: Sequence[str], *, n_pairs: int, seed: int) -> list[dict]:
    """Pares de segmentos de 0,5 s en estado normal (fallo N) del mismo grupo (modelo, índice de
    grabación del dron, fondo), concatenados después en 1 s; grupo = modelo + índice del dron
    (el reparto de los autores es por segmento y tiene fugas)."""
    cells: dict[tuple, list[str]] = {}
    for name in sorted(names):
        meta = parse_yi_name(name)
        if meta is None or meta["fault"] != "N" or not name.lower().endswith(".wav"):
            continue
        cells.setdefault((meta["model"], meta["drone"], meta["background"]), []).append(name)
    usable = {k: v for k, v in cells.items() if len(v) >= 2}
    if not usable:
        raise ValueError("no hay segmentos normales emparejables")
    rng = np.random.default_rng(seed)
    keys = sorted(usable)
    out = []
    for k in range(n_pairs):
        key = keys[int(rng.integers(len(keys)))]
        a, b = rng.choice(len(usable[key]), size=2, replace=False)
        out.append(
            {
                "id": f"yi{k:05d}",
                "segments": [usable[key][int(a)], usable[key][int(b)]],
                "group_id": f"yi_kaist:{key[0]}_{key[1]}",
                "synthetic_concat": True,
            }
        )
    return out


def concat_with_crossfade(parts: Sequence[np.ndarray], sr: int, fade_s: float = 0.01) -> np.ndarray:
    """Concatena con fundido cruzado lineal (evita el clic de la unión)."""
    fade = int(fade_s * sr)
    out = np.asarray(parts[0], dtype=np.float64)
    for p in parts[1:]:
        q = np.asarray(p, dtype=np.float64)
        f = min(fade, len(out), len(q))
        ramp = np.linspace(0, 1, f) if f else np.zeros(0)
        mixed = out[len(out) - f :] * (1 - ramp) + q[:f] * ramp
        out = np.concatenate([out[: len(out) - f], mixed, q[f:]])
    return out.astype(np.float32)


In [ ]:
# Svanström: licencia original antes de descargar datos (hosts y redirecciones restringidos).
SVAN_HOSTS = {"raw.githubusercontent.com", "api.github.com"}
def read_source(url):
    return read_https(url, SVAN_HOSTS, timeout=90)

ROOT_URL = f"https://raw.githubusercontent.com/DroneDetectionThesis/Drone-detection-dataset/{COMMIT}"
license_bytes = read_source(ROOT_URL + "/LICENSE")
assert hashlib.sha256(license_bytes).hexdigest() == PILOT["license_sha256"]
assert b"CC0 1.0 Universal" in license_bytes
print("Licencia CC0 original verificada por SHA-256.")


In [ ]:
# Svanström: selección reproducible y descarga con SHA-256 por archivo. Solo datos públicos.
# Decisión del usuario 2026-11-11 («Pero te lo autorizo, estoy despierto»): SOLO prueba técnica aislada
# en Colab; los WAV y el modelo nunca entran en el producto. Los manifiestos del repo siguen sin aprobar.
SOURCE_INTAKE_DOWNLOAD_APPROVED = True
assert SOURCE_INTAKE_DOWNLOAD_APPROVED, "Procedencia por WAV sin verificar: no repetir la ingesta."
assert 1 <= N_PER_CLASS <= 30
randomizer = random.Random(SEED)
selected = []
for prefix in ("BACKGROUND", "HELICOPTER", "DRONE"):
    candidates = sorted((c for c in PILOT["clips"] if c["name"].startswith(prefix + "_")),
                        key=lambda c: c["name"])
    assert len(candidates) == 30
    candidates = [c for c in candidates if not c["training_excluded"]]
    unique = sorted({c["sha256"]: c for c in candidates}.values(), key=lambda c: c["name"])
    assert len(unique) == len(candidates), "duplicado exacto sin marcar training_excluded"
    selected += randomizer.sample(unique, min(N_PER_CLASS, len(unique)))
assert len({c["sha256"] for c in selected}) == len(selected)

CATALOG = []  # una fila por clip: fuente, id único, ruta local, etiqueta, grupo, partición fijada
(BASE / "svanstrom").mkdir(exist_ok=True)
DOWNLOAD_WORKERS = 8  # descargas concurrentes; cada fichero se verifica por hash antes de usarse

def fetch_svanstrom(item):
    # Punto de control: un WAV ya presente solo se reutiliza si su SHA-256 coincide.
    name = item["name"]
    assert name.replace("_", "").replace(".", "").isalnum() and name.endswith(".wav")
    path = BASE / "svanstrom" / name
    if verified_cached_bytes(path, item["sha256"]) is None:
        raw = read_source(f"{ROOT_URL}/Data/Audio/{name}")
        assert len(raw) == item["bytes"] and hashlib.sha256(raw).hexdigest() == item["sha256"], name
        write_atomic(path, raw)
    return path

svan_paths = run_concurrent(selected, fetch_svanstrom, workers=DOWNLOAD_WORKERS,
                            on_done=Progress("Svanström", len(selected), every=30))
for item, path in zip(selected, svan_paths):
    name = item["name"]
    with wave.open(str(path), "rb") as wav:
        assert wav.getframerate() == 44100 and wav.getsampwidth() == 2
        assert wav.getnchannels() in (1, 2) and 9.9 <= wav.getnframes()/44100 <= 10.1
    CATALOG.append({"clip": f"svanstrom/{name}", "fuente": "svanstrom", "path": path,
                    "clase": name.split("_")[0], "dron": int(name.startswith("DRONE_")),
                    "grupo": "svanstrom:" + item["group_id"], "split": None})
print(f"Svanström: {len(selected)} WAV verificados ({sum(c['bytes'] for c in selected)/1e6:.1f} MB).")


In [ ]:
# ESC-10 (CC BY 3.0) en el commit fijado; nunca ESC-50 completo (CC BY-NC).
# Licencia: Git blob del LICENSE + texto CC BY 3.0 para ESC-10; meta/esc50.csv por SHA-256; cada
# fichero marcado esc10=True. Solo clases admitidas: sin crying_baby ni sneezing (sonidos humanos)
# ni licencias por clip CC Sampling+. Por fichero: tamaño y Git blob SHA-1 publicados en el commit
# (identificador de contenido de git; no es control criptográfico) y SHA-256 calculado e impreso.
ESC = PILOT["esc10"]
ESC_URL = f"https://raw.githubusercontent.com/karolpiczak/ESC-50/{ESC['commit']}"
def git_blob_sha1(data):
    return hashlib.sha1(b"blob %d\0" % len(data) + data, usedforsecurity=False).hexdigest()  # nosemgrep
esc_license = read_https(ESC_URL + "/LICENSE", SVAN_HOSTS, limit=1 << 21)
assert git_blob_sha1(esc_license) == ESC["license_git_blob_sha1"]
assert b"The ESC-10 subset is licensed as a Creative Commons Attribution 3.0" in esc_license
esc_meta = read_https(ESC_URL + "/meta/esc50.csv", SVAN_HOSTS, limit=1 << 21)
assert hashlib.sha256(esc_meta).hexdigest() == ESC["metadata_csv_sha256"]
esc10_rows = {r["filename"]: r for r in csv.DictReader(io.StringIO(esc_meta.decode("utf-8")))}
assert all(esc10_rows[f["path"].split("/")[-1]]["esc10"] == "True" for f in ESC["files"])
assert not {f["category"] for f in ESC["files"]} & {"crying_baby", "sneezing"}
(BASE / "esc10").mkdir(exist_ok=True)
esc_inventory = hashlib.sha256()

def fetch_esc10(f):
    # Punto de control: se reutiliza solo si tamaño y Git blob coinciden; si no, se descarga.
    path = BASE / "esc10" / f["path"].split("/")[-1]
    raw = path.read_bytes() if path.is_file() else b""
    if len(raw) != f["bytes"] or git_blob_sha1(raw) != f["blob"]:
        raw = read_https(f"{ESC_URL}/{f['path']}", SVAN_HOSTS, limit=1 << 20)
        assert len(raw) == f["bytes"] and git_blob_sha1(raw) == f["blob"], path.name
        write_atomic(path, raw)
    return path, raw

esc_fetched = run_concurrent(ESC["files"], fetch_esc10, workers=DOWNLOAD_WORKERS,
                             on_done=Progress("ESC-10", len(ESC["files"]), every=80))
for f, (path, raw) in zip(ESC["files"], esc_fetched):
    name = path.name
    esc_inventory.update(f"{hashlib.sha256(raw).hexdigest()}  {name}\n".encode())
    CATALOG.append({"clip": f"esc10/{name}", "fuente": "esc10", "path": path, "clase": f["category"],
                    "dron": 0, "grupo": f["group_id"], "split": None})
print(f"ESC-10: licencia CC BY 3.0 verificada; {len(ESC['files'])} clips admitidos verificados "
      f"({sum(f['bytes'] for f in ESC['files'])/1e6:.1f} MB); SHA-256 del inventario {esc_inventory.hexdigest()}")


In [ ]:
# FSD50K (Zenodo 4060432): licencia POR CLIP verificada en los metadatos de la fuente original.
# Se descargan doc/ground_truth/metadata (SHA-256 fijado), se relee la licencia de cada clip fijado
# en clips_info (solo CC0 y CC BY), se comprueba que no tenga etiquetas humanas ni de música ni
# mencione drones, y se leen SOLO esos clips del ZIP original por rangos HTTP (directorio central
# por SHA-256; cada clip por tamaño, CRC-32 y SHA-256).
FSD = PILOT["fsd50k"]
ZENODO_HOSTS = {"zenodo.org"}
def zenodo_file(record, key):
    return f"https://zenodo.org/api/records/{record}/files/{key}/content"
fsd_record = json.loads(read_https(f"https://zenodo.org/api/records/{FSD['record']}", ZENODO_HOSTS, limit=1 << 22))
assert fsd_record["doi"] == "10.5281/zenodo.4060432"
remote_sizes = {f["key"]: f["size"] for f in fsd_record["files"]}
fsd_zips = {}
for item in FSD["metadata_archives"]:
    assert remote_sizes[item["key"]] == item["bytes"], item["key"]
    raw = read_https(zenodo_file(FSD["record"], item["key"]), ZENODO_HOSTS, limit=item["bytes"])
    assert len(raw) == item["bytes"] and hashlib.sha256(raw).hexdigest() == item["sha256"], item["key"]
    fsd_zips[item["key"]] = zipfile.ZipFile(io.BytesIO(raw))
fsd_license = fsd_zips["FSD50K.doc.zip"].read("FSD50K.doc/LICENSE-DATASET")
assert hashlib.sha256(fsd_license).hexdigest() == FSD["license_sha256"]
assert b"Attribution 4.0 International (CC BY 4.0)" in fsd_license
fsd_gt = {p: read_fsd50k_ground_truth(fsd_zips["FSD50K.ground_truth.zip"].read(
              f"FSD50K.ground_truth/{p}.csv").decode("utf-8"), p) for p in ("dev", "eval")}
fsd_info = {p: json.loads(fsd_zips["FSD50K.metadata.zip"].read(
                f"FSD50K.metadata/{p}_clips_info_FSD50K.json")) for p in ("dev", "eval")}
problems = verify_fsd50k_pins(FSD["clips"], fsd_gt, fsd_info)
assert not problems, problems[:10]
licenses = pd.Series([ALLOWED_CLIP_LICENSES[fsd_info[c["split"]][c["fname"]]["license"]] for c in FSD["clips"]])
print("FSD50K: licencia por clip verificada en la fuente:", licenses.value_counts().to_dict())

cds = {}
for part, info in FSD["audio_archives"].items():
    for vol in info["volumes"]:
        assert remote_sizes[vol["key"]] == vol["bytes"], vol["key"]
    cd = info["central_directory"]
    key = info["volumes"][cd["disk"]]["key"]
    raw = read_https(zenodo_file(FSD["record"], key), ZENODO_HOSTS, byte_range=(cd["offset"], cd["offset"] + cd["bytes"]))
    assert hashlib.sha256(raw).hexdigest() == cd["sha256"], f"directorio central {part}"
    cds[part] = {e["name"]: e for e in parse_central_directory(raw, cd["n_entries"])}
(BASE / "fsd50k").mkdir(exist_ok=True)
# Descarga concurrente (DOWNLOAD_WORKERS hilos) con un límite GLOBAL de 120 peticiones/minuto
# compartido entre hilos (Zenodo admite ~133) y reintentos con Retry-After en read_https. Cada
# clip ya presente se reutiliza solo si su SHA-256 coincide (punto de control en /content).
ZENODO_LIMITER = RateLimiter(120)

def zenodo_range(url, byte_range):
    ZENODO_LIMITER.acquire()
    return read_https(url, ZENODO_HOSTS, byte_range=byte_range)

def fetch_fsd50k(clip):
    part = clip["split"]
    entry = cds[part][clip["member"]]
    assert all(entry[k] == clip[k] for k in ("disk", "offset", "size", "method", "crc32"))
    assert entry["compressed_size"] == clip["compressed_bytes"]
    path = BASE / "fsd50k" / f"{int(clip['fname'])}.wav"
    if verified_cached_bytes(path, clip["sha256"]) is not None:
        return path
    vols = FSD["audio_archives"][part]["volumes"]
    sizes = [v["bytes"] for v in vols]
    span = LOCAL_HEADER_LEN + len(clip["member"].encode()) + 512 + clip["compressed_bytes"]
    span = min(span, sum(sizes[clip["disk"]:]) - clip["offset"])
    raw = b"".join(zenodo_range(zenodo_file(FSD["record"], vols[d]["key"]), (a, b))
                   for d, a, b in volume_ranges(clip["disk"], clip["offset"], span, sizes))
    head = local_header_length(raw, clip["member"])
    data = decode_member(raw[head:head + clip["compressed_bytes"]], method=clip["method"],
                         size=clip["size"], crc32=clip["crc32"])
    assert hashlib.sha256(data).hexdigest() == clip["sha256"], clip["fname"]
    write_atomic(path, data)
    return path

already = sum(verified_cached_bytes(BASE / "fsd50k" / f"{int(c['fname'])}.wav", c["sha256"]) is not None
              for c in FSD["clips"])
print(f"FSD50K: {already}/{len(FSD['clips'])} clips ya verificados en /content; se descargan el resto")
fsd_paths = run_concurrent(FSD["clips"], fetch_fsd50k, workers=DOWNLOAD_WORKERS,
                           on_done=Progress("FSD50K", len(FSD["clips"]), every=50))
for clip, path in zip(FSD["clips"], fsd_paths):
    CATALOG.append({"clip": f"fsd50k/{clip['fname']}.wav", "fuente": "fsd50k", "path": path,
                    "clase": clip["target_label"], "dron": 0, "grupo": clip["group_id"], "split": None})
print(f"FSD50K: {len(FSD['clips'])} clips verificados ({sum(c['compressed_bytes'] for c in FSD['clips'])/1e6:.0f} MB "
      f"descargados); {len({c['group_id'] for c in FSD['clips']})} autores (grupos).")


In [ ]:
# ITU ARIS Lab (Zenodo 22682339, CC BY 4.0): licencia en la API y en el LICENSE del ZIP; ZIP por
# SHA-256; cada WAV contra el SHA256SUMS de los autores (fijado por SHA-256). Grupos y partición de
# los autores (groups.json, split interval_disjoint): segmentos solapados al 50 %, nunca al azar.
ARIS = PILOT["aris"]
aris_record = json.loads(read_https(f"https://zenodo.org/api/records/{ARIS['record']}", ZENODO_HOSTS, limit=1 << 22))
assert aris_record["metadata"]["license"]["id"] == "cc-by-4.0"
assert aris_record["doi"] == "10.5281/zenodo.22682339"
arch = ARIS["archive"]
assert {f["key"]: f["size"] for f in aris_record["files"]}[arch["key"]] == arch["bytes"]
zip_path = BASE / arch["key"]
if verified_cached_bytes(zip_path, arch["sha256"]) is None:  # punto de control: ZIP verificado
    raw = read_https(zenodo_file(ARIS["record"], arch["key"]), ZENODO_HOSTS, limit=arch["bytes"])
    assert len(raw) == arch["bytes"] and hashlib.sha256(raw).hexdigest() == arch["sha256"]
    write_atomic(zip_path, raw)
    del raw
else:
    print("ITU ARIS: ZIP ya verificado en /content")
aris_zip = zipfile.ZipFile(zip_path)
inner = {}
for name, digest in ARIS["inner_files_sha256"].items():
    inner[name] = aris_zip.read(ARIS["inner_prefix"] + name)
    assert hashlib.sha256(inner[name]).hexdigest() == digest, name
assert b"Creative Commons Attribution 4.0 International (CC BY 4.0)" in inner["LICENSE"]
aris_sums = parse_sha256sums(inner["SHA256SUMS"].decode("utf-8"))
aris_rows = aris_segments(json.loads(inner["groups.json"]), json.loads(inner[ARIS["split_used"]]))
assert len(aris_rows) == ARIS["segment_count"]
(BASE / "aris").mkdir(exist_ok=True)
aris_progress = Progress("ITU ARIS", len(aris_rows), every=1000)
for n, seg in enumerate(aris_rows, 1):
    path = BASE / "aris" / seg["path"].split("/")[-1]
    if verified_cached_bytes(path, aris_sums[seg["path"]]) is None:
        data = aris_zip.read(ARIS["inner_prefix"] + seg["path"])
        assert hashlib.sha256(data).hexdigest() == aris_sums[seg["path"]], seg["path"]
        write_atomic(path, data)
    aris_progress(n, len(aris_rows))
    CATALOG.append({"clip": "aris/" + seg["path"].split("/")[-1], "fuente": "aris", "path": path,
                    "clase": "ARIS_" + ("DRONE" if seg["label"] else "BACKGROUND"), "dron": seg["label"],
                    "grupo": seg["group_id"], "split": seg["split"]})
print(f"ITU ARIS: licencia CC BY 4.0 verificada; {len(aris_rows)} segmentos verificados contra SHA256SUMS; "
      f"{len({s['group_id'] for s in aris_rows})} grupos (intervalos/bloques).")
catalog = pd.DataFrame(CATALOG)
assert catalog["clip"].is_unique
display(catalog.groupby(["fuente", "dron"]).agg(clips=("clip", "size"), grupos=("grupo", "nunique")))


In [ ]:
# Fuentes de DRON adicionales (iteración 6). Sus manifiestos siguen con intake_approved=false: solo
# se descargan si EXTRA_AUTHORIZATION tiene la cita de la decisión del usuario para cada una.
# Cada fichero se verifica por tamaño y MD5/SHA-256 publicados por la fuente original.
EXTRA_ENABLED = enabled_extra_sources(EXTRA_AUTHORIZATION)
EXTRA_INVENTORY = []
if not EXTRA_ENABLED:
    print("Fuentes de dron extra: NINGUNA (sin autorización registrada). C-LOSO quedará NO EVALUABLE en",
          "los pliegues con una sola fuente de dron entrenando.")
else:
    step("Fuentes de dron extra autorizadas: " + ", ".join(EXTRA_ENABLED))
import soundfile as sf
if "salford" in EXTRA_ENABLED:
    SAL = PILOT["extra"]["salford"]
    FIGSHARE_HOSTS = {"api.figshare.com", "ndownloader.figshare.com", "s3-eu-west-1.amazonaws.com"}
    record = json.loads(read_https(SAL["api"], FIGSHARE_HOSTS, limit=1 << 22))
    assert record["license"]["name"] == "CC BY 4.0", record["license"]
    remote_files = {f["id"]: (f["computed_md5"], f["size"]) for f in record["files"]}
    (BASE / "salford").mkdir(exist_ok=True)
    def fetch_salford(f):
        assert remote_files[f["figshare_file_id"]] == (f["md5"], f["bytes"]), f["path"]
        path = BASE / "salford" / f["path"]
        return path, stream_download(f["url"], FIGSHARE_HOSTS, path, size=f["bytes"], md5=f["md5"])
    got = run_concurrent(SAL["files"], fetch_salford, workers=4, on_done=Progress("Salford", len(SAL["files"]), every=20))
    for f, (path, digest) in zip(SAL["files"], got):
        CATALOG.append({"clip": f"salford/{f['path']}", "fuente": "salford", "path": path, "clase": "SALFORD_" + f["suas_code"],
                        "dron": 1, "grupo": f["group_id"], "split": None, "recorte": "energia"})
        EXTRA_INVENTORY.append((f"salford/{f['path']}", digest))
    step(f"Salford: {len(SAL['files'])} sobrevuelos verificados (MD5 de Figshare; SHA-256 calculado)")
if "droneaudioset" in EXTRA_ENABLED:
    DAS = PILOT["extra"]["droneaudioset"]
    HF_HOSTS = {"huggingface.co", "cdn-lfs.hf.co", "cdn-lfs-us-1.hf.co", "cas-bridge.xethub.hf.co", "cdn-lfs.huggingface.co"}
    das_info = json.loads(read_https(DAS["info_url"], HF_HOSTS, limit=1 << 22))
    assert das_info.get("sha") == DAS["revision"] and das_info.get("cardData", {}).get("license") == "mit"
    import pyarrow.parquet as pq
    (BASE / "droneaudioset").mkdir(exist_ok=True)
    for f in DAS["files"]:
        path = BASE / "droneaudioset" / f["path"].split("/")[-1]
        digest = stream_download(f["url"], HF_HOSTS, path, size=f["bytes"], sha256=f["sha256"])
        table = pq.read_table(path).to_pylist()
        wanted = {r["upstream_file_path"]: r for r in f["rows"]}
        for row in table:
            meta_row = wanted.get(row.get("file_path"))
            if meta_row is None:
                continue  # silence u otra toma no admitida
            audio = next(v for v in row.values() if isinstance(v, dict) and ("bytes" in v or "array" in v))
            if audio.get("bytes"):
                data, sr = sf.read(io.BytesIO(audio["bytes"]), always_2d=True)
            else:
                data, sr = np.asarray(audio["array"], dtype=np.float32).reshape(-1, 1), int(audio["sampling_rate"])
            mono = data[:, 0].astype(np.float32)  # un canal (micrófono de referencia de cada toma)
            stem = meta_row["upstream_file_path"].replace("/", "__").removesuffix(".wav")
            for k, start in enumerate(chunk_starts(len(mono), sr)):
                out = BASE / "droneaudioset" / f"{stem}__{k}.wav"
                chunk = mono[start:start + int(EXTRA_CHUNK_SECONDS * sr)]
                if not out.is_file():
                    sf.write(str(out), chunk, sr, subtype="FLOAT")
                CATALOG.append({"clip": f"droneaudioset/{out.name}", "fuente": "droneaudioset", "path": out,
                                "clase": "DAS_" + meta_row["drone"], "dron": 1, "grupo": meta_row["group_id"], "split": None})
        EXTRA_INVENTORY.append((f"droneaudioset/{f['path']}", digest))
        del table
    step(f"DroneAudioSet: {len(DAS['files'])} parquet verificados por SHA-256 (una toma por sesión)")
if "yi_kaist" in EXTRA_ENABLED:
    YI = PILOT["extra"]["yi_kaist"]
    yi_record = json.loads(read_https(YI["api"], ZENODO_HOSTS, limit=1 << 22))
    assert yi_record["metadata"]["license"]["id"] == "cc-by-4.0"
    tar_path = BASE / "yi_kaist" / YI["file"]["path"]
    digest = stream_download(YI["file"]["url"], ZENODO_HOSTS, tar_path, size=YI["file"]["bytes"], md5=YI["file"]["md5"])
    EXTRA_INVENTORY.append((f"yi_kaist/{YI['file']['path']}", digest))
    normal = {}
    with tarfile.open(tar_path, mode="r|") as archive:
        for member in archive:
            name = member.name
            meta_name = parse_yi_name(name)
            if not member.isfile() or meta_name is None or meta_name["fault"] != "N" or not name.lower().endswith(".wav"):
                continue
            assert not name.startswith("/") and ".." not in name.split("/"), name
            normal[name] = archive.extractfile(member).read()
    pairs = yi_selection(sorted(normal), n_pairs=YI["n_pairs"], seed=SEED)
    (BASE / "yi_kaist" / "pares").mkdir(parents=True, exist_ok=True)
    for p in pairs:
        parts = [sf.read(io.BytesIO(normal[s]), always_2d=True) for s in p["segments"]]
        sr = parts[0][1]
        joined = concat_with_crossfade([d[:, 0] for d, _ in parts], sr)
        out = BASE / "yi_kaist" / "pares" / f"{p['id']}.wav"
        sf.write(str(out), joined, sr, subtype="FLOAT")
        CATALOG.append({"clip": f"yi_kaist/{p['id']}.wav", "fuente": "yi_kaist", "path": out, "clase": "YI_A_NORMAL",
                        "dron": 1, "grupo": p["group_id"], "split": None})
    del normal
    step(f"Yi-KAIST: {len(pairs)} pares de segmentos normales concatenados (1 s; synthetic_concat=true)")
catalog = pd.DataFrame(CATALOG)
assert catalog["clip"].is_unique
assert not set(catalog["fuente"]) - {"svanstrom", "esc10", "fsd50k", "aris", *EXTRA_ENABLED}
display(catalog.groupby(["fuente", "dron"]).agg(clips=("clip", "size"), grupos=("grupo", "nunique")))


## Revisión de privacidad

Escucha al menos una muestra de cada fuente y clase y comprueba que no haya voz identificable u
otros datos personales. Tras YAMNet, una **puerta automática** excluye del entrenamiento y la
evaluación los clips con `max_speech_score >= 0,30` en **todas** las fuentes y los lista; no
sustituye esta escucha. Si la escucha detecta voz en un clip no excluido, no entrenes; documenta
el clip y resuelve su tratamiento antes de seguir. El audio permanece solo en el almacenamiento
temporal de este entorno de Colab.


In [ ]:
# Reproducción de un clip por fuente y clase para revisión humana.
for (source, label), part in catalog.groupby(["fuente", "dron"]):
    row = part.sort_values("clip").iloc[0]
    audio, sr = librosa.load(row["path"], sr=None, mono=True)
    print(source, "dron" if label else "no dron", row["clip"], f"{len(audio)/sr:.1f} s")
    display(Audio(audio, rate=sr))


In [ ]:
# YAMNet + rasgos por lotes con puntos de control (código probado: data_quality, channel_norm,
# harmonic, model_export.window_features). Por clip: calidad (corrupto/silencioso/saturado), mono
# 16 kHz y DOS vistas: (a) línea base de la iteración 5, YAMNet sobre el clip entero SIN
# normalización (embedding medio y puntuación de voz por clip); (b) iteración 6, ventanas de 0,96 s
# con salto de 0,48 s (máx. MAX_WINDOWS_CLIP) -> normalización de canal -> YAMNet -> rasgo
# armónico: EXACTAMENTE la ruta del Mac y de «Prueba tu audio».
MODEL_URL = "https://tfhub.dev/google/yamnet/1"
if "yamnet" not in globals():
    yamnet = hub.load(MODEL_URL)
class_names = pd.read_csv(yamnet.class_map_path().numpy().decode())["display_name"].tolist()
assert len(class_names) == 521
speech_columns = [i for i, label in enumerate(class_names)
                  if label in {"Speech", "Conversation", "Narration, monologue", "Child speech, kid speaking"}]
assert len(speech_columns) == 4
SPEECH_NAMES = [class_names[i] for i in speech_columns]

def yamnet_tf(window):
    scores, embeddings, _ = yamnet(tf.convert_to_tensor(np.asarray(window, dtype=np.float32)))
    return scores.numpy(), embeddings.numpy()

def pick_windows(n_samples, limit):
    starts = frame_starts(n_samples)
    if len(starts) > limit:
        starts = [starts[i] for i in np.unique(np.linspace(0, len(starts) - 1, limit).round().astype(int))]
    return starts

def windows_of_signal(signal16k, limit):
    feats = [window_features(extract_window(signal16k, s), yamnet_tf, speech_columns) for s in pick_windows(len(signal16k), limit)]
    x = np.stack([np.concatenate([f["embedding"], f["armonico"]]) for f in feats]).astype(np.float32)
    return x, np.array([f["voz"] for f in feats], dtype=np.float32)

PATHS = {c["clip"]: c["path"] for c in CATALOG}
CLIP_SOURCE = {c["clip"]: c["fuente"] for c in CATALOG}
CROP = {c["clip"]: c.get("recorte") for c in CATALOG}
FILE_SHA256 = {c["clip"]: hashlib.sha256(Path(c["path"]).read_bytes()).hexdigest() for c in CATALOG}
DATA_INVENTORY_SHA256 = inventory_sha256(FILE_SHA256.items())
FEAT_CONTEXT = f"{MODEL_URL}|{TARGET_SR}|{CODE_VERSION}|{norm_config_sha256()}|{json.dumps(HARMONIC_CONFIG, sort_keys=True)}|{MAX_WINDOWS_CLIP}"
FEAT_SHARD = 128
step(f"Inventario de datos SHA-256: {DATA_INVENTORY_SHA256} ({len(CATALOG)} clips)")

def load_signal(clip):
    raw, sr = load_audio(PATHS[clip])
    signal = normalize_format(raw, sr)
    return energy_crop(signal, TARGET_SR) if CROP.get(clip) == "energia" else signal

def feature_shard(items):
    rows, quality_dropped, means5, blocks6, speech6 = [], [], [], [], []
    for item in items:
        try:
            raw, sr = load_audio(item["path"])
            quality = clip_quality(raw, sr)
        except Exception as exc:  # ilegible => corrupto (se informa; nunca se corrige)
            quality = {"status": "corrupto", "error": type(exc).__name__}
        if quality["status"] != "ok":
            quality_dropped.append({"clip": item["clip"], "fuente": item["fuente"], "motivo": "calidad: " + quality["status"],
                                    "detalle": json.dumps(quality, default=str, sort_keys=True)})
            continue
        signal = normalize_format(raw, sr)
        if item.get("recorte") == "energia":
            signal = energy_crop(signal, TARGET_SR)
        padded = signal if len(signal) >= 15600 else np.pad(signal, (0, 15600 - len(signal)))
        scores5, emb5 = yamnet_tf(padded)
        x6, voz6 = windows_of_signal(signal, MAX_WINDOWS_CLIP)
        means5.append(emb5.mean(axis=0))
        blocks6.append(x6)
        speech6.append(voz6)
        rows.append({"clip": item["clip"], "fuente": item["fuente"], "clase": item["clase"], "dron": int(item["dron"]),
                     "grupo": item["grupo"], "tipo": "dron" if item["dron"] else negative_type(item["fuente"], item["clase"]),
                     "segundos": round(len(signal) / TARGET_SR, 3), "ventanas": int(len(x6)),
                     "max_speech_score": round(float(scores5[:, speech_columns].max()), 3),
                     "pcm_sha256": pcm_sha256(signal), "file_sha256": FILE_SHA256[item["clip"]]})
    x6, off6 = pack_frames(blocks6) if blocks6 else (np.zeros((0, 1032), np.float32), np.zeros(1, np.int64))
    sp6, _ = pack_frames([v[:, None] for v in speech6]) if speech6 else (np.zeros((0, 1), np.float32), None)
    arrays = {"means5": np.asarray(means5, np.float32).reshape(-1, 1024), "x6": x6, "off6": off6, "voz6": sp6[:, 0]}
    return arrays, {"rows": rows, "quality": quality_dropped}

rows, QUALITY_DROPPED, parts = [], [], []
ranges = shard_ranges(len(CATALOG), FEAT_SHARD)
reused = 0
for n, (a, b) in enumerate(ranges, 1):
    items = CATALOG[a:b]
    key = shard_key([(c["clip"], FILE_SHA256[c["clip"]]) for c in items], FEAT_CONTEXT)
    path = CHECKPOINTS / f"rasgos6_{a:06d}_{key[:16]}.npz"
    loaded = load_shard(path, key)
    if loaded is None:
        loaded = feature_shard(items)
        save_shard(path, key, *loaded)  # guardado incremental: un corte solo pierde el lote en curso
    else:
        reused += 1
    arrays, meta = loaded
    rows += meta["rows"]
    QUALITY_DROPPED += meta["quality"]
    parts.append(arrays)
    step(f"Rasgos: lote {n}/{len(ranges)} ({b}/{len(CATALOG)} clips; {reused} lotes reutilizados)")
MEANS5 = np.vstack([p["means5"] for p in parts])
X6, OFF6 = pack_frames([unpack_frames(p["x6"], p["off6"], i) for p in parts for i in range(len(p["off6"]) - 1)])
VOZ6 = np.concatenate([p["voz6"] for p in parts])
del parts
assert len(MEANS5) == len(rows) == len(OFF6) - 1 and len(VOZ6) == len(X6)
info = pd.DataFrame(rows)
step(f"Rasgos listos: {len(info)} clips, {len(X6)} ventanas normalizadas; descartados por calidad: {len(QUALITY_DROPPED)}")
display(info.groupby("fuente")[["segundos", "ventanas"]].sum())


In [ ]:
# Puerta automática de privacidad (código probado: apply_speech_gate) en TODAS las fuentes, igual
# que en la iteración 5 (puntuación de voz de YAMNet sobre el clip completo). Además, en la
# iteración 6 las VENTANAS con voz >= umbral se descartan como en el Mac (puerta por ventana).
HUMAN_CLEARED = {
    "svanstrom/BACKGROUND_006.wav": "usuario, 2026-11-11: escuchado; crujido de madera, no es voz",
}
SPEECH_GATE = apply_speech_gate(info["clip"].tolist(), info["max_speech_score"].tolist(),
                                threshold=SPEECH_EXCLUSION_THRESHOLD, human_cleared=HUMAN_CLEARED)
SPEECH_DROPPED = [{"clip": e["clip"], "fuente": CLIP_SOURCE[e["clip"]], "motivo": "posible voz",
                   "detalle": f"{e['motivo']}; max_speech_score={e['max_speech_score']}"}
                  for e in SPEECH_GATE["excluded"]]
print(f"Excluidos por posible voz (umbral {SPEECH_EXCLUSION_THRESHOLD:.2f}): {len(SPEECH_DROPPED)}; "
      f"reincorporados por revisión humana: {SPEECH_GATE['human_cleared']}")
print(SPEECH_GATE["note"])
keep = list(SPEECH_GATE["kept_indices"])
info = info.iloc[keep].reset_index(drop=True)
MEANS5 = MEANS5[keep]
blocks = [unpack_frames(X6, OFF6, i) for i in keep]
voz_blocks = [VOZ6[OFF6[i]:OFF6[i + 1]] for i in keep]
X6, OFF6 = pack_frames(blocks)
VOZ6 = np.concatenate(voz_blocks)
WINDOW_SPEECH = VOZ6 >= SPEECH_EXCLUSION_THRESHOLD
print(f"Ventanas con voz descartadas (puerta por ventana): {int(WINDOW_SPEECH.sum())} de {len(VOZ6)}")
assert len(MEANS5) == len(info) == len(OFF6) - 1


In [ ]:
# Intake pendiente: los manifiestos del repo siguen con intake_approved=false.
# Decisión del usuario 2026-11-11 (prueba técnica) y plan del hito 2 aprobado el 2026-11-16 (H2.5):
# entrenamiento SOLO en esta prueba técnica aislada; nada entra en el producto.
TRAINING_ALLOWED = True
assert TRAINING_ALLOWED, "Intake sin aprobar: derechos por fichero, revisión de voz, data card y agrupación."


In [ ]:
# Duplicados exactos (hash del PCM) y casi duplicados (coseno de la media de embeddings YAMNet de la
# línea base >= NEAR_DUPLICATE_COSINE entre grupos distintos), como en la iteración 5.
assert TRAINING_ALLOWED, "Entrenamiento bloqueado"
exact = exact_duplicates(info["pcm_sha256"].tolist(), info["clip"].tolist())
pairs = [(i, j, s) for i, j, s in near_duplicate_pairs(MEANS5, threshold=NEAR_DUPLICATE_COSINE, groups=info["grupo"].tolist())
         if i not in exact and j not in exact]
near = resolve_near_duplicates(pairs, info["grupo"].tolist(), info["clip"].tolist())
DUP_DROPPED = [{"clip": info["clip"][i], "fuente": info["fuente"][i], "motivo": "duplicado exacto",
                "detalle": "se conserva " + info["clip"][d["kept"]]} for i, d in exact.items()]
DUP_DROPPED += [{"clip": info["clip"][i], "fuente": info["fuente"][i], "motivo": "casi duplicado entre grupos",
                 "detalle": f"coseno >= {NEAR_DUPLICATE_COSINE}; grupo conservado {g}"} for i, g in near["dropped"].items()]
drop = set(exact) | set(near["dropped"])
keep = [i for i in range(len(info)) if i not in drop]
info = info.iloc[keep].reset_index(drop=True)
MEANS5 = MEANS5[keep]
voz_blocks = [VOZ6[OFF6[i]:OFF6[i + 1]] for i in keep]
X6, OFF6 = pack_frames([unpack_frames(X6, OFF6, i) for i in keep])
VOZ6 = np.concatenate(voz_blocks)
WINDOW_SPEECH = VOZ6 >= SPEECH_EXCLUSION_THRESHOLD
residual = near_duplicate_pairs(MEANS5, threshold=NEAR_DUPLICATE_COSINE, groups=info["grupo"].tolist())
assert not residual, "quedan casi duplicados entre grupos distintos"
DISCARDS = sorted(QUALITY_DROPPED + SPEECH_DROPPED + DUP_DROPPED, key=lambda r: (r["motivo"], r["clip"]))
buffer = io.StringIO()
writer = csv.DictWriter(buffer, fieldnames=["clip", "fuente", "motivo", "detalle"], lineterminator="\n")
writer.writeheader()
writer.writerows(DISCARDS)
DISCARDS_CSV = buffer.getvalue().encode("utf-8")
DISCARDS_SHA256 = hashlib.sha256(DISCARDS_CSV).hexdigest()
(BASE / "descartes.csv").write_bytes(DISCARDS_CSV)
y = info["dron"].to_numpy().astype(int)
groups = info["grupo"].tolist()
src = info["fuente"].tolist()
DRONE_SOURCES = sorted({s for s, label in zip(src, y) if label == 1})
step(f"Clips finales: {len(info)}; fuentes con dron: {DRONE_SOURCES}; descartes.csv SHA-256 {DISCARDS_SHA256}")
display(info.groupby(["fuente", "dron"]).agg(clips=("clip", "size"), grupos=("grupo", "nunique"),
                                             horas=("segundos", lambda s: round(s.sum() / 3600, 2))))


In [ ]:
# Aumento SOLO para entrenamiento (código probado: augment.plan_cross_mixes, channel_augment):
# - N_CROSS_MIXES mezclas CRUZADAS: dron de una fuente sobre fondo de OTRA, cada una con un canal
#   aleatorio encima (random_channel por semilla);
# - N_CHANNEL_AUG copias de clips de AMBAS clases con canal aleatorio (el canal deja de delatar
#   la clase). En cada pliegue solo entran las que vienen de clips de entrenamiento (fallo cerrado).
print(SYNTHETIC_NOTE)
print(CHANNEL_SYNTHETIC_NOTE)
positives = [i for i in range(len(y)) if y[i] == 1]
negatives = [i for i in range(len(y)) if y[i] == 0]
CROSS_PLAN = plan_cross_mixes(positives, negatives, src, info["segundos"].tolist(), n=N_CROSS_MIXES, seed=SEED)
CHAN_PLAN = plan_channel_augment(list(range(len(y))), src, n=N_CHANNEL_AUG, seed=SEED + 1, labels=y.tolist())
def _named(plan, keys):
    return [{**m, **{k: info["clip"][m[k]] for k in keys}} for m in plan]
_, CROSS_PLAN_SHA256 = canonical_json(_named(CROSS_PLAN, ("drone", "background")))
_, CHAN_PLAN_SHA256 = canonical_json(_named(CHAN_PLAN, ("clip",)))
AUG_SHARD = 100

def render_cross(m):
    mix = render_mix(m, load_signal(info["clip"][m["drone"]]), load_signal(info["clip"][m["background"]]), TARGET_SR)
    return apply_channel(mix, random_channel(SEED + int(m["id"][-5:])))

def render_channel(p):
    signal = load_signal(info["clip"][p["clip"]])
    length = min(len(signal), int(5.0 * TARGET_SR))
    start = int((len(signal) - length) * p["offset"])
    return apply_channel(signal[start:start + length], random_channel(p["seed"]))

def augmented_features(plan, render, tag):
    blocks, valid = [], []
    for a, b in shard_ranges(len(plan), AUG_SHARD):
        items = [(m["id"], canonical_json(_named([m], ("drone", "background") if "drone" in m else ("clip",)))[1]) for m in plan[a:b]]
        key = shard_key(items, FEAT_CONTEXT + "|" + tag)
        path = CHECKPOINTS / f"{tag}_{a:06d}_{key[:16]}.npz"
        loaded = load_shard(path, key)
        if loaded is None:
            part, ok = [], []
            for m in plan[a:b]:
                try:
                    x, _ = windows_of_signal(render(m), MAX_WINDOWS_AUG)
                    part.append(x)
                    ok.append(1)
                except (SilentMixError, ValueError):
                    part.append(np.zeros((0, 1032), np.float32))
                    ok.append(0)
            frames, offsets = pack_frames(part)
            loaded = ({"x": frames, "off": offsets, "ok": np.asarray(ok, np.int8)}, {})
            save_shard(path, key, *loaded)
        arrays, _ = loaded
        blocks += [unpack_frames(arrays["x"], arrays["off"], i) for i in range(len(arrays["off"]) - 1)]
        valid += arrays["ok"].astype(bool).tolist()
        step(f"  {tag}: {b}/{len(plan)}")
    frames, offsets = pack_frames(blocks)
    return frames, offsets, np.asarray(valid, dtype=bool)

CROSS_X, CROSS_OFF, CROSS_OK = augmented_features(CROSS_PLAN, render_cross, "cruzadas6")
CHAN_X, CHAN_OFF, CHAN_OK = augmented_features(CHAN_PLAN, render_channel, "canal6")
step(f"Aumento: {int(CROSS_OK.sum())}/{len(CROSS_PLAN)} mezclas cruzadas y {int(CHAN_OK.sum())}/{len(CHAN_PLAN)} copias de canal válidas")


In [ ]:
# Evaluación de la iteración 6. LOSO es la métrica PRINCIPAL (cada fuente entera es test nunca
# vista); StratifiedGroupKFold (5 folds) es secundaria y solo para la línea base y el principal.
# Por variante: regresión logística por VENTANA sobre rasgos (embedding normalizado [+ armónico]),
# puntuación de clip = persistencia 3 de 5 (equivale a la alerta del Mac), Platt en validación,
# umbral por PEOR FUENTE de validación (FPR <= MAX_FPR_ON en cada fuente). La línea base de la
# iteración 5 se reentrena con su receta exacta (media de embeddings sin normalizar, pesos por
# clase y fuente, umbral con recall >= 0,90). En cada partición: grupos disjuntos y ningún casi
# duplicado cruza particiones; aumento y mezclas solo desde clips de entrenamiento.
from sklearn.linear_model import LogisticRegression
assert TRAINING_ALLOWED, "Entrenamiento bloqueado"
CAPTION = metric_caption(provisional=PILOT["group_status"] != "verified")
EMB = slice(0, 1024)
VARIANTS6 = {
    "iter6 · normalización": {"harm": False, "aug": False, "inlp": 0, "dro": False},
    "iter6 · + canal + cruzadas": {"harm": False, "aug": True, "inlp": 0, "dro": False},
    "iter6 · + armónico": {"harm": True, "aug": True, "inlp": 0, "dro": False},
    "iter6 · + INLP": {"harm": True, "aug": True, "inlp": INLP_ROUNDS, "dro": False},
    MAIN_VARIANT: {"harm": True, "aug": True, "inlp": INLP_ROUNDS, "dro": True},
}
VARIANTS = (BASELINE_VARIANT, *VARIANTS6)
SGKF_VARIANTS = (BASELINE_VARIANT, MAIN_VARIANT)

def cols(x, spec):
    return x if spec["harm"] else x[:, EMB]

def clip_windows(i, use_speech_gate=True):
    w = unpack_frames(X6, OFF6, i)
    if use_speech_gate:
        mask = ~WINDOW_SPEECH[OFF6[i]:OFF6[i + 1]]
        if mask.any():
            return w[mask]
    return w

def training_windows(train, held_out, spec, centers=None):
    xs, ys, doms = [], [], []
    for i in train:
        w = clip_windows(i)
        if len(w) > MAX_FRAMES_TRAIN:
            w = w[np.unique(np.linspace(0, len(w) - 1, MAX_FRAMES_TRAIN).round().astype(int))]
        xs.append(w); ys += [int(y[i])] * len(w); doms += [src[i]] * len(w)
    n_aug = 0
    if spec["aug"]:
        cross = [k for k in cross_mixes_for_fold(CROSS_PLAN, train, held_out=held_out) if CROSS_OK[k]]
        chan = [k for k in channel_augs_for_training(CHAN_PLAN, train) if CHAN_OK[k]]
        assert_channel_augs_training_only(CHAN_PLAN, chan, train)
        for k in cross:
            w = unpack_frames(CROSS_X, CROSS_OFF, k)
            xs.append(w); ys += [1] * len(w); doms += ["cruzada:" + CROSS_PLAN[k]["background_source"]] * len(w)
        for k in chan:
            w = unpack_frames(CHAN_X, CHAN_OFF, k)
            xs.append(w); ys += [int(y[CHAN_PLAN[k]["clip"]])] * len(w); doms += ["canal:" + src[CHAN_PLAN[k]["clip"]]] * len(w)
        n_aug = len(cross) + len(chan)
    x = np.vstack(xs).astype(np.float64)
    if centers is not None:
        x = x - np.vstack([centers[d.split(":")[-1]] for d in doms])
    return cols(x, spec), np.asarray(ys), doms, n_aug

def fit_variant(train, held_out, spec, centers=None):
    x, labels, doms, n_aug = training_windows(train, held_out, spec, centers)
    if spec["dro"]:
        model, _ = group_dro_fit(x, labels, [f"{d}|{v}" for d, v in zip(doms, labels)], seed=SEED, rounds=DRO_ROUNDS,
                                 eta=DRO_ETA, c=LR_C, domains=doms, inlp_rounds=spec["inlp"])
    else:
        model = fit_linear(x, labels, class_source_weights(labels, doms), seed=SEED, c=LR_C, domains=doms,
                           inlp_rounds=spec["inlp"])
    return model, n_aug

def clip_scores(model, indices, spec, centers=None):
    out = []
    for i in indices:
        w = clip_windows(i).astype(np.float64)
        if centers is not None:
            w = w - centers[src[i]]
        out.append(persistence_score(linear_proba(model, cols(w, spec)), 3, 5))
    return np.asarray(out)

def baseline_scores(train, parts):
    train = list(train)
    w = class_source_weights(y[train], [src[i] for i in train])
    model = LogisticRegression(max_iter=3000, random_state=SEED).fit(MEANS5[train], y[train], sample_weight=w)
    return [model.predict_proba(MEANS5[list(p)])[:, 1] for p in parts]

PRED_ROWS, RULES, AUG_COUNT = [], {}, {}

def record(variant, scheme, partition, validation, test, pv, pt):
    yv = y[list(validation)]
    platt, iso = fit_calibrator(yv, pv, "platt"), fit_calibrator(yv, pv, "isotonica")
    if platt["a"] <= 0:  # validación degenerada: Platt decreciente no se usa (se informa)
        platt = {"method": "platt", "a": 1.0, "b": 0.0, "nota": "Platt no creciente en validación; identidad"}
    cv_ = apply_calibration(platt, pv)
    if variant == BASELINE_VARIANT:
        rule = choose_threshold_for_recall(yv, cv_, target_recall=TARGET_RECALL)
        thr, detail = rule["threshold"], {"regla": "recall >= 0,90 (iteración 5)"}
    else:
        h = hysteresis_thresholds(yv, cv_, [src[i] for i in validation], max_fpr_on=MAX_FPR_ON, max_fpr_off=MAX_FPR_OFF)
        thr, detail = h["theta_on"], {"regla": "peor fuente", "theta_off": h["theta_off"],
                                      "fuente_limitante": h["detalle_on"]["fuente_limitante"]}
    RULES[(variant, scheme, partition)] = {"platt": platt, "umbral": float(thr), **detail}
    for i, a, b, c in zip(test, apply_calibration(platt, pt), pt, apply_calibration(iso, pt)):
        PRED_ROWS.append({"modelo": variant, "esquema": scheme, "particion": partition, "fuente": src[i],
                          "clip": info["clip"][i], "grupo": groups[i], "tipo": info["tipo"][i], "dron": int(y[i]),
                          "puntuacion": float(a), "puntuacion_cruda": float(b), "puntuacion_isotonica": float(c),
                          "umbral": float(thr)})

def check_partition(train, validation, test):
    assign = np.full(len(y), "fuera", dtype=object)
    for name, part in (("train", train), ("val", validation), ("test", test)):
        assign[list(part)] = name
    seen = [{groups[i] for i in part} for part in (train, validation, test)]
    assert not (seen[0] & seen[1] or seen[0] & seen[2] or seen[1] & seen[2]), "un grupo cruza particiones"
    used = np.flatnonzero(assign != "fuera")
    crossing = near_duplicate_pairs(MEANS5[used], threshold=NEAR_DUPLICATE_COSINE, groups=assign[used].tolist())
    assert_no_cross_partition_pairs(crossing, assign[used].tolist())

def evaluate(scheme, partition, train, validation, test, held_out, variants):
    check_partition(train, validation, test)
    for variant in variants:
        if variant == BASELINE_VARIANT:
            pv, pt = baseline_scores(train, [validation, test])
        else:
            spec = VARIANTS6[variant]
            model, AUG_COUNT[(variant, scheme, partition)] = fit_variant(train, held_out, spec)
            pv, pt = clip_scores(model, validation, spec), clip_scores(model, test, spec)
        record(variant, scheme, partition, validation, test, pv, pt)
    step(f"  {scheme} {partition}: train/val/test = {len(train)}/{len(validation)}/{len(test)} clips")

LOSO = leave_one_source_out(y.tolist(), groups, src, seed=SEED)
for split in LOSO:
    evaluate("loso", "sin_" + split["fuente"], split["train"], split["validation"], split["test"], split["fuente"], VARIANTS)
folds = stratified_group_folds(y.tolist(), groups, k=N_FOLDS, seed=SEED)
for f, (tr, va, te) in enumerate(rotating_folds(folds, N_FOLDS)):
    evaluate("sgkf5", f"fold{f}", tr, va, te, None, SGKF_VARIANTS)


In [ ]:
# Variante de SITIO (D-7), SIEMPRE aparte y fuera de C-LOSO: para cada fuente retenida con
# negativos se apartan CAL_GROUPS grupos negativos como «calibración del sitio» (fuera del test);
# los rasgos se centran por fuente con la media de sus negativos (entrenamiento: negativos de
# entrenamiento de cada fuente; fuente retenida: solo los de calibración). Esquema «loso_cal».
spec = VARIANTS6[MAIN_VARIANT]
for split in LOSO:
    held = split["fuente"]
    held_neg_groups = sorted({groups[i] for i in split["test"] if y[i] == 0})
    if len(held_neg_groups) <= CAL_GROUPS:
        print(f"  loso_cal sin_{held}: {len(held_neg_groups)} grupos negativos; no hay bastantes para calibrar")
        continue
    rng = np.random.default_rng(SEED)
    cal_groups = set(rng.choice(held_neg_groups, size=CAL_GROUPS, replace=False).tolist())
    cal = [i for i in split["test"] if groups[i] in cal_groups]
    test = [i for i in split["test"] if groups[i] not in cal_groups]
    rows_neg = [(src[i], clip_windows(i)) for i in list(split["train"]) + cal if y[i] == 0]
    centers = {}
    for s in sorted({s for s, _ in rows_neg}):
        centers[s] = np.vstack([w for t, w in rows_neg if t == s]).astype(np.float64).mean(axis=0)
    for s in set(src) - set(centers):  # fuente sin negativos de entrenamiento (solo dron): sin centrar
        centers[s] = np.zeros(X6.shape[1])
    model, _ = fit_variant(split["train"], held, spec, centers)
    pv = clip_scores(model, split["validation"], spec, centers)
    pt = clip_scores(model, test, spec, centers)
    record(MAIN_VARIANT, "loso_cal", "sin_" + held, split["validation"], test, pv, pt)
    step(f"  loso_cal sin_{held}: {len(cal)} clips de calibración apartados del test")
pred = pd.DataFrame(PRED_ROWS)
assert pred.groupby(["modelo", "esquema"])["clip"].apply(lambda c: c.is_unique).all()
step(f"Predicciones fuera de fold: {len(pred)} filas {CAPTION}")


In [ ]:
# Resumen LOSO (principal) con IC 95 % por grupos, criterio C-LOSO (ADR 0018) y D-6 de referencia.
# ROC-AUC (independiente de la prevalencia) con IC por remuestreo de grupos completos.
from sklearn.metrics import average_precision_score

def interval(boot, name):
    if boot["status"] != "OK":
        return f"NO DISPONIBLE ({boot['n_groups']} grupos)"
    iv = boot["intervals"][name]
    return "—" if iv is None else f"[{iv[0]:.3f}, {iv[1]:.3f}]"

def auc_interval(sub, n_boot=300):
    yy, pp, gg = sub["dron"].to_numpy(), sub["puntuacion"].to_numpy(), sub["grupo"].to_numpy()
    unique = np.unique(gg)
    if len(unique) < 10 or len(set(yy.tolist())) < 2:
        return "NO DISPONIBLE"
    members = {g: np.flatnonzero(gg == g) for g in unique}
    rng = np.random.default_rng(SEED)
    vals = []
    for _ in range(n_boot):
        idx = np.concatenate([members[unique[k]] for k in rng.choice(len(unique), len(unique))])
        v = safe_roc_auc(yy[idx], pp[idx])
        if v is not None:
            vals.append(v)
    return f"[{np.percentile(vals, 2.5):.3f}, {np.percentile(vals, 97.5):.3f}]" if len(vals) >= n_boot // 2 else "NO DISPONIBLE"

def summarize(sub):
    yy = sub["dron"].to_numpy()
    alarm = (sub["puntuacion"] >= sub["umbral"]).to_numpy().astype(float)
    m = binary_metrics(yy, alarm, 0.5)
    boot = group_bootstrap(yy, alarm, sub["grupo"].tolist(), 0.5, n_boot=500, seed=SEED)
    both = len(set(yy.tolist())) == 2
    r = lambda v: None if v is None or (isinstance(v, float) and np.isnan(v)) else round(float(v), 3)
    return {"n": len(sub), "positivos": int(yy.sum()), "recall": r(m["recall"]), "IC95_recall": interval(boot, "recall"),
            "fpr": r(m["fpr"]), "IC95_fpr": interval(boot, "fpr"),
            "roc_auc": r(safe_roc_auc(yy, sub["puntuacion"])) if both else None, "IC95_roc_auc": auc_interval(sub) if both else "—",
            "pr_auc": r(average_precision_score(yy, sub["puntuacion"])) if both else None,
            "ece_platt": r(expected_calibration_error(yy, sub["puntuacion"]))}

loso_pred = pred[pred["esquema"] == "loso"]
LOSO_SUMMARY = pd.DataFrame([{"modelo": v, "fuente_no_vista": p.replace("sin_", ""), **summarize(g)}
                             for (v, p), g in loso_pred.groupby(["modelo", "particion"])])
print("Leave-one-source-out (MÉTRICA PRINCIPAL; fuente nunca vista)", CAPTION)
display(LOSO_SUMMARY)

def c_loso(model):
    out = []
    for s, g in loso_pred[loso_pred["modelo"] == model].groupby("fuente"):
        yy = g["dron"].to_numpy()
        alarm = (g["puntuacion"] >= g["umbral"]).to_numpy()
        training_drone = [d for d in DRONE_SOURCES if d != s]
        e = {"fuente_retenida": s, "fuentes_dron_entrenando": training_drone, "positivos": int(yy.sum()),
             "roc_auc": safe_roc_auc(yy, g["puntuacion"]), "recall": float(alarm[yy == 1].mean()) if yy.any() else None,
             "fpr": float(alarm[yy == 0].mean()) if (yy == 0).any() else None}
        checks = []
        if e["positivos"]:
            if len(training_drone) < 2:
                e["estado"] = "NO EVALUABLE"
            else:
                checks.append(e["recall"] >= 0.50)
                if e["fpr"] is not None:  # ROC-AUC solo si la fuente tiene también negativos
                    checks.append(e["roc_auc"] is not None and e["roc_auc"] >= 0.85)
        if e["fpr"] is not None:
            checks.append(e["fpr"] <= 0.10)
        e.setdefault("estado", "CUMPLE" if all(checks) else "NO CUMPLE")
        out.append(e)
    states = {e["estado"] for e in out}
    overall = "CUMPLE" if states == {"CUMPLE"} else ("NO EVALUABLE" if "NO EVALUABLE" in states and "NO CUMPLE" not in states else "NO CUMPLE")
    return overall, out

CRITERIA = {}
for v in (MAIN_VARIANT, BASELINE_VARIANT):
    overall, table = c_loso(v)
    by = {e["fuente_retenida"]: e for e in table}
    sub = loso_pred[loso_pred["modelo"] == v]
    aris, fsd = sub[sub["fuente"] == "aris"], sub[sub["fuente"] == "fsd50k"]
    d6 = {"pr_auc_aris": float(average_precision_score(aris["dron"], aris["puntuacion"])),
          "recall_aris": float((aris.loc[aris["dron"] == 1, "puntuacion"] >= aris.loc[aris["dron"] == 1, "umbral"]).mean()),
          "fpr_fsd50k": float((fsd["puntuacion"] >= fsd["umbral"]).mean())}
    d6["cumple"] = d6["pr_auc_aris"] >= 0.85 and d6["recall_aris"] >= 0.50 and d6["fpr_fsd50k"] <= 0.10
    CRITERIA[v] = {"c_loso": overall, "por_fuente": table, "d6_referencia": d6}
    print(f"\n{v}\n  C-LOSO (ADR 0018, propuesto): {overall}")
    display(pd.DataFrame(table))
    print(f"  D-6 (referencia; depende de la prevalencia de ARIS): PR-AUC ARIS {d6['pr_auc_aris']:.3f}, "
          f"recall ARIS {d6['recall_aris']:.3f}, FPR FSD50K {d6['fpr_fsd50k']:.3f} -> {'cumple' if d6['cumple'] else 'NO cumple'}")
GENERALIZES = CRITERIA[MAIN_VARIANT]["c_loso"] == "CUMPLE"
if not GENERALIZES:
    print("\nAVISO: el modelo principal NO cumple C-LOSO: la demo debe rotularse «modelo que no generaliza» y se itera.")
cal_pred = pred[pred["esquema"] == "loso_cal"]
if len(cal_pred):
    print("\nVariante de sitio (D-7; aparte, NO cuenta para C-LOSO):")
    display(pd.DataFrame([{"fuente_no_vista": p.replace("sin_", ""), **summarize(g)} for p, g in cal_pred.groupby("particion")]))
cv_pred = pred[pred["esquema"] == "sgkf5"]
print("\nStratifiedGroupKFold (secundaria, fuentes vistas):")
display(pd.DataFrame([{"modelo": v, **summarize(cv_pred[cv_pred["modelo"] == v])} for v in SGKF_VARIANTS]))
print("Línea base exacta de la iteración 5 (otra ejecución): predicciones.csv SHA-256 "
      "8ec5d2281682dd02cb11c3a0d9a74feb59df508325afdaa99a715d39a7c67407; aquí se reentrena con su receta.")


In [ ]:
# Modelo FINAL (el que se exporta): receta del principal con TODAS las fuentes. Partición por grupos:
# fold 0 = TEST final (nunca visto: autocomprobación de «Prueba tu audio»), fold 1 = validación
# (Platt y umbrales por peor fuente con histéresis), resto = entrenamiento.
spec = VARIANTS6[MAIN_VARIANT]
FINAL_FOLDS = stratified_group_folds(y.tolist(), groups, k=N_FOLDS, seed=SEED + 7)
FINAL_TEST = [i for i in range(len(y)) if FINAL_FOLDS[i] == 0]
FINAL_VAL = [i for i in range(len(y)) if FINAL_FOLDS[i] == 1]
FINAL_TRAIN = [i for i in range(len(y)) if FINAL_FOLDS[i] >= 2]
check_partition(FINAL_TRAIN, FINAL_VAL, FINAL_TEST)
FINAL_MODEL, _ = fit_variant(FINAL_TRAIN, None, spec)
FINAL_MODEL = {**FINAL_MODEL, "nombre": MAIN_VARIANT}
pv = clip_scores(FINAL_MODEL, FINAL_VAL, spec)
FINAL_PLATT = fit_calibrator(y[FINAL_VAL], pv, "platt")
assert FINAL_PLATT["a"] > 0, "Platt no creciente: no se exporta un modelo con calibración invertida"
FINAL_TH = hysteresis_thresholds(y[FINAL_VAL], apply_calibration(FINAL_PLATT, pv), [src[i] for i in FINAL_VAL],
                                 max_fpr_on=MAX_FPR_ON, max_fpr_off=MAX_FPR_OFF)
FINAL_ALERT = {"theta_on": FINAL_TH["theta_on"], "theta_off": FINAL_TH["theta_off"], **ALERT_DEFAULTS}
pt = apply_calibration(FINAL_PLATT, clip_scores(FINAL_MODEL, FINAL_TEST, spec))
yt = y[FINAL_TEST]
FINAL_TEST_METRICS = {"roc_auc": safe_roc_auc(yt, pt), "recall": float((pt[yt == 1] >= FINAL_ALERT["theta_on"]).mean()),
                      "fpr": float((pt[yt == 0] >= FINAL_ALERT["theta_on"]).mean()), "n": len(yt)}
step(f"Modelo final: {FINAL_MODEL['inlp_direcciones']} direcciones INLP; theta_on {FINAL_ALERT['theta_on']:.4f}, "
     f"theta_off {FINAL_ALERT['theta_off']:.4f}; test final (fuentes vistas) {FINAL_TEST_METRICS}")


In [ ]:
# YAMNet -> ONNX con tf2onnx (opset fijo) y PARIDAD frente a TensorFlow en >= 50 ventanas reales
# normalizadas (y las entradas doradas). Si la conversión falla o la diferencia supera PARITY_TOL,
# NO se exporta yamnet.onnx ni modelo_hito2.json (fallo cerrado) y «Prueba tu audio» usa TensorFlow.
YAMNET_ONNX = BASE / "yamnet.onnx"
ONNX_INFO = {"origen": MODEL_URL, "opset": ONNX_OPSET, "entrada": "waveform float32 [n] a 16 kHz (ventana de 15 600 muestras)",
             "salidas": ["scores (n, 521)", "embeddings (n, 1024)"], "paridad_tol": PARITY_TOL}
PARITY_OK, yamnet_onnx = False, None
try:
    import onnxruntime as ort
    import tf2onnx
    sig = (tf.TensorSpec([None], tf.float32, name="waveform"),)
    @tf.function(input_signature=sig)
    def yamnet_serving(waveform):
        scores, embeddings, _ = yamnet(waveform)
        return {"scores": scores, "embeddings": embeddings}
    tf2onnx.convert.from_function(yamnet_serving, input_signature=sig, opset=ONNX_OPSET, output_path=str(YAMNET_ONNX))
    session = ort.InferenceSession(str(YAMNET_ONNX), providers=["CPUExecutionProvider"])
    input_name = session.get_inputs()[0].name
    def yamnet_onnx(window):
        outs = session.run(None, {input_name: np.asarray(window, dtype=np.float32)})
        by_dim = {o.shape[-1]: o for o in outs if o.ndim == 2}
        return by_dim[521], by_dim[1024]
    rng = np.random.default_rng(SEED)
    probe = rng.choice(len(info), size=min(PARITY_WINDOWS, len(info)), replace=False)
    windows = [normalize_window(extract_window(load_signal(info["clip"][i]), 0)) for i in probe]
    windows += [normalize_window(to_mono_16k(golden_input(s, k), GOLDEN_SR)) for s, k in zip((101, 202, 303), GOLDEN_KINDS)]
    diffs_e, diffs_s = [], []
    for w in windows:
        s_tf, e_tf = yamnet_tf(w)
        s_ox, e_ox = yamnet_onnx(w)
        diffs_e.append(float(np.max(np.abs(e_tf - e_ox))))
        diffs_s.append(float(np.max(np.abs(s_tf - s_ox))))
    ONNX_INFO.update({"onnx_sha256": hashlib.sha256(YAMNET_ONNX.read_bytes()).hexdigest(), "bytes": YAMNET_ONNX.stat().st_size,
                      "paridad_ventanas": len(windows), "paridad_max_embedding": max(diffs_e), "paridad_max_scores": max(diffs_s),
                      "versiones": dependency_versions(["tensorflow", "tf2onnx", "onnx", "onnxruntime"])})
    PARITY_OK = max(diffs_e) <= PARITY_TOL and max(diffs_s) <= PARITY_TOL
    step(f"ONNX: {ONNX_INFO['bytes'] / 1e6:.1f} MB; paridad máx. embedding {max(diffs_e):.2e}, voz/clases {max(diffs_s):.2e} "
         f"en {len(windows)} ventanas -> {'OK' if PARITY_OK else 'FALLA'}")
except Exception as exc:  # conversión no disponible: se informa y no se exporta
    ONNX_INFO["error"] = f"{type(exc).__name__}: {exc}"[:300]
    print("ONNX NO disponible:", ONNX_INFO["error"])
ONNX_INFO["paridad_ok"] = bool(PARITY_OK)
if not PARITY_OK:
    print("FALLO CERRADO: sin paridad ONNX <= 1e-4 no se exporta el artefacto para el Mac.")
    YAMNET_ONNX.unlink(missing_ok=True)


In [ ]:
# Exportación: predicciones.csv (código probado: predictions), registro_iter6.json (experiment) y,
# SOLO con paridad ONNX, modelo_hito2.json (model_export: pesos, Platt, alerta, normalización,
# índices de voz, vectores dorados calculados con la sesión ONNX que usará el Mac) y SHA256SUMS.
csv_bytes, csv_sha256 = predictions_csv(PRED_ROWS, columns=EXTENDED_COLUMNS)
(BASE / "predicciones.csv").write_bytes(csv_bytes)
DEPENDENCIES = dependency_versions(["tensorflow", "tensorflow-hub", "tf-keras", "tf2onnx", "onnx", "onnxruntime", "librosa",
                                    "soundfile", "scikit-learn", "scipy", "numpy", "pandas", "matplotlib"])
EXPERIMENT = experiment_record(
    config={"iteracion": 6, "semilla": SEED, "n_folds": N_FOLDS, "modelo_principal": MAIN_VARIANT, "variantes": list(VARIANTS),
            "max_ventanas_clip": MAX_WINDOWS_CLIP, "max_ventanas_aumento": MAX_WINDOWS_AUG, "max_ventanas_entreno": MAX_FRAMES_TRAIN,
            "n_cruzadas": N_CROSS_MIXES, "n_canal": N_CHANNEL_AUG, "max_fpr_on": MAX_FPR_ON, "max_fpr_off": MAX_FPR_OFF,
            "inlp_rondas": INLP_ROUNDS, "dro_rondas": DRO_ROUNDS, "dro_eta": DRO_ETA, "lr_c": LR_C, "cal_grupos": CAL_GROUPS,
            "normalizacion_sha256": norm_config_sha256(), "armonico": HARMONIC_CONFIG, "voz_umbral": SPEECH_EXCLUSION_THRESHOLD,
            "fuentes_extra": EXTRA_ENABLED, "autorizaciones_extra": EXTRA_AUTHORIZATION},
    data={"inventario_sha256": DATA_INVENTORY_SHA256, "inventario_extra": sorted(EXTRA_INVENTORY), "descartes_csv_sha256": DISCARDS_SHA256,
          "plan_cruzadas_sha256": CROSS_PLAN_SHA256, "plan_canal_sha256": CHAN_PLAN_SHA256, "clips_evaluados": len(info),
          "fuentes_dron": DRONE_SOURCES, "group_status_svanstrom": PILOT["group_status"]},
    dependencies=DEPENDENCIES, predictions_sha256=csv_sha256, notebook_code_sha256=NOTEBOOK_CODE_SHA256,
    extra={"modelo_principal": MAIN_VARIANT, "criterios": CRITERIA, "generaliza": GENERALIZES,
           "reglas": {"|".join(k): v for k, v in RULES.items()}, "onnx": ONNX_INFO, "test_final": FINAL_TEST_METRICS,
           "puerta_voz": {"umbral": SPEECH_GATE["threshold"], "nota": SPEECH_GATE["note"]},
           "excluidos_por_voz": SPEECH_GATE["excluded"], "revisiones_humanas": SPEECH_GATE["human_cleared"],
           "nota_sintetico": [SYNTHETIC_NOTE, CHANNEL_SYNTHETIC_NOTE], "filas_predicciones": len(PRED_ROWS),
           "linea_base_iter5_csv_sha256": "8ec5d2281682dd02cb11c3a0d9a74feb59df508325afdaa99a715d39a7c67407"})
assert verify_record(EXPERIMENT)
(BASE / "registro_iter6.json").write_text(json.dumps(EXPERIMENT, ensure_ascii=False, indent=1))
summary_criteria = {"c_loso": CRITERIA[MAIN_VARIANT]["c_loso"], "d6_referencia": CRITERIA[MAIN_VARIANT]["d6_referencia"],
                    "generaliza": GENERALIZES, "rotulo_demo": "" if GENERALIZES else "modelo que no generaliza"}
ARTIFACT = build_artifact(model=FINAL_MODEL, platt=FINAL_PLATT, alert=FINAL_ALERT, yamnet_info=ONNX_INFO,
                          speech_indices=speech_columns, speech_names=SPEECH_NAMES, speech_threshold=SPEECH_EXCLUSION_THRESHOLD,
                          criteria=summary_criteria, predictions_sha256=csv_sha256, record_sha256=EXPERIMENT["registro_sha256"],
                          yamnet_fn=yamnet_onnx if PARITY_OK else yamnet_tf)
EXPORTS = ["predicciones.csv", "registro_iter6.json", "descartes.csv"]
if PARITY_OK:
    data = json.dumps(ARTIFACT, ensure_ascii=False, sort_keys=True, indent=1).encode("utf-8")
    assert len(data) <= 200_000, f"modelo_hito2.json demasiado grande: {len(data)} bytes"
    (BASE / "modelo_hito2.json").write_bytes(data)
    assert check_golden(json.loads(data), yamnet_onnx) <= GOLDEN_TOLERANCE
    EXPORTS = ["modelo_hito2.json", "yamnet.onnx", *EXPORTS]
sums = "".join(f"{hashlib.sha256((BASE / n).read_bytes()).hexdigest()}  {n}\n" for n in EXPORTS)
(BASE / "SHA256SUMS").write_text(sums)
print(sums)
print("Recalcular en el repo: uv run python recompute_metrics.py predicciones.csv --sha256", csv_sha256,
      "--record registro_iter6.json" + (" --artifact modelo_hito2.json" if PARITY_OK else ""), "--criterio c-loso")
try:
    from google.colab import files  # descarga al navegador del usuario; no monta Drive
    for name in [*EXPORTS, "SHA256SUMS"]:
        files.download(str(BASE / name))
except ImportError:
    pass


## Prueba tu audio

Sube un fichero de audio (wav, mp3, m4a, ogg o flac) y el cuaderno te dice si detecta un dron,
usando **exactamente** la ruta del modelo exportado: remuestreo a 16 kHz → ventanas de 0,96 s con
salto de 0,48 s → normalización de canal → YAMNet (ONNX si la paridad pasó) → **puerta de voz**
(ventanas con voz ≥ 0,30 se descartan y se cuentan) → rasgo armónico → clasificador calibrado →
**alerta 3 de 5 con histéresis**.

**Usa solo audio propio o con una licencia que lo permita. No descargues audio de YouTube ni de
otras plataformas.** El fichero se analiza en esta sesión y se **borra** al terminar; no se guarda
en Drive ni en el repositorio. Resultado de una prueba técnica: **no** es un vuelo real verificado.


In [ ]:
# Autocomprobación de punta a punta: un clip de DRON y uno de FONDO del TEST final (nunca vistos al
# entrenar ni al calibrar) pasan por analyze_audio (la función de «Prueba tu audio») y su veredicto
# debe coincidir con la decisión de la ruta por lotes del modelo final. Se eligen clips con >= 5
# ventanas y <= MAX_WINDOWS_CLIP (misma regla 3 de 5 en ambas rutas).
USER_YAMNET = yamnet_onnx if PARITY_OK else yamnet_tf
def batch_decision(i):
    p = apply_calibration(FINAL_PLATT, np.array([persistence_score(linear_proba(FINAL_MODEL, clip_windows(i, use_speech_gate=False).astype(np.float64)), 3, 5)]))[0]
    return p >= FINAL_ALERT["theta_on"], p
eligible = [i for i in FINAL_TEST if 5 <= info["ventanas"][i] <= MAX_WINDOWS_CLIP
            and not WINDOW_SPEECH[OFF6[i]:OFF6[i + 1]].any()]
chosen = {}
for want in (1, 0):
    cands = [i for i in eligible if y[i] == want and batch_decision(i)[0] == bool(want)]
    if cands:
        chosen[want] = cands[0]
SELF_CHECK = []
for want, i in sorted(chosen.items(), reverse=True):
    signal = load_signal(info["clip"][i])
    report = analyze_audio(signal, TARGET_SR, artifact=ARTIFACT, yamnet_fn=USER_YAMNET)
    expected = VERDICT_YES if want else VERDICT_NO
    SELF_CHECK.append({"clip": info["clip"][i], "esperado": expected, "obtenido": report["veredicto"],
                       "prob_lotes": round(float(batch_decision(i)[1]), 4), "persistencia_ruta_usuario": report["persistencia"]})
    assert report["veredicto"] == expected, SELF_CHECK[-1]
display(pd.DataFrame(SELF_CHECK))
if len(chosen) < 2:
    print("AVISO: no hubo en el test final un clip de dron detectado y uno de fondo rechazado con >= 5 ventanas;",
          "la autocomprobación es parcial:", sorted(chosen))
else:
    step("Autocomprobación OK: la ruta de «Prueba tu audio» reproduce la decisión del modelo final")


In [ ]:
# Prueba tu audio: sube uno o varios ficheros. Se analizan en memoria (los no WAV se escriben un
# instante en /content/prueba_audio para que ffmpeg/soundfile los decodifiquen) y se BORRAN después.
print(USER_AUDIO_NOTE)
def show_report(name, signal, sr, report):
    print(f"\n=== {name}: {report['veredicto']} ===  ({report['rotulo']})")
    if report["prob_max"] is not None:
        print(f"Probabilidad máx. {report['prob_max']:.3f} · media {report['prob_media']:.3f} · "
              f"{100 * report['fraccion_sobre_umbral']:.0f} % de ventanas sobre el umbral ({report['theta_on']:.3f}) · "
              f"{report['alertas']} alerta(s)")
    print(f"{report['ventanas']} ventanas; {report['ventanas_voz_descartadas']} descartadas por voz")
    for w in report["avisos"]:
        print("AVISO:", w)
    mono = to_16k_any(signal, sr)
    fig, (a, b) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    a.specgram(mono, NFFT=512, Fs=16000, noverlap=352, cmap="magma")
    a.set(ylabel="Hz", title=f"{name} · espectrograma (16 kHz)")
    t = np.array(report["tiempos_s"]) + 0.48
    probs = [np.nan if p is None else p for p in report["prob_por_ventana"]]
    b.plot(t, probs, "o-", label="probabilidad calibrada por ventana")
    b.axhline(report["theta_on"], color="red", ls="--", label="umbral de alerta")
    b.axhline(report["theta_off"], color="orange", ls=":", label="fin de alerta (histéresis)")
    b.set(ylim=(0, 1), xlabel="Tiempo (s)", ylabel="Probabilidad")
    b.legend(fontsize=8)
    plt.tight_layout(); plt.show()

try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    uploaded = {}
    print("Fuera de Colab: no hay subida de ficheros.")
work = Path("/content/prueba_audio")
for name, data in uploaded.items():
    path = work / Path(name).name
    try:
        if path.suffix.lower() == ".wav":
            try:
                signal, sr = decode_wav_bytes(data)
            except AudioDecodeError:
                signal = None
        else:
            signal = None
        if signal is None:
            work.mkdir(parents=True, exist_ok=True)
            path.write_bytes(data)
            signal, sr = load_user_audio(path, fallback_loader=load_audio)
        report = analyze_audio(signal, sr, artifact=ARTIFACT, yamnet_fn=USER_YAMNET)
        show_report(name, signal, sr, report)
    except (AudioDecodeError, ValueError) as exc:
        print(f"\n=== {name}: no se pudo analizar ({exc}) ===")
    finally:
        path.unlink(missing_ok=True)  # borrado explícito: nada queda en /content
        signal = data = None
uploaded = None
print("Ficheros subidos borrados de la sesión.")


In [ ]:
# Ninguna fuente prohibida aparece en las descargadas.
DADS_ENABLED = False
assert not DADS_ENABLED
assert set(catalog["fuente"]) <= {"svanstrom", "esc10", "fsd50k", "aris", *EXTRA_ENABLED}
assert not set(EXTRA_ENABLED) - {"salford", "droneaudioset", "yi_kaist"}
print("DADS y demás fuentes prohibidas bloqueadas:", ", ".join(PROHIBITED))
step("Fin del cuaderno de la iteración 6")


**Lectura de resultados:** describen solo estos clips, fuentes y repartos. ITU ARIS es una sola
sesión; FSD50K y ESC-10 son clips cortos de Freesound. Las mezclas y copias con canal aleatorio son
sintéticas (`synthetic=true`) y nunca se puntúan. No se ha medido vuelo real, distancia, falsas
alarmas por hora en un emplazamiento ni el micrófono del Mac (eso es el protocolo en sala del
hito 2, H2.14). Si C-LOSO no se cumple, la demo se rotula «modelo que no generaliza».
